# Module 6 - Secure Deployment & Infrastructure

---

<details>
<summary><b>Module Overview — Click to Expand</b></summary>
<br>

## Module Overview

This module explores how artificial intelligence and machine learning systems can be securely deployed and operated in production environments. Unlike attacks that directly target training data or model behavior, deployment attacks often exploit weaknesses in the infrastructure, services, and processes surrounding the model.

You will learn how attackers can exploit:

- Insecure AI APIs and inference endpoints
- Weak identity and access controls
- Exposed secrets and credentials
- Vulnerable containers and runtime environments
- Misconfigured cloud and network infrastructure
- Compromised model artifacts and deployment pipelines

You will also implement defenses such as:

- Authentication and least-privilege access
- Secure secrets and configuration management
- Container and infrastructure hardening
- Network segmentation and zero-trust principles
- Model integrity and provenance verification
- Logging, monitoring, and incident response

---

</details>

<details>
<summary><b>Learning Objectives — Click to Expand</b></summary>
<br>

# Learning Objectives

By the end of this module, learners should be able to:

1. **Identify the major components of a deployed AI/ML system** and explain how APIs, model servers, databases, cloud infrastructure, containers, model registries, and monitoring systems contribute to the overall attack surface.

2. **Analyze the attack surface of an AI deployment** by identifying exposed services, trust boundaries, communication paths, external dependencies, and potential entry points for attackers.

3. **Explain how inference APIs and model endpoints can be secured** using authentication, authorization, request validation, rate limiting, secure communication, and appropriate response handling.

4. **Apply identity and access management principles to AI infrastructure**, including authentication, authorization, role-based access control, service identities, separation of duties, and the principle of least privilege.

5. **Recognize insecure handling of secrets and credentials**, including hardcoded API keys, passwords, access tokens, cloud credentials, and improperly protected configuration information.

6. **Describe secure approaches for managing secrets and configuration**, including centralized secret management, credential rotation, short-lived credentials, secure defaults, and automated configuration validation.

7. **Evaluate the security of containerized AI workloads** by identifying risks involving vulnerable base images, unnecessary packages, excessive privileges, insecure runtime configurations, and untrusted container images.

8. **Explain how network segmentation and zero-trust principles can reduce the blast radius of an AI system compromise** by restricting unnecessary communication between infrastructure components.

9. **Describe security considerations associated with cloud-hosted AI systems**, including public and private endpoints, storage security, service-to-service communication, cloud permissions, multi-tenancy, and workload isolation.

10. **Verify the integrity and provenance of AI model artifacts** using techniques such as cryptographic hashing, digital signatures, model versioning, lineage tracking, and trusted model registries.

11. **Explain how secure MLOps and CI/CD practices protect AI deployment pipelines**, including security testing, approval gates, separation of duties, artifact verification, deployment authorization, and rollback procedures.

12. **Differentiate between infrastructure monitoring and AI-specific monitoring**, including resource utilization, authentication activity, inference behavior, model performance, query patterns, and data or model drift.

13. **Recognize indicators of a potential AI security incident** and determine appropriate actions for investigation, containment, credential revocation, model rollback, recovery, and post-incident analysis.

14. **Apply defense-in-depth principles to AI deployment architecture** by combining multiple security controls to reduce attack surface, restrict attacker movement, and improve the resilience of production AI systems.

15. **Evaluate an AI deployment and recommend appropriate security improvements** based on identified vulnerabilities, threats, trust relationships, and operational requirements.

</details>

---

<details>
<summary><b>Section 1 — AI/ML Deployment Architecture and Attack Surface — Click to Expand</b></summary>
<br>

## Section 1 — AI/ML Deployment Architecture and Attack Surface

Securing an AI model does not stop when training is complete. Once a model is deployed, it becomes part of a larger system that may include APIs, applications, databases, cloud services, containers, authentication systems, model repositories, and monitoring tools. Each additional component creates new connections, dependencies, and potential attack paths.

This section introduces the architecture of a deployed AI system and explains how deployment changes the security problem. You will examine the major components surrounding a production model, identify trust boundaries and communication paths, and learn how traditional cybersecurity risks combine with AI-specific threats to create the overall AI system attack surface.

The goal is to begin thinking about AI security at the **system level rather than only at the model level**.

<br>

<details>
<summary><b>1.1 From Model Development to Deployment</b></summary>
<br>

### From Model Development to Deployment

During development, a machine learning model may exist primarily inside a controlled environment such as a research notebook, development workstation, or training server. Developers collect data, train the model, evaluate its performance, and save the resulting model artifact.

At this stage, attention is often focused on whether the model performs correctly. Security concerns may involve the integrity of the training data, adversarial manipulation, model privacy, or other threats discussed in previous modules.

Deployment changes the environment significantly.

A production model must normally interact with other systems so that users or applications can make predictions. The model may be placed inside a container, hosted on a cloud platform, exposed through an API, connected to databases, or integrated into a larger application.

A simplified deployment may look like:

**User → Application → API → Model → Prediction**

A production system may be considerably more complex:

**User → API Gateway → Authentication → Inference Service → Model → Database → Logging and Monitoring**

The security of the model now depends partly on the security of every component surrounding it.

For example, a model may be resistant to adversarial manipulation but still be exposed through an API that has no authentication. An organization may carefully protect its training dataset while accidentally deploying cloud storage that is publicly accessible. A secure model may also execute inside a container containing vulnerable software dependencies.

For this reason, deployment expands the security problem from **protecting a model** to **protecting an AI system**.

NIST's AI Risk Management Framework emphasizes managing AI risk throughout the lifecycle rather than treating security as a single activity performed during development [1]. NIST's adversarial machine learning taxonomy similarly considers attacks in relation to the broader AI lifecycle and the systems in which models operate [2].

A useful principle for this module is:

**A secure model does not automatically create a secure AI system.**

</details>

<br>

<details>
<summary><b>1.2 What Makes Up a Deployed AI System?</b></summary>
<br>

### What Makes Up a Deployed AI System?

A deployed AI system contains more than the machine learning model itself. Production environments commonly combine traditional software components with AI-specific resources.

The **model artifact** contains the trained parameters and structure needed to perform inference. Depending on the technology being used, this could be a serialized model file, neural-network checkpoint, or another model format.

The **inference service** loads the model and executes predictions. When an application submits an input, the inference service prepares the request, sends the appropriate data through the model, and returns the result.

An **API or inference endpoint** provides a way for users, applications, or other services to communicate with the inference service. Because this endpoint may be reachable across a network, it can become one of the most important externally accessible components of the AI system.

An **authentication and authorization system** determines who or what is permitted to access the AI service and what operations those identities may perform.

A **database or data store** may contain information required by the application, including user data, inference records, metadata, embeddings, or other information used during processing.

A **model registry or repository** may store approved model versions and associated metadata. Deployment systems can retrieve models from the registry when new versions are released.

**Containers and runtime infrastructure** provide the environment in which the model and inference software execute.

**Cloud and network infrastructure** connects these services and determines which systems are able to communicate.

Finally, **logging and monitoring systems** collect information about requests, authentication events, model behavior, resource consumption, and security events.

A simplified architecture might therefore contain:

**Client → API Gateway → Authentication → Inference Service → Model**

while the inference service may also communicate with:

- Databases
- Model storage
- Logging services
- Secret management systems
- External APIs
- Monitoring platforms

Each component performs a legitimate function, but each component also represents something that must be protected.

The security question therefore becomes more than:

**"Is the model secure?"**

It becomes:

**"Is every component that stores, executes, communicates with, or controls access to the model appropriately secured?"**

[1], [2]

</details>

<br>

<details>
<summary><b>1.3 Understanding the AI/ML Deployment Architecture</b></summary>
<br>

### Understanding the AI/ML Deployment Architecture

Before an AI system can be secured, defenders must understand how its components communicate.

Security teams frequently create architecture diagrams that identify systems, communication paths, external dependencies, and security boundaries. The same practice is valuable when analyzing AI deployments.

Consider the following simplified architecture:

**Internet User**

↓

**API Gateway**

↓

**Authentication Service**

↓

**AI Inference Service**

↓

**Production Model**

The inference service might also communicate with:

**Database | Model Registry | Logging System | Secret Manager**

Each arrow represents a communication path. Each communication path creates security questions.

For example:

- Who is allowed to communicate across this connection?
- How is the communicating service authenticated?
- Is the traffic encrypted?
- What information is transmitted?
- Can the destination modify sensitive resources?
- Is communication logged?
- What happens if one side of the connection becomes compromised?

Modern AI applications may consist of multiple services instead of a single application. NIST guidance for securing microservice architectures emphasizes protecting service-to-service communication and controlling access between individual application components [5], [6].

Understanding the architecture therefore allows defenders to determine not only **what exists**, but also **how compromise could spread through the system**.

Later sections of this module will return to these communication paths when examining API security, identity, container security, network segmentation, and monitoring.

</details>

<br>

<details>
<summary><b>1.4 Understanding the AI Attack Surface</b></summary>
<br>

### Understanding the AI Attack Surface

An **attack surface** is the collection of points where an attacker may attempt to interact with, manipulate, or gain unauthorized access to a system.

For a traditional application, the attack surface might include web interfaces, APIs, network ports, authentication mechanisms, databases, operating systems, and software dependencies.

AI systems contain these same components but introduce additional assets and interfaces associated with machine learning.

Examples include:

- Training datasets
- Model files
- Model registries
- Inference endpoints
- ML frameworks and libraries
- Feature stores
- Embedding or vector databases
- Model outputs
- Training infrastructure
- Model deployment pipelines

NIST's adversarial machine learning taxonomy demonstrates that attacks can occur against different components and stages of an AI system rather than only against the final model [2].

Consider an externally accessible inference API. An attacker may attempt to exploit the API itself through traditional application-security weaknesses. The same endpoint may simultaneously provide access that can be used to conduct model extraction, membership inference, adversarial probing, or resource-exhaustion attacks.

The endpoint therefore belongs to both the **traditional software attack surface** and the **AI-specific attack surface**.

This overlap is one reason securing production AI systems requires knowledge from both cybersecurity and machine learning security.

When evaluating an AI attack surface, defenders should ask:

- **What can an attacker reach?**
- **What can an attacker learn?**
- **What can an attacker modify?**
- **What resources can an attacker consume?**
- **Where could an attacker move next?**

These questions shift attention away from individual vulnerabilities and toward the overall exposure of the system.

</details>

<br>

<details>
<summary><b>1.5 Traditional vs. AI-Specific Attack Surfaces</b></summary>
<br>

### Traditional vs. AI-Specific Attack Surfaces

AI systems do not replace traditional cybersecurity problems. Instead, they add new security concerns on top of existing ones.

A deployed inference server is still a server. It may contain operating-system vulnerabilities, insecure network services, vulnerable software libraries, weak credentials, exposed secrets, or configuration mistakes.

An AI API is still an API. It may suffer from broken authentication, insufficient authorization, unrestricted resource consumption, or security misconfiguration. OWASP's API Security Top 10 documents many of these risks [31].

However, the API also provides access to a machine learning model.

That access may enable attacks that would not exist in a traditional application.

For example, repeated model queries may help an attacker approximate or steal model functionality. Tramèr et al. demonstrated that prediction APIs can provide enough information for attackers to reconstruct machine learning models through carefully designed queries [10].

The same interface could therefore experience:

**Traditional API Threat**

Unrestricted requests → Resource exhaustion

**AI-Specific Threat**

Unrestricted model queries → Model extraction

Another example is model storage. Traditional security concerns include unauthorized file access or modification. AI-specific consequences could include replacing the production model, introducing a backdoored model, or stealing proprietary model parameters.

This means defenders should avoid treating AI security and infrastructure security as separate problems.

A more complete way to think about the environment is:

**Traditional Cybersecurity Risks + AI-Specific Risks = AI System Security**

The remainder of this module will repeatedly use this idea.

</details>

<br>

<details>
<summary><b>1.6 Components, Connections, and Trust Boundaries</b></summary>
<br>

### Components, Connections, and Trust Boundaries

A **trust boundary** exists where data, requests, or control passes between components operating under different levels of trust or authority.

Trust boundaries are important because attackers often attempt to cross them.

One obvious trust boundary exists between the public Internet and an organization's AI API. However, trust boundaries can also exist internally.

For example:

**Internet → API Gateway**

**API Gateway → Inference Service**

**Inference Service → Database**

**Inference Service → Model Registry**

**CI/CD Pipeline → Production Environment**

Each transition should require defenders to consider what security controls are necessary.

Traditional network security sometimes assumed that systems inside an organization's internal network could be trusted more than external systems. Zero-trust architecture challenges this assumption.

NIST SP 800-207 explains that trust should not be granted simply because a user or system is located inside a particular network [3]. Access decisions should instead be based on identities, resources, policies, and other security information.

Applied to AI infrastructure, an inference service should not automatically receive unrestricted access to training data, model registries, credentials, or administrative services simply because all of those resources exist inside the same cloud environment.

Instead, each connection should have a legitimate purpose and an appropriate level of access.

This becomes especially important after a compromise.

If an attacker compromises the inference service, the attacker may inherit whatever access that service possesses.

The permissions and network connections assigned to the compromised service therefore help determine the **blast radius** of the attack.

</details>

<br>

<details>
<summary><b>1.7 Mapping Attacks to Deployment Components</b></summary>
<br>

### Mapping Attacks to Deployment Components

The attacks introduced in earlier modules can be connected directly to the components of a deployed AI system.

This allows defenders to move from asking:

**"How does this attack work?"**

to asking:

**"Which part of my production environment could enable this attack, and where should I place defenses?"**

Consider several examples.

#### Inference Endpoint

An exposed inference endpoint could enable:

- Model extraction
- Membership inference
- Adversarial probing
- Resource exhaustion
- Unauthorized model usage

#### Training Data Storage

Compromised data storage could enable:

- Data poisoning
- Dataset manipulation
- Unauthorized disclosure
- Training-data theft

#### Model Registry

A compromised model registry could enable:

- Model replacement
- Malicious model distribution
- Model theft
- Deployment of outdated or vulnerable models

#### Deployment Pipeline

A compromised deployment pipeline could enable:

- Unauthorized model deployment
- Artifact substitution
- Configuration manipulation
- Credential theft
- Supply-chain compromise

#### Container or Runtime

A vulnerable runtime could enable:

- Privilege escalation
- Container escape
- Resource abuse
- Access to credentials
- Modification or theft of model artifacts

#### Logging and Monitoring Infrastructure

Weak monitoring could allow attacks to continue without detection or make it difficult to determine what happened after an incident.

Mapping attacks to components is valuable because the appropriate defense often exists outside the model itself.

For example, model extraction may be an AI-specific attack, but important controls include API authentication, rate limiting, query monitoring, and access logging [10], [31].

This is a central theme of secure AI deployment:

**Many AI attacks require infrastructure-level defenses.**

</details>

<br>

<details>
<summary><b>1.8 Defense in Depth for AI Infrastructure</b></summary>
<br>

### Defense in Depth for AI Infrastructure

No single security control can protect an entire AI deployment.

Authentication can restrict access to an inference API, but stolen credentials may still allow an attacker to authenticate.

Rate limiting can restrict excessive queries, but it does not prevent every malicious request.

Network segmentation can limit communication between services, but it cannot guarantee that an individual service will never become compromised.

Container hardening reduces runtime risk, but vulnerable application code may still exist inside the container.

For this reason, AI infrastructure should use **defense in depth**.

Defense in depth places multiple security controls between an attacker and sensitive resources so that failure of one control does not automatically result in complete compromise.

Consider an attacker attempting to reach a production model:

**Internet → Firewall → API Gateway → Authentication → Authorization → Rate Limiting → Inference Service → Model**

If one layer fails, another may still prevent or detect the attack.

The same principle should exist throughout the rest of the architecture.

For example, even if an inference container becomes compromised:

- Network segmentation may prevent it from reaching the training database.
- Least-privilege permissions may prevent it from modifying models in the registry.
- Monitoring may detect abnormal activity.
- Model integrity verification may prevent an unauthorized artifact from being deployed.

NIST's AI RMF emphasizes managing AI risk throughout the lifecycle [1], while zero-trust architecture emphasizes limiting implicit trust between resources [3]. Together, these principles support an architecture in which multiple controls protect AI assets at different stages and boundaries.

The goal is not to assume that every component will remain secure forever.

Instead, secure infrastructure should be designed to answer:

**"If this component is compromised, what can the attacker do next?"**

A well-designed AI system limits that answer as much as possible.

</details>

<br>

<details>
<summary><b>Section 1 Summary</b></summary>
<br>

### Section 1 Summary

Deploying an AI model transforms it from an isolated model artifact into part of a larger software and infrastructure ecosystem. APIs, containers, cloud services, databases, model registries, credentials, networks, and monitoring platforms all contribute to the security of the final system.

A production AI environment therefore contains both **traditional cybersecurity attack surfaces** and **AI-specific attack surfaces**.

Throughout this section, you learned that:

- A secure model does not automatically create a secure AI system.
- Deployment introduces additional components, dependencies, and communication paths.
- Each externally or internally accessible component contributes to the attack surface.
- Traditional software vulnerabilities can enable AI-specific attacks.
- Trust boundaries identify where communication crosses between different security contexts.
- Compromised services should not automatically provide access to the rest of the AI environment.
- Previous AI attacks can be mapped to specific deployment components.
- Defense in depth uses multiple security layers to reduce the likelihood and impact of compromise.
- Secure architecture should limit the blast radius if an individual component becomes compromised.

The remainder of Module 6 will apply these principles to individual layers of the deployment architecture, beginning with one of the most important externally accessible components: **the AI inference API**.

</details>

</details>

In [ ]:

# @title Section 1 — AI/ML Deployment Architecture and Attack Surface Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.colors import ListedColormap
from IPython.display import display, HTML
from io import BytesIO
import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"
TEAL = "#0891B2"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):
    buffer = BytesIO()
    fig.savefig(
        buffer, format="png", dpi=135,
        bbox_inches="tight", facecolor="white"
    )
    plt.close(fig)

    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    buffer.close()

    display(HTML(f"""
    <details style="margin:12px 0;border:1px solid #cbd5e1;
                    border-radius:10px;overflow:hidden;">
      <summary style="cursor:pointer;padding:16px;font-size:16px;
                      font-weight:bold;background:#eff6ff;
                      color:#1e3a8a;border-left:5px solid {color};">
        {html.escape(title)} — Click to Expand
      </summary>
      <div style="padding:15px;background:white;text-align:center;">
        <img src="data:image/png;base64,{encoded}"
             style="width:100%;max-width:950px;height:auto;">
        <div style="max-width:850px;margin:15px auto;padding:16px;
                    background:#f8fafc;border-radius:8px;color:#334155;
                    font-size:14px;line-height:1.7;text-align:left;">
          {explanation}
        </div>
      </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def setup_diagram(ax, title, xlim=(0, 12), ylim=(0, 10)):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.axis("off")
    ax.set_title(title, fontsize=16, fontweight="bold", pad=18)


def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC", title_size=10):
    card = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.12",
        facecolor=background, edgecolor=accent, linewidth=1.8
    )
    ax.add_patch(card)

    ax.text(
        x+w/2, y+h*(0.66 if subtitle else 0.5), title,
        ha="center", va="center", fontsize=title_size,
        fontweight="bold", color=DARK
    )

    if subtitle:
        ax.text(
            x+w/2, y+h*0.28, subtitle,
            ha="center", va="center", fontsize=8.5, color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):
    ax.add_patch(FancyArrowPatch(
        start, end, arrowstyle="-|>", mutation_scale=16,
        linewidth=1.8, color=color
    ))


# ============================================================
# VISUALIZATION 1
# FROM MODEL DEVELOPMENT TO DEPLOYMENT
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
fig.suptitle(
    "From Model Development to Production Deployment",
    fontsize=16, fontweight="bold"
)

# Development environment
ax = axes[0]
setup_diagram(ax, "Development Environment", (0, 6), (0, 10))

development = [
    ("Training Data", "Collected and prepared", BLUE),
    ("Model Training", "Learn patterns from data", PURPLE),
    ("Model Evaluation", "Measure model performance", GREEN),
    ("Saved Model Artifact", "Export trained model", ORANGE)
]

for i, (title, subtitle, color) in enumerate(development):
    y = 8.0 - i*2.0
    add_card(ax, 0.6, y, 4.8, 1.25, title, subtitle, color)

    if i < len(development)-1:
        add_arrow(ax, (3, y-0.05), (3, y-0.7))

# Production environment
ax = axes[1]
setup_diagram(ax, "Production Environment", (0, 6), (0, 10))

production = [
    ("External Users", "Submit prediction requests", RED),
    ("API Gateway", "Expose a controlled endpoint", BLUE),
    ("Authentication", "Verify identity and access", GREEN),
    ("Inference Service", "Execute the deployed model", PURPLE),
    ("Production Model", "Generate predictions", ORANGE)
]

for i, (title, subtitle, color) in enumerate(production):
    y = 8.4 - i*1.65
    add_card(ax, 0.6, y, 4.8, 1.15, title, subtitle, color)

    if i < len(production)-1:
        add_arrow(ax, (3, y-0.05), (3, y-0.48))

plt.tight_layout()

show_collapsible_figure(
    "1.1 — From Model Development to Deployment",
    fig,
    """
    During model development, attention is often
    focused on preparing data, training the model,
    and evaluating prediction performance.

    <br><br>

    The resulting model may be stored as a file
    or checkpoint called a <b>model artifact</b>.

    <br><br>

    Deployment changes the environment because
    the model becomes accessible to users and
    applications through additional software
    and infrastructure.

    <br><br>

    A production system may introduce API gateways,
    authentication services, inference servers,
    containers, databases, and monitoring tools.

    <br><br>

    These components create additional opportunities
    for attackers to interact with the system.

    <br><br>

    For example, a model may perform well against
    adversarial examples but still be deployed
    behind an API with weak authentication.

    <br><br>

    Similarly, a carefully trained model may
    be stored in an insecure model registry.

    <br><br>

    The diagram simplifies development and production
    into two workflows. Real systems may include
    additional stages and supporting services.

    <br><br>

    <b>Key takeaway:</b> A secure model does not
    automatically create a secure AI system.
    Deployment expands security responsibilities
    to the surrounding infrastructure.
    """,
    BLUE
)


# ============================================================
# VISUALIZATION 2
# COMPONENTS OF A DEPLOYED AI SYSTEM
# ============================================================

fig, ax = plt.subplots(figsize=(13, 9))
setup_diagram(
    ax,
    "Components of a Production AI System",
    xlim=(0, 12), ylim=(0, 11)
)

# Main request path
main_components = [
    (0.4, 8.0, "Client", "User or application", GREEN),
    (3.3, 8.0, "API Gateway", "Request entry point", BLUE),
    (6.2, 8.0, "Auth Service", "Identity and permissions", PURPLE),
    (9.1, 8.0, "Inference Service", "Runs predictions", ORANGE)
]

for x, y, title, subtitle, color in main_components:
    add_card(ax, x, y, 2.5, 1.3, title, subtitle, color, title_size=9.5)

for x in [2.9, 5.8, 8.7]:
    add_arrow(ax, (x, 8.65), (x+0.4, 8.65))

# Supporting infrastructure
supporting = [
    (0.4, 4.5, "Model Registry", "Approved model artifacts", PURPLE),
    (3.3, 4.5, "Data Store", "Application information", RED),
    (6.2, 4.5, "Secret Manager", "Credentials and tokens", ORANGE),
    (9.1, 4.5, "Monitoring", "Logs, metrics, and alerts", GREEN)
]

for x, y, title, subtitle, color in supporting:
    add_card(ax, x, y, 2.5, 1.3, title, subtitle, color, title_size=9.5)

# Supporting services are connected to the inference service.
for x in [1.65, 4.55, 7.45, 10.35]:
    add_arrow(ax, (10.35, 8.0), (x, 5.8), GRAY)

add_card(
    ax, 3.3, 1.6, 5.4, 1.2,
    "Container / Runtime Infrastructure",
    "Hosts inference software and model execution",
    TEAL
)

add_arrow(ax, (10.35, 4.5), (7.2, 2.8), TEAL)

ax.text(
    6, 0.7,
    "Each component and communication path introduces security responsibilities",
    ha="center", fontsize=10.5, fontweight="bold", color=DARK
)

plt.tight_layout()

show_collapsible_figure(
    "1.2 — Components of a Deployed AI System",
    fig,
    """
    Production AI systems consist of multiple
    connected components rather than a model alone.

    <br><br>

    The <b>client</b> submits requests through
    an API gateway.

    <br><br>

    The <b>authentication and authorization service</b>
    helps determine whether the requester is
    permitted to access the application.

    <br><br>

    The <b>inference service</b> loads the model
    and executes predictions.

    <br><br>

    Supporting services provide capabilities
    required for production operation.

    <br><br>

    A <b>model registry</b> stores model versions,
    while a <b>data store</b> contains application
    information.

    <br><br>

    A <b>secret manager</b> protects credentials
    needed to access other services.

    <br><br>

    <b>Monitoring infrastructure</b> collects
    operational and security information.

    <br><br>

    Containers and runtime infrastructure
    provide the environment in which
    inference software executes.

    <br><br>

    This diagram is conceptual: not every
    application uses every component, and
    actual communication paths may differ.

    <br><br>

    <b>Key takeaway:</b> Securing an AI deployment
    requires protecting the components that
    store, execute, communicate with, and
    control access to the model.
    """,
    PURPLE
)


# ============================================================
# VISUALIZATION 3
# TRADITIONAL VS AI-SPECIFIC ATTACK SURFACES
# ============================================================

categories = [
    "API / Endpoint",
    "Model Storage",
    "Training Data",
    "Inference Runtime",
    "Deployment Pipeline",
    "Network Services"
]

# 1 means the category can be relevant to that risk class.
# This is a conceptual classification, not measured risk.

risk_matrix = np.array([
    [1, 1],
    [1, 1],
    [1, 1],
    [1, 1],
    [1, 1],
    [1, 0]
])

fig, ax = plt.subplots(figsize=(11, 7))

ax.imshow(
    risk_matrix,
    cmap=ListedColormap(["#F1F5F9", "#DBEAFE"]),
    vmin=0, vmax=1, aspect="auto"
)

ax.set_xticks([0, 1])
ax.set_xticklabels([
    "Traditional Cybersecurity Risks",
    "AI-Specific Security Risks"
])

ax.set_yticks(range(len(categories)))
ax.set_yticklabels(categories)

ax.set_title(
    "Traditional and AI-Specific Attack Surface Overlap",
    fontsize=16, fontweight="bold", pad=18
)

for i in range(len(categories)):
    for j in range(2):
        ax.text(
            j, i,
            "RELEVANT" if risk_matrix[i, j] else "INDIRECT",
            ha="center", va="center",
            fontsize=10, fontweight="bold",
            color=BLUE if risk_matrix[i, j] else GRAY
        )

ax.set_xticks(np.arange(-0.5, 2, 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(categories), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=3)
ax.tick_params(which="minor", bottom=False, left=False)

plt.tight_layout()

show_collapsible_figure(
    "1.3 — Traditional vs. AI-Specific Attack Surfaces",
    fig,
    """
    Deployed AI systems face both traditional
    cybersecurity threats and threats
    associated with machine learning.

    <br><br>

    Traditional risks include weak authentication,
    insecure network services, exposed credentials,
    vulnerable software, and configuration mistakes.

    <br><br>

    AI-specific threats include model extraction,
    membership inference, adversarial probing,
    data poisoning, and model replacement.

    <br><br>

    These threats frequently overlap.

    <br><br>

    For example, an inference API may be vulnerable
    to ordinary resource-exhaustion attacks.

    <br><br>

    The same API may also allow an attacker to
    repeatedly query the model and attempt
    to reconstruct its functionality.

    <br><br>

    A compromised model registry may expose
    ordinary storage-security weaknesses,
    while also allowing unauthorized model
    substitution.

    <br><br>

    The matrix identifies conceptual relevance,
    not the likelihood or severity of attacks.
    An indirect classification does not mean
    that a component is safe.

    <br><br>

    <b>Key takeaway:</b> AI security adds
    new risks to traditional infrastructure
    security rather than replacing it.
    """,
    ORANGE
)


# ============================================================
# VISUALIZATION 4
# TRUST BOUNDARIES AND COMMUNICATION PATHS
# ============================================================

fig, ax = plt.subplots(figsize=(12, 8))
setup_diagram(
    ax,
    "Trust Boundaries in a Deployed AI System",
    xlim=(0, 12), ylim=(0, 10)
)

zones = [
    (0.3, 6.4, 3.4, 2.7, "External Zone", "#FEE2E2", RED),
    (4.3, 6.4, 3.4, 2.7, "Application Zone", "#DBEAFE", BLUE),
    (8.3, 6.4, 3.4, 2.7, "Protected Zone", "#DCFCE7", GREEN)
]

for x, y, w, h, label, background, color in zones:
    ax.add_patch(FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.04,rounding_size=0.15",
        facecolor=background, edgecolor=color,
        linewidth=2, linestyle="--"
    ))
    ax.text(
        x+w/2, y+h-0.35, label,
        ha="center", va="center",
        fontsize=11, fontweight="bold", color=DARK
    )

add_card(ax, 0.7, 7.0, 2.6, 1.0, "Internet Client", accent=RED)
add_card(ax, 4.7, 7.0, 2.6, 1.0, "API Gateway", accent=BLUE)
add_card(ax, 8.7, 7.0, 2.6, 1.0, "Inference Service", accent=GREEN)

add_arrow(ax, (3.3, 7.5), (4.7, 7.5), RED)
add_arrow(ax, (7.3, 7.5), (8.7, 7.5), BLUE)

add_card(
    ax, 8.7, 3.3, 2.6, 1.0,
    "Model Registry", accent=PURPLE
)

add_card(
    ax, 4.7, 3.3, 2.6, 1.0,
    "Data Store", accent=ORANGE
)

add_arrow(ax, (10.0, 7.0), (10.0, 4.3), GREEN)
add_arrow(ax, (8.7, 7.2), (6.0, 4.3), ORANGE)

ax.text(
    4.0, 8.1,
    "Boundary 1",
    ha="center", fontsize=9, color=RED
)

ax.text(
    8.0, 8.1,
    "Boundary 2",
    ha="center", fontsize=9, color=BLUE
)

ax.text(
    6.0, 1.7,
    "Every boundary requires an explicit access decision",
    ha="center", fontsize=11, fontweight="bold", color=DARK
)

plt.tight_layout()

show_collapsible_figure(
    "1.4 — Trust Boundaries and Communication Paths",
    fig,
    """
    A trust boundary separates components
    that operate under different levels
    of trust or authority.

    <br><br>

    The first boundary in this example
    separates external clients from
    the application's API gateway.

    <br><br>

    Another boundary separates the
    application entry point from
    protected inference infrastructure.

    <br><br>

    Additional boundaries may exist
    between the inference service,
    databases, model registries,
    and other internal services.

    <br><br>

    Defenders should evaluate every
    communication path by asking:

    <br><br>

    Who is communicating? Is the identity
    verified? Is access authorized?
    What information is exchanged?
    Is communication protected and logged?

    <br><br>

    Zero-trust principles discourage
    automatically trusting services
    merely because they operate
    inside the same network.

    <br><br>

    A compromised inference service
    should not automatically receive
    unrestricted access to the
    model registry or training data.

    <br><br>

    <b>Key takeaway:</b> Trust boundaries
    help defenders identify where
    authentication, authorization,
    encryption, and monitoring
    controls should be enforced.
    """,
    BLUE
)


# ============================================================
# VISUALIZATION 5
# MAPPING ATTACKS TO DEPLOYMENT COMPONENTS
# ============================================================

components = [
    "Inference Endpoint",
    "Training Data Store",
    "Model Registry",
    "Deployment Pipeline",
    "Container / Runtime",
    "Monitoring System"
]

attacks = [
    "Extraction",
    "Poisoning",
    "Model Swap",
    "Supply Chain",
    "Privilege Abuse",
    "Detection Gap"
]

# Illustrative mapping of primary attack relationships.
mapping = np.array([
    [1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 1]
])

fig, ax = plt.subplots(figsize=(12, 7))

ax.imshow(
    mapping,
    cmap=ListedColormap(["#F1F5F9", "#FECACA"]),
    vmin=0, vmax=1,
    aspect="auto"
)

ax.set_xticks(range(len(attacks)))
ax.set_xticklabels(attacks, rotation=15, ha="right")

ax.set_yticks(range(len(components)))
ax.set_yticklabels(components)

ax.set_title(
    "Mapping AI Security Attacks to Deployment Components",
    fontsize=16, fontweight="bold", pad=18
)

for i in range(len(components)):
    for j in range(len(attacks)):
        if mapping[i, j]:
            ax.text(
                j, i, "X",
                ha="center", va="center",
                fontsize=14, fontweight="bold", color=RED
            )

ax.set_xticks(np.arange(-0.5, len(attacks), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(components), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", bottom=False, left=False)

plt.tight_layout()

show_collapsible_figure(
    "1.5 — Mapping Attacks to Deployment Components",
    fig,
    """
    Understanding how an attack works
    is only one part of defending
    a deployed AI system.

    <br><br>

    Defenders must also identify
    which infrastructure components
    could enable the attack.

    <br><br>

    An <b>inference endpoint</b>
    may provide access for model
    extraction or adversarial probing.

    <br><br>

    A compromised <b>training data store</b>
    may allow attackers to manipulate
    information used during training.

    <br><br>

    A compromised <b>model registry</b>
    may allow unauthorized replacement
    of a production model.

    <br><br>

    A vulnerable <b>deployment pipeline</b>
    may permit artifact substitution
    or supply-chain compromise.

    <br><br>

    Weak <b>runtime isolation</b>
    may enable privilege abuse.

    <br><br>

    Insufficient <b>monitoring</b>
    may prevent defenders from
    identifying ongoing attacks.

    <br><br>

    The X markers highlight selected
    primary relationships. The chart
    is not an exhaustive attack matrix:
    individual attacks may affect
    several components.

    <br><br>

    <b>Key takeaway:</b> Many AI-specific
    attacks require infrastructure-level
    defenses, including access controls,
    integrity checks, logging,
    and secure deployment practices.
    """,
    RED
)


# ============================================================
# VISUALIZATION 6
# DEFENSE IN DEPTH AND BLAST RADIUS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
fig.suptitle(
    "Defense in Depth and Compromise Blast Radius",
    fontsize=16, fontweight="bold"
)

# Left: broad access
ax = axes[0]
setup_diagram(ax, "Overly Permissive Architecture", (0, 6), (0, 10))

add_card(
    ax, 1.3, 7.6, 3.4, 1.2,
    "Compromised Inference Service",
    "Broad permissions",
    RED
)

targets = [
    (0.4, 4.7, "Training Data"),
    (3.2, 4.7, "Model Registry"),
    (0.4, 2.1, "Secret Store"),
    (3.2, 2.1, "Admin Services")
]

for x, y, title in targets:
    add_card(
        ax, x, y, 2.4, 1.1,
        title, "Potentially reachable", RED,
        title_size=9
    )
    add_arrow(ax, (3, 7.6), (x+1.2, y+1.1), RED)

# Right: restricted access
ax = axes[1]
setup_diagram(ax, "Least-Privilege Architecture", (0, 6), (0, 10))

add_card(
    ax, 1.3, 7.6, 3.4, 1.2,
    "Compromised Inference Service",
    "Restricted permissions",
    ORANGE
)

for x, y, title in targets:
    add_card(
        ax, x, y, 2.4, 1.1,
        title, "Access restricted", GREEN,
        title_size=9
    )

ax.text(
    3, 6.4,
    "Authorization and network controls",
    ha="center", fontsize=10,
    fontweight="bold", color=GREEN
)

ax.plot(
    [0.4, 5.6], [6.0, 6.0],
    linestyle="--", linewidth=3, color=GREEN
)

ax.text(
    3, 0.9,
    "Smaller potential blast radius",
    ha="center", fontsize=10,
    fontweight="bold", color=GREEN
)

plt.tight_layout()

show_collapsible_figure(
    "1.6 — Defense in Depth and Blast Radius Reduction",
    fig,
    """
    Defense in depth uses multiple
    independent security controls
    to protect important resources.

    <br><br>

    This visualization compares two
    hypothetical deployment architectures.

    <br><br>

    On the left, a compromised inference
    service has broad access to other
    infrastructure components.

    <br><br>

    Depending on its permissions,
    the attacker may be able to
    reach training data, model
    storage, credentials, or
    administrative services.

    <br><br>

    On the right, the inference
    service operates with more
    restricted permissions.

    <br><br>

    Independent authorization,
    network segmentation, and
    least-privilege access reduce
    the resources available
    after a compromise.

    <br><br>

    The <b>blast radius</b> describes
    the potential scope and impact
    of a security incident.

    <br><br>

    Reducing the blast radius
    does not guarantee that an
    individual service cannot
    be compromised.

    <br><br>

    Instead, it helps prevent
    one compromised component
    from providing unrestricted
    access to the entire environment.

    <br><br>

    Additional defenses such as
    authentication, rate limiting,
    container hardening, model
    integrity verification, and
    monitoring provide further
    protection.

    <br><br>

    <b>Key takeaway:</b> Secure
    infrastructure should be designed
    around the question: If one
    component is compromised,
    what can the attacker do next?
    """,
    GREEN
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("Section 1 — AI/ML Deployment Architecture and Attack Surface Visualizations")
print("-------------------------------------------------------------------------")
print("Six collapsible visualizations generated.")
print()
print("Concepts Covered")
print("----------------")
print("1. From Model Development to Deployment")
print("2. Components of a Deployed AI System")
print("3. Traditional vs. AI-Specific Attack Surfaces")
print("4. Trust Boundaries and Communication Paths")
print("5. Mapping Attacks to Deployment Components")
print("6. Defense in Depth and Blast Radius")
print()
print("All diagrams are conceptual teaching examples.")
print("No external services, model downloads, or attacks were used.")


<details>

<summary><b>Section 1 Lab — Click to Expand</b></summary>

<br>

A company is preparing to deploy a machine learning model that detects potentially malicious files.

During development, the model existed primarily as:

    DATA
      ↓
    MODEL
      ↓
    PREDICTION

Production deployment is more complicated.

The deployed system now contains:

    INTERNET USER
          ↓
       WEB API
          ↓
    MODEL SERVER
          ↓
       MODEL

The model server also communicates with:

    DATABASE

    MODEL REGISTRY

    LOGGING SERVICE

Each new component creates additional connections and potential attack paths.

Your job is to examine this architecture from a security perspective.

You will:

    Identify the components

    Map communication paths

    Identify exposed components

    Locate trust boundaries

    Simulate a compromised component

    Examine possible attack paths

The goal is to understand why securing the model alone does not secure the complete AI system.

This simulation uses only standard Python and should run almost instantly.

</details>

In [ ]:
# @title ▶ Part 1: Build the Deployment Architecture { display-mode: "form" }

components = {
    "Internet User": {
        "type": "External",
        "trusted": False
    },

    "Web API": {
        "type": "Application",
        "trusted": True
    },

    "Model Server": {
        "type": "AI Infrastructure",
        "trusted": True
    },

    "Model Registry": {
        "type": "Model Storage",
        "trusted": True
    },

    "Database": {
        "type": "Data Storage",
        "trusted": True
    },

    "Logging Service": {
        "type": "Monitoring",
        "trusted": True
    }
}

print("=" * 70)
print("AI DEPLOYMENT COMPONENTS")
print("=" * 70)

for name, information in components.items():

    print(f"\nComponent: {name}")
    print(f"Type:      {information['type']}")
    print(f"Trusted:   {information['trusted']}")

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

The deployed system contains much more than the machine learning model.

The environment now includes:

    EXTERNAL USERS

    APPLICATION INTERFACES

    MODEL INFRASTRUCTURE

    DATA STORAGE

    MODEL STORAGE

    MONITORING

Each component serves a legitimate purpose, but each component may also introduce security considerations.

Notice that the Internet user is marked:

    Trusted = False

External input should not automatically be trusted simply because the application accepts it.

The internal components are currently considered trusted, but later in the lab we will examine what happens if one of those components becomes compromised.

</details>

In [ ]:
# @title ▶ Part 2: Map Communication Paths { display-mode: "form" }

connections = [
    ("Internet User", "Web API"),
    ("Web API", "Model Server"),
    ("Model Server", "Model Registry"),
    ("Model Server", "Database"),
    ("Model Server", "Logging Service")
]

print("=" * 70)
print("AI SYSTEM COMMUNICATION PATHS")
print("=" * 70)

for source, destination in connections:

    print(
        f"\n{source}"
        f"\n   ↓"
        f"\n{destination}"
    )

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

Security analysis should consider not only which components exist but also how they communicate.

The primary inference path is:

    INTERNET USER
          ↓
       WEB API
          ↓
    MODEL SERVER

However, the model server also communicates with several internal systems.

This creates additional paths:

    MODEL SERVER
          ↓
    MODEL REGISTRY

    MODEL SERVER
          ↓
       DATABASE

    MODEL SERVER
          ↓
    LOGGING SERVICE

Communication paths matter because a compromised component may provide an attacker with opportunities to interact with additional systems.

The architecture should therefore be analyzed as a connected system rather than a collection of independent components.

</details>

In [ ]:
# @title ▶ Part 3: Identify the Attack Surface { display-mode: "form" }

attack_surface = {
    "Web API": [
        "Unauthorized requests",
        "Malicious input",
        "Excessive requests"
    ],

    "Model Server": [
        "Runtime compromise",
        "Unauthorized model access"
    ],

    "Model Registry": [
        "Model replacement",
        "Unauthorized artifact access"
    ],

    "Database": [
        "Sensitive data exposure",
        "Unauthorized queries"
    ],

    "Logging Service": [
        "Log manipulation",
        "Sensitive information exposure"
    ]
}

print("=" * 70)
print("AI DEPLOYMENT ATTACK SURFACE")
print("=" * 70)

for component, risks in attack_surface.items():

    print(f"\n{component}")

    for risk in risks:
        print(f"  • {risk}")

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

Each component introduces different security risks.

The Web API is especially important because it is directly exposed to requests originating outside the trusted environment.

However, internal components should not be ignored.

The model registry contains model artifacts.

The database may contain sensitive information.

The model server executes the AI workload.

The logging service contains information about system activity.

The complete attack surface is therefore larger than:

    ATTACKER
        ↓
      MODEL

A better representation is:

    ATTACKER
        ↓
      API
        ↓
    APPLICATION
        ↓
    MODEL SERVER
       ↙   ↓   ↘
    MODEL  DATA  LOGS

An attacker may attempt to compromise one component and then use that access to reach another.

</details>

In [ ]:
# @title ▶ Part 4: Identify Trust Boundaries { display-mode: "form" }

print("=" * 70)
print("TRUST BOUNDARY ANALYSIS")
print("=" * 70)

for source, destination in connections:

    source_trust = components[source]["trusted"]
    destination_trust = components[destination]["trusted"]

    crosses_boundary = (
        source_trust != destination_trust
    )

    print(
        f"\n{source} → {destination}"
    )

    if crosses_boundary:
        print(
            "⚠ TRUST BOUNDARY CROSSED"
        )
    else:
        print(
            "Internal communication path"
        )

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

A particularly important connection appears between:

    INTERNET USER
          ↓
       WEB API

Information moves from an untrusted external environment into a trusted application environment.

This represents a **trust boundary**.

Crossing a trust boundary should usually involve security controls.

For example:

    UNTRUSTED REQUEST
          ↓
    AUTHENTICATION
          ↓
    AUTHORIZATION
          ↓
    INPUT VALIDATION
          ↓
    TRUSTED APPLICATION

Later sections of this module will examine these controls in greater detail.

For now, the important skill is recognizing where trust changes within the architecture.

</details>

In [ ]:
# @title ▶ Part 5: Simulate a Compromise { display-mode: "form" }

compromised_component = "Model Server"

print("=" * 70)
print("COMPROMISE SIMULATION")
print("=" * 70)

print(
    f"\nCompromised Component: "
    f"{compromised_component}"
)

reachable_components = []

for source, destination in connections:

    if source == compromised_component:
        reachable_components.append(
            destination
        )

print(
    "\nSystems directly reachable "
    "from the compromised component:"
)

for component in reachable_components:
    print(f"  • {component}")

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

The model server has now been treated as compromised.

Look at the systems it can directly communicate with.

The compromised model server can reach:

    MODEL REGISTRY

    DATABASE

    LOGGING SERVICE

This demonstrates why internal architecture matters.

An attacker who compromises one component may attempt to use its existing communication paths to reach additional resources.

The security question becomes:

> **If this component is compromised, what else can the attacker reach?**

This concept will become especially important when we examine network segmentation, cloud infrastructure, identity, and least privilege later in the module.

</details>

In [ ]:
# @title ▶ Part 6: Compare Deployment Designs { display-mode: "form" }

designs = {
    "Design A — Broad Access": {
        "Model Registry": True,
        "Database": True,
        "Logging Service": True,
        "Admin Service": True,
        "Training Storage": True
    },

    "Design B — Restricted Access": {
        "Model Registry": True,
        "Database": True,
        "Logging Service": True,
        "Admin Service": False,
        "Training Storage": False
    }
}

print("=" * 70)
print("DEPLOYMENT ACCESS COMPARISON")
print("=" * 70)

for design, permissions in designs.items():

    accessible = sum(
        permissions.values()
    )

    print(f"\n{design}")
    print("-" * 50)

    for resource, allowed in permissions.items():

        status = (
            "Accessible"
            if allowed
            else "Blocked"
        )

        print(
            f"{resource:<20} {status}"
        )

    print(
        f"\nTotal Reachable Resources: "
        f"{accessible}"
    )

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

The two architectures provide the model server with different levels of access.

### Design A

The model server can communicate with resources that it may not actually require during inference.

If the model server becomes compromised, those unnecessary connections increase the attacker's opportunities.

### Design B

The model server can reach only the resources required for its intended operation.

This reduces the potential impact of a compromise.

This introduces a principle that will appear repeatedly throughout this module:

> **A component should only have access to the systems and resources required for its intended function.**

Reducing unnecessary connections reduces the attack surface and can make lateral movement more difficult.

</details>

In [ ]:
# @title ▶ Part 7: Student Investigation { display-mode: "form" }

test_components = [
    "Web API",
    "Model Server",
    "Model Registry",
    "Database",
    "Logging Service"
]

print("=" * 70)
print("STUDENT INVESTIGATION")
print("=" * 70)

for target in test_components:

    reachable = []

    for source, destination in connections:

        if source == target:
            reachable.append(
                destination
            )

    print(f"\n{target}")
    print("-" * 40)

    if reachable:

        print("Can directly reach:")

        for resource in reachable:
            print(f"  • {resource}")

    else:
        print(
            "No outgoing connections "
            "in this architecture."
        )

<details>

<summary><b>Part 7 Explanation — Click to Expand</b></summary>

<br>

Review the output and compare the different components.

Ask yourself:

Which component is directly exposed to an external user?

Which component has the most internal connections?

Which systems contain valuable assets?

Which component would create the greatest opportunities for additional access if it were compromised?

You can also modify the `connections` list from Part 2.

Try adding:

    ("Web API", "Database")

Then rerun the investigation.

Ask yourself:

> Does the Web API actually need direct database access?

Every unnecessary connection can create another potential attack path.

</details>

In [ ]:
# @title ▶ Part 8: Knowledge Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. Which component was directly exposed to the Internet user?",
        "options": [
            "A. Web API",
            "B. Model Registry",
            "C. Database",
            "D. Logging Service"
        ],
        "answer": "A"
    },

    {
        "question":
        "2. Where did the main trust boundary appear?",
        "options": [
            "A. Between the Internet user and Web API",
            "B. Inside the model file",
            "C. Between training epochs",
            "D. Inside the database"
        ],
        "answer": "A"
    },

    {
        "question":
        "3. Why was compromise of the Model Server important?",
        "options": [
            "A. It had connections to multiple internal resources",
            "B. It automatically retrained the model",
            "C. It removed the Internet",
            "D. It had no access to anything"
        ],
        "answer": "A"
    },

    {
        "question":
        "4. Which design reduced the number of reachable resources?",
        "options": [
            "A. Restricted Access",
            "B. Broad Access"
        ],
        "answer": "A"
    },

    {
        "question":
        "5. What is the main lesson from this lab?",
        "options": [
            "A. AI security requires examining the complete deployment architecture",
            "B. Only model accuracy matters after deployment",
            "C. Internal services never need security controls",
            "D. Every component should communicate with every other component"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("LAB 1 — KNOWLEDGE CHECK")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1

    else:
        print(
            f"✗ Incorrect. "
            f"Correct answer: {q['answer']}"
        )

percentage = (
    score / len(questions)
) * 100

print("\n" + "=" * 70)
print(
    f"Lab Score: "
    f"{score}/{len(questions)}"
)

print(
    f"Percentage: "
    f"{percentage:.1f}%"
)

if percentage >= 80:
    print("✓ Lab complete!")

else:
    print(
        "△ Review the lab explanations "
        "and try again."
    )

<details>

<summary><b>Lab 1 — Practical Takeaways — Click to Expand</b></summary>

<br>

## Lab 1 — Practical Takeaways

This lab demonstrated how deployment changes the AI security problem.

During development, it may be tempting to think primarily about:

    DATA
      ↓
    MODEL
      ↓
    PREDICTION

A production environment contains many additional components:

    USER
      ↓
     API
      ↓
    MODEL SERVER
       ↓
    MODEL

along with connections to:

    DATABASES

    MODEL REGISTRIES

    LOGGING SYSTEMS

    CLOUD SERVICES

    OTHER INFRASTRUCTURE

Each component and communication path can increase the attack surface.

You also observed what happens when the model server is treated as compromised.

Because it could communicate with several internal resources, compromise of one component could potentially create opportunities to reach other parts of the environment.

Restricting unnecessary communication reduced the number of reachable resources.

The main lesson is:

> **Securing a deployed AI system requires protecting the architecture surrounding the model, not only the model itself.**

As you move through this module, the following sections will introduce specific controls for protecting these components and communication paths.

</details>

In [ ]:
# @title ▶ Section 1 Checkpoint — Deployment Architecture & Attack Surface { display-mode: "form" }

questions = [
    {
        "question":
        "1. Why does deploying an AI model increase its attack surface?",
        "options": [
            "A. The model becomes connected to additional systems and services",
            "B. Deployment automatically changes the training dataset",
            "C. Deployment removes all authentication",
            "D. The model becomes larger"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. Which is an example of a component surrounding a deployed AI model?",
        "options": [
            "A. API",
            "B. Database",
            "C. Model registry",
            "D. All of the above"
        ],
        "answer": "D"
    },
    {
        "question":
        "3. What is a trust boundary?",
        "options": [
            "A. A point where data or requests move between components with different trust levels",
            "B. The physical size of a model",
            "C. The number of training epochs",
            "D. A model accuracy threshold"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Why are communication paths important during security analysis?",
        "options": [
            "A. They show how information and requests move between components",
            "B. They determine model accuracy",
            "C. They replace authentication",
            "D. They eliminate vulnerabilities"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. What should defenders consider when securing a deployed AI system?",
        "options": [
            "A. Only the model",
            "B. Only the training dataset",
            "C. The complete system surrounding the model",
            "D. Only the user interface"
        ],
        "answer": "C"
    }
]

score = 0

print("=" * 70)
print("SECTION 1 CHECKPOINT")
print("AI/ML DEPLOYMENT ARCHITECTURE AND ATTACK SURFACE")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. "
            f"Correct answer: {q['answer']}"
        )

percentage = (
    score / len(questions)
) * 100

print("\n" + "=" * 70)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Checkpoint complete!")
else:
    print(
        "△ Review Section 1 and try again."
    )

---

<details>
<summary><b>Section 2 — Securing AI APIs and Inference Endpoints — Click to Expand</b></summary>
<br>

## Section 2 — Securing AI APIs and Inference Endpoints

Once an AI model is deployed, users and applications need a way to interact with it. In many production environments, this interaction occurs through an **Application Programming Interface (API)** or inference endpoint.

The inference endpoint may be one of the most exposed components of an AI system because it accepts requests from users, applications, or other services and passes those requests to the model. If the endpoint is poorly secured, attackers may be able to access the model without authorization, submit large numbers of queries, consume excessive computing resources, collect sensitive model outputs, or use the interface to support attacks such as model extraction.

This section examines how established API security controls can be applied specifically to AI inference systems. The goal is not only to protect the API itself, but also to control how users and applications interact with the model behind it.

<br>

<details>
<summary><b>2.1 Why AI Models Need Secure Interfaces</b></summary>
<br>

### Why AI Models Need Secure Interfaces

A trained model normally cannot be useful to an application unless some mechanism allows the application to provide input and receive a prediction or generated response.

An inference API provides this interface.

A simplified interaction may look like:

**User → API → Model → Prediction → User**

For example, a phishing-detection model might receive a request containing information about an email and return a classification:

**Input:** Email content

**Output:** Phishing probability: 0.94

The API makes the model useful, but it also makes the model reachable.

If the endpoint is publicly accessible without appropriate security controls, an attacker may be able to interact with the model repeatedly or in ways that were never intended by the system designers.

This can create several risks:

- Unauthorized use of the model
- Model extraction
- Information leakage
- Resource exhaustion
- Automated probing
- Abuse of expensive inference resources
- Exposure of sensitive input or output data

Tramèr et al. demonstrated that prediction APIs can expose enough information for attackers to approximate or reconstruct machine learning models through repeated queries [10]. Research into cloud-hosted ML systems has similarly shown that exposing machine learning capabilities through remote services creates additional security and privacy concerns [11].

Therefore, an inference endpoint should not simply be treated as a convenient doorway to the model.

It should be treated as a **security boundary**.

</details>

<br>

<details>
<summary><b>2.2 Understanding the AI Inference Endpoint</b></summary>
<br>

### Understanding the AI Inference Endpoint

An **inference endpoint** is a network-accessible interface that allows a client to submit data to a deployed model and receive the model's result.

A request may travel through several components before reaching the model:

**Client → API Gateway → Authentication → Inference Service → Model**

The model then generates a result that travels back through the system:

**Model → Inference Service → API → Client**

Depending on the application, requests may contain:

- Images
- Text
- Documents
- Numerical features
- Audio
- Embeddings
- Application metadata

Responses may contain:

- Predicted classes
- Probability scores
- Confidence values
- Generated text
- Embeddings
- Recommendations
- Other model-derived information

Both directions create security concerns.

Incoming requests may be malicious, oversized, malformed, unauthorized, or intentionally designed to probe the model.

Outgoing responses may reveal more information than the user actually needs.

This means endpoint security must control both:

**What enters the AI system**

and

**What leaves the AI system.**

Securing the inference endpoint therefore requires multiple controls rather than a single protection mechanism.

</details>

<br>

<details>
<summary><b>2.3 Authentication vs. Authorization</b></summary>
<br>

### Authentication vs. Authorization

Two important controls for protecting AI APIs are **authentication** and **authorization**.

Although these terms are closely related, they answer different questions.

**Authentication asks:**

**"Who are you?"**

Authentication verifies the identity of the user, application, or service making the request.

Examples include:

- API keys
- Access tokens
- OAuth tokens
- Signed credentials
- Service identities
- Certificates

After an identity has been verified, the system must determine what that identity is allowed to do.

**Authorization asks:**

**"What are you allowed to do?"**

For example, an authenticated user may be allowed to request predictions but not:

- Upload a new model
- Delete a production model
- Read administrative logs
- Modify API configuration
- Access training data

Authentication without authorization can still create serious security problems.

A secure AI service should therefore follow:

**Request → Authenticate Identity → Check Authorization → Allow or Deny**

OWASP identifies broken authentication and broken authorization as major API security risks [31]. These problems become especially important when APIs provide access to valuable or sensitive AI resources.

Identity and access control will be explored more deeply in Section 3.

</details>

<br>

<details>
<summary><b>2.4 Protecting AI Endpoints with Access Controls</b></summary>
<br>

### Protecting AI Endpoints with Access Controls

Not every AI endpoint should be available to every user.

Different endpoints may provide very different capabilities.

Consider an AI deployment containing:

**/predict**

**/model/upload**

**/model/delete**

**/logs**

**/admin/config**

The `/predict` endpoint may need to be accessible to authorized application users.

The `/model/upload` endpoint may only need to be accessible to the deployment pipeline.

The `/logs` endpoint may only need to be accessible to security personnel or administrators.

Giving every authenticated user access to every endpoint would violate the principle of least privilege.

Instead, access should be limited according to the user's or service's legitimate responsibilities.

For example:

**Application User → Prediction Access**

**Deployment Service → Model Deployment Access**

**Security Team → Security Log Access**

**Administrator → Configuration Access**

An attacker who compromises a normal user account would then receive only the capabilities associated with that account rather than administrative control over the AI environment.

This is an example of reducing **blast radius** through authorization.

OWASP's API security guidance identifies broken function-level authorization as a significant risk when users can access functions that should be restricted to other roles [31].

For AI systems, access controls should protect not only prediction functionality but also model management, administrative operations, training resources, logs, and configuration interfaces.

</details>

<br>

<details>
<summary><b>2.5 Request and Input Validation</b></summary>
<br>

### Request and Input Validation

An inference service should never assume that incoming requests are safe simply because they arrived through the expected API.

Requests should be validated before they reach sensitive application logic or the model.

Validation may include checking:

- Required fields
- Data types
- File formats
- Input dimensions
- Request size
- Character limits
- Allowed parameter values
- Content types
- Unexpected fields

Suppose an image-classification API expects:

**224 × 224 RGB images**

The API should not automatically accept arbitrary files of unlimited size.

Similarly, if a text classifier expects a limited amount of text, the service may reject inputs that greatly exceed the expected length.

A basic request path should therefore resemble:

**Request**

↓

**Validate**

↓

**Valid?**

**No → Reject**

**Yes → Continue to Model**

Input validation cannot prevent every AI-specific attack. An adversarial example, for instance, may still satisfy all normal format requirements.

However, validation reduces unnecessary exposure by preventing obviously invalid, malformed, or excessive requests from reaching deeper parts of the system.

This demonstrates defense in depth: **API validation and model-level defenses perform different security functions and should work together.**

</details>

<br>

<details>
<summary><b>2.6 Rate Limiting and Resource Protection</b></summary>
<br>

### Rate Limiting and Resource Protection

AI inference can consume significant computational resources.

Depending on the model, a single request may require CPU time, GPU time, memory, network bandwidth, or access to external services. Large generative models may make individual inference requests especially expensive.

An API that allows unlimited requests can therefore create both security and availability risks.

**Rate limiting** restricts how frequently a user, application, IP address, token, or other identity may submit requests.

For example:

**Normal User → 60 requests per minute**

instead of:

**User → Unlimited requests**

Rate limiting can help protect against:

- Resource exhaustion
- Automated API abuse
- Excessive inference costs
- Model extraction
- Automated reconnaissance
- Large-scale probing

OWASP identifies unrestricted resource consumption as a major API security risk because API requests can consume resources such as CPU, memory, storage, bandwidth, and third-party services [31].

Rate limiting can be combined with other controls, including:

- Maximum request size
- Maximum batch size
- Execution timeouts
- Query quotas
- Concurrent-request limits
- Spending limits
- Resource quotas

These controls are particularly important for AI systems because attackers may not need to exploit a traditional software vulnerability to cause damage.

Simply forcing the model to perform excessive amounts of legitimate computation may be enough to affect availability or cost.

</details>

<br>

<details>
<summary><b>2.7 Secure Communication with TLS/HTTPS</b></summary>
<br>

### Secure Communication with TLS/HTTPS

AI inference requests may contain sensitive information.

Depending on the application, users could submit:

- Personal information
- Business documents
- Financial information
- Images
- Security telemetry
- Proprietary data
- Internal communications

If this information travels across a network without encryption, an attacker with access to the communication path may be able to observe or manipulate it.

AI APIs should therefore use encrypted communication such as **HTTPS using Transport Layer Security (TLS)**.

Instead of:

**Client → Unencrypted Request → AI API**

the system should use:

**Client → Encrypted TLS Connection → AI API**

Encryption in transit helps provide confidentiality and integrity for network communication.

TLS can protect:

- Authentication tokens
- API keys transmitted with requests
- User input
- Model predictions
- Sensitive metadata

Encryption should also be considered for **service-to-service communication** inside the environment.

An inference service communicating with another internal service should not automatically assume that the network is trusted simply because both systems are hosted in the same cloud environment.

This follows the zero-trust principle introduced in Section 1 [3].

</details>

<br>

<details>
<summary><b>2.8 API Gateways and Inference Gateways</b></summary>
<br>

### API Gateways and Inference Gateways

As AI deployments become more complex, organizations may place an **API gateway** between external clients and internal AI services.

Instead of allowing users to communicate directly with the inference server:

**Internet → Inference Server**

the architecture may use:

**Internet → API Gateway → Inference Server**

The gateway becomes a controlled entry point into the AI environment.

Depending on the implementation, gateways may enforce:

- Authentication
- Authorization
- Rate limiting
- Request validation
- TLS
- Logging
- Routing
- Request quotas
- Traffic policies

This reduces the number of security responsibilities that must be implemented independently by every model service.

AI environments may also use specialized **inference gateways** designed to manage traffic to multiple models or inference workloads.

For example:

**Users**

↓

**Inference Gateway**

↓

**Model A | Model B | Model C**

The gateway can determine which model receives a request while applying security and operational policies before the request reaches the inference workload.

However, gateways should not be considered a complete security solution.

If an attacker can bypass the gateway and communicate directly with an internal inference service, many gateway protections may become ineffective.

The architecture should therefore ensure that protected services are reachable only through approved communication paths whenever possible.

</details>

<br>

<details>
<summary><b>2.9 Preventing Information Leakage Through API Responses</b></summary>
<br>

### Preventing Information Leakage Through API Responses

Securing an inference API requires controlling not only requests but also responses.

A model may produce more information internally than the user actually needs.

Consider a classification model that internally produces:

**Cat: 0.03**

**Dog: 0.94**

**Wolf: 0.02**

**Other: 0.01**

The application may only need to return:

**Prediction: Dog**

Returning every confidence score may provide useful functionality in some applications, but unnecessary information can also help attackers understand model behavior.

Tramèr et al. demonstrated that prediction APIs can leak information useful for reconstructing model functionality [10].

Developers should therefore ask:

**What information does the user actually need?**

Responses should avoid unnecessarily exposing:

- Internal model details
- Debug information
- Stack traces
- File paths
- Infrastructure information
- Sensitive metadata
- Unnecessary confidence values
- Administrative information

This does not mean confidence scores should never be returned. Some applications legitimately require them.

The security principle is **data minimization**:

**Return the information required for the legitimate use case without unnecessarily exposing internal system information.**

</details>

<br>

<details>
<summary><b>2.10 Securing Connections to Third-Party APIs</b></summary>
<br>

### Securing Connections to Third-Party APIs

Modern AI applications frequently depend on external services.

An AI application may send requests to:

- Cloud model providers
- External LLM APIs
- Search services
- Databases
- Content-processing services
- Authentication providers
- Storage services

These connections create additional trust relationships.

A simplified application might look like:

**User → Application → External AI Provider**

The application is now dependent on the security and availability of another service.

OWASP warns that applications may become vulnerable when they trust data received from third-party APIs more than data received directly from users [31].

External responses should therefore still be treated as untrusted input.

Organizations should also consider:

- Whether communication uses TLS
- How third-party credentials are stored
- What information is being transmitted
- Whether sensitive data is permitted to leave the organization
- What permissions external credentials provide
- How failures and timeouts are handled
- Whether responses are validated
- Whether third-party usage is logged

Third-party AI services can provide valuable capabilities, but using an external service expands the system's attack surface and introduces another trust boundary.

</details>

<br>

<details>
<summary><b>2.11 Connecting API Vulnerabilities to AI Attacks</b></summary>
<br>

### Connecting API Vulnerabilities to AI Attacks

One of the most important lessons from this section is that traditional API weaknesses can make AI-specific attacks easier to perform.

Consider several examples.

#### No Authentication

If an inference endpoint does not require authentication, anyone who discovers the endpoint may be able to query the model.

**No Authentication → Unauthorized Model Access**

This may enable reconnaissance, model extraction, or unauthorized use.

#### No Rate Limiting

An attacker may submit thousands or millions of queries.

**Unlimited Queries → Automated Model Probing**

This can support model extraction, inference attacks, or resource exhaustion.

#### Poor Input Validation

Malformed or unexpectedly large inputs may reach the inference service.

**Unvalidated Input → Increased Attack Surface**

This can expose application weaknesses or consume unnecessary resources.

#### Verbose Responses

Detailed confidence scores, debug messages, or internal metadata may reveal unnecessary information.

**Verbose Responses → Information Leakage**

That information may help an attacker better understand the system or model.

#### Unencrypted Communication

Credentials, inference inputs, or model responses may become exposed during transmission.

**No TLS → Data or Credential Exposure**

#### Excessive Authorization

A user who only needs inference access may also receive model-management privileges.

**Excessive Permissions → Larger Blast Radius**

The key idea is:

**AI-specific attacks are often enabled or amplified by ordinary infrastructure weaknesses.**

This is why protecting an AI model requires more than implementing defenses inside the model itself.

The infrastructure that controls access to the model is part of the defense.

</details>

<br>

<details>
<summary><b>Section 2 Summary</b></summary>
<br>

### Section 2 Summary

Inference APIs make AI models accessible to users and applications, but that accessibility also creates one of the most important attack surfaces in a production AI system.

Throughout this section, you learned that:

- Inference endpoints should be treated as security boundaries.
- Authentication verifies who or what is making a request.
- Authorization determines what an authenticated identity is allowed to do.
- Access should be restricted according to legitimate responsibilities.
- Requests should be validated before reaching the model.
- Rate limiting and resource controls can reduce automated abuse and resource exhaustion.
- TLS protects sensitive inference traffic while it moves across networks.
- API and inference gateways can provide centralized security controls.
- API responses should avoid unnecessarily exposing model or infrastructure information.
- Third-party APIs introduce additional trust relationships and attack surfaces.
- Traditional API vulnerabilities can enable or amplify AI-specific attacks such as model extraction, inference attacks, and resource exhaustion.

The next section examines **Identity, Access Control, and Least Privilege** in greater detail. These concepts determine which users, applications, and services can access sensitive AI resources and help limit what an attacker can reach if an identity becomes compromised.

</details>

</details>

In [ ]:

# @title Section 2 — Securing AI APIs and Inference Endpoints Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import ListedColormap
from IPython.display import display, HTML
from io import BytesIO
import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"
TEAL = "#0891B2"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):
    buffer = BytesIO()
    fig.savefig(
        buffer, format="png", dpi=135,
        bbox_inches="tight", facecolor="white"
    )
    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")
    buffer.close()

    display(HTML(f"""
    <details style="margin:12px 0;border:1px solid #cbd5e1;
                    border-radius:10px;overflow:hidden;">
      <summary style="cursor:pointer;padding:16px;font-size:16px;
                      font-weight:bold;background:#eff6ff;
                      color:#1e3a8a;border-left:5px solid {color};">
        {html.escape(title)} — Click to Expand
      </summary>
      <div style="padding:15px;background:white;text-align:center;">
        <img src="data:image/png;base64,{encoded}"
             style="width:100%;max-width:950px;height:auto;">
        <div style="max-width:850px;margin:15px auto;padding:16px;
                    background:#f8fafc;border-radius:8px;color:#334155;
                    font-size:14px;line-height:1.7;text-align:left;">
          {explanation}
        </div>
      </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def setup_diagram(ax, title, xlim=(0, 12), ylim=(0, 10)):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.axis("off")
    ax.set_title(
        title, fontsize=16, fontweight="bold", pad=18
    )


def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC",
             title_size=10):
    card = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.12",
        facecolor=background, edgecolor=accent, linewidth=1.8
    )
    ax.add_patch(card)

    ax.text(
        x+w/2, y+h*(0.66 if subtitle else 0.5),
        title, ha="center", va="center",
        fontsize=title_size, fontweight="bold", color=DARK
    )

    if subtitle:
        ax.text(
            x+w/2, y+h*0.28, subtitle,
            ha="center", va="center",
            fontsize=8.5, color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):
    ax.add_patch(FancyArrowPatch(
        start, end, arrowstyle="-|>",
        mutation_scale=16, linewidth=1.8, color=color
    ))


# ============================================================
# VISUALIZATION 1
# AI INFERENCE ENDPOINT ARCHITECTURE
# ============================================================

fig, ax = plt.subplots(figsize=(12, 9))

setup_diagram(
    ax,
    "AI Inference API — Request and Response Lifecycle",
    xlim=(0, 12), ylim=(0, 11)
)

components = [
    ("Client Application", "Submits an inference request", GREEN),
    ("API Gateway", "Receives and routes requests", BLUE),
    ("Authentication / Authorization", "Verifies identity and permissions", PURPLE),
    ("Request Validation", "Checks format, size, and parameters", ORANGE),
    ("Inference Service", "Processes validated model input", BLUE),
    ("Trained AI Model", "Produces a prediction or response", GREEN)
]

for i, (title, subtitle, accent) in enumerate(components):
    y = 9.4 - i*1.65

    add_card(
        ax, 2.0, y, 8.0, 1.15,
        title, subtitle, accent
    )

    if i < len(components)-1:
        add_arrow(
            ax, (6, y-0.05), (6, y-0.45)
        )

ax.text(
    10.6, 5.3,
    "Request\npath",
    ha="center", va="center",
    fontsize=11, fontweight="bold", color=BLUE
)

ax.text(
    0.9, 5.3,
    "Response\nreturns\nthrough\nservice",
    ha="center", va="center",
    fontsize=9, color=GRAY
)

plt.tight_layout()

show_collapsible_figure(
    "2.1 — Understanding the AI Inference Endpoint",
    fig,
    """
    An inference endpoint allows a user or application
    to submit data to a deployed AI model and receive
    a prediction or generated response.

    <br><br>

    The client normally communicates with an API
    rather than accessing the model directly.

    <br><br>

    In a production architecture, the request may
    pass through an API gateway, authentication
    and authorization controls, and input validation
    before reaching the inference service.

    <br><br>

    The inference service loads or accesses the
    trained model and executes the prediction.

    <br><br>

    The result is then returned through the
    application to the requesting client.

    <br><br>

    Both directions introduce security concerns.

    <br><br>

    Incoming requests may be unauthorized,
    malformed, oversized, or intended to
    probe the model.

    <br><br>

    Outgoing responses may expose sensitive
    information or unnecessary model details.

    <br><br>

    <b>Key takeaway:</b> The inference endpoint
    is a security boundary that must control
    both what enters the AI system and what
    information leaves it.
    """,
    BLUE
)


# ============================================================
# VISUALIZATION 2
# AUTHENTICATION, AUTHORIZATION, AND ENDPOINT ACCESS
# ============================================================

roles = [
    "Application User",
    "Deployment Service",
    "Security Reviewer",
    "Administrator"
]

endpoints = [
    "/predict",
    "/model/upload",
    "/model/delete",
    "/logs",
    "/admin/config"
]

# Illustrative role permissions.
# 1 = permitted, 0 = denied.

permissions = np.array([
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0],
    [1, 1, 1, 1, 1]
])

fig, ax = plt.subplots(figsize=(12, 7))

ax.imshow(
    permissions,
    cmap=ListedColormap(["#FEE2E2", "#DCFCE7"]),
    vmin=0, vmax=1, aspect="auto"
)

ax.set_xticks(range(len(endpoints)))
ax.set_xticklabels(endpoints, rotation=12, ha="right")

ax.set_yticks(range(len(roles)))
ax.set_yticklabels(roles)

ax.set_title(
    "Authentication vs. Authorization — Example API Permissions",
    fontsize=16, fontweight="bold", pad=18
)

for i in range(len(roles)):
    for j in range(len(endpoints)):
        allowed = permissions[i, j] == 1

        ax.text(
            j, i,
            "ALLOW" if allowed else "DENY",
            ha="center", va="center",
            fontsize=9, fontweight="bold",
            color=GREEN if allowed else RED
        )

ax.set_xticks(
    np.arange(-0.5, len(endpoints), 1), minor=True
)
ax.set_yticks(
    np.arange(-0.5, len(roles), 1), minor=True
)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", bottom=False, left=False)

plt.tight_layout()

show_collapsible_figure(
    "2.2 — Authentication, Authorization, and Endpoint Permissions",
    fig,
    """
    Authentication and authorization are related
    security controls, but they perform
    different functions.

    <br><br>

    <b>Authentication</b> verifies the identity
    of a user, application, or service.

    <br><br>

    <b>Authorization</b> determines which
    operations that identity may perform.

    <br><br>

    For example, an authenticated application
    user may be permitted to request predictions
    but should not automatically receive
    permission to upload or delete models.

    <br><br>

    Similarly, a deployment service may need
    permission to upload approved model
    artifacts without needing access to
    security logs.

    <br><br>

    The matrix illustrates a hypothetical
    role-based access control policy.

    <br><br>

    Green cells indicate permitted operations.
    Red cells indicate operations that should
    be denied under the example policy.

    <br><br>

    Actual permissions should reflect
    organizational responsibilities and
    the principle of least privilege.
    Administrative access should also
    be limited to necessary functions.

    <br><br>

    <b>Key takeaway:</b> Successfully
    authenticating a user does not mean
    that user is authorized to access
    every API endpoint.
    """,
    GREEN
)


# ============================================================
# VISUALIZATION 3
# REQUEST VALIDATION AND SECURE PROCESSING
# ============================================================

fig, ax = plt.subplots(figsize=(11, 9))

setup_diagram(
    ax,
    "Inference API — Request Validation Workflow",
    xlim=(0, 10), ylim=(0, 11)
)

add_card(
    ax, 3.0, 9.0, 4.0, 1.1,
    "Incoming Request",
    "Image, text, or numerical input",
    BLUE
)

add_card(
    ax, 3.0, 7.0, 4.0, 1.1,
    "Validation Checks",
    "Type, size, schema, and parameters",
    ORANGE
)

add_arrow(ax, (5, 9.0), (5, 8.1))

add_card(
    ax, 0.4, 4.7, 4.0, 1.2,
    "Invalid Request",
    "Reject with a controlled error",
    RED
)

add_card(
    ax, 5.6, 4.7, 4.0, 1.2,
    "Valid Request",
    "Continue to inference",
    GREEN
)

add_arrow(ax, (4.1, 7.0), (2.4, 5.9), RED)
add_arrow(ax, (5.9, 7.0), (7.6, 5.9), GREEN)

add_card(
    ax, 5.6, 2.3, 4.0, 1.2,
    "Inference Service",
    "Process accepted input",
    BLUE
)

add_arrow(ax, (7.6, 4.7), (7.6, 3.5), GREEN)

ax.text(
    2.4, 3.6,
    "No model execution",
    ha="center", fontsize=10,
    fontweight="bold", color=RED
)

ax.text(
    5.0, 0.9,
    "Validation reduces exposure but does not guarantee adversarial robustness",
    ha="center", fontsize=9.5, color=GRAY
)

plt.tight_layout()

show_collapsible_figure(
    "2.3 — Request Validation and Secure Inference Processing",
    fig,
    """
    Inference APIs should validate incoming
    requests before passing them to
    the deployed model.

    <br><br>

    Validation checks whether the request
    matches the format and constraints
    expected by the application.

    <br><br>

    Common checks include required fields,
    data types, supported file formats,
    request sizes, input dimensions,
    and allowed parameter values.

    <br><br>

    For example, an image classifier
    expecting 224 × 224 RGB images
    should not accept arbitrary
    oversized files without validation.

    <br><br>

    Requests that fail validation
    should be rejected with controlled
    error responses.

    <br><br>

    Valid requests may proceed to
    the inference service.

    <br><br>

    However, validation does not
    guarantee that an input is harmless.

    <br><br>

    An adversarial example can satisfy
    all ordinary format requirements
    while still attempting to manipulate
    model predictions.

    <br><br>

    <b>Key takeaway:</b> Input validation
    protects the API and surrounding
    software, while model-level
    defenses address different risks.
    """,
    ORANGE
)


# ============================================================
# VISUALIZATION 4
# RATE LIMITING AND RESOURCE PROTECTION
# ============================================================

# Synthetic request counts per minute.
# The example quota is 60 requests per minute.

minutes = np.arange(1, 16)

requested = np.array([
    22, 28, 35, 42, 54,
    75, 110, 140, 125, 98,
    82, 60, 45, 38, 30
])

quota = 60

accepted = np.minimum(requested, quota)
rejected = requested - accepted

fig, ax = plt.subplots(figsize=(12, 7))

ax.bar(
    minutes, accepted,
    color=GREEN, label="Accepted Requests"
)

ax.bar(
    minutes, rejected,
    bottom=accepted,
    color=RED, label="Rate-Limited Requests"
)

ax.axhline(
    quota,
    color=ORANGE,
    linestyle="--",
    linewidth=2,
    label="60 Requests/Minute Quota"
)

ax.set_xticks(minutes)
ax.set_xlabel("Minute")
ax.set_ylabel("Requests")

ax.set_title(
    "Illustrative Rate Limiting for an AI Inference API",
    fontsize=16, fontweight="bold"
)

ax.legend()
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "2.4 — Rate Limiting and Resource Protection",
    fig,
    """
    AI inference can consume substantial
    computing resources, including CPU,
    GPU memory, and processing time.

    <br><br>

    If an API accepts unlimited requests,
    attackers may attempt to overwhelm
    the service or generate excessive
    inference costs.

    <br><br>

    Rate limiting restricts how many
    requests a client or identity may
    submit during a defined period.

    <br><br>

    This example uses a hypothetical
    quota of 60 requests per minute.

    <br><br>

    Green bars represent requests
    accepted under the quota.

    <br><br>

    Red bars represent requests
    that exceed the quota and
    would be rejected or deferred
    by this simplified policy.

    <br><br>

    The numbers are synthetic and
    do not represent measured traffic.

    <br><br>

    Real systems may use token buckets,
    rolling windows, per-user quotas,
    concurrency limits, execution
    timeouts, and spending limits.

    <br><br>

    Rate limiting can make large-scale
    automated probing and model extraction
    more difficult, but it cannot
    guarantee that these attacks
    will be prevented.

    <br><br>

    <b>Key takeaway:</b> Resource
    protection is an important
    part of AI security because
    excessive legitimate-looking
    requests can still cause harm.
    """,
    RED
)


# ============================================================
# VISUALIZATION 5
# API GATEWAYS, TLS, AND TRUST BOUNDARIES
# ============================================================

fig, ax = plt.subplots(figsize=(12, 9))

setup_diagram(
    ax,
    "Secure Communication and API Gateway Architecture",
    xlim=(0, 12), ylim=(0, 11)
)

add_card(
    ax, 0.5, 8.0, 3.0, 1.3,
    "External Client",
    "Sends inference requests",
    RED
)

add_card(
    ax, 4.5, 8.0, 3.0, 1.3,
    "API Gateway",
    "Authentication, validation, quotas",
    BLUE
)

add_card(
    ax, 8.5, 8.0, 3.0, 1.3,
    "Inference Service",
    "Protected model execution",
    GREEN
)

add_arrow(ax, (3.5, 8.65), (4.5, 8.65), GREEN)
add_arrow(ax, (7.5, 8.65), (8.5, 8.65), GREEN)

ax.text(
    4.0, 9.1,
    "HTTPS/TLS",
    ha="center", fontsize=9,
    fontweight="bold", color=GREEN
)

ax.text(
    8.0, 9.1,
    "Protected channel",
    ha="center", fontsize=9,
    fontweight="bold", color=GREEN
)

add_card(
    ax, 4.5, 4.8, 3.0, 1.3,
    "Gateway Security Policies",
    "Enforced before model access",
    PURPLE
)

add_arrow(ax, (6.0, 8.0), (6.0, 6.1))

add_card(
    ax, 8.5, 4.8, 3.0, 1.3,
    "External AI Provider",
    "Third-party trust boundary",
    ORANGE
)

add_arrow(ax, (10.0, 8.0), (10.0, 6.1), ORANGE)

add_card(
    ax, 0.5, 4.8, 3.0, 1.3,
    "Direct Access Attempt",
    "Bypass gateway controls",
    RED
)

ax.text(
    6.0, 2.7,
    "Protect inference services from direct, unauthorized network access",
    ha="center", fontsize=10.5,
    fontweight="bold", color=DARK
)

ax.text(
    6.0, 1.7,
    "TLS protects data in transit; authorization controls who may access resources",
    ha="center", fontsize=9.5, color=GRAY
)

plt.tight_layout()

show_collapsible_figure(
    "2.5 — API Gateways, TLS, and Third-Party Security",
    fig,
    """
    An API gateway provides a controlled
    entry point into an AI inference environment.

    <br><br>

    Depending on its configuration,
    the gateway may enforce authentication,
    authorization, input validation,
    rate limiting, logging, and routing.

    <br><br>

    HTTPS using Transport Layer Security
    protects information traveling
    between clients and the API.

    <br><br>

    This can include authentication
    tokens, submitted data, and
    model predictions.

    <br><br>

    Internal service-to-service
    communication should also be
    protected when appropriate.

    <br><br>

    Some applications send requests
    to third-party AI providers.

    <br><br>

    These connections introduce
    additional trust boundaries
    and require careful management
    of credentials, transmitted data,
    permissions, and responses.

    <br><br>

    A gateway is not sufficient
    if attackers can bypass it
    and directly reach an
    unprotected inference service.

    <br><br>

    <b>Important:</b> TLS provides
    communication protection.
    It does not replace authentication,
    authorization, or request validation.

    <br><br>

    <b>Key takeaway:</b> Secure API
    architectures combine protected
    communication with independently
    enforced access controls.
    """,
    PURPLE
)


# ============================================================
# VISUALIZATION 6
# INFORMATION LEAKAGE AND LAYERED API DEFENSES
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

fig.suptitle(
    "API Response Minimization and Layered Security",
    fontsize=16, fontweight="bold"
)

# Left panel: information disclosure
ax = axes[0]
setup_diagram(
    ax,
    "Illustrative Classification Response",
    xlim=(0, 6), ylim=(0, 10)
)

classes = ["Cat", "Dog", "Wolf", "Other"]
probabilities = [0.03, 0.94, 0.02, 0.01]

ax.barh(
    classes, probabilities,
    color=[GRAY, GREEN, GRAY, GRAY]
)

ax.set_xlim(0, 1.05)
ax.set_xlabel("Model Confidence")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.2)

ax.text(
    0.5, 4.5,
    "Full probability vector",
    ha="center", fontsize=10,
    fontweight="bold", color=ORANGE
)

ax.text(
    0.5, 5.3,
    "May reveal more than the client needs",
    ha="center", fontsize=9, color=GRAY
)

ax.text(
    0.5, 7.0,
    "Minimized response: Prediction = Dog",
    ha="center", fontsize=10,
    fontweight="bold", color=GREEN
)

# Right panel: defense layers
ax = axes[1]
setup_diagram(
    ax,
    "Layered API Security Controls",
    xlim=(0, 6), ylim=(0, 10)
)

layers = [
    ("Authentication", BLUE),
    ("Authorization", GREEN),
    ("Request Validation", ORANGE),
    ("Rate and Resource Limits", PURPLE),
    ("Response Minimization", TEAL),
    ("Logging and Monitoring", BLUE)
]

for i, (title, accent) in enumerate(layers):
    y = 8.3 - i*1.3

    add_card(
        ax, 0.5, y, 5.0, 0.95,
        title, accent=accent
    )

    if i < len(layers)-1:
        add_arrow(
            ax, (3, y-0.05), (3, y-0.35)
        )

plt.tight_layout()

show_collapsible_figure(
    "2.6 — Information Leakage and Defense in Depth",
    fig,
    """
    Securing an AI API requires
    controlling both incoming
    requests and outgoing responses.

    <br><br>

    A classification model may
    internally produce a probability
    for every possible class.

    <br><br>

    However, the application may
    only need to return the
    predicted class.

    <br><br>

    Returning unnecessary confidence
    values can sometimes provide
    attackers with additional
    information about model behavior.

    <br><br>

    This information may support
    repeated probing or attempts
    to approximate model functionality.

    <br><br>

    Some legitimate applications
    require confidence scores.
    The correct response format
    therefore depends on the
    application's requirements.

    <br><br>

    <b>Data minimization</b> means
    returning the information
    necessary for the intended task
    without unnecessarily exposing
    internal details.

    <br><br>

    The right-hand diagram shows
    how response minimization works
    alongside authentication,
    authorization, input validation,
    resource limits, and monitoring.

    <br><br>

    Each control addresses
    a different security concern.

    <br><br>

    <b>Key takeaway:</b> A secure
    inference API uses multiple
    layers of protection rather
    than relying on any
    single defensive mechanism.
    """,
    GREEN
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("Section 2 — Securing AI APIs and Inference Endpoints Visualizations")
print("------------------------------------------------------------------")
print("Six collapsible visualizations generated.")

print()
print("Concepts Covered")
print("----------------")
print("1. AI Inference Endpoint Architecture")
print("2. Authentication and Authorization")
print("3. Request Validation")
print("4. Rate Limiting and Resource Protection")
print("5. API Gateways, TLS, and Third-Party Trust Boundaries")
print("6. Information Leakage and Layered API Defenses")

print()
print("All metrics, permissions, and example traffic are illustrative.")
print("No external APIs, model downloads, or attacks were used.")


<details>

<summary><b>Section 2 Lab — Click to Expand</b></summary>

<br>

A company has deployed a phishing-detection model.

Applications submit email information to an inference endpoint and receive a phishing prediction.

The intended architecture is:

    CLIENT
      ↓
    API
      ↓
    INFERENCE SERVICE
      ↓
    MODEL
      ↓
    PREDICTION

The development team initially focused on making the model available and did not implement strong API security controls.

The first version allows requests to reach the inference service with very few restrictions.

Your task is to examine the insecure endpoint and progressively add:

    AUTHENTICATION

    AUTHORIZATION

    INPUT VALIDATION

    RATE LIMITING

    RESPONSE CONTROL

The goal is not to build a production web server.

Instead, this lab simulates the security decisions that should occur before and after an inference request reaches a deployed model.

All code runs locally and should execute almost instantly.

</details>

In [ ]:
# @title ▶ Part 1: Create the Inference Service { display-mode: "form" }

def phishing_model(email_text):

    suspicious_words = [
        "urgent",
        "password",
        "verify",
        "account",
        "click"
    ]

    text = email_text.lower()

    matches = sum(
        word in text
        for word in suspicious_words
    )

    probability = min(
        0.10 + (matches * 0.18),
        0.99
    )

    classification = (
        "PHISHING"
        if probability >= 0.50
        else "LEGITIMATE"
    )

    return {
        "classification": classification,
        "probability": round(probability, 2)
    }


print("=" * 70)
print("PHISHING MODEL READY")
print("=" * 70)

test_email = (
    "Urgent: verify your account password "
    "by clicking this link."
)

print("\nTest Input:")
print(test_email)

print("\nPrediction:")
print(phishing_model(test_email))

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

This simple function represents the deployed phishing model.

It accepts:

    EMAIL TEXT

and returns:

    CLASSIFICATION

    PROBABILITY

The model itself is intentionally simple because this lab focuses on the infrastructure surrounding inference rather than machine learning accuracy.

The important question is:

> **What security decisions occur before a request is allowed to reach this function?**

At the moment, none exist.

</details>

In [ ]:
# @title ▶ Part 2: Create the Vulnerable Endpoint { display-mode: "form" }

def vulnerable_api(request):

    email_text = request.get(
        "email",
        ""
    )

    return phishing_model(
        email_text
    )


print("=" * 70)
print("VULNERABLE API")
print("=" * 70)

request = {
    "email":
        "Urgent! Verify your account password."
}

response = vulnerable_api(
    request
)

print("\nRequest:")
print(request)

print("\nResponse:")
print(response)

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

The first endpoint performs almost no security checks.

Its path is simply:

    REQUEST
       ↓
      API
       ↓
     MODEL

The endpoint does not determine:

    WHO SENT THE REQUEST

    WHETHER THE USER IS AUTHORIZED

    WHETHER THE REQUEST IS VALID

    HOW MANY REQUESTS WERE SENT

This means anyone who can reach the endpoint can potentially interact with the model.

The following parts will add security controls one layer at a time.

</details>

In [ ]:
# @title ▶ Part 3: Add Authentication { display-mode: "form" }

valid_api_keys = {
    "key-user-123": "ApplicationUser",
    "key-admin-999": "Administrator"
}


def authenticate(api_key):

    if api_key not in valid_api_keys:
        return None

    return valid_api_keys[
        api_key
    ]


test_keys = [
    "invalid-key",
    "key-user-123",
    "key-admin-999"
]

print("=" * 70)
print("AUTHENTICATION TEST")
print("=" * 70)

for key in test_keys:

    identity = authenticate(key)

    print(f"\nAPI Key: {key}")

    if identity:
        print(
            f"✓ Authenticated as: "
            f"{identity}"
        )
    else:
        print(
            "✗ Authentication failed"
        )

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

Authentication introduces the question:

> **Who is making this request?**

The simulation uses API keys.

A valid key maps to an identity.

The request path can now become:

    REQUEST
       ↓
    API KEY
       ↓
    AUTHENTICATE
       ↓
    VALID?
     ↙   ↘
    NO   YES
    ↓     ↓
  REJECT CONTINUE

An invalid credential should prevent the request from reaching the inference service.

Authentication establishes identity.

It does **not** determine what that identity is allowed to do.

That requires authorization.

</details>

In [ ]:
# @title ▶ Part 4: Add Endpoint Authorization { display-mode: "form" }

permissions = {
    "ApplicationUser": [
        "/predict"
    ],

    "Administrator": [
        "/predict",
        "/logs",
        "/admin/config"
    ]
}


def authorize(role, endpoint):

    allowed_endpoints = permissions.get(
        role,
        []
    )

    return endpoint in allowed_endpoints


tests = [
    ("ApplicationUser", "/predict"),
    ("ApplicationUser", "/admin/config"),
    ("Administrator", "/predict"),
    ("Administrator", "/admin/config")
]

print("=" * 70)
print("AUTHORIZATION TEST")
print("=" * 70)

for role, endpoint in tests:

    allowed = authorize(
        role,
        endpoint
    )

    print(
        f"\nRole:     {role}"
        f"\nEndpoint: {endpoint}"
    )

    if allowed:
        print("✓ ACCESS ALLOWED")
    else:
        print("✗ ACCESS DENIED")

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

Authorization answers a different question:

> **What is this identity allowed to do?**

An ApplicationUser can access:

    /predict

but cannot access:

    /admin/config

An Administrator has additional permissions.

This demonstrates:

    AUTHENTICATION
          ↓
    WHO ARE YOU?

followed by:

    AUTHORIZATION
          ↓
    WHAT CAN YOU DO?

A valid API key should not automatically grant access to every function exposed by the AI system.

Restricting functionality according to role also reduces the blast radius if a normal user credential becomes compromised.

</details>

In [ ]:
# @title ▶ Part 5: Validate Inference Requests { display-mode: "form" }

MAX_TEXT_LENGTH = 300


def validate_request(request):

    if "email" not in request:
        return False, "Missing email field"

    if not isinstance(
        request["email"],
        str
    ):
        return False, "Email must be text"

    if len(request["email"]) == 0:
        return False, "Email cannot be empty"

    if len(request["email"]) > MAX_TEXT_LENGTH:
        return False, "Input exceeds maximum size"

    return True, "Valid request"


test_requests = [
    {
        "email":
            "Please review the attached report."
    },

    {
        "wrong_field":
            "Missing expected field"
    },

    {
        "email":
            12345
    },

    {
        "email":
            "A" * 500
    }
]

print("=" * 70)
print("INPUT VALIDATION TEST")
print("=" * 70)

for request in test_requests:

    valid, message = validate_request(
        request
    )

    print("\n" + "-" * 50)
    print(f"Valid:  {valid}")
    print(f"Result: {message}")

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

The inference service now checks whether the request matches its expected format.

The validator checks:

    REQUIRED FIELD

    DATA TYPE

    EMPTY INPUT

    MAXIMUM SIZE

A request that fails validation is rejected before reaching the model.

The path becomes:

    REQUEST
       ↓
    VALIDATE
       ↓
    VALID?
     ↙   ↘
    NO   YES
    ↓     ↓
  REJECT MODEL

Input validation does not stop every AI-specific attack.

For example, a carefully constructed adversarial input might still satisfy all normal format requirements.

However, validation prevents malformed or excessive requests from unnecessarily reaching deeper parts of the AI system.

</details>

In [ ]:
# @title ▶ Part 6: Simulate API Abuse { display-mode: "form" }

normal_requests = 5
attacker_requests = 1000

cost_per_request = 0.002


normal_cost = (
    normal_requests
    * cost_per_request
)

attack_cost = (
    attacker_requests
    * cost_per_request
)


print("=" * 70)
print("INFERENCE RESOURCE SIMULATION")
print("=" * 70)

print(
    f"\nNormal User Requests: "
    f"{normal_requests}"
)

print(
    f"Estimated Cost: "
    f"${normal_cost:.3f}"
)

print(
    f"\nAutomated Requests: "
    f"{attacker_requests}"
)

print(
    f"Estimated Cost: "
    f"${attack_cost:.2f}"
)

print(
    "\nWithout limits, automated clients "
    "can consume significantly more "
    "inference resources."
)

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

AI inference consumes resources.

Depending on the system, requests may consume:

    CPU

    GPU

    MEMORY

    NETWORK BANDWIDTH

    EXTERNAL API COSTS

An attacker who can submit large numbers of requests may increase resource consumption or operating costs.

Repeated queries may also support other attacks such as automated probing or model extraction.

This is why production AI APIs often need limits on how frequently clients can interact with the inference service.

</details>

In [ ]:
# @title ▶ Part 7: Add Rate Limiting { display-mode: "form" }

REQUEST_LIMIT = 5

request_counts = {}


def check_rate_limit(user):

    current = request_counts.get(
        user,
        0
    )

    if current >= REQUEST_LIMIT:
        return False

    request_counts[user] = (
        current + 1
    )

    return True


print("=" * 70)
print("RATE LIMIT TEST")
print("=" * 70)

user = "client_42"

for request_number in range(1, 9):

    allowed = check_rate_limit(
        user
    )

    if allowed:
        result = "ALLOWED"
    else:
        result = "RATE LIMITED"

    print(
        f"Request {request_number}: "
        f"{result}"
    )

<details>

<summary><b>Part 7 Explanation — Click to Expand</b></summary>

<br>

The simulated rate limiter allows:

    5 REQUESTS

before additional requests are blocked.

Instead of:

    CLIENT
      ↓
    UNLIMITED REQUESTS
      ↓
     MODEL

the system now follows:

    CLIENT
      ↓
    REQUEST COUNT
      ↓
    WITHIN LIMIT?
     ↙       ↘
    NO       YES
    ↓         ↓
  REJECT     MODEL

Real systems usually measure limits over time, such as:

    Requests per second

    Requests per minute

    Requests per hour

They may also use different quotas for different users or services.

The principle remains the same: one client should not automatically receive unlimited access to expensive inference resources.

</details>

In [ ]:
# @title ▶ Part 8: Reduce Response Exposure { display-mode: "form" }

model_result = {
    "classification": "PHISHING",
    "probability": 0.94,
    "internal_model": "phishing_model_v7",
    "server": "gpu-prod-02",
    "debug_score": 0.938271
}


def public_response(result):

    return {
        "classification":
            result["classification"],

        "probability":
            result["probability"]
    }


print("=" * 70)
print("RESPONSE CONTROL")
print("=" * 70)

print("\nInternal Model Result:")
print(model_result)

print("\nClient Response:")
print(
    public_response(
        model_result
    )
)

<details>

<summary><b>Part 8 Explanation — Click to Expand</b></summary>

<br>

Endpoint security applies to both directions.

Earlier controls protected:

    WHAT ENTERS THE SYSTEM

This part considers:

    WHAT LEAVES THE SYSTEM

The internal result contained information such as:

    MODEL VERSION

    SERVER NAME

    DEBUG VALUES

The client does not necessarily need this information.

Returning unnecessary internal details may reveal information about the deployment environment or model.

The public response therefore contains only the information required by the client.

This applies the principle of data minimization to API responses.

</details>

In [ ]:
# @title ▶ Part 9: Build the Secured Endpoint { display-mode: "form" }

request_counts = {}


def secure_inference_api(
    api_key,
    endpoint,
    request
):

    # Authentication
    role = authenticate(
        api_key
    )

    if role is None:
        return {
            "status": 401,
            "message": "Authentication failed"
        }

    # Authorization
    if not authorize(
        role,
        endpoint
    ):
        return {
            "status": 403,
            "message": "Access denied"
        }

    # Rate limiting
    if not check_rate_limit(
        api_key
    ):
        return {
            "status": 429,
            "message": "Rate limit exceeded"
        }

    # Input validation
    valid, message = validate_request(
        request
    )

    if not valid:
        return {
            "status": 400,
            "message": message
        }

    # Model inference
    result = phishing_model(
        request["email"]
    )

    # Response minimization
    return {
        "status": 200,
        "result": public_response({
            **result,
            "internal_model":
                "phishing_model_v7",
            "server":
                "gpu-prod-02",
            "debug_score":
                result["probability"]
        })
    }


print("=" * 70)
print("SECURED INFERENCE ENDPOINT READY")
print("=" * 70)

<details>

<summary><b>Part 9 Explanation — Click to Expand</b></summary>

<br>

The controls have now been combined into one inference path.

A request follows:

    REQUEST
       ↓
    AUTHENTICATION
       ↓
    AUTHORIZATION
       ↓
    RATE LIMIT
       ↓
    INPUT VALIDATION
       ↓
      MODEL
       ↓
    RESPONSE CONTROL
       ↓
     CLIENT

Notice that no individual control is expected to provide complete security.

Authentication does not replace authorization.

Authorization does not replace validation.

Validation does not replace rate limiting.

Rate limiting does not protect sensitive response data.

The controls work together as **defense in depth**.

</details>

In [ ]:
# @title ▶ Part 10: Security Test { display-mode: "form" }

request_counts = {}

tests = [
    {
        "name": "Valid prediction request",
        "key": "key-user-123",
        "endpoint": "/predict",
        "request": {
            "email":
                "Urgent! Verify your account."
        }
    },

    {
        "name": "Invalid API key",
        "key": "fake-key",
        "endpoint": "/predict",
        "request": {
            "email":
                "Test message"
        }
    },

    {
        "name": "Unauthorized admin access",
        "key": "key-user-123",
        "endpoint": "/admin/config",
        "request": {
            "email":
                "Test message"
        }
    },

    {
        "name": "Oversized input",
        "key": "key-user-123",
        "endpoint": "/predict",
        "request": {
            "email":
                "A" * 500
        }
    }
]

print("=" * 70)
print("SECURED API TEST")
print("=" * 70)

for test in tests:

    response = secure_inference_api(
        test["key"],
        test["endpoint"],
        test["request"]
    )

    print("\n" + "-" * 60)
    print(test["name"])
    print(f"Result: {response}")

<details>

<summary><b>Part 10 Explanation — Click to Expand</b></summary>

<br>

The secured endpoint now responds differently depending on the security problem.

A legitimate prediction request receives:

    200 — SUCCESS

An invalid identity receives:

    401 — AUTHENTICATION FAILED

An authenticated user attempting an unauthorized function receives:

    403 — ACCESS DENIED

An invalid request receives:

    400 — BAD REQUEST

Repeated requests can eventually receive:

    429 — RATE LIMIT EXCEEDED

These different controls protect different stages of the inference process.

The model is only reached after the request passes the required security checks.

</details>

In [ ]:
# @title ▶ Part 11: Compare API Security { display-mode: "form" }

print("=" * 70)
print("INFERENCE API SECURITY COMPARISON")
print("=" * 70)

print("""
VULNERABLE ENDPOINT

Request
   ↓
Model
   ↓
Full Response


SECURED ENDPOINT

Request
   ↓
Authentication
   ↓
Authorization
   ↓
Rate Limiting
   ↓
Input Validation
   ↓
Model
   ↓
Response Minimization
   ↓
Client
""")

<details>

<summary><b>Part 11 Explanation — Click to Expand</b></summary>

<br>

The vulnerable design prioritized accessibility:

    REQUEST
       ↓
     MODEL

The secured design introduces several boundaries before the request reaches the model.

These controls reduce different risks.

    AUTHENTICATION
    verifies identity.

    AUTHORIZATION
    restricts capabilities.

    VALIDATION
    rejects malformed or excessive input.

    RATE LIMITING
    restricts repeated requests.

    RESPONSE CONTROL
    limits unnecessary information exposure.

The important lesson is:

> **An inference API should be treated as a controlled security boundary between clients and the deployed model.**

Making a model available through an API should not mean giving every client unrestricted access to the model or the infrastructure surrounding it.

</details>

In [ ]:
# @title ▶ Part 12: Knowledge Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. What was the main weakness of the original endpoint?",
        "options": [
            "A. Requests reached the model with almost no security checks",
            "B. It required too much authentication",
            "C. It had strict rate limits",
            "D. It returned too little information"
        ],
        "answer": "A"
    },

    {
        "question":
        "2. Which control verified the API key?",
        "options": [
            "A. Authentication",
            "B. Authorization",
            "C. Rate limiting",
            "D. Input validation"
        ],
        "answer": "A"
    },

    {
        "question":
        "3. Which control prevented a normal user from accessing "
        "/admin/config?",
        "options": [
            "A. Authorization",
            "B. Authentication",
            "C. Model training",
            "D. Response formatting"
        ],
        "answer": "A"
    },

    {
        "question":
        "4. Which control rejected the 500-character test input?",
        "options": [
            "A. Input validation",
            "B. Authentication",
            "C. Model accuracy",
            "D. Logging"
        ],
        "answer": "A"
    },

    {
        "question":
        "5. What was the purpose of the request limit?",
        "options": [
            "A. Reduce excessive inference usage",
            "B. Increase unlimited access",
            "C. Remove authentication",
            "D. Expose internal model information"
        ],
        "answer": "A"
    },

    {
        "question":
        "6. Why did the public response remove internal server details?",
        "options": [
            "A. The client did not need that information",
            "B. The model could not generate it",
            "C. Authentication required it",
            "D. Rate limiting deleted it"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("LAB 2 — KNOWLEDGE CHECK")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. "
            f"Correct answer: {q['answer']}"
        )

percentage = (
    score / len(questions)
) * 100

print("\n" + "=" * 70)
print(f"Lab Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Lab complete!")
else:
    print(
        "△ Review the lab explanations "
        "and try again."
    )

<details>

<summary><b>Lab 2 — Practical Takeaways — Click to Expand</b></summary>

<br>

## Lab 2 — Practical Takeaways

This lab demonstrated how security controls can be placed around an AI inference endpoint.

The vulnerable system followed:

    CLIENT
      ↓
    MODEL

The secured system followed:

    CLIENT
      ↓
    AUTHENTICATION
      ↓
    AUTHORIZATION
      ↓
    RATE LIMITING
      ↓
    INPUT VALIDATION
      ↓
    MODEL
      ↓
    RESPONSE CONTROL
      ↓
    CLIENT

Each layer serves a different purpose.

Authentication establishes identity.

Authorization determines what that identity may access.

Input validation controls what enters the system.

Rate limiting reduces excessive use of inference resources.

Response control limits what information leaves the system.

Together, these controls demonstrate **defense in depth**.

The central lesson is:

> **A deployed model should not be directly exposed to every request that reaches the application.**

The inference endpoint acts as a security boundary where access, requests, resource consumption, and responses can be controlled before an interaction becomes a security problem.

</details>

In [ ]:
# @title ▶ Section 2 Checkpoint — Securing AI APIs & Inference Endpoints { display-mode: "form" }

questions = [
    {
        "question":
        "1. Why should an AI inference endpoint be treated as a security boundary?",
        "options": [
            "A. It controls access between clients and the deployed model",
            "B. It automatically trains the model",
            "C. It removes the need for authentication",
            "D. It stores every training dataset"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. What question does authentication answer?",
        "options": [
            "A. What are you allowed to do?",
            "B. Who are you?",
            "C. How accurate is the model?",
            "D. How large is the request?"
        ],
        "answer": "B"
    },
    {
        "question":
        "3. An authenticated normal user attempts to access /admin/config. "
        "Which control should prevent this?",
        "options": [
            "A. Authorization",
            "B. Model training",
            "C. Data augmentation",
            "D. Quantization"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Why should requests be validated before reaching the model?",
        "options": [
            "A. To reject malformed, unexpected, or excessive requests",
            "B. To guarantee that adversarial examples are impossible",
            "C. To retrain the model",
            "D. To increase every request size"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. What is one security benefit of rate limiting?",
        "options": [
            "A. It restricts excessive automated requests",
            "B. It gives users unlimited inference access",
            "C. It removes authentication",
            "D. It exposes more model information"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. Which is an example of resource protection?",
        "options": [
            "A. Maximum request size",
            "B. Query quota",
            "C. Execution timeout",
            "D. All of the above"
        ],
        "answer": "D"
    }
]

score = 0

print("=" * 70)
print("SECTION 2 CHECKPOINT")
print("SECURING AI APIs AND INFERENCE ENDPOINTS")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input("\nYour answer: ").strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(f"✗ Incorrect. Correct answer: {q['answer']}")

percentage = score / len(questions) * 100

print("\n" + "=" * 70)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Checkpoint complete!")
else:
    print("△ Review Section 2 and try again.")

---

<details>
<summary><b>Section 3 — Identity, Access Control, and Least Privilege — Click to Expand</b></summary>
<br>

## Section 3 — Identity, Access Control, and Least Privilege

A deployed AI system may contain many sensitive resources, including production models, training datasets, databases, model registries, secrets, logs, cloud services, and deployment pipelines. Not every user or service should have access to every resource.

Identity and Access Management (IAM) provides the mechanisms used to determine **who or what is requesting access, how that identity is verified, and what actions it is permitted to perform**.

In AI environments, identities are not limited to human users. Applications, containers, inference services, training jobs, CI/CD pipelines, and other workloads may also require identities and permissions.

This section explores how authentication, authorization, roles, attributes, service identities, separation of duties, and least privilege can be used to protect AI infrastructure and reduce the impact of compromised accounts or workloads.

<br>

<details>
<summary><b>3.1 Identity and Access Management in AI Systems</b></summary>
<br>

### Identity and Access Management in AI Systems

Identity and Access Management, commonly called **IAM**, controls access to systems and resources.

A simplified access decision can be represented as:

**Identity → Authentication → Authorization → Resource**

The first question is:

**Who or what is requesting access?**

The second is:

**Can that identity prove who it claims to be?**

The third is:

**What is that identity allowed to do?**

These questions are important because a production AI environment may contain highly valuable resources.

Examples include:

- Training datasets
- Production models
- Model registries
- Inference endpoints
- Databases
- Vector stores
- API credentials
- Cloud storage
- Deployment pipelines
- Security logs
- Monitoring systems

Consider an organization with a production model registry.

A machine learning developer may need permission to upload a candidate model.

An automated deployment service may need permission to retrieve an approved model.

An inference service may only need permission to read the currently deployed model.

A normal application user may not need direct access to the model registry at all.

Giving every identity the same level of access would create unnecessary risk.

IAM provides a way to assign access based on legitimate responsibilities rather than simply giving broad access to anyone inside the environment.

NIST's Zero Trust Architecture emphasizes that access decisions should focus on protecting individual resources and should not automatically trust identities simply because they are located inside an organization's network [3].

</details>

<br>

<details>
<summary><b>3.2 Human and Workload Identities</b></summary>
<br>

### Human and Workload Identities

When discussing identity, it is easy to think only about human users.

However, modern AI systems contain both **human identities** and **workload identities**.

Human identities may include:

- Developers
- Data scientists
- Security analysts
- Cloud administrators
- System administrators
- Application users

Workload identities may include:

- Inference services
- Training jobs
- Containers
- Virtual machines
- Automated deployment systems
- CI/CD pipelines
- Monitoring services
- Data-processing applications

For example, an inference service may need to automatically retrieve a production model when it starts.

Instead of storing a human administrator's username and password inside the application, the inference service should have its own identity.

The interaction could look like:

**Inference Service**

↓

**Service Identity**

↓

**Authentication**

↓

**Model Storage**

The service identity can then receive only the permissions necessary for inference.

This separation is important because services often operate continuously and automatically. Sharing human credentials with applications makes it difficult to determine who performed an action and may provide the application with permissions it does not need.

Secure AI environments should therefore ask:

**Which humans require access?**

and

**Which workloads require access?**

Both types of identities should be authenticated and authorized appropriately.

NIST guidance for microservice environments emphasizes securing communication and access between individual services rather than assuming internal workloads can automatically trust one another [5], [6].

</details>

<br>

<details>
<summary><b>3.3 Authentication and Strong Identity Verification</b></summary>
<br>

### Authentication and Strong Identity Verification

Authentication verifies that an identity is who or what it claims to be.

As introduced in Section 2, an API may authenticate a user before allowing access to an inference endpoint. However, authentication must occur throughout the AI infrastructure.

Examples may include:

**Developer → Model Registry**

**Inference Service → Model Storage**

**Deployment Pipeline → Production Environment**

**Monitoring Service → Logging Platform**

**Administrator → Cloud Management Interface**

Different authentication mechanisms may be appropriate for different situations.

Human authentication may involve:

- Passwords
- Multi-factor authentication
- Hardware security keys
- Single sign-on
- Identity-provider authentication

Workload authentication may involve:

- Service accounts
- Short-lived tokens
- Certificates
- Signed credentials
- Managed workload identities

Authentication should provide confidence that the identity requesting access is legitimate.

However, authentication alone does not determine what the identity is allowed to do.

A user can successfully prove their identity and still attempt to access a resource they should not be permitted to use.

For this reason:

**Authentication ≠ Authorization**

A secure system requires both.

</details>

<br>

<details>
<summary><b>3.4 Authorization and Permission Management</b></summary>
<br>

### Authorization and Permission Management

After an identity has been authenticated, the system must determine what actions that identity is permitted to perform.

This process is called **authorization**.

Permissions commonly describe actions such as:

- Read
- Write
- Create
- Update
- Execute
- Delete
- Deploy
- Approve
- Administer

Consider a model registry containing:

**model_v1**

**model_v2**

**model_v3**

A production inference service may require:

**READ model_v3**

It probably does not require:

**DELETE model_v3**

or:

**UPLOAD new models**

Similarly, an application user may need permission to execute inference:

**EXECUTE prediction**

but not:

**READ model file**

Authorization allows these differences to be enforced.

Poor authorization can turn a small security incident into a much larger compromise.

If an attacker steals the credentials of an application user who can only perform inference, the attacker's capabilities remain limited.

If that same account also has administrative permissions over models, databases, logs, and deployment infrastructure, the attacker gains significantly more power.

Authorization therefore directly affects the **blast radius** of compromised identities.

</details>

<br>

<details>
<summary><b>3.5 Role-Based Access Control (RBAC)</b></summary>
<br>

### Role-Based Access Control (RBAC)

One common method for managing authorization is **Role-Based Access Control**, or RBAC.

Instead of assigning every permission directly to every individual user, permissions are assigned to roles.

Users are then assigned to the appropriate roles.

A simplified example might look like:

**User → Role → Permissions → Resource**

For an AI environment, roles could include:

### Data Scientist

- Read approved training data
- Train candidate models
- Submit models for evaluation

### Deployment Engineer

- Read approved model artifacts
- Deploy approved models
- Perform rollback operations

### Security Analyst

- Read security logs
- Review alerts
- Investigate security events

### Application User

- Request inference
- View authorized prediction results

This is easier to manage than independently configuring every permission for every user.

Sandhu et al. describe formal RBAC models in which permissions are associated with roles and users receive permissions through their role assignments [7]. Earlier work by Ferraiolo and Kuhn also demonstrated how role-based controls can support organizational security policies and separation of duties [8].

RBAC is particularly useful when many users perform similar responsibilities.

However, roles should still be designed carefully.

A role called:

**AI-Administrator**

with permission to perform every action in the environment may be convenient, but widespread assignment of that role would create significant security risk.

RBAC should support least privilege rather than becoming a shortcut for granting excessive access.

</details>

<br>

<details>
<summary><b>3.6 Attribute-Based Access Control (ABAC)</b></summary>
<br>

### Attribute-Based Access Control (ABAC)

Role-based access control is useful, but sometimes access decisions require more context than a user's role.

**Attribute-Based Access Control (ABAC)** makes authorization decisions using attributes associated with subjects, resources, requested actions, and potentially the surrounding environment [4].

Examples of attributes might include:

**Subject Attributes**

- Department
- Job function
- Security clearance
- Service identity

**Resource Attributes**

- Production
- Development
- Confidential
- Public
- Model version

**Action Attributes**

- Read
- Deploy
- Delete
- Approve

**Environmental Attributes**

- Time
- Network location
- Device state
- Authentication strength

For example, a policy might state:

**Allow model deployment only when:**

- User role = Deployment Engineer
- Environment = Production
- Model status = Approved
- Authentication = MFA verified

ABAC can therefore provide more fine-grained access control than simply asking whether someone belongs to a particular role.

In AI environments, this can be useful when different models, datasets, environments, or operations require different security policies.

For example:

**Development Model → Broader Developer Access**

while:

**Production Model → Restricted Production Access**

The goal is to make authorization decisions match the actual security requirements of the resource.

</details>

<br>

<details>
<summary><b>3.7 The Principle of Least Privilege</b></summary>
<br>

### The Principle of Least Privilege

The **principle of least privilege** states that users and services should receive only the permissions necessary to perform their legitimate responsibilities.

Consider an AI inference service.

Its job may be to:

1. Load the approved production model.
2. Receive authorized inference requests.
3. Execute predictions.
4. Write operational logs.

The inference service may therefore need:

- Read access to the production model
- Permission to perform inference
- Permission to write logs

It probably does not need:

- Permission to modify training data
- Permission to delete models
- Permission to create administrative accounts
- Permission to disable logging
- Permission to change cloud network settings
- Access to unrelated application secrets

A poorly configured service might have:

**Inference Service → Administrator Permissions**

A least-privilege design might instead have:

**Inference Service → Read Production Model + Execute Inference + Write Logs**

This difference becomes extremely important if the inference service is compromised.

With administrator permissions:

**Compromised Service → Entire Environment**

With least privilege:

**Compromised Service → Limited Resources**

Least privilege does not necessarily prevent the initial compromise.

Instead, it helps **limit what happens next**.

This is one of the most important defense-in-depth principles in secure AI deployment.

</details>

<br>

<details>
<summary><b>3.8 Service Accounts and AI Workloads</b></summary>
<br>

### Service Accounts and AI Workloads

Automated AI services often need to interact with other resources without a human manually authenticating each request.

A **service account** or workload identity allows an application or service to authenticate as itself.

For example:

**Inference Container**

↓

**Service Account**

↓

**Model Storage**

The service account may receive permission to:

**READ production_model**

but not:

**WRITE production_model**

This is preferable to embedding an administrator's credentials inside the application.

Different workloads should also use different identities whenever practical.

For example:

**Training Service → Training Service Identity**

**Inference Service → Inference Service Identity**

**Monitoring Service → Monitoring Service Identity**

**Deployment Pipeline → Deployment Identity**

If every workload shares the same identity, defenders may struggle to determine which service performed an action.

Shared identities can also increase blast radius.

If the credential is compromised:

**One Credential → Access Used by Multiple Services**

Separate workload identities provide greater control:

**Compromised Inference Identity → Inference Permissions Only**

NIST's microservices guidance supports service-specific authentication and authorization mechanisms for controlling communication between distributed components [5].

</details>

<br>

<details>
<summary><b>3.9 Separation of Duties</b></summary>
<br>

### Separation of Duties

Some sensitive actions should require involvement from more than one person, role, or process.

This principle is known as **separation of duties**.

Consider a data scientist who develops a new production model.

If the same person can:

**Train → Evaluate → Approve → Deploy**

then a single account has control over the entire process.

A stronger design could separate responsibilities:

**Data Scientist → Develop Model**

↓

**Evaluation Process → Validate Model**

↓

**Approver → Authorize Release**

↓

**Deployment Service → Deploy Approved Model**

This makes it more difficult for one compromised account or malicious insider to introduce an unauthorized model directly into production.

Separation of duties can be particularly valuable for:

- Model approval
- Production deployment
- Access-policy modification
- Secret management
- Security-log administration
- Model deletion
- Emergency rollback

RBAC systems can help enforce this separation by assigning different responsibilities to different roles [7], [8].

The goal is not to create unnecessary administrative complexity.

Instead, separation of duties should be applied to actions where compromise or misuse could have significant consequences.

</details>

<br>

<details>
<summary><b>3.10 Zero Trust and AI Resource Access</b></summary>
<br>

### Zero Trust and AI Resource Access

Traditional network architectures sometimes treated internal systems as trusted because they were located behind an organization's firewall.

Modern cloud and distributed environments make this assumption increasingly dangerous.

NIST's Zero Trust Architecture states that trust should not automatically be granted based only on network location or ownership [3].

For AI infrastructure, this means:

**Inside the Cloud Network ≠ Automatically Trusted**

An inference service should still authenticate before accessing model storage.

A deployment pipeline should still be authorized before modifying production resources.

A developer should still require appropriate permissions before accessing sensitive training data.

A monitoring service should not automatically receive administrative control over the systems it observes.

A simplified zero-trust interaction is:

**Identity**

↓

**Authenticate**

↓

**Evaluate Policy**

↓

**Authorize Specific Resource**

↓

**Allow Minimum Required Access**

Access should be continuously controlled around resources rather than assuming that everything inside the environment can communicate freely.

This complements the least-privilege principle.

**Zero Trust asks:**

**"Why should this identity be trusted to access this resource?"**

**Least privilege asks:**

**"What is the minimum access this identity actually needs?"**

Together, they can significantly reduce unnecessary trust inside an AI environment.

</details>

<br>

<details>
<summary><b>3.11 Limiting Blast Radius Through Access Control</b></summary>
<br>

### Limiting Blast Radius Through Access Control

One of the most useful ways to understand IAM security is to consider what happens **after an account or service is compromised**.

Imagine an attacker compromises the production inference service.

In a poorly configured environment, the service account can:

- Read training data
- Modify datasets
- Upload models
- Delete models
- Read secrets
- Change logging settings
- Access databases
- Modify infrastructure

The compromise may therefore spread:

**Inference Service Compromised**

↓

**Credentials Stolen**

↓

**Model Registry Access**

↓

**Database Access**

↓

**Secrets Access**

↓

**Broader Environment Compromise**

Now consider a least-privilege configuration.

The inference service can only:

- Read the approved production model
- Perform inference
- Write operational logs

The attacker compromises the same service.

However:

**Modify Training Data → DENIED**

**Upload Model → DENIED**

**Delete Model → DENIED**

**Read Administrative Secrets → DENIED**

**Disable Security Logging → DENIED**

The service is still compromised, and incident response is still necessary.

But the attacker's ability to move through the environment has been significantly reduced.

This illustrates an important security principle:

**Access control does not only determine who gets in. It also determines how far an attacker can go after something goes wrong.**

</details>

<br>

<details>
<summary><b>Section 3 Summary</b></summary>
<br>

### Section 3 Summary

Identity and access management determines which users and workloads can interact with sensitive AI resources and what actions they are permitted to perform.

Throughout this section, you learned that:

- AI environments contain both human and workload identities.
- Authentication verifies identity, while authorization determines permitted actions.
- Access should be granted according to legitimate responsibilities.
- RBAC assigns permissions through organizational roles.
- ABAC can make more contextual and fine-grained access decisions.
- Least privilege limits identities to the minimum permissions necessary.
- AI workloads should use appropriately scoped service identities rather than shared administrator credentials.
- Separation of duties can prevent a single identity from controlling sensitive processes from beginning to end.
- Zero trust avoids granting implicit trust simply because a service exists inside an internal network.
- Strong access controls can significantly reduce the blast radius of a compromised user, service, or credential.

A well-designed AI environment should therefore assume that credentials and workloads can eventually become compromised and restrict their permissions accordingly.

The next section examines another closely related deployment risk: **Secrets, Credentials, and Configuration Security**. You will explore how API keys, tokens, cloud credentials, passwords, and insecure configuration settings can expose an otherwise well-protected AI environment.

</details>

</details>

In [ ]:

# @title Section 3 — Identity, Access Control, and Least Privilege Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import ListedColormap
from IPython.display import display, HTML
from io import BytesIO
import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"
TEAL = "#0891B2"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):
    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    buffer.close()

    display(HTML(f"""
    <details style="
        margin:12px 0;
        border:1px solid #cbd5e1;
        border-radius:10px;
        overflow:hidden;
    ">
        <summary style="
            cursor:pointer;
            padding:16px;
            font-size:16px;
            font-weight:bold;
            background:#eff6ff;
            color:#1e3a8a;
            border-left:5px solid {color};
        ">
            {html.escape(title)} — Click to Expand
        </summary>

        <div style="
            padding:15px;
            background:white;
            text-align:center;
        ">
            <img
                src="data:image/png;base64,{encoded}"
                style="width:100%;max-width:950px;height:auto;"
            >

            <div style="
                max-width:850px;
                margin:15px auto;
                padding:16px;
                background:#f8fafc;
                border-radius:8px;
                color:#334155;
                font-size:14px;
                line-height:1.7;
                text-align:left;
            ">
                {explanation}
            </div>
        </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def setup_diagram(ax, title, xlim=(0, 12), ylim=(0, 10)):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.axis("off")
    ax.set_title(title, fontsize=16, fontweight="bold", pad=18)


def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC",
             title_size=10):

    card = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.8
    )

    ax.add_patch(card)

    ax.text(
        x+w/2,
        y+h*(0.66 if subtitle else 0.5),
        title,
        ha="center",
        va="center",
        fontsize=title_size,
        fontweight="bold",
        color=DARK
    )

    if subtitle:
        ax.text(
            x+w/2, y+h*0.28,
            subtitle,
            ha="center",
            va="center",
            fontsize=8.5,
            color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):
    ax.add_patch(FancyArrowPatch(
        start, end,
        arrowstyle="-|>",
        mutation_scale=16,
        linewidth=1.8,
        color=color
    ))


# ============================================================
# VISUALIZATION 1
# IDENTITY AND ACCESS MANAGEMENT ARCHITECTURE
# ============================================================

fig, ax = plt.subplots(figsize=(12, 9))

setup_diagram(
    ax,
    "Identity and Access Management in AI Infrastructure",
    xlim=(0, 12),
    ylim=(0, 11)
)

add_card(
    ax, 0.5, 8.3, 3.2, 1.2,
    "Human User",
    "Developer, analyst, or administrator",
    BLUE
)

add_card(
    ax, 0.5, 5.4, 3.2, 1.2,
    "Workload Identity",
    "Application or deployment service",
    PURPLE
)

add_card(
    ax, 4.4, 7.0, 3.2, 1.5,
    "IAM Policy Engine",
    "Authenticate and authorize requests",
    ORANGE
)

add_card(
    ax, 8.3, 8.3, 3.2, 1.2,
    "Inference API",
    "Model prediction access",
    GREEN
)

add_card(
    ax, 8.3, 5.4, 3.2, 1.2,
    "Model Registry",
    "Model artifact management",
    BLUE
)

add_card(
    ax, 8.3, 2.5, 3.2, 1.2,
    "Protected Data Store",
    "Training and application data",
    RED
)

add_arrow(ax, (3.7, 8.9), (4.4, 8.0), BLUE)
add_arrow(ax, (3.7, 6.0), (4.4, 7.4), PURPLE)

for y in [8.9, 6.0, 3.1]:
    add_arrow(ax, (7.6, 7.75), (8.3, y), GREEN)

ax.text(
    6.0, 4.1,
    "Access decisions depend on verified identity and policy",
    ha="center",
    fontsize=10,
    fontweight="bold",
    color=DARK
)

plt.tight_layout()

show_collapsible_figure(
    "3.1 — Identity and Access Management Architecture",
    fig,
    """
    Identity and Access Management, commonly called
    <b>IAM</b>, determines who or what can access
    resources within an AI environment.

    <br><br>

    A deployed AI system may contain inference
    endpoints, model registries, training datasets,
    deployment pipelines, and administrative interfaces.

    <br><br>

    Each resource requires appropriate access controls.

    <br><br>

    Human users and automated workloads submit
    requests using their identities.

    <br><br>

    The IAM system verifies the identity and
    evaluates whether the requested operation
    is permitted under the applicable policies.

    <br><br>

    For example, a developer may be allowed
    to inspect a model but not delete a
    production model artifact.

    <br><br>

    A deployment service may be permitted
    to retrieve an approved artifact but
    not read unrelated employee records.

    <br><br>

    The diagram represents a conceptual
    authorization architecture. Actual systems
    may distribute these controls across
    identity providers, gateways, and services.

    <br><br>

    <b>Key takeaway:</b> IAM protects AI
    infrastructure by connecting identities,
    permissions, and protected resources.
    """,
    BLUE
)


# ============================================================
# VISUALIZATION 2
# HUMAN VS WORKLOAD IDENTITIES
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

fig.suptitle(
    "Human Identities vs. Workload Identities",
    fontsize=16,
    fontweight="bold"
)

examples = [
    (
        "Human Identity",
        [
            ("Human User", "Developer or administrator", BLUE),
            ("Identity Provider", "Passwordless login or MFA", PURPLE),
            ("Access Token", "Short-lived authenticated session", ORANGE),
            ("AI Resource", "Authorized user operation", GREEN)
        ]
    ),
    (
        "Workload Identity",
        [
            ("AI Application", "Running service or workload", BLUE),
            ("Workload Authentication", "Managed identity or certificate", PURPLE),
            ("Scoped Credentials", "Short-lived service permissions", ORANGE),
            ("AI Resource", "Authorized service operation", GREEN)
        ]
    )
]

for ax, (heading, stages) in zip(axes, examples):

    setup_diagram(ax, heading, (0, 6), (0, 10))

    for i, (title, subtitle, accent) in enumerate(stages):
        y = 8.1 - i*2.0

        add_card(
            ax, 0.6, y, 4.8, 1.2,
            title, subtitle, accent
        )

        if i < len(stages)-1:
            add_arrow(
                ax, (3.0, y-0.05),
                (3.0, y-0.75)
            )

plt.tight_layout()

show_collapsible_figure(
    "3.2 — Human and Workload Identities",
    fig,
    """
    AI environments contain both human
    identities and workload identities.

    <br><br>

    A <b>human identity</b> represents
    a person accessing the environment,
    such as a developer, researcher,
    security reviewer, or administrator.

    <br><br>

    Human authentication may use
    multifactor authentication, identity
    providers, and temporary sessions.

    <br><br>

    A <b>workload identity</b> represents
    an application, container, inference
    service, or deployment pipeline.

    <br><br>

    Automated workloads cannot rely on
    a person manually signing in for
    every operation.

    <br><br>

    Instead, they may authenticate using
    managed identities, workload federation,
    certificates, or short-lived tokens.

    <br><br>

    Workloads should not share one
    unrestricted administrative identity.

    <br><br>

    Assigning separate identities allows
    defenders to grant permissions based
    on each service's actual responsibilities.

    <br><br>

    <b>Key takeaway:</b> Both humans and
    automated services require verified,
    individually accountable identities.
    """,
    PURPLE
)


# ============================================================
# VISUALIZATION 3
# ROLE-BASED ACCESS CONTROL (RBAC)
# ============================================================

roles = [
    "Researcher",
    "ML Developer",
    "Deployment Service",
    "Security Reviewer",
    "Administrator"
]

resources = [
    "Read\nDatasets",
    "Train\nModels",
    "Deploy\nModels",
    "Read\nSecurity Logs",
    "Manage\nIAM"
]

# Hypothetical RBAC policy.
# 1 = allowed; 0 = denied.

permissions = np.array([
    [1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 1, 1]
])

fig, ax = plt.subplots(figsize=(12, 7))

ax.imshow(
    permissions,
    cmap=ListedColormap(["#FEE2E2", "#DCFCE7"]),
    vmin=0,
    vmax=1,
    aspect="auto"
)

ax.set_xticks(range(len(resources)))
ax.set_xticklabels(resources)

ax.set_yticks(range(len(roles)))
ax.set_yticklabels(roles)

ax.set_title(
    "Example Role-Based Access Control Policy",
    fontsize=16,
    fontweight="bold",
    pad=18
)

for i in range(len(roles)):
    for j in range(len(resources)):

        allowed = permissions[i, j] == 1

        ax.text(
            j, i,
            "ALLOW" if allowed else "DENY",
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
            color=GREEN if allowed else RED
        )

ax.set_xticks(
    np.arange(-0.5, len(resources), 1),
    minor=True
)

ax.set_yticks(
    np.arange(-0.5, len(roles), 1),
    minor=True
)

ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", bottom=False, left=False)

plt.tight_layout()

show_collapsible_figure(
    "3.3 — Role-Based Access Control (RBAC)",
    fig,
    """
    Role-Based Access Control, or
    <b>RBAC</b>, assigns permissions
    according to an identity's role.

    <br><br>

    Rather than assigning every
    permission separately to every user,
    an organization defines roles
    with specific responsibilities.

    <br><br>

    For example, a researcher may
    require access to approved datasets.

    <br><br>

    An ML developer may need
    permission to train models.

    <br><br>

    A deployment service may
    require permission to deploy
    approved model artifacts.

    <br><br>

    Security reviewers may need
    access to security logs
    without permission to
    modify production models.

    <br><br>

    The matrix shows an illustrative
    RBAC policy. Actual permissions
    should be determined by
    organizational requirements.

    <br><br>

    RBAC reduces unnecessary access
    and makes permission management
    easier to review.

    <br><br>

    <b>Key takeaway:</b> RBAC grants
    access according to defined
    responsibilities rather than
    allowing every authenticated
    identity to access every resource.
    """,
    GREEN
)


# ============================================================
# VISUALIZATION 4
# ATTRIBUTE-BASED ACCESS CONTROL (ABAC)
# ============================================================

fig, ax = plt.subplots(figsize=(12, 9))

setup_diagram(
    ax,
    "Attribute-Based Access Control — Policy Evaluation",
    xlim=(0, 12),
    ylim=(0, 11)
)

attributes = [
    (0.4, 8.3, "Subject Attributes",
     "Role, department, identity", BLUE),
    (4.4, 8.3, "Resource Attributes",
     "Model, classification, owner", PURPLE),
    (8.4, 8.3, "Environment Attributes",
     "Time, device, network", ORANGE)
]

for x, y, title, subtitle, color in attributes:
    add_card(
        ax, x, y, 3.2, 1.2,
        title, subtitle, color
    )

add_card(
    ax, 4.4, 5.5, 3.2, 1.2,
    "ABAC Policy Engine",
    "Evaluate all required conditions",
    TEAL
)

for x in [2.0, 6.0, 10.0]:
    add_arrow(ax, (x, 8.3), (6.0, 6.7))

add_card(
    ax, 1.1, 2.4, 3.5, 1.2,
    "ALLOW",
    "All required conditions satisfied",
    GREEN
)

add_card(
    ax, 7.4, 2.4, 3.5, 1.2,
    "DENY",
    "One or more conditions not satisfied",
    RED
)

add_arrow(ax, (5.4, 5.5), (2.85, 3.6), GREEN)
add_arrow(ax, (6.6, 5.5), (9.15, 3.6), RED)

plt.tight_layout()

show_collapsible_figure(
    "3.4 — Attribute-Based Access Control (ABAC)",
    fig,
    """
    Attribute-Based Access Control,
    or <b>ABAC</b>, evaluates access
    using multiple characteristics
    of a request.

    <br><br>

    These characteristics are
    called attributes.

    <br><br>

    <b>Subject attributes</b>
    describe the requesting identity,
    such as its department, role,
    or security clearance.

    <br><br>

    <b>Resource attributes</b>
    describe the protected object,
    such as a model's owner
    or data classification.

    <br><br>

    <b>Action attributes</b>
    describe the requested operation,
    such as reading or deploying a model.

    <br><br>

    <b>Environmental attributes</b>
    describe relevant conditions,
    such as the device, network,
    or time of access.

    <br><br>

    Consider a policy allowing
    production model deployment
    only when the requester is
    an approved deployment identity,
    the artifact is authorized,
    and the environment satisfies
    the required security conditions.

    <br><br>

    If a required condition
    is not satisfied, the
    request should be denied.

    <br><br>

    <b>Key takeaway:</b> ABAC supports
    more context-sensitive access
    decisions than role membership
    alone.
    """,
    ORANGE
)


# ============================================================
# VISUALIZATION 5
# LEAST PRIVILEGE AND SEPARATION OF DUTIES
# ============================================================

tasks = [
    "Read Data",
    "Train Model",
    "Approve Model",
    "Deploy Model",
    "Modify IAM",
    "Delete Model"
]

# Example: developer identity.
# Broad permissions vs. task-specific permissions.

broad_permissions = np.ones(len(tasks))
least_privilege = np.array([1, 1, 0, 0, 0, 0])

x = np.arange(len(tasks))
width = 0.36

fig, ax = plt.subplots(figsize=(12, 7))

ax.bar(
    x-width/2,
    broad_permissions,
    width,
    color=RED,
    label="Overly Broad Permissions"
)

ax.bar(
    x+width/2,
    least_privilege,
    width,
    color=GREEN,
    label="Least-Privilege Developer"
)

ax.set_xticks(x)
ax.set_xticklabels(tasks, rotation=15, ha="right")

ax.set_yticks([0, 1])
ax.set_yticklabels(["Not Granted", "Granted"])
ax.set_ylim(0, 1.25)

ax.set_title(
    "Least Privilege — Restricting a Developer Identity",
    fontsize=16,
    fontweight="bold"
)

ax.legend()
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "3.5 — Least Privilege and Separation of Duties",
    fig,
    """
    The <b>principle of least privilege</b>
    states that an identity should
    receive only the permissions
    necessary for its assigned tasks.

    <br><br>

    The red bars represent an
    overly permissive developer
    identity that can perform
    every operation.

    <br><br>

    The green bars represent
    a hypothetical developer
    identity restricted to
    reading approved data
    and training models.

    <br><br>

    The developer cannot independently
    approve a model, deploy it,
    change IAM permissions,
    or delete production artifacts.

    <br><br>

    <b>Separation of duties</b>
    builds on this principle
    by distributing sensitive
    responsibilities across
    different identities or roles.

    <br><br>

    For example, the identity
    that trains a model
    should not necessarily
    be the same identity
    that approves and deploys it.

    <br><br>

    This reduces the chance
    that one compromised account
    can control the entire
    deployment process.

    <br><br>

    The permissions shown are
    conceptual and may differ
    between organizations.

    <br><br>

    <b>Key takeaway:</b> Least privilege
    limits unnecessary capabilities,
    while separation of duties
    prevents excessive control
    from being concentrated
    in one identity.
    """,
    GREEN
)


# ============================================================
# VISUALIZATION 6
# ZERO TRUST AND BLAST RADIUS REDUCTION
# ============================================================

# Conceptual example:
# Which resources might be reachable after
# one inference workload is compromised?

resources = [
    "Inference API",
    "Model Registry",
    "Training Data",
    "Secret Store",
    "IAM Administration",
    "Deployment Pipeline"
]

unrestricted = np.array([1, 1, 1, 1, 1, 1])
restricted = np.array([1, 0, 0, 0, 0, 0])

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

fig.suptitle(
    "Zero Trust and Blast Radius Reduction",
    fontsize=16,
    fontweight="bold"
)

for ax, values, title in zip(
    axes,
    [unrestricted, restricted],
    ["Broadly Trusted Workload", "Zero-Trust / Least-Privilege Design"]
):

    colors = [
        RED if allowed else GREEN
        for allowed in values
    ]

    bars = ax.barh(
        resources,
        values,
        color=colors,
        height=0.65
    )

    ax.invert_yaxis()
    ax.set_xlim(0, 1.3)
    ax.set_xticks([])
    ax.set_title(title, fontweight="bold")

    for bar, allowed in zip(bars, values):

        ax.text(
            0.52,
            bar.get_y()+bar.get_height()/2,
            "REACHABLE" if allowed else "RESTRICTED",
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
            color="white" if allowed else DARK
        )

plt.tight_layout()

show_collapsible_figure(
    "3.6 — Zero Trust and Limiting the Blast Radius",
    fig,
    """
    Zero Trust is a security approach
    that avoids granting access
    solely because an identity
    or service operates inside
    a trusted network.

    <br><br>

    Every request should be
    evaluated using appropriate
    identity, authorization,
    and contextual controls.

    <br><br>

    Consider an attacker who
    compromises an AI inference
    service.

    <br><br>

    In an overly permissive
    architecture, the compromised
    workload may possess credentials
    or network access to multiple
    sensitive resources.

    <br><br>

    These resources might include
    model registries, training data,
    secrets, and deployment services.

    <br><br>

    In a more restrictive
    architecture, the workload
    receives only the permissions
    required for inference.

    <br><br>

    Independent authorization,
    network segmentation, workload
    identity, and short-lived
    credentials can help reduce
    unnecessary access.

    <br><br>

    The chart is a conceptual
    comparison, not a measured
    compromise simulation.

    <br><br>

    <b>Blast radius</b> describes
    the potential scope of damage
    following a compromise.

    <br><br>

    Reducing the blast radius
    does not prevent every
    initial compromise, but
    it can limit the attacker's
    ability to move into
    additional resources.

    <br><br>

    <b>Key takeaway:</b> Zero Trust
    and least privilege work
    together to limit what
    a compromised identity
    can access or control.
    """,
    RED
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("Section 3 — Identity, Access Control, and Least Privilege Visualizations")
print("------------------------------------------------------------------------")
print("Six collapsible visualizations generated.")

print()
print("Concepts Covered")
print("----------------")
print("1. Identity and Access Management Architecture")
print("2. Human and Workload Identities")
print("3. Role-Based Access Control (RBAC)")
print("4. Attribute-Based Access Control (ABAC)")
print("5. Least Privilege and Separation of Duties")
print("6. Zero Trust and Blast Radius Reduction")

print()
print("All permissions and access relationships are illustrative.")
print("No external services, credentials, or model downloads were used.")


<details>

<summary><b>Section 3 Lab — Click to Expand</b></summary>

<br>

A company operates several services as part of its production AI environment.

The environment contains:

    TRAINING SERVICE

    INFERENCE SERVICE

    DEPLOYMENT SERVICE

    MONITORING SERVICE

These workloads interact with sensitive resources such as:

    TRAINING DATA

    MODEL REGISTRY

    PRODUCTION MODEL

    SECURITY LOGS

    DEPLOYMENT SYSTEM

A developer initially configured the environment using very broad permissions.

This makes deployment easier, but it creates an important security problem.

If one service becomes compromised, the attacker may inherit all of the permissions assigned to that service.

In this lab, you will:

    Create workload identities

    Examine role-based permissions

    Test authorization decisions

    Compare excessive permissions with least privilege

    Simulate a compromised inference service

    Measure the resulting blast radius

    Apply separation of duties

The goal is to see that IAM does more than decide whether someone can access a system.

IAM also determines:

> **How far can an attacker go if an identity is compromised?**

This simulation uses standard Python and should run almost instantly.

</details>

In [ ]:
# @title ▶ Part 1: Create Workload Identities { display-mode: "form" }

identities = {
    "training-service": {
        "type": "Workload",
        "role": "Trainer"
    },

    "inference-service": {
        "type": "Workload",
        "role": "Inference"
    },

    "deployment-service": {
        "type": "Workload",
        "role": "Deployer"
    },

    "monitoring-service": {
        "type": "Workload",
        "role": "Monitor"
    },

    "alice": {
        "type": "Human",
        "role": "DataScientist"
    }
}

print("=" * 70)
print("AI ENVIRONMENT IDENTITIES")
print("=" * 70)

for identity, info in identities.items():

    print(f"\nIdentity: {identity}")
    print(f"Type:     {info['type']}")
    print(f"Role:     {info['role']}")

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

The environment contains both:

    HUMAN IDENTITIES

and:

    WORKLOAD IDENTITIES

The inference service does not use Alice's account.

Instead, it has its own identity:

    inference-service

Likewise, the training, deployment, and monitoring services have separate identities.

This allows security policies to distinguish between different workloads.

For example:

    INFERENCE SERVICE
           ↓
    INFERENCE PERMISSIONS

instead of:

    MULTIPLE SERVICES
           ↓
    SHARED ADMINISTRATOR ACCOUNT

Separate identities make it easier to restrict permissions and determine which service performed an action.

</details>

In [ ]:
# @title ▶ Part 2: Configure RBAC Permissions { display-mode: "form" }

role_permissions = {

    "Trainer": {
        "training_data": ["read"],
        "model_registry": ["upload"]
    },

    "Inference": {
        "production_model": ["read"],
        "inference": ["execute"],
        "security_logs": ["write"]
    },

    "Deployer": {
        "model_registry": ["read"],
        "production": ["deploy", "rollback"]
    },

    "Monitor": {
        "security_logs": ["read"]
    },

    "DataScientist": {
        "training_data": ["read"],
        "model_registry": ["read"]
    }
}

print("=" * 70)
print("ROLE-BASED ACCESS CONTROL")
print("=" * 70)

for role, resources in role_permissions.items():

    print(f"\n{role}")
    print("-" * 40)

    for resource, actions in resources.items():

        print(
            f"{resource:<20} "
            f"{', '.join(actions)}"
        )

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

This configuration demonstrates **Role-Based Access Control (RBAC)**.

Instead of assigning permissions individually to every identity, permissions are associated with roles.

For example:

    inference-service
           ↓
       Inference Role
           ↓
    READ production_model
    EXECUTE inference
    WRITE security_logs

The inference service does not need permission to train models, upload new models, or deploy infrastructure.

The permissions are based on the workload's actual responsibility.

This makes access policies easier to manage while supporting least privilege.

</details>

In [ ]:
# @title ▶ Part 3: Test Authorization { display-mode: "form" }

def authorize(identity, resource, action):

    if identity not in identities:
        return False

    role = identities[identity]["role"]

    permissions = role_permissions.get(
        role,
        {}
    )

    allowed_actions = permissions.get(
        resource,
        []
    )

    return action in allowed_actions


tests = [
    (
        "inference-service",
        "production_model",
        "read"
    ),

    (
        "inference-service",
        "model_registry",
        "upload"
    ),

    (
        "deployment-service",
        "production",
        "deploy"
    ),

    (
        "monitoring-service",
        "security_logs",
        "read"
    ),

    (
        "alice",
        "production",
        "deploy"
    )
]

print("=" * 70)
print("AUTHORIZATION TEST")
print("=" * 70)

for identity, resource, action in tests:

    allowed = authorize(
        identity,
        resource,
        action
    )

    print(
        f"\nIdentity: {identity}"
        f"\nResource: {resource}"
        f"\nAction:   {action}"
    )

    if allowed:
        print("✓ ALLOWED")
    else:
        print("✗ DENIED")

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

The authorization function evaluates:

    IDENTITY
       ↓
      ROLE
       ↓
    PERMISSIONS
       ↓
    RESOURCE + ACTION

The inference service is allowed to read the production model because that permission is necessary for inference.

However:

    inference-service
           ↓
    UPLOAD model
           ↓
         DENIED

Alice can access resources needed for her data-science responsibilities but cannot deploy directly to production.

Authorization therefore controls not only which resources an identity can reach but also which **actions** it can perform on those resources.

</details>

In [ ]:
# @title ▶ Part 4: Create an Overprivileged Service { display-mode: "form" }

admin_permissions = {
    "training_data": [
        "read", "write", "delete"
    ],

    "model_registry": [
        "read", "upload", "delete"
    ],

    "production_model": [
        "read", "write", "delete"
    ],

    "security_logs": [
        "read", "write", "delete"
    ],

    "production": [
        "deploy", "rollback"
    ],

    "cloud_admin": [
        "modify"
    ],

    "secrets": [
        "read"
    ]
}

print("=" * 70)
print("OVERPRIVILEGED INFERENCE SERVICE")
print("=" * 70)

for resource, actions in admin_permissions.items():

    print(
        f"\n{resource:<20}"
        f"{', '.join(actions)}"
    )

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

Imagine that an administrator gave the inference service broad administrator permissions because it was convenient during deployment.

The service can now:

    MODIFY TRAINING DATA

    UPLOAD OR DELETE MODELS

    DELETE SECURITY LOGS

    DEPLOY TO PRODUCTION

    MODIFY CLOUD RESOURCES

    READ SECRETS

None of these permissions are necessary for normal inference.

This violates the principle of least privilege.

The danger becomes clearer when we assume that the inference service has been compromised.

</details>

In [ ]:
# @title ▶ Part 5: Measure the Blast Radius { display-mode: "form" }

least_privilege_permissions = (
    role_permissions["Inference"]
)


def count_capabilities(permissions):

    capabilities = []

    for resource, actions in permissions.items():

        for action in actions:

            capabilities.append(
                f"{action} {resource}"
            )

    return capabilities


admin_capabilities = count_capabilities(
    admin_permissions
)

limited_capabilities = count_capabilities(
    least_privilege_permissions
)


print("=" * 70)
print("COMPROMISED SERVICE — BLAST RADIUS")
print("=" * 70)

print("\nOVERPRIVILEGED CONFIGURATION")

for capability in admin_capabilities:
    print(f"  • {capability}")

print(
    f"\nTotal attacker capabilities: "
    f"{len(admin_capabilities)}"
)


print("\n" + "=" * 70)

print("\nLEAST-PRIVILEGE CONFIGURATION")

for capability in limited_capabilities:
    print(f"  • {capability}")

print(
    f"\nTotal attacker capabilities: "
    f"{len(limited_capabilities)}"
)

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

The attack itself did not change.

In both scenarios:

    INFERENCE SERVICE
           ↓
       COMPROMISED

What changed was the set of permissions available to the compromised identity.

With broad administrator permissions, the attacker inherits many capabilities across the environment.

With least privilege, the compromised service is restricted primarily to its legitimate inference responsibilities.

This demonstrates an important idea:

> **Least privilege may not prevent the initial compromise, but it can dramatically reduce what the attacker can do afterward.**

This reduced impact is referred to as a smaller **blast radius**.

</details>

In [ ]:
# @title ▶ Part 6: Compare Security Impact { display-mode: "form" }

broad_count = len(
    admin_capabilities
)

limited_count = len(
    limited_capabilities
)

reduction = (
    (broad_count - limited_count)
    / broad_count
) * 100


print("=" * 70)
print("BLAST RADIUS COMPARISON")
print("=" * 70)

print(
    f"\nOverprivileged capabilities: "
    f"{broad_count}"
)

print(
    f"Least-privilege capabilities: "
    f"{limited_count}"
)

print(
    f"Capability reduction: "
    f"{reduction:.1f}%"
)

print("\nSecurity Result:")

if limited_count < broad_count:

    print(
        "✓ Least privilege reduced "
        "the attacker's available actions."
    )

else:

    print(
        "✗ Permissions were not reduced."
    )

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

The comparison provides a simple way to visualize blast radius.

The overprivileged service exposes many more capabilities to an attacker.

The least-privilege service exposes only a small set of actions required for its intended function.

The goal is not necessarily:

    ZERO PERMISSIONS

The inference service still needs legitimate access to operate.

Instead, the goal is:

    ONLY REQUIRED PERMISSIONS

This balance allows the AI system to continue functioning while reducing unnecessary security exposure.

</details>

In [ ]:
# @title ▶ Part 7: Test Separation of Duties { display-mode: "form" }

model_release_process = {
    "Develop Model":
        "DataScientist",

    "Approve Model":
        "ModelApprover",

    "Deploy Model":
        "Deployer"
}


def can_complete_entire_release(role):

    required_roles = set(
        model_release_process.values()
    )

    return (
        len(required_roles) == 1
        and role in required_roles
    )


print("=" * 70)
print("MODEL RELEASE — SEPARATION OF DUTIES")
print("=" * 70)

for step, role in model_release_process.items():

    print(
        f"\n{step:<20} → {role}"
    )


print("\nCan one role control the entire release?")

if can_complete_entire_release(
    "DataScientist"
):
    print("⚠ YES — Separation is weak.")

else:
    print(
        "✓ NO — Responsibilities are separated."
    )

<details>

<summary><b>Part 7 Explanation — Click to Expand</b></summary>

<br>

Sensitive production changes should not always be controlled by one identity.

This example separates the model-release process:

    DATA SCIENTIST
          ↓
    DEVELOP MODEL

          ↓

    MODEL APPROVER
          ↓
    APPROVE MODEL

          ↓

    DEPLOYMENT SERVICE
          ↓
    DEPLOY MODEL

If the Data Scientist's account becomes compromised, the attacker cannot immediately complete the entire production release process using that identity alone.

Separation of duties is especially valuable for high-impact actions such as model approval, deployment, access-policy changes, secret management, and deletion of production resources.

</details>

In [ ]:
# @title ▶ Part 8: Add Context-Aware Access Control { display-mode: "form" }

def production_deployment_policy(
    role,
    environment,
    model_status,
    mfa_verified
):

    if role != "DeploymentEngineer":
        return False

    if environment != "Production":
        return False

    if model_status != "Approved":
        return False

    if not mfa_verified:
        return False

    return True


tests = [
    {
        "role": "DeploymentEngineer",
        "environment": "Production",
        "model_status": "Approved",
        "mfa_verified": True
    },

    {
        "role": "DeploymentEngineer",
        "environment": "Production",
        "model_status": "Unapproved",
        "mfa_verified": True
    },

    {
        "role": "DeploymentEngineer",
        "environment": "Production",
        "model_status": "Approved",
        "mfa_verified": False
    }
]


print("=" * 70)
print("ATTRIBUTE-BASED ACCESS CONTROL")
print("=" * 70)

for test in tests:

    allowed = production_deployment_policy(
        **test
    )

    print("\n" + "-" * 50)

    for attribute, value in test.items():
        print(f"{attribute}: {value}")

    print(
        "\nDecision:",
        "✓ ALLOW" if allowed
        else "✗ DENY"
    )

<details>

<summary><b>Part 8 Explanation — Click to Expand</b></summary>

<br>

RBAC asks whether an identity belongs to an appropriate role.

ABAC can consider additional context.

In this example, production deployment requires:

    ROLE = DeploymentEngineer

    ENVIRONMENT = Production

    MODEL STATUS = Approved

    MFA = Verified

Having the correct role alone is not enough.

The system evaluates multiple attributes before allowing the sensitive operation.

This can provide more precise control over high-risk AI resources and actions.

</details>

In [ ]:
# @title ▶ Part 9: Apply Zero Trust { display-mode: "form" }

requests = [
    {
        "identity": "inference-service",
        "authenticated": True,
        "resource": "production_model",
        "action": "read"
    },

    {
        "identity": "inference-service",
        "authenticated": True,
        "resource": "model_registry",
        "action": "upload"
    },

    {
        "identity": "unknown-service",
        "authenticated": False,
        "resource": "production_model",
        "action": "read"
    }
]


print("=" * 70)
print("ZERO-TRUST ACCESS TEST")
print("=" * 70)

for request in requests:

    print("\n" + "-" * 60)

    print(
        f"Identity: {request['identity']}"
    )

    print(
        f"Request:  "
        f"{request['action']} "
        f"{request['resource']}"
    )

    if not request["authenticated"]:

        print("✗ DENIED — Authentication required")
        continue

    allowed = authorize(
        request["identity"],
        request["resource"],
        request["action"]
    )

    if allowed:
        print(
            "✓ ALLOWED — Explicitly authorized"
        )
    else:
        print(
            "✗ DENIED — Permission not granted"
        )

<details>

<summary><b>Part 9 Explanation — Click to Expand</b></summary>

<br>

Zero Trust does not assume that a request should be trusted simply because it originates from an internal workload.

Each request must answer:

    WHO IS REQUESTING ACCESS?

    HAS THE IDENTITY BEEN AUTHENTICATED?

    WHAT RESOURCE IS BEING REQUESTED?

    WHAT ACTION IS BEING REQUESTED?

    IS THAT ACTION AUTHORIZED?

The inference service is an internal production workload.

However:

    INTERNAL SERVICE ≠ UNLIMITED TRUST

It must still authenticate and receive explicit authorization for the requested resource.

This complements least privilege by removing unnecessary implicit trust between AI infrastructure components.

</details>

In [ ]:
# @title ▶ Part 10: Knowledge Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. Why did each AI workload receive a separate identity?",
        "options": [
            "A. To assign permissions based on each workload's needs",
            "B. To give every workload administrator access",
            "C. To eliminate authentication",
            "D. To hide workload activity"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. Why was the inference service denied model upload permission?",
        "options": [
            "A. Uploading models is unnecessary for inference",
            "B. The model registry was unavailable",
            "C. Workloads cannot use RBAC",
            "D. Inference services cannot authenticate"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. What happened when the overprivileged inference service "
        "was treated as compromised?",
        "options": [
            "A. The attacker inherited many unnecessary capabilities",
            "B. The attacker lost all permissions",
            "C. The model automatically retrained",
            "D. Nothing changed"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. What did least privilege reduce?",
        "options": [
            "A. The blast radius of the compromise",
            "B. The need for identities",
            "C. The number of models",
            "D. The need for authorization"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. Why were model development, approval, and deployment separated?",
        "options": [
            "A. To prevent one identity from controlling the entire process",
            "B. To eliminate production security",
            "C. To allow anonymous deployment",
            "D. To give Data Scientists administrator access"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. What additional information did the ABAC example consider?",
        "options": [
            "A. Role, environment, model status, and MFA",
            "B. Only username",
            "C. Only network location",
            "D. Only model accuracy"
        ],
        "answer": "A"
    },
    {
        "question":
        "7. What was the main Zero Trust lesson?",
        "options": [
            "A. Internal services still require authentication and authorization",
            "B. Internal services should automatically receive administrator access",
            "C. Authentication is only needed outside the network",
            "D. Every workload should share one identity"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("LAB 3 — KNOWLEDGE CHECK")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. "
            f"Correct answer: {q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 70)
print(f"Lab Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Lab complete!")
else:
    print(
        "△ Review the lab explanations "
        "and try again."
    )

<details>

<summary><b>Lab 3 — Practical Takeaways — Click to Expand</b></summary>

<br>

## Lab 3 — Practical Takeaways

This lab demonstrated that Identity and Access Management is not simply about deciding whether someone can log in.

IAM controls:

    WHO OR WHAT HAS AN IDENTITY

    HOW THAT IDENTITY IS VERIFIED

    WHICH RESOURCES IT CAN ACCESS

    WHICH ACTIONS IT CAN PERFORM

You created separate identities for AI workloads and used RBAC to give each workload permissions based on its responsibilities.

You then compared:

    OVERPRIVILEGED SERVICE

with:

    LEAST-PRIVILEGE SERVICE

The same inference service was considered compromised in both cases.

The important difference was what happened afterward.

    BROAD PERMISSIONS
          ↓
    LARGE BLAST RADIUS

while:

    LEAST PRIVILEGE
          ↓
    LIMITED BLAST RADIUS

You also used separation of duties to prevent one identity from developing, approving, and deploying a production model by itself.

Finally, you examined ABAC and Zero Trust to see how access decisions can consider additional context and require explicit authorization even for internal services.

The central lesson is:

> **Assume that an identity may eventually become compromised and limit what that identity can do before the compromise occurs.**

Strong IAM does not only control who gets access.

It also controls **how far an attacker can go after something goes wrong.**

</details>

In [ ]:
# @title ▶ Section 3 Checkpoint — Identity, Access Control & Least Privilege { display-mode: "form" }

questions = [
    {
        "question":
        "1. What does authentication determine?",
        "options": [
            "A. Who or what the identity is",
            "B. Every resource the identity owns",
            "C. Which model has the highest accuracy",
            "D. How many GPUs are available"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. What does authorization determine?",
        "options": [
            "A. Whether an identity can prove who it is",
            "B. What an authenticated identity is allowed to do",
            "C. How a model was trained",
            "D. How long a password is"
        ],
        "answer": "B"
    },
    {
        "question":
        "3. Which is an example of a workload identity?",
        "options": [
            "A. Data scientist",
            "B. Security analyst",
            "C. Inference service",
            "D. Application user"
        ],
        "answer": "C"
    },
    {
        "question":
        "4. What is the main goal of least privilege?",
        "options": [
            "A. Give every service administrator access",
            "B. Give identities only the permissions they need",
            "C. Eliminate authentication",
            "D. Allow every internal service to trust each other"
        ],
        "answer": "B"
    },
    {
        "question":
        "5. What is a major benefit of separate service identities?",
        "options": [
            "A. Every service receives the same permissions",
            "B. Compromise can be limited to that service's permissions",
            "C. Authentication is no longer required",
            "D. Logs are no longer necessary"
        ],
        "answer": "B"
    },
    {
        "question":
        "6. What does separation of duties help prevent?",
        "options": [
            "A. One identity controlling an entire sensitive process",
            "B. Models performing inference",
            "C. Services generating logs",
            "D. Users authenticating"
        ],
        "answer": "A"
    },
    {
        "question":
        "7. Which statement best represents Zero Trust?",
        "options": [
            "A. Internal systems should automatically trust each other",
            "B. Access should be explicitly authenticated and authorized",
            "C. Administrators should share credentials",
            "D. Network location alone proves identity"
        ],
        "answer": "B"
    }
]

score = 0

print("=" * 70)
print("SECTION 3 CHECKPOINT")
print("IDENTITY, ACCESS CONTROL, AND LEAST PRIVILEGE")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 70)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Checkpoint complete!")
else:
    print("△ Review Section 3 and try again.")

---

<details>
<summary><b>Section 4 — Secrets, Credentials, and Configuration Security — Click to Expand</b></summary>
<br>

## Section 4 — Secrets, Credentials, and Configuration Security

AI systems frequently depend on sensitive credentials and configuration information to communicate with cloud platforms, databases, model providers, storage systems, registries, and other services.

These secrets may include API keys, access tokens, passwords, private keys, certificates, cloud credentials, and service credentials. If attackers obtain them, they may be able to access AI resources without directly attacking the model itself.

Configuration settings can create similar risks. A secure model may still be exposed if cloud storage is accidentally made public, administrative interfaces are exposed to the Internet, logging is disabled, or services are configured with unnecessary permissions.

This section examines how secrets and configuration information become exposed and how organizations can securely store, rotate, validate, and manage them throughout the AI deployment lifecycle.

<br>

<details>
<summary><b>4.1 Secrets in Deployed AI Systems</b></summary>
<br>

### Secrets in Deployed AI Systems

A **secret** is sensitive information used to authenticate identities, authorize access, encrypt information, or establish trust between systems.

Modern AI deployments may depend on many different secrets.

Examples include:

- Model-provider API keys
- Cloud access credentials
- Database usernames and passwords
- Service-account credentials
- Access tokens
- Model-registry credentials
- Vector-database credentials
- Storage-access keys
- Private cryptographic keys
- TLS certificates and associated private keys
- CI/CD deployment credentials
- Monitoring and logging tokens

Consider an AI application that uses an external model provider.

The application may require:

**AI Application → API Key → External Model Provider**

The API key may allow the application to submit requests and potentially consume paid resources.

If an attacker steals the key, the attacker may be able to make requests while impersonating the legitimate application.

Similarly, compromised cloud credentials could potentially provide access to model storage, databases, deployment systems, or other infrastructure depending on the permissions associated with those credentials.

Secrets therefore act as **keys to other parts of the AI environment**.

Protecting the model while leaving its credentials exposed would provide little security.

</details>

<br>

<details>
<summary><b>4.2 How Secrets Become Exposed</b></summary>
<br>

### How Secrets Become Exposed

Secrets can become exposed in many different ways.

One of the most common mistakes is placing credentials directly inside application source code.

However, secrets may also appear in:

- Configuration files
- Git repositories
- Notebook cells
- Container images
- Application logs
- Shell scripts
- CI/CD configuration
- Error messages
- Web applications
- Documentation
- Shared files

A developer may accidentally write:

`API_KEY = "example-secret-key"`

and later upload the project to a public repository.

Even if the credential is eventually removed from the current version of the file, it may remain accessible through version-control history.

Meli, McNiece, and Reaves conducted a large-scale study of public GitHub repositories and found widespread secret leakage involving API keys and cryptographic credentials [15].

Sinha et al. similarly examined techniques for detecting and mitigating secret-key leakage in source-code repositories [16].

These studies demonstrate that credential leakage is not merely a theoretical security problem. Development workflows themselves can become a source of exposure.

AI development can create additional opportunities for this problem because researchers and developers frequently experiment with external APIs, cloud platforms, notebooks, model repositories, and data services.

A credential used temporarily during experimentation can become a production security issue if it is accidentally committed, shared, or reused.

</details>

<br>

<details>
<summary><b>4.3 Hardcoded Credentials</b></summary>
<br>

### Hardcoded Credentials

A **hardcoded credential** is a secret placed directly inside source code or another application artifact.

For example:

`MODEL_API_KEY = "abc123-secret-key"`

This approach may appear convenient because the application immediately has access to the credential.

However, it creates several security problems.

Anyone who can read the source code may be able to read the secret.

If the application is shared with another developer, the credential is shared as well.

If the source code is uploaded to a repository, the credential may become part of repository history.

If the same code is copied into a container image, the secret may also become embedded inside the image.

Hardcoded credentials are also difficult to rotate.

Suppose an API key becomes compromised.

If the credential exists directly inside the application, developers may need to modify the code and redeploy the application simply to replace the key.

A better design separates:

**Application Code**

from:

**Secret Value**

The application knows **which secret it needs**, but the actual credential is obtained securely at runtime.

This separation makes it easier to control access, rotate credentials, and prevent accidental exposure.

A useful rule is:

**Secrets should not be treated as normal application code or configuration data.**

</details>

<br>

<details>
<summary><b>4.4 Secrets and Version Control</b></summary>
<br>

### Secrets and Version Control

Version-control systems such as Git are essential for modern software and machine learning development.

However, they can also preserve information that developers believed they had deleted.

Consider the following sequence:

**Commit 1**

`API_KEY = "SECRET123"`

A developer later notices the mistake.

**Commit 2**

`API_KEY = ""`

The current file no longer contains the credential.

However, the original value may still exist in:

**Git History → Commit 1 → SECRET123**

This means simply deleting a leaked credential from the latest version of a file may not be sufficient.

If a real secret has been exposed, organizations should generally assume that the credential may have been copied.

The important response is therefore not simply:

**Delete the secret from the file**

but:

**Revoke or rotate the exposed credential**

followed by correcting the development process that caused the exposure.

Secret-scanning tools can also examine repositories for patterns that resemble API keys, passwords, tokens, private keys, and other credentials.

Meli et al. demonstrated the scale at which secrets can appear in public repositories [15], while Sinha et al. explored automated approaches for identifying and mitigating these exposures [16].

Repository security should therefore include both **prevention and detection**.

</details>

<br>

<details>
<summary><b>4.5 Environment Variables and Configuration Files</b></summary>
<br>

### Environment Variables and Configuration Files

A common improvement over hardcoding credentials is to separate configuration information from application source code.

For example, instead of:

`API_KEY = "SECRET123"`

an application might retrieve the value from an environment variable:

`API_KEY = environment["API_KEY"]`

The source code now contains the name of the required configuration value rather than the secret itself.

This provides several advantages.

The same application code can run in:

- Development
- Testing
- Staging
- Production

while using different credentials and configuration settings in each environment.

However, moving a secret into an environment variable does **not automatically make it secure**.

Secrets may still be exposed through:

- Process information
- Debugging tools
- Misconfigured container environments
- CI/CD logs
- Application logs
- Environment dumps
- Administrative interfaces

Configuration files can create similar risks.

A file such as:

`config.env`

may contain:

- Database passwords
- API keys
- Cloud credentials
- Service URLs

If that file is accidentally committed to version control or included in a container image, the secrets may still become exposed.

Environment variables and external configuration files are useful for separating code from configuration, but sensitive credentials should ideally be backed by a dedicated secret-management mechanism when appropriate.

</details>

<br>

<details>
<summary><b>4.6 Secret Management Systems</b></summary>
<br>

### Secret Management Systems

A **secret management system** provides centralized mechanisms for storing and controlling access to sensitive credentials.

Instead of embedding a database password inside an AI application:

**Application → Hardcoded Password → Database**

the architecture can use:

**Application**

↓

**Authenticated Workload Identity**

↓

**Secret Manager**

↓

**Authorized Secret**

↓

**Database**

The application first proves its identity.

The secret-management system then determines whether that workload is authorized to retrieve the requested credential.

Secret-management platforms may provide capabilities such as:

- Encrypted secret storage
- Fine-grained access control
- Audit logging
- Secret versioning
- Credential rotation
- Temporary credentials
- Access monitoring

This approach also works well with the IAM principles introduced in Section 3.

For example:

**Inference Service → May Read Inference Database Credential**

while:

**Inference Service → Cannot Read Deployment Administrator Credential**

A secret manager does not eliminate the need for access control. Instead, it provides a controlled location where secrets can be stored and distributed according to identity and policy.

The goal is to avoid scattering sensitive credentials across source code, notebooks, configuration files, containers, and deployment scripts.

</details>

<br>

<details>
<summary><b>4.7 Credential Rotation and Short-Lived Credentials</b></summary>
<br>

### Credential Rotation and Short-Lived Credentials

No credential should be assumed to remain secret forever.

Passwords can be exposed.

API keys can be accidentally committed.

Tokens can be stolen.

Service credentials can be extracted from compromised workloads.

One way to reduce the impact of credential exposure is **credential rotation**.

Rotation replaces an existing credential with a new one.

A simplified lifecycle is:

**Create → Use → Rotate → Revoke**

Suppose an AI service uses the same API key for three years.

If an attacker steals that key during the first year and the organization never detects the theft, the credential may remain useful for a long period.

Shorter-lived credentials reduce this opportunity.

Instead of:

**Credential Valid for Years**

a system may use:

**Temporary Credential → Expires Automatically**

Short-lived credentials can be especially useful for automated workloads and cloud services.

For example, an inference workload may authenticate using its service identity and receive a temporary token rather than storing a permanent cloud password.

Credential rotation also becomes extremely important after a suspected leak.

If a credential appears in a public repository, log file, exposed container, or compromised system, defenders should assume that simply removing the visible copy does not guarantee that the secret remains private.

The compromised credential should be **revoked or replaced**.

</details>

<br>

<details>
<summary><b>4.8 Secure Configuration and Secure Defaults</b></summary>
<br>

### Secure Configuration and Secure Defaults

Credentials are only one part of deployment configuration.

AI services may also rely on configuration settings that determine:

- Which network interfaces are exposed
- Whether authentication is required
- Which users can access storage
- Whether logging is enabled
- Whether encryption is required
- Which services may communicate
- Whether administrative interfaces are public
- Which permissions are assigned to workloads
- Which model version is deployed

A system can contain secure code and still become vulnerable because of insecure configuration.

For example:

**Authentication Feature Exists**

but:

**Authentication = Disabled**

The software supports security, but the deployment does not use it.

Secure systems should therefore use **secure defaults** whenever possible.

Instead of:

**Public unless restricted**

a safer default is:

**Private unless explicitly exposed**

Instead of:

**Access allowed unless denied**

a safer policy is:

**Access denied unless explicitly authorized**

Instead of:

**Logging optional**

security-relevant logging should be enabled according to organizational requirements.

Secure defaults reduce the likelihood that a forgotten configuration setting will expose sensitive AI resources.

</details>

<br>

<details>
<summary><b>4.9 Cloud Misconfiguration</b></summary>
<br>

### Cloud Misconfiguration

Cloud platforms make it possible to rapidly deploy AI infrastructure, but their flexibility also creates many opportunities for configuration mistakes.

Potential examples include:

- Publicly accessible storage
- Public databases
- Exposed administrative interfaces
- Overly permissive firewall rules
- Excessive cloud permissions
- Unencrypted storage
- Disabled logging
- Unrestricted network access
- Public model repositories
- Default credentials

Consider a model stored in cloud object storage.

The model itself may be protected from model-level attacks.

However, if the storage configuration allows public access:

**Internet → Public Storage → Production Model**

an attacker may be able to obtain the model without attacking the inference service at all.

Similarly, a private training dataset may become exposed if its storage permissions are configured incorrectly.

This demonstrates why cloud configuration is part of AI security.

The organization must protect not only:

**The Model**

but also:

**The Infrastructure That Stores and Runs the Model**

Configuration reviews, automated policy checks, infrastructure-as-code practices, and continuous monitoring can help identify dangerous settings before or after deployment.

</details>

<br>

<details>
<summary><b>4.10 Configuration Drift and Continuous Validation</b></summary>
<br>

### Configuration Drift and Continuous Validation

A system may be securely configured when it is first deployed but gradually become less secure over time.

This problem is known as **configuration drift**.

Imagine that an inference service is originally configured as:

**Public Access: Disabled**

**Authentication: Required**

**Logging: Enabled**

**Admin Access: Restricted**

Several months later, troubleshooting changes are made:

**Public Access: Enabled**

**Authentication: Required**

**Logging: Disabled**

**Admin Access: Broadened**

The application code may not have changed at all, but the security posture has changed significantly.

Configuration drift can occur because of:

- Manual administrative changes
- Emergency troubleshooting
- Temporary exceptions
- Infrastructure updates
- Testing changes
- Inconsistent environments
- Deployment errors

Organizations can reduce configuration drift by defining infrastructure and security requirements in repeatable ways.

**Infrastructure as Code (IaC)** allows infrastructure configuration to be represented in files that can be versioned, reviewed, tested, and deployed consistently.

Security policies can also be automatically evaluated against these configurations.

A simplified process might be:

**Configuration Change**

↓

**Automated Security Validation**

↓

**Policy Compliant?**

**No → Block or Alert**

**Yes → Deploy**

Continuous validation is important because security configuration should not be treated as something checked only once during initial deployment.

</details>

<br>

<details>
<summary><b>4.11 Responding to an Exposed Credential</b></summary>
<br>

### Responding to an Exposed Credential

Discovering an exposed secret should be treated as a security event.

Suppose a developer discovers a cloud API key inside a public repository.

A poor response would be:

**Delete Key from File → Commit Change → Done**

The key may already have been copied by someone else.

A stronger response is:

**Identify Exposure**

↓

**Revoke or Disable Credential**

↓

**Issue Replacement Credential if Necessary**

↓

**Determine Associated Permissions**

↓

**Review Logs for Unauthorized Use**

↓

**Remove Credential from Code and History Where Appropriate**

↓

**Correct the Process That Caused the Leak**

The investigation should determine what the exposed credential could access.

For example:

**Credential A**

may only allow:

**Read Public Model Metadata**

while:

**Credential B**

may allow:

**Read/Write Cloud Storage + Modify Models + Access Database**

The second exposure has a much larger potential impact.

This connects credential security directly to least privilege.

If a secret becomes exposed, the permissions attached to that credential help determine the possible **blast radius**.

Security teams should therefore ask:

- What secret was exposed?
- When was it exposed?
- Where was it exposed?
- What systems could it access?
- What permissions did it have?
- Was it used unexpectedly?
- Has it been revoked?
- What other credentials may also be affected?
- How can the same type of exposure be prevented in the future?

Credential incidents demonstrate why prevention, monitoring, least privilege, and incident response must work together.

</details>

<br>

<details>
<summary><b>4.12 Connecting Secrets and Configuration to AI Security</b></summary>
<br>

### Connecting Secrets and Configuration to AI Security

Secrets and configuration vulnerabilities may appear to be traditional cybersecurity problems, but they can directly enable attacks against AI resources.

Consider several examples.

#### Exposed Model Provider API Key

**Leaked API Key**

↓

**Unauthorized Inference Requests**

↓

**Resource Abuse / Financial Cost**

#### Exposed Model Registry Credential

**Stolen Registry Credential**

↓

**Unauthorized Registry Access**

↓

**Model Theft or Model Replacement**

#### Exposed Cloud Credential

**Stolen Cloud Credential**

↓

**Cloud Resource Access**

↓

**Models / Data / Infrastructure Exposed**

#### Public Model Storage

**Storage Misconfiguration**

↓

**Unauthorized Model Download**

↓

**Potential Model Theft**

#### Public Training Data Storage

**Storage Misconfiguration**

↓

**Unauthorized Dataset Access**

↓

**Privacy or Confidentiality Exposure**

#### Excessive Credential Permissions

**Credential Compromised**

↓

**Broad Administrative Access**

↓

**Large Blast Radius**

The attacker may never need to directly defeat the model's security mechanisms.

Instead, the attacker may simply use a leaked credential or insecure configuration to access the infrastructure surrounding the model.

This reinforces one of the major themes of Module 6:

**AI security depends on the security of the entire deployment environment, not only the model.**

</details>

<br>

<details>
<summary><b>Section 4 Summary</b></summary>
<br>

### Section 4 Summary

Secrets and configuration information allow AI services to communicate with databases, cloud platforms, model providers, registries, deployment systems, and other resources. If these secrets or configurations are exposed, attackers may gain legitimate-looking access to sensitive AI infrastructure.

Throughout this section, you learned that:

- AI deployments may contain many different credentials, tokens, keys, and service secrets.
- Secrets can become exposed through source code, repositories, notebooks, containers, logs, and configuration files.
- Hardcoded credentials tightly couple sensitive information to application code and make secure rotation difficult.
- Removing a secret from the current version of a repository does not necessarily remove it from version history.
- Environment variables separate configuration from code but are not a complete secret-management solution.
- Secret-management systems can centralize storage, access control, auditing, and credential distribution.
- Credential rotation and short-lived credentials can reduce the impact of stolen secrets.
- Secure defaults reduce the likelihood that configuration mistakes expose AI resources.
- Cloud misconfiguration can expose models, datasets, databases, and administrative interfaces.
- Configuration drift can gradually weaken an initially secure deployment.
- Exposed credentials should be revoked or rotated and investigated rather than simply deleted from source code.
- Least privilege limits the potential blast radius when credentials are compromised.

The next section examines **Container and Runtime Security**, where you will explore how vulnerable images, insecure base images, excessive privileges, unnecessary software, and weak runtime isolation can expose deployed AI workloads.

</details>

</details>

In [ ]:

# @title Section 4 — Secrets, Credentials, and Configuration Security Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import ListedColormap
from IPython.display import display, HTML
from io import BytesIO
import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"
TEAL = "#0891B2"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):
    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    buffer.close()

    display(HTML(f"""
    <details style="
        margin:12px 0;
        border:1px solid #cbd5e1;
        border-radius:10px;
        overflow:hidden;
    ">
        <summary style="
            cursor:pointer;
            padding:16px;
            font-size:16px;
            font-weight:bold;
            background:#eff6ff;
            color:#1e3a8a;
            border-left:5px solid {color};
        ">
            {html.escape(title)} — Click to Expand
        </summary>

        <div style="
            padding:15px;
            background:white;
            text-align:center;
        ">
            <img
                src="data:image/png;base64,{encoded}"
                style="width:100%;max-width:950px;height:auto;"
            >

            <div style="
                max-width:850px;
                margin:15px auto;
                padding:16px;
                background:#f8fafc;
                border-radius:8px;
                color:#334155;
                font-size:14px;
                line-height:1.7;
                text-align:left;
            ">
                {explanation}
            </div>
        </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def setup_diagram(ax, title, xlim=(0, 12), ylim=(0, 10)):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.axis("off")
    ax.set_title(
        title,
        fontsize=16,
        fontweight="bold",
        pad=18
    )


def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC",
             title_size=10):

    card = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.03,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.8
    )

    ax.add_patch(card)

    ax.text(
        x+w/2,
        y+h*(0.66 if subtitle else 0.5),
        title,
        ha="center",
        va="center",
        fontsize=title_size,
        fontweight="bold",
        color=DARK
    )

    if subtitle:
        ax.text(
            x+w/2,
            y+h*0.28,
            subtitle,
            ha="center",
            va="center",
            fontsize=8.5,
            color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):
    ax.add_patch(FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=16,
        linewidth=1.8,
        color=color
    ))


# ============================================================
# VISUALIZATION 1
# SECRETS AND CREDENTIAL EXPOSURE PATHS
# ============================================================

fig, ax = plt.subplots(figsize=(12, 9))

setup_diagram(
    ax,
    "Common Secret Exposure Paths in AI Infrastructure",
    xlim=(0, 12),
    ylim=(0, 11)
)

add_card(
    ax, 4.2, 8.7, 3.6, 1.2,
    "Sensitive Credentials",
    "API keys, tokens, passwords",
    RED
)

exposure_paths = [
    (0.3, 6.0, "Source Code", "Hardcoded credentials"),
    (4.4, 6.0, "Version Control", "Committed secret history"),
    (8.5, 6.0, "Logs and Errors", "Accidental disclosure"),
    (0.3, 2.8, "Configuration Files", "Insecure file access"),
    (4.4, 2.8, "Container Images", "Embedded secrets"),
    (8.5, 2.8, "Cloud Storage", "Excessive permissions")
]

for x, y, title, subtitle in exposure_paths:
    add_card(
        ax, x, y, 3.2, 1.2,
        title, subtitle, RED
    )

for x, y, _, _ in exposure_paths:
    add_arrow(
        ax,
        (6.0, 8.7),
        (x+1.6, y+1.2),
        RED
    )

ax.text(
    6.0, 1.0,
    "A secret can be exposed through storage, logging, deployment, or access mistakes",
    ha="center",
    fontsize=10,
    fontweight="bold",
    color=DARK
)

plt.tight_layout()

show_collapsible_figure(
    "4.1 — Secrets and Credential Exposure Paths",
    fig,
    """
    AI applications frequently require secrets
    to communicate with external services
    and protected infrastructure.

    <br><br>

    Examples include API keys for hosted models,
    database passwords, model registry tokens,
    and cloud access credentials.

    <br><br>

    These secrets can become exposed at
    several points in the development
    and deployment process.

    <br><br>

    A developer might accidentally place
    an API key directly inside source code.

    <br><br>

    If that code is committed to a repository,
    the credential may remain accessible
    through version control history even
    after the current file is corrected.

    <br><br>

    Secrets may also appear in application
    logs, error messages, configuration
    files, or container images.

    <br><br>

    In cloud environments, improperly
    configured storage permissions may
    expose sensitive configuration data.

    <br><br>

    The diagram shows conceptual exposure
    paths rather than actual incidents.

    <br><br>

    <b>Key takeaway:</b> Protecting secrets
    requires examining their entire lifecycle,
    including creation, storage, distribution,
    use, rotation, and revocation.
    """,
    RED
)


# ============================================================
# VISUALIZATION 2
# HARDCODED CREDENTIALS VS SECRET MANAGEMENT
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

fig.suptitle(
    "Hardcoded Credentials vs. Managed Secrets",
    fontsize=16,
    fontweight="bold"
)

# Insecure architecture
ax = axes[0]
setup_diagram(
    ax,
    "Insecure Credential Handling",
    xlim=(0, 6),
    ylim=(0, 10)
)

insecure_steps = [
    ("Application Code", "Contains embedded API key", RED),
    ("Source Repository", "Stores code and credential", RED),
    ("Container Image", "May preserve the credential", ORANGE),
    ("External AI Service", "Receives credential", BLUE)
]

for i, (title, subtitle, color) in enumerate(insecure_steps):
    y = 8.1 - i*2.0

    add_card(
        ax, 0.6, y, 4.8, 1.2,
        title, subtitle, color
    )

    if i < len(insecure_steps)-1:
        add_arrow(
            ax, (3.0, y-0.05),
            (3.0, y-0.75),
            RED
        )

# Managed secrets architecture
ax = axes[1]
setup_diagram(
    ax,
    "Managed Credential Handling",
    xlim=(0, 6),
    ylim=(0, 10)
)

secure_steps = [
    ("AI Application", "Uses workload identity", BLUE),
    ("Secret Manager", "Checks identity and permissions", PURPLE),
    ("Scoped Credential", "Retrieved securely when needed", GREEN),
    ("External AI Service", "Authorized service access", GREEN)
]

for i, (title, subtitle, color) in enumerate(secure_steps):
    y = 8.1 - i*2.0

    add_card(
        ax, 0.6, y, 4.8, 1.2,
        title, subtitle, color
    )

    if i < len(secure_steps)-1:
        add_arrow(
            ax, (3.0, y-0.05),
            (3.0, y-0.75),
            GREEN
        )

plt.tight_layout()

show_collapsible_figure(
    "4.2 — Hardcoded Credentials vs. Secure Secret Management",
    fig,
    """
    Hardcoded credentials are secrets
    written directly into application code.

    <br><br>

    For example, a developer might place
    a model provider API key inside a
    Python script.

    <br><br>

    This is dangerous because anyone
    who gains access to the source
    code may also gain access
    to the credential.

    <br><br>

    The problem can become worse
    when the code is committed
    to version control or packaged
    into a container image.

    <br><br>

    A safer architecture separates
    secrets from application code.

    <br><br>

    The application authenticates
    using a workload identity and
    retrieves an authorized secret
    from a dedicated secret manager.

    <br><br>

    Secret management systems can
    provide access controls, auditing,
    encryption, and rotation support.

    <br><br>

    However, using a secret manager
    does not automatically eliminate
    every risk.

    <br><br>

    The application must still
    protect credentials while
    they are in memory and avoid
    exposing them through logs
    or error messages.

    <br><br>

    <b>Key takeaway:</b> Credentials
    should be managed independently
    of application source code
    and distributed only to
    authorized workloads.
    """,
    GREEN
)


# ============================================================
# VISUALIZATION 3
# SECRET MANAGEMENT ARCHITECTURE
# ============================================================

fig, ax = plt.subplots(figsize=(12, 9))

setup_diagram(
    ax,
    "Secure Secret Retrieval for an AI Workload",
    xlim=(0, 12),
    ylim=(0, 11)
)

add_card(
    ax, 0.5, 8.2, 3.2, 1.3,
    "AI Inference Service",
    "Requests access to a secret",
    BLUE
)

add_card(
    ax, 4.4, 8.2, 3.2, 1.3,
    "Identity Provider",
    "Verifies workload identity",
    PURPLE
)

add_card(
    ax, 8.3, 8.2, 3.2, 1.3,
    "Secret Manager",
    "Enforces access policy",
    GREEN
)

add_arrow(ax, (3.7, 8.85), (4.4, 8.85), BLUE)
add_arrow(ax, (7.6, 8.85), (8.3, 8.85), PURPLE)

add_card(
    ax, 8.3, 5.0, 3.2, 1.3,
    "Protected Secret",
    "Encrypted at rest",
    GREEN
)

add_arrow(ax, (9.9, 8.2), (9.9, 6.3), GREEN)

add_card(
    ax, 4.4, 2.1, 3.2, 1.3,
    "Authorized Service Access",
    "Credential used for permitted operation",
    TEAL
)

add_arrow(ax, (8.3, 5.6), (6.0, 3.4), GREEN)

add_card(
    ax, 0.5, 5.0, 3.2, 1.3,
    "Unauthorized Workload",
    "Insufficient permissions",
    RED
)

ax.text(
    2.1, 4.3,
    "Access denied",
    ha="center",
    fontsize=10,
    fontweight="bold",
    color=RED
)

ax.text(
    6.0, 0.9,
    "Authentication + authorization + protected delivery + auditing",
    ha="center",
    fontsize=10,
    fontweight="bold",
    color=DARK
)

plt.tight_layout()

show_collapsible_figure(
    "4.3 — Secret Management Architecture",
    fig,
    """
    A secret management system provides
    a centralized method for storing
    and distributing sensitive credentials.

    <br><br>

    Consider an AI inference service
    that requires a credential
    to access a hosted model provider.

    <br><br>

    Rather than storing the credential
    in its source code, the service
    authenticates using its
    workload identity.

    <br><br>

    The secret manager evaluates
    whether that identity is
    authorized to retrieve
    the requested secret.

    <br><br>

    If the request is permitted,
    the credential can be delivered
    through a protected communication
    channel.

    <br><br>

    If the identity lacks permission,
    access should be denied.

    <br><br>

    Secret managers can also record
    access events, helping defenders
    investigate unusual secret
    retrieval activity.

    <br><br>

    A production implementation
    should consider short-lived
    credentials, encryption,
    least-privilege policies,
    and secret rotation.

    <br><br>

    The exact architecture depends
    on the cloud provider or
    secret management technology.

    <br><br>

    <b>Key takeaway:</b> A secure
    secret manager protects
    credentials by controlling
    which identities can retrieve
    and use them.
    """,
    PURPLE
)


# ============================================================
# VISUALIZATION 4
# CREDENTIAL ROTATION AND EXPIRATION
# ============================================================

# Synthetic timeline representing two credential policies.
# Values are conceptual exposure windows in days.

days = np.arange(0, 91)

long_lived_active = np.ones_like(days)
short_lived_active = (days % 15 < 15).astype(int)

fig, ax = plt.subplots(figsize=(12, 7))

# Long-lived credential
ax.broken_barh(
    [(0, 90)],
    (2.0, 0.65),
    facecolors=RED
)

# Rotated credentials
for start in range(0, 90, 15):
    ax.broken_barh(
        [(start, 14.5)],
        (0.7, 0.65),
        facecolors=GREEN
    )

for start in range(15, 90, 15):
    ax.axvline(
        start,
        color=GREEN,
        linestyle=":",
        alpha=0.45
    )

ax.set_xlim(0, 90)
ax.set_ylim(0.3, 3.0)

ax.set_yticks([1.025, 2.325])
ax.set_yticklabels([
    "Rotated Credentials",
    "Long-Lived Credential"
])

ax.set_xticks(np.arange(0, 91, 15))
ax.set_xlabel("Illustrative Timeline (Days)")

ax.set_title(
    "Credential Lifetime and Rotation Windows",
    fontsize=16,
    fontweight="bold"
)

ax.grid(axis="x", alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "4.4 — Credential Rotation and Expiration",
    fig,
    """
    Credential rotation replaces
    an existing credential with
    a new one.

    <br><br>

    Rotation helps limit the
    period during which a
    compromised credential
    may remain useful.

    <br><br>

    The top bar represents a
    hypothetical credential
    that remains valid for
    90 days.

    <br><br>

    The bottom bars represent
    credentials replaced
    every 15 days.

    <br><br>

    Each segment represents
    a separate credential
    validity period.

    <br><br>

    The shorter rotation
    schedule can reduce
    the maximum lifetime
    of an individual credential.

    <br><br>

    However, rotation alone
    does not guarantee that
    an attacker loses access.

    <br><br>

    If an attacker controls
    the workload that retrieves
    new credentials, they may
    be able to obtain replacements.

    <br><br>

    Strong credential security
    also requires monitoring,
    workload identity protection,
    revocation, and least privilege.

    <br><br>

    The timelines are illustrative
    and are not recommended
    universal rotation intervals.

    <br><br>

    <b>Key takeaway:</b> Short-lived
    credentials and effective
    rotation reduce reliance
    on long-lived secrets
    and can limit exposure
    after credential theft.
    """,
    ORANGE
)


# ============================================================
# VISUALIZATION 5
# SECURE VS INSECURE CONFIGURATION
# ============================================================

settings = [
    "Public Model Storage",
    "Debug Mode Enabled",
    "Excessive IAM Permissions",
    "Unencrypted Communication",
    "Unrestricted Network Access",
    "Secrets in Configuration"
]

# 1 = insecure condition present
# 0 = insecure condition absent
#
# Hypothetical configurations.

insecure = np.ones(len(settings))
hardened = np.zeros(len(settings))

matrix = np.column_stack([insecure, hardened])

fig, ax = plt.subplots(figsize=(12, 7))

ax.imshow(
    matrix,
    cmap=ListedColormap(["#DCFCE7", "#FEE2E2"]),
    vmin=0,
    vmax=1,
    aspect="auto"
)

ax.set_xticks([0, 1])
ax.set_xticklabels([
    "Insecure Deployment",
    "Hardened Deployment"
])

ax.set_yticks(range(len(settings)))
ax.set_yticklabels(settings)

ax.set_title(
    "Example Configuration Security Comparison",
    fontsize=16,
    fontweight="bold",
    pad=18
)

for i in range(len(settings)):
    for j in range(2):

        insecure_condition = matrix[i, j] == 1

        ax.text(
            j, i,
            "PRESENT" if insecure_condition else "MITIGATED",
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold",
            color=RED if insecure_condition else GREEN
        )

ax.set_xticks(np.arange(-0.5, 2, 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(settings), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", bottom=False, left=False)

plt.tight_layout()

show_collapsible_figure(
    "4.5 — Secure vs. Insecure Configuration",
    fig,
    """
    Security configuration determines
    how deployed infrastructure
    behaves and which resources
    are accessible.

    <br><br>

    Even when application code
    is secure, configuration
    mistakes can introduce
    serious vulnerabilities.

    <br><br>

    Examples include publicly
    accessible model storage,
    excessive IAM permissions,
    unnecessary debug features,
    and unrestricted network access.

    <br><br>

    An AI model registry
    configured for public
    write access may allow
    unauthorized artifact
    modification.

    <br><br>

    A cloud storage bucket
    containing training data
    may expose sensitive
    information if public
    access is enabled.

    <br><br>

    A hardened deployment
    uses secure defaults,
    restricted permissions,
    protected communication,
    and controlled configuration.

    <br><br>

    The matrix is an
    illustrative comparison.
    Real configurations require
    context-specific evaluation.

    <br><br>

    Some services may legitimately
    require public access,
    but that access should
    be intentional and
    appropriately controlled.

    <br><br>

    <b>Key takeaway:</b> Secure
    configuration reduces
    unnecessary exposure
    before an attacker
    attempts to exploit
    the system.
    """,
    GREEN
)


# ============================================================
# VISUALIZATION 6
# CONFIGURATION DRIFT AND CONTINUOUS VALIDATION
# ============================================================

# Synthetic configuration compliance values.
# Each point represents the percentage of
# checked controls matching the approved baseline.

days = np.arange(1, 31)

compliance = np.array([
    100, 100, 100, 99, 98,
    98, 97, 96, 94, 93,
    91, 90, 88, 85, 83,
    82, 80, 79, 78, 76,
    75, 74, 96, 98, 99,
    100, 100, 99, 100, 100
])

threshold = 90

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(
    days,
    compliance,
    marker="o",
    linewidth=2.5,
    color=BLUE,
    label="Configuration Compliance"
)

ax.axhline(
    threshold,
    color=RED,
    linestyle="--",
    linewidth=2,
    label="Illustrative Review Threshold"
)

ax.fill_between(
    days,
    compliance,
    threshold,
    where=compliance < threshold,
    color=RED,
    alpha=0.15
)

ax.annotate(
    "Configuration drift",
    xy=(18, 79),
    xytext=(10, 68),
    fontsize=10,
    fontweight="bold",
    color=RED,
    arrowprops=dict(
        arrowstyle="->",
        color=RED,
        linewidth=1.5
    )
)

ax.annotate(
    "Remediation",
    xy=(23, 96),
    xytext=(24, 83),
    fontsize=10,
    fontweight="bold",
    color=GREEN,
    arrowprops=dict(
        arrowstyle="->",
        color=GREEN,
        linewidth=1.5
    )
)

ax.set_xlim(1, 30)
ax.set_ylim(60, 105)

ax.set_xlabel("Day")
ax.set_ylabel("Controls Matching Baseline (%)")

ax.set_title(
    "Continuous Configuration Validation and Drift Detection",
    fontsize=16,
    fontweight="bold"
)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "4.6 — Configuration Drift and Continuous Validation",
    fig,
    """
    Configuration drift occurs
    when deployed settings
    gradually differ from
    an approved configuration
    baseline.

    <br><br>

    Drift can result from
    manual changes, software
    updates, new deployments,
    or infrastructure modifications.

    <br><br>

    Some changes may be
    authorized and harmless.

    <br><br>

    Others may weaken security
    by enabling public access,
    expanding permissions,
    or disabling important
    protective controls.

    <br><br>

    This graph shows synthetic
    configuration compliance
    over a 30-day period.

    <br><br>

    Compliance decreases as
    more settings deviate
    from the approved baseline.

    <br><br>

    A review threshold
    highlights when additional
    investigation may be needed.

    <br><br>

    After remediation,
    the configuration returns
    closer to the baseline.

    <br><br>

    Real organizations may
    use infrastructure-as-code,
    policy-as-code, automated
    configuration scanning,
    and continuous monitoring
    to detect unauthorized changes.

    <br><br>

    A compliance percentage
    alone does not measure
    overall security risk.
    Individual control failures
    may have very different impacts.

    <br><br>

    <b>Key takeaway:</b> Secure
    configuration requires
    continuous verification,
    not simply an initial
    hardening checklist.
    """,
    PURPLE
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("Section 4 — Secrets, Credentials, and Configuration Security Visualizations")
print("----------------------------------------------------------------------------")
print("Six collapsible visualizations generated.")

print()
print("Concepts Covered")
print("----------------")
print("1. Secrets and Credential Exposure Paths")
print("2. Hardcoded Credentials vs. Secret Management")
print("3. Secret Management Architecture")
print("4. Credential Rotation and Expiration")
print("5. Secure vs. Insecure Configuration")
print("6. Configuration Drift and Continuous Validation")

print()
print("All scenarios and numerical values are illustrative.")
print("No real credentials, cloud services, or external APIs were used.")


<details>

<summary><b>Section 4 Lab — Click to Expand</b></summary>

<br>

A development team has created an AI application that connects to several external resources.

The application uses:

    MODEL PROVIDER API

    DATABASE

    MODEL REGISTRY

    CLOUD STORAGE

During development, the team prioritized getting the application working quickly.

As a result, some credentials were placed directly inside application code and several deployment settings were configured insecurely.

The system works.

However:

    FUNCTIONAL ≠ SECURE

In this lab, you will perform a basic security review of the deployment.

You will:

    Find hardcoded secrets

    Examine why source-code exposure matters

    Separate secrets from application code

    Simulate a secret-management system

    Rotate compromised credentials

    Compare long-lived and short-lived credentials

    Identify insecure configuration

    Detect configuration drift

    Respond to a simulated credential exposure

The goal is to understand that attackers do not necessarily need to attack the AI model directly.

Sometimes the easiest path into an AI environment is simply:

    FIND CREDENTIAL
          ↓
    USE CREDENTIAL
          ↓
    ACCESS AI RESOURCE

All exercises use standard Python and should run almost instantly.

</details>

In [ ]:
# @title ▶ Part 1: Inspect the Application { display-mode: "form" }

application_config = {
    "MODEL_API_KEY": "sk-demo-123456",
    "DATABASE_PASSWORD": "AI_DB_PASSWORD_2026",
    "REGISTRY_TOKEN": "registry-token-demo",
    "MODEL_ENDPOINT": "https://model.example",
    "DEBUG_MODE": True,
    "PUBLIC_STORAGE": True,
    "LOGGING_ENABLED": False
}

print("=" * 70)
print("AI APPLICATION CONFIGURATION")
print("=" * 70)

for key, value in application_config.items():

    print(
        f"{key:<22} = {value}"
    )

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

This configuration contains several obvious security problems.

Credentials appear directly beside normal application configuration.

Examples include:

    MODEL_API_KEY

    DATABASE_PASSWORD

    REGISTRY_TOKEN

Anyone who receives this file may also receive the credentials.

The configuration also contains potentially dangerous settings:

    DEBUG_MODE = True

    PUBLIC_STORAGE = True

    LOGGING_ENABLED = False

This demonstrates an important distinction.

Some security problems involve:

    SECRETS

while others involve:

    CONFIGURATION

A deployment can therefore be vulnerable even if the underlying AI model itself is secure.

</details>

In [ ]:
# @title ▶ Part 2: Scan for Exposed Secrets { display-mode: "form" }

sensitive_keywords = [
    "KEY",
    "PASSWORD",
    "TOKEN",
    "SECRET"
]

detected_secrets = []

for name, value in application_config.items():

    if any(
        keyword in name.upper()
        for keyword in sensitive_keywords
    ):
        detected_secrets.append(name)


print("=" * 70)
print("SECRET SCAN")
print("=" * 70)

if detected_secrets:

    print("\nPotential secrets detected:")

    for secret in detected_secrets:
        print(f"  ⚠ {secret}")

else:

    print(
        "\n✓ No obvious secrets detected."
    )

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

The scanner searches configuration names for terms commonly associated with credentials.

It detected:

    API KEY

    PASSWORD

    TOKEN

This is a simplified version of the idea behind repository secret scanning.

Real secret-scanning tools use much more sophisticated techniques, including patterns associated with specific credential formats.

The important concept is that development repositories and notebooks should be checked for accidentally exposed credentials.

However, detection does not solve the entire problem.

If a real credential has already been exposed, finding it is only the beginning.

</details>

In [ ]:
# @title ▶ Part 3: Simulate Version-Control History { display-mode: "form" }

git_history = [
    {
        "commit": "Commit 1",
        "content":
            'MODEL_API_KEY = "sk-demo-123456"'
    },

    {
        "commit": "Commit 2",
        "content":
            'MODEL_API_KEY = ""'
    }
]

print("=" * 70)
print("SIMULATED GIT HISTORY")
print("=" * 70)

for commit in git_history:

    print(
        f"\n{commit['commit']}"
    )

    print(
        commit["content"]
    )

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

The current version of the file no longer contains the API key.

However, the previous commit still does.

This represents an important version-control problem.

The sequence was:

    COMMIT SECRET
          ↓
    NOTICE MISTAKE
          ↓
    DELETE SECRET
          ↓
    COMMIT AGAIN

The current file appears safe.

But:

    VERSION HISTORY
          ↓
    ORIGINAL SECRET

may still exist.

Therefore, if a real credential was committed to a repository, simply deleting the value should not be considered sufficient.

The exposed credential should generally be treated as compromised and revoked or rotated.

</details>

In [ ]:
# @title ▶ Part 4: Use Environment Variables { display-mode: "form" }

import os

# Simulate credentials being provided
# by the deployment environment.
os.environ["MODEL_API_KEY"] = "runtime-key-789"
os.environ["DATABASE_PASSWORD"] = "runtime-db-password"


def load_application_config():

    return {
        "MODEL_API_KEY":
            os.environ.get("MODEL_API_KEY"),

        "DATABASE_PASSWORD":
            os.environ.get("DATABASE_PASSWORD"),

        "MODEL_ENDPOINT":
            "https://model.example"
    }


secure_config = load_application_config()

print("=" * 70)
print("RUNTIME CONFIGURATION")
print("=" * 70)

print(
    "\nApplication code requests:"
)

print("  MODEL_API_KEY")
print("  DATABASE_PASSWORD")

print(
    "\nCredentials are loaded "
    "at runtime rather than "
    "written directly into the code."
)

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

The application now refers to the **name** of the required secret instead of containing the actual credential directly in its source code.

Conceptually:

    APPLICATION CODE
          ↓
    REQUEST SECRET
          ↓
    RUNTIME ENVIRONMENT
          ↓
    SECRET VALUE

This separates:

    CODE

from:

    CREDENTIAL

Environment variables can be an improvement over hardcoded credentials, but they should not be treated as a complete secret-management solution.

Poorly configured environments, debugging tools, logs, or container settings may still expose their values.

The important improvement here is separation.

The application no longer needs the credential embedded directly inside its source code.

</details>

In [ ]:
# @title ▶ Part 5: Use a Secret Management System { display-mode: "form" }

secret_store = {
    "prod/model-api-key": {
        "value": "managed-key-v1",
        "allowed_services": [
            "inference-service"
        ]
    },

    "prod/database-password": {
        "value": "managed-db-v1",
        "allowed_services": [
            "database-service"
        ]
    }
}


def get_secret(service, secret_name):

    secret = secret_store.get(
        secret_name
    )

    if secret is None:
        return "SECRET NOT FOUND"

    if service not in secret[
        "allowed_services"
    ]:
        return "ACCESS DENIED"

    return secret["value"]


print("=" * 70)
print("SECRET MANAGER TEST")
print("=" * 70)

tests = [
    (
        "inference-service",
        "prod/model-api-key"
    ),

    (
        "inference-service",
        "prod/database-password"
    ),

    (
        "database-service",
        "prod/database-password"
    )
]

for service, secret in tests:

    result = get_secret(
        service,
        secret
    )

    print(
        f"\nService: {service}"
        f"\nSecret:  {secret}"
        f"\nResult:  {result}"
    )

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

The secret-management simulation adds access control around the credentials.

The inference service can retrieve:

    MODEL API KEY

but cannot retrieve:

    DATABASE PASSWORD

The database service can retrieve its own credential.

This follows the same least-privilege principle introduced in Section 3.

Instead of:

    EVERY SERVICE
         ↓
    EVERY SECRET

we have:

    SERVICE IDENTITY
         ↓
    AUTHORIZED SECRET
         ↓
    REQUIRED CREDENTIAL

A secret-management system can also support capabilities such as centralized auditing, rotation, access policies, and credential lifecycle management.

</details>

In [ ]:
# @title ▶ Part 6: Rotate a Credential { display-mode: "form" }

old_key = secret_store[
    "prod/model-api-key"
]["value"]

print("=" * 70)
print("CREDENTIAL ROTATION")
print("=" * 70)

print(
    f"\nOld credential: {old_key}"
)

# Rotate credential
secret_store[
    "prod/model-api-key"
]["value"] = "managed-key-v2"

new_key = secret_store[
    "prod/model-api-key"
]["value"]

print(
    f"New credential: {new_key}"
)

print("\nCredential changed:")

if old_key != new_key:
    print("✓ ROTATION SUCCESSFUL")
else:
    print("✗ ROTATION FAILED")

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

Credential rotation replaces an existing secret with a new value.

Before rotation:

    managed-key-v1

After rotation:

    managed-key-v2

If the old credential had been stolen, replacing it reduces the attacker's ability to continue using that credential.

This is especially important after confirmed or suspected exposure.

A useful response sequence is:

    DETECT EXPOSURE
          ↓
    REVOKE / ROTATE
          ↓
    UPDATE AUTHORIZED SERVICE
          ↓
    INVESTIGATE USAGE
          ↓
    CORRECT ROOT CAUSE

Rotation should be treated as part of credential lifecycle management rather than something performed only after an incident.

</details>

In [ ]:
# @title ▶ Part 7: Compare Credential Lifetimes { display-mode: "form" }

credentials = {
    "Long-Lived Key": {
        "lifetime_hours": 8760
    },

    "Short-Lived Token": {
        "lifetime_hours": 1
    }
}

print("=" * 70)
print("CREDENTIAL LIFETIME COMPARISON")
print("=" * 70)

for name, info in credentials.items():

    print(
        f"\n{name}"
        f"\nLifetime: "
        f"{info['lifetime_hours']} hour(s)"
    )


long_lived = credentials[
    "Long-Lived Key"
]["lifetime_hours"]

short_lived = credentials[
    "Short-Lived Token"
]["lifetime_hours"]

reduction = (
    1 - (short_lived / long_lived)
) * 100

print(
    f"\nPotential credential lifetime "
    f"reduction: {reduction:.2f}%"
)

<details>

<summary><b>Part 7 Explanation — Click to Expand</b></summary>

<br>

A long-lived credential may remain valid for months or even years.

If that credential is stolen, an attacker may continue using it until it is manually revoked or rotated.

A short-lived credential automatically becomes invalid after a limited period.

Conceptually:

    STOLEN LONG-LIVED KEY
             ↓
    LONG ATTACK WINDOW

compared with:

    STOLEN SHORT-LIVED TOKEN
             ↓
    LIMITED ATTACK WINDOW
             ↓
          EXPIRES

Short-lived credentials do not prevent credential theft.

They reduce how long a stolen credential may remain useful.

</details>

In [ ]:
# @title ▶ Part 8: Audit Security Configuration { display-mode: "form" }

deployment_config = {
    "debug_mode": True,
    "public_storage": True,
    "logging_enabled": False,
    "tls_enabled": True,
    "admin_interface_public": True,
    "authentication_required": True
}


secure_requirements = {
    "debug_mode": False,
    "public_storage": False,
    "logging_enabled": True,
    "tls_enabled": True,
    "admin_interface_public": False,
    "authentication_required": True
}


print("=" * 70)
print("CONFIGURATION SECURITY AUDIT")
print("=" * 70)

problems = 0

for setting, expected in secure_requirements.items():

    actual = deployment_config[
        setting
    ]

    if actual == expected:

        status = "✓ SECURE"

    else:

        status = "⚠ MISCONFIGURED"
        problems += 1

    print(
        f"\n{setting:<28}"
        f"Current: {str(actual):<6} "
        f"Expected: {str(expected):<6} "
        f"{status}"
    )


print(
    f"\nConfiguration problems detected: "
    f"{problems}"
)

<details>

<summary><b>Part 8 Explanation — Click to Expand</b></summary>

<br>

The audit compares the deployed configuration against an approved secure configuration.

It detects several problems:

    DEBUG MODE ENABLED

    PUBLIC STORAGE ENABLED

    LOGGING DISABLED

    PUBLIC ADMIN INTERFACE

None of these problems require attacking the AI model.

They result from insecure infrastructure configuration.

This demonstrates why secure deployment should define an expected configuration and automatically compare deployed systems against that standard.

A secure default means the system begins from the safer state rather than depending on someone to remember to enable security later.

</details>

In [ ]:
# @title ▶ Part 9: Apply Secure Defaults { display-mode: "form" }

deployment_config = {
    "debug_mode": False,
    "public_storage": False,
    "logging_enabled": True,
    "tls_enabled": True,
    "admin_interface_public": False,
    "authentication_required": True
}


problems = 0

print("=" * 70)
print("SECURE CONFIGURATION VALIDATION")
print("=" * 70)

for setting, expected in secure_requirements.items():

    actual = deployment_config[
        setting
    ]

    if actual == expected:

        print(
            f"✓ {setting}"
        )

    else:

        print(
            f"⚠ {setting}"
        )

        problems += 1


print("\n" + "=" * 70)

if problems == 0:

    print(
        "✓ Deployment matches the "
        "approved secure configuration."
    )

else:

    print(
        f"⚠ {problems} configuration "
        "problem(s) remain."
    )

<details>

<summary><b>Part 9 Explanation — Click to Expand</b></summary>

<br>

The deployment now matches the expected secure configuration.

Instead of relying on a developer to manually inspect every setting, the configuration can be validated automatically.

This is important because production environments may contain many settings across:

    CLOUD SERVICES

    CONTAINERS

    STORAGE

    NETWORKS

    AI SERVICES

    LOGGING SYSTEMS

    ACCESS CONTROLS

Automated validation helps identify insecure settings before or after deployment.

</details>

In [ ]:
# @title ▶ Part 10: Detect Configuration Drift { display-mode: "form" }

# Simulate a change made after deployment
deployment_config[
    "public_storage"
] = True

deployment_config[
    "logging_enabled"
] = False


print("=" * 70)
print("CONFIGURATION DRIFT CHECK")
print("=" * 70)

drift_detected = []

for setting, expected in secure_requirements.items():

    actual = deployment_config[
        setting
    ]

    if actual != expected:

        drift_detected.append(
            setting
        )


if drift_detected:

    print(
        "\n⚠ CONFIGURATION DRIFT DETECTED"
    )

    for setting in drift_detected:
        print(f"  • {setting}")

else:

    print(
        "\n✓ No configuration drift detected."
    )

<details>

<summary><b>Part 10 Explanation — Click to Expand</b></summary>

<br>

The system was initially deployed securely.

Later, two settings changed:

    PUBLIC STORAGE ENABLED

    LOGGING DISABLED

The deployment now differs from the approved security baseline.

This is **configuration drift**.

The important lesson is that:

    SECURE AT DEPLOYMENT

does not automatically mean:

    SECURE FOREVER

Configurations can change because of:

    MANUAL ADMINISTRATOR CHANGES

    AUTOMATION

    SOFTWARE UPDATES

    TROUBLESHOOTING

    DEPLOYMENT ERRORS

    ATTACKER ACTIVITY

Continuous configuration validation can identify when the deployed environment no longer matches the approved secure state.

</details>

In [ ]:
# @title ▶ Part 11: Credential Exposure Response { display-mode: "form" }

incident = {
    "credential":
        "prod/model-api-key",

    "exposed":
        True,

    "revoked":
        False,

    "rotated":
        False,

    "logs_reviewed":
        False,

    "root_cause_fixed":
        False
}


print("=" * 70)
print("CREDENTIAL INCIDENT RESPONSE")
print("=" * 70)

if incident["exposed"]:

    print(
        "\n⚠ Credential exposure detected."
    )

    # Step 1
    incident["revoked"] = True
    print("✓ Old credential revoked")

    # Step 2
    incident["rotated"] = True
    print("✓ Replacement credential created")

    # Step 3
    incident["logs_reviewed"] = True
    print("✓ Access logs reviewed")

    # Step 4
    incident["root_cause_fixed"] = True
    print("✓ Exposure source corrected")


print("\nIncident Status:")

for step, complete in incident.items():

    if step == "credential":
        continue

    print(
        f"{step:<20} {complete}"
    )

<details>

<summary><b>Part 11 Explanation — Click to Expand</b></summary>

<br>

Finding an exposed credential requires more than removing it from a file.

The simulated response followed:

    DETECT EXPOSURE
          ↓
    REVOKE OLD CREDENTIAL
          ↓
    CREATE REPLACEMENT
          ↓
    REVIEW ACCESS LOGS
          ↓
    FIX ROOT CAUSE

Reviewing logs is important because defenders should determine whether the exposed credential was actually used.

Fixing the root cause is also important.

If the credential leaked because developers were placing secrets inside source code, simply replacing the credential without changing that practice may cause the same problem again.

Incident response should therefore address both:

    IMMEDIATE EXPOSURE

and:

    WHY THE EXPOSURE OCCURRED

</details>

In [ ]:
# @title ▶ Part 12: Knowledge Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. What was wrong with the original application configuration?",
        "options": [
            "A. Secrets were stored directly with application configuration",
            "B. It used too many short-lived credentials",
            "C. It had no configuration values",
            "D. All storage was private"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. Why was deleting the API key from the newest simulated Git commit insufficient?",
        "options": [
            "A. The secret still appeared in earlier history",
            "B. Git automatically changes passwords",
            "C. The model required the old key",
            "D. Environment variables cannot contain secrets"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. What did the secret manager add?",
        "options": [
            "A. Controlled access to stored secrets",
            "B. Public access to every credential",
            "C. Unlimited administrator privileges",
            "D. Model training"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Why rotate an exposed credential?",
        "options": [
            "A. To invalidate the old credential and replace it",
            "B. To make the leaked credential permanent",
            "C. To disable authentication",
            "D. To publish the credential"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. What advantage did the short-lived credential provide?",
        "options": [
            "A. A smaller useful window if stolen",
            "B. Permanent access",
            "C. Unlimited permissions",
            "D. No expiration"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. What did the configuration audit detect?",
        "options": [
            "A. Settings that differed from the approved secure configuration",
            "B. Model accuracy",
            "C. Training labels",
            "D. GPU temperature only"
        ],
        "answer": "A"
    },
    {
        "question":
        "7. What caused configuration drift in the simulation?",
        "options": [
            "A. Security-related settings changed after deployment",
            "B. The model became more accurate",
            "C. The API key expired",
            "D. The user logged out"
        ],
        "answer": "A"
    },
    {
        "question":
        "8. What should happen after detecting an exposed credential?",
        "options": [
            "A. Revoke or rotate it, investigate usage, and fix the exposure source",
            "B. Keep using it",
            "C. Disable monitoring",
            "D. Share it with additional services"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("LAB 4 — KNOWLEDGE CHECK")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. "
            f"Correct answer: {q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 70)
print(f"Lab Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Lab complete!")
else:
    print(
        "△ Review the lab explanations "
        "and try again."
    )

<details>

<summary><b>Lab 4 — Practical Takeaways — Click to Expand</b></summary>

<br>

## Lab 4 — Practical Takeaways

This lab demonstrated two closely related parts of deployment security:

    SECRET SECURITY

and:

    CONFIGURATION SECURITY

You began with an application containing credentials directly alongside normal configuration.

This created a path such as:

    SOURCE CODE
        ↓
    EXPOSED SECRET
        ↓
    ATTACKER OBTAINS CREDENTIAL
        ↓
    ACCESS TO AI RESOURCE

You then separated credentials from application code and simulated a secret-management system that controlled which workload could retrieve each secret.

You also rotated an exposed credential and examined how short-lived credentials can reduce the amount of time a stolen credential remains useful.

The second half of the lab examined configuration security.

You compared:

    DEPLOYED CONFIGURATION

against:

    APPROVED SECURE CONFIGURATION

and detected insecure settings.

After correcting them, you simulated configuration drift and discovered that a previously secure deployment had changed over time.

This creates an important operational cycle:

    DEFINE SECURE CONFIGURATION
              ↓
           DEPLOY
              ↓
           VALIDATE
              ↓
           MONITOR
              ↓
        DETECT DRIFT
              ↓
          CORRECT

The central lesson is:

> **A secure AI model can still be compromised by insecure credentials or infrastructure configuration.**

Secrets should be protected throughout their lifecycle, and deployment configurations should be continuously validated rather than assumed to remain secure after initial deployment.

</details>

In [ ]:
# @title ▶ Section 4 Checkpoint — Secrets, Credentials & Configuration Security { display-mode: "form" }

questions = [
    {
        "question":
        "1. Why are secrets important to AI system security?",
        "options": [
            "A. They may provide access to models, databases, cloud services, and other resources",
            "B. They only affect model accuracy",
            "C. They are only used during training",
            "D. They are always public"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. What is a major problem with hardcoded credentials?",
        "options": [
            "A. Anyone with access to the code may be able to obtain them",
            "B. They automatically expire",
            "C. They cannot access services",
            "D. They prevent source-code sharing"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. A secret was committed to Git and later deleted from the current file. What should happen?",
        "options": [
            "A. Assume the secret may still be exposed and rotate or revoke it",
            "B. Nothing, because deleting the line completely solves the problem",
            "C. Make the repository public",
            "D. Reuse the same credential elsewhere"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Why can secret-management systems be safer than hardcoding secrets?",
        "options": [
            "A. Secrets can be stored and accessed separately from application code",
            "B. They make every secret public",
            "C. They eliminate authorization",
            "D. They permanently embed secrets into containers"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. What is one advantage of short-lived credentials?",
        "options": [
            "A. A stolen credential has a smaller useful lifetime",
            "B. They never expire",
            "C. They provide unlimited permissions",
            "D. They eliminate authentication"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. What is configuration drift?",
        "options": [
            "A. A deployed configuration gradually differs from the approved secure state",
            "B. A model becoming more accurate",
            "C. A credential being encrypted",
            "D. A user logging out"
        ],
        "answer": "A"
    },
    {
        "question":
        "7. What should generally happen after a credential is confirmed exposed?",
        "options": [
            "A. Revoke or rotate it and investigate the exposure",
            "B. Continue using it indefinitely",
            "C. Publish it for other developers",
            "D. Disable security logging"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("SECTION 4 CHECKPOINT")
print("SECRETS, CREDENTIALS, AND CONFIGURATION SECURITY")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 70)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Checkpoint complete!")
else:
    print("△ Review Section 4 and try again.")

---

<details>
<summary><b>Section 5 — Container and Runtime Security — Click to Expand</b></summary>
<br>

## Section 5 — Container and Runtime Security

Modern AI applications are frequently packaged and deployed using **containers**. A container can package the inference application, machine learning framework, supporting libraries, and other dependencies into a portable environment that can run consistently across development, testing, and production systems.

Containers provide important deployment benefits, but they also introduce security considerations. Vulnerable base images, outdated dependencies, excessive privileges, malicious images, exposed secrets, and insecure runtime settings can all place an AI workload at risk.

A secure model running inside an insecure container is still part of an insecure AI system.

This section examines how containers are used in AI deployments, where container security risks originate, and how techniques such as minimal images, vulnerability scanning, least privilege, runtime isolation, resource restrictions, trusted registries, and regular patching can reduce those risks.

<br>

<details>
<summary><b>5.1 Why AI Systems Use Containers</b></summary>
<br>

### Why AI Systems Use Containers

Machine learning applications often depend on more than a single model file.

An inference application may require:

- Python
- Machine learning frameworks
- Model-serving libraries
- Application code
- Operating-system libraries
- Data-processing libraries
- API frameworks
- Configuration information
- Supporting dependencies

For example, an application may work correctly on a developer's computer because that computer contains a specific version of Python, TensorFlow, PyTorch, scikit-learn, or another dependency.

Moving the application to another server may produce unexpected problems if the environment is different.

Containers help address this problem by packaging the application and its dependencies together.

A simplified AI container might contain:

**Container**

- Inference application
- ML framework
- Python libraries
- System libraries
- Model-serving code

The container can then be moved between compatible environments while maintaining a more consistent runtime configuration.

This makes containers useful for:

- AI inference services
- Training workloads
- Data-processing jobs
- Model evaluation
- Microservices
- Cloud deployment
- CI/CD pipelines

However, packaging dependencies together also means vulnerabilities in those dependencies may become part of the deployed workload.

The container therefore becomes another component of the AI system's attack surface.

</details>

<br>

<details>
<summary><b>5.2 Containers vs. Virtual Machines</b></summary>
<br>

### Containers vs. Virtual Machines

Containers and virtual machines both provide ways to isolate workloads, but they do not operate in exactly the same way.

A virtual machine commonly contains:

**Application**

↓

**Libraries**

↓

**Guest Operating System**

↓

**Hypervisor**

↓

**Host**

Containers typically share the host operating system's kernel:

**Container Application**

↓

**Container Libraries**

↓

**Shared Host Kernel**

↓

**Host**

This difference is important for security.

A container may appear to the application like an isolated environment, but it is not simply a smaller virtual machine.

Combe, Martin, and Di Pietro examined Docker from a security perspective and highlighted important architectural differences between containerized and virtualized environments [17].

Because containers share the host kernel, isolation mechanisms must prevent one container from improperly interacting with other containers or the host.

This creates an important security principle:

**Containerization provides isolation, but containers should not automatically be treated as perfect security boundaries.**

Secure deployment therefore requires both:

**Secure Container Image**

and

**Secure Container Runtime Configuration**

</details>

<br>

<details>
<summary><b>5.3 Understanding the Container Attack Surface</b></summary>
<br>

### Understanding the Container Attack Surface

A containerized AI workload introduces several components that attackers may attempt to target.

A simplified deployment might look like:

**Container Registry**

↓

**Container Image**

↓

**Container Runtime**

↓

**AI Inference Application**

↓

**Production Model**

Security weaknesses can appear at each stage.

### Container Registry

An attacker may attempt to distribute or substitute an untrusted image.

### Container Image

The image may contain vulnerable packages, malicious software, embedded secrets, or outdated dependencies.

### Container Runtime

The container may run with unnecessary privileges or insecure isolation settings.

### Application

The inference application itself may contain software vulnerabilities.

### Host

A serious container compromise may potentially affect the underlying system depending on the vulnerability and runtime configuration.

The container attack surface therefore extends beyond the code written by the AI developer.

A container may contain hundreds of software packages that were indirectly included through a base image or dependency.

Research examining Docker Hub images has repeatedly identified vulnerabilities and insecure components within publicly available container images [18]–[21].

For defenders, the important question becomes:

**What exactly are we deploying along with the model?**

</details>

<br>

<details>
<summary><b>5.4 Container Images and Base Images</b></summary>
<br>

### Container Images and Base Images

Containers are normally created from **container images**.

An image provides the files, libraries, packages, and configuration needed to start the container.

Images are often built from another image called a **base image**.

For example:

`FROM python:3.12`

The application developer may then add:

- Application code
- Python packages
- ML libraries
- Model files
- Configuration

This creates an inheritance relationship:

**Base Image**

↓

**Operating-System Packages**

↓

**Python Runtime**

↓

**ML Dependencies**

↓

**Application**

↓

**Final AI Image**

A vulnerability in the base image may therefore become a vulnerability in every application image built from it.

Haque and Babar examined vulnerabilities inherited through container base images and demonstrated how base-image security can affect large numbers of downstream containerized applications [21].

This means developers should not only ask:

**"Is my application code secure?"**

They should also ask:

**"What did my application inherit from its base image?"**

Base images should therefore be selected carefully, maintained regularly, and updated when security fixes become available.

</details>

<br>

<details>
<summary><b>5.5 Vulnerable and Malicious Container Images</b></summary>
<br>

### Vulnerable and Malicious Container Images

Container registries allow developers to quickly obtain images created by other organizations and community members.

This convenience creates a supply-chain trust problem.

A downloaded image may contain:

- Known software vulnerabilities
- Outdated packages
- Unnecessary services
- Insecure configuration
- Embedded secrets
- Malicious software

Research into Docker Hub has demonstrated that publicly available images can contain significant security risks.

Studies have identified vulnerable packages across large collections of container images [18]–[21].

Research into Docker Hub security has also identified malicious images associated with activities such as remote code execution and cryptocurrency mining [18].

This creates a simple but important lesson:

**Available for download ≠ Trusted**

and:

**Popular ≠ Secure**

AI developers may be particularly tempted to download preconfigured images because installing GPU libraries, ML frameworks, and other dependencies can be complicated.

However, convenience should not replace verification.

Before deploying a third-party image, organizations should consider:

- Who created the image?
- Is the publisher trusted?
- When was it last updated?
- Does it contain known vulnerabilities?
- Is the image actually the expected artifact?
- Are unnecessary packages included?
- Can its origin be verified?

Container images should be treated as **software supply-chain artifacts**.

</details>

<br>

<details>
<summary><b>5.6 Building Minimal AI Containers</b></summary>
<br>

### Building Minimal AI Containers

A container should contain the software required to perform its intended function without unnecessarily including unrelated tools and packages.

Consider an inference service that only needs:

- Python
- An API framework
- A machine learning library
- The inference application
- The production model

The container probably does not need:

- Compilers
- Development tools
- Text editors
- Debugging utilities
- Unused network tools
- Unnecessary package managers
- Unrelated software libraries

Every additional package can potentially introduce additional vulnerabilities.

This leads to the principle of **container minimization**.

Instead of:

**Large General-Purpose Image**

↓

**Hundreds of Packages**

↓

**Large Attack Surface**

prefer:

**Minimal Base Image**

↓

**Required Dependencies**

↓

**AI Application**

↓

**Smaller Attack Surface**

Glatard et al. analyzed vulnerabilities in container images used for scientific data analysis and found that updating packages and removing unnecessary components could significantly reduce vulnerability exposure [20].

This is particularly relevant to AI and data-science environments, where convenience images may include many packages that are useful during development but unnecessary during production inference.

A useful distinction is:

**Development Container → Tools Needed to Build and Debug**

**Production Container → Only What Is Needed to Run**

Reducing unnecessary software does not guarantee security, but it reduces the number of components that must be maintained and protected.

</details>

<br>

<details>
<summary><b>5.7 Container Vulnerability Scanning</b></summary>
<br>

### Container Vulnerability Scanning

Because container images may contain many packages, manually reviewing every dependency is often impractical.

**Container vulnerability scanners** can examine images for software components associated with known vulnerabilities.

A simplified deployment pipeline might use:

**Build Image**

↓

**Scan Image**

↓

**Known Critical Vulnerabilities?**

**Yes → Block / Review**

**No → Continue**

↓

**Deploy**

Scanning can help identify vulnerabilities in:

- Operating-system packages
- Language libraries
- Application dependencies
- Base images

Research examining thousands of Docker images has found significant numbers of known vulnerabilities within publicly available images [18]–[21].

This makes vulnerability scanning an important security gate before production deployment.

However, scanning has limitations.

A scanner may:

- Miss newly discovered vulnerabilities
- Produce false positives
- Disagree with another scanner
- Identify a vulnerability that is not exploitable in the specific configuration

Therefore:

**Scanner Result ≠ Complete Security Assessment**

Scanning should be treated as one part of defense in depth.

It can help answer:

**"What known vulnerabilities are present in this artifact?"**

but additional controls are still needed to determine whether the container is securely configured and operated.

</details>

<br>

<details>
<summary><b>5.8 Running Containers with Least Privilege</b></summary>
<br>

### Running Containers with Least Privilege

The principle of least privilege from Section 3 also applies to containers.

A container should receive only the privileges necessary to perform its intended function.

Consider an AI inference container.

Its legitimate responsibilities may be:

- Read the production model
- Receive inference requests
- Execute predictions
- Write approved logs

It probably does not need permission to:

- Modify host operating-system files
- Control other containers
- Change host network settings
- Mount arbitrary host directories
- Access unrelated secrets
- Install kernel modules

A poorly configured container may receive far more privileges than necessary.

If the inference application becomes compromised, the attacker may inherit those privileges.

This creates:

**Application Compromise**

↓

**Container Privileges**

↓

**Potential Additional Access**

Least privilege reduces this opportunity.

A hardened container should therefore limit:

- Filesystem access
- Linux capabilities
- Mounted directories
- Device access
- Network access
- Secret access
- Runtime permissions

The same security question introduced earlier applies:

**If this container becomes compromised, what can the attacker do next?**

</details>

<br>

<details>
<summary><b>5.9 Root vs. Non-Root Containers</b></summary>
<br>

### Root vs. Non-Root Containers

Processes inside containers may execute as different users.

A common security mistake is allowing an application to run as the **root user** when root privileges are unnecessary.

Consider:

**AI Application → Running as Root**

If the application contains a vulnerability and an attacker gains control of the process, the attacker may inherit powerful privileges inside the container.

A stronger configuration is:

**AI Application → Dedicated Non-Root User**

The application receives only the permissions needed to execute inference.

For example, the application may require permission to:

- Read the model
- Read required configuration
- Bind to an appropriate application port
- Write to a designated logging location

It does not necessarily need unrestricted administrative control.

Running as a non-root user is not a complete defense against container compromise.

However, it provides another layer of protection by reducing the privileges available to a compromised application.

This follows the same principle introduced in Section 3:

**Do not provide a workload with more authority than its legitimate function requires.**

</details>

<br>

<details>
<summary><b>5.10 Runtime Isolation and Container Escape</b></summary>
<br>

### Runtime Isolation and Container Escape

Containers are intended to isolate workloads from one another and from the host system.

However, vulnerabilities or insecure configurations can weaken this isolation.

A **container escape** occurs when activity inside a container breaks through the intended isolation boundary and gains unauthorized access to the host or other resources outside the container.

A simplified security boundary is:

**Attacker**

↓

**Compromised AI Application**

↓

**Container**

↓

**X Isolation Boundary X**

↓

**Host System**

Secure configuration attempts to prevent the attacker from crossing that boundary.

Risk may increase when containers receive capabilities such as:

- Privileged execution
- Sensitive host-directory mounts
- Direct access to container-runtime sockets
- Excessive Linux capabilities
- Unnecessary device access

Container-security research emphasizes that container isolation depends on multiple operating-system and runtime mechanisms rather than a completely separate guest operating system [17].

This is why production AI containers should avoid unnecessarily powerful runtime configurations.

The objective is to make compromise of the AI application remain:

**Inside the Container**

rather than becoming:

**AI Application → Container → Host → Other Workloads**

</details>

<br>

<details>
<summary><b>5.11 Resource Limits</b></summary>
<br>

### Resource Limits

AI workloads can consume significant computational resources.

Depending on the model, inference may require:

- CPU
- GPU
- Memory
- Storage
- Network bandwidth

If a container has unrestricted access to resources, an overloaded or compromised application may consume enough resources to affect other workloads.

Consider two AI services sharing a host.

Without resource controls:

**Container A → Consumes Most Available Memory**

↓

**Container B → Performance Degradation or Failure**

Resource limits can restrict how much of a resource a container is permitted to consume.

Examples include:

- CPU limits
- Memory limits
- GPU allocation
- Process limits
- Storage quotas

These controls complement the API rate limiting introduced in Section 2.

Rate limiting controls:

**How many requests reach the application**

while container resource limits control:

**How many infrastructure resources the workload can consume**

Together, they provide stronger protection against resource-exhaustion scenarios.

This demonstrates defense in depth:

**API Rate Limit + Container Resource Limit + Monitoring**

is stronger than relying on any one of these controls by itself.

</details>

<br>

<details>
<summary><b>5.12 Container Registries and Image Trust</b></summary>
<br>

### Container Registries and Image Trust

Container images are commonly stored in **container registries**.

A deployment process may look like:

**Developer**

↓

**Build Container Image**

↓

**Container Registry**

↓

**Production Deployment**

The registry therefore becomes an important part of the software supply chain.

If an attacker can modify or replace an image stored in the registry, the deployment system may distribute the attacker's artifact to production.

Organizations should therefore consider:

- Who can upload images?
- Who can modify image tags?
- Who can delete images?
- Who can retrieve production images?
- Is access logged?
- Are images scanned?
- Can image origin and integrity be verified?
- Are trusted registries required?

Production systems should avoid automatically trusting an image simply because it has a familiar name.

For example:

`ai-inference:latest`

is a human-readable label.

It does not by itself prove that the underlying image is the same artifact that was previously reviewed.

This concept will become especially important in Section 7 when model and deployment artifact integrity are examined in greater detail.

The key principle is:

**Deployment systems should verify what they are deploying rather than trusting names alone.**

</details>

<br>

<details>
<summary><b>5.13 Patching and Updating AI Containers</b></summary>
<br>

### Patching and Updating AI Containers

A container that is secure when created may not remain secure forever.

New vulnerabilities are continuously discovered in:

- Operating systems
- Python packages
- ML frameworks
- API frameworks
- System libraries
- Container runtimes
- Supporting dependencies

Research examining scientific container images found that software updates could remove a substantial portion of known vulnerabilities [20].

This means container security requires ongoing maintenance.

A simplified lifecycle is:

**Build**

↓

**Scan**

↓

**Test**

↓

**Deploy**

↓

**Monitor for New Vulnerabilities**

↓

**Update**

↓

**Rebuild**

↓

**Redeploy**

A common mistake is assuming that a running container will somehow become patched automatically because an upstream image was updated.

If the production image contains vulnerable packages, the organization may need to rebuild and redeploy the workload using updated components.

This is one reason reproducible deployment pipelines are important.

Instead of manually modifying production containers, organizations can rebuild tested artifacts from controlled source configurations.

Patching must also be balanced with model and application compatibility. Updating a machine learning framework or dependency could potentially affect application behavior.

Updates should therefore be:

**Identified → Tested → Verified → Deployed**

rather than ignored indefinitely or pushed blindly into production.

</details>

<br>

<details>
<summary><b>5.14 Connecting Container Security to AI Security</b></summary>
<br>

### Connecting Container Security to AI Security

Container vulnerabilities are traditional infrastructure problems, but compromising a container can provide access to AI-specific assets.

Consider an inference container containing:

- Inference application
- Machine learning framework
- Production model
- API configuration
- Service identity
- Logging connection

If an attacker compromises the container, the attacker may attempt to:

- Steal the model
- Modify the model
- Access service credentials
- Manipulate inference behavior
- Intercept model inputs
- Observe predictions
- Abuse computing resources
- Move toward other infrastructure

For example:

**Vulnerable Library**

↓

**Inference Application Compromised**

↓

**Container Access**

↓

**Production Model Exposed**

A different configuration might allow:

**Compromised Container**

↓

**Stolen Service Credential**

↓

**Model Storage Access**

↓

**Additional AI Assets**

The model itself did not necessarily contain a vulnerability.

Instead, the infrastructure used to execute the model became the attack path.

This reinforces a recurring principle from Module 6:

**Protecting the model requires protecting the environment that executes the model.**

</details>

<br>

<details>
<summary><b>Section 5 Summary</b></summary>
<br>

### Section 5 Summary

Containers provide a portable and consistent way to package and deploy AI applications, but they also introduce additional software, configuration, and runtime components that must be secured.

Throughout this section, you learned that:

- Containers package AI applications and their dependencies into portable runtime environments.
- Containers and virtual machines use different isolation models.
- Containers commonly share the host operating-system kernel.
- Container images may inherit vulnerabilities from their base images.
- Publicly available images may contain vulnerable, outdated, or malicious components.
- Image popularity should not be treated as evidence of security.
- Minimal production images reduce unnecessary software and attack surface.
- Vulnerability scanning can identify known security issues before deployment.
- Containers should receive only the privileges required for their intended function.
- AI applications should run as non-root users when administrative privileges are unnecessary.
- Runtime isolation helps prevent a compromised container from affecting the host or other workloads.
- Resource limits can reduce the impact of resource-exhaustion attacks.
- Container registries should be treated as important software supply-chain components.
- Containers must be continuously monitored, updated, rebuilt, and redeployed as vulnerabilities are discovered.
- A compromised container can expose models, credentials, inputs, outputs, and other AI resources even when the model itself is secure.

The next section examines **Network and Cloud Infrastructure Security**, where you will explore how segmentation, firewalls, ingress and egress controls, secure service communication, zero-trust architecture, and cloud security controls can prevent a compromised AI workload from reaching the rest of the environment.

</details>

</details>

In [ ]:

# @title Section 5 — Container and Runtime Security Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import ListedColormap
from IPython.display import display, HTML
from io import BytesIO
import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"
TEAL = "#0891B2"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):
    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")
    buffer.close()

    display(HTML(f"""
    <details style="margin:12px 0;border:1px solid #cbd5e1;
                    border-radius:10px;overflow:hidden;">
      <summary style="cursor:pointer;padding:16px;font-size:16px;
                      font-weight:bold;background:#eff6ff;
                      color:#1e3a8a;border-left:5px solid {color};">
        {html.escape(title)} — Click to Expand
      </summary>
      <div style="padding:15px;background:white;text-align:center;">
        <img src="data:image/png;base64,{encoded}"
             style="width:100%;max-width:950px;height:auto;">
        <div style="max-width:850px;margin:15px auto;padding:16px;
                    background:#f8fafc;border-radius:8px;color:#334155;
                    font-size:14px;line-height:1.7;text-align:left;">
          {explanation}
        </div>
      </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def setup_diagram(ax, title, xlim=(0, 12), ylim=(0, 10)):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.axis("off")
    ax.set_title(title, fontsize=15, fontweight="bold", pad=18)


def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC", title_size=10):
    card = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.8
    )
    ax.add_patch(card)

    ax.text(
        x+w/2, y+h*(0.66 if subtitle else 0.5),
        title, ha="center", va="center",
        fontsize=title_size, fontweight="bold", color=DARK
    )

    if subtitle:
        ax.text(
            x+w/2, y+h*0.28, subtitle,
            ha="center", va="center",
            fontsize=8.5, color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):
    ax.add_patch(FancyArrowPatch(
        start, end, arrowstyle="-|>",
        mutation_scale=16, linewidth=1.8, color=color
    ))


# ============================================================
# VISUALIZATION 1
# CONTAINERS VS VIRTUAL MACHINES
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
fig.suptitle(
    "Container and Virtual Machine Architecture",
    fontsize=16, fontweight="bold"
)

ax = axes[0]
setup_diagram(ax, "Virtual Machine", (0, 6), (0, 10))

vm_layers = [
    ("AI Inference Application", "Model-serving application", BLUE),
    ("Application Libraries", "Python and ML dependencies", PURPLE),
    ("Guest Operating System", "Independent guest OS", ORANGE),
    ("Hypervisor", "Virtualization layer", TEAL),
    ("Host Hardware", "CPU, memory, storage", GRAY)
]

for i, (title, subtitle, color) in enumerate(vm_layers):
    y = 8.4 - i*1.65
    add_card(ax, 0.5, y, 5.0, 1.15, title, subtitle, color)

ax = axes[1]
setup_diagram(ax, "Container", (0, 6), (0, 10))

container_layers = [
    ("AI Inference Application", "Model-serving application", BLUE),
    ("Application Libraries", "Packaged dependencies", PURPLE),
    ("Container Runtime", "Container management", TEAL),
    ("Shared Host Kernel", "OS kernel shared with host", ORANGE),
    ("Host Hardware", "CPU, memory, storage", GRAY)
]

for i, (title, subtitle, color) in enumerate(container_layers):
    y = 8.4 - i*1.65
    add_card(ax, 0.5, y, 5.0, 1.15, title, subtitle, color)

plt.tight_layout()

show_collapsible_figure(
    "5.1 — Containers vs. Virtual Machines",
    fig,
    """
    Containers and virtual machines both provide
    environments for running AI applications,
    but they differ in how they isolate workloads.

    <br><br>

    A <b>virtual machine</b> typically runs a
    guest operating system above a hypervisor.
    The guest OS provides a separate operating
    system environment for its applications.

    <br><br>

    A <b>container</b> packages the application,
    machine learning libraries, and supporting
    dependencies while sharing the host's
    operating system kernel.

    <br><br>

    This makes containers convenient for
    deploying AI inference services consistently
    across compatible environments.

    <br><br>

    For example, a container may package a
    Python application, an API framework,
    PyTorch or scikit-learn, and a model artifact.

    <br><br>

    However, sharing the host kernel introduces
    important security considerations.
    A kernel vulnerability or insecure runtime
    configuration may affect container isolation.

    <br><br>

    Virtual machines and containers are not
    interchangeable security boundaries.
    Both require appropriate hardening,
    maintenance, and access controls.

    <br><br>

    The diagrams are simplified architecture
    representations rather than complete
    descriptions of every virtualization technology.

    <br><br>

    <b>Key takeaway:</b> Containers simplify
    AI deployment, but container isolation
    should not be assumed to provide
    complete protection from the host
    or other workloads.
    """,
    BLUE
)


# ============================================================
# VISUALIZATION 2
# CONTAINER ATTACK SURFACE AND IMAGE DEPENDENCIES
# ============================================================

fig, ax = plt.subplots(figsize=(12, 10))

setup_diagram(
    ax,
    "Containerized AI Workload — Supply Chain and Attack Surface",
    xlim=(0, 12), ylim=(0, 12)
)

stages = [
    ("Container Registry", "Image source and publisher trust", PURPLE),
    ("Base Image", "Inherited OS packages and libraries", ORANGE),
    ("AI Dependencies", "ML frameworks and Python packages", BLUE),
    ("Final Container Image", "Application, model, and configuration", TEAL),
    ("Container Runtime", "Privileges, mounts, and isolation", RED),
    ("Production AI Service", "Inference API and model execution", GREEN)
]

for i, (title, subtitle, color) in enumerate(stages):
    y = 10.2 - i*1.7
    add_card(ax, 2.1, y, 7.8, 1.15, title, subtitle, color)

    if i < len(stages)-1:
        add_arrow(
            ax, (6.0, y-0.05),
            (6.0, y-0.5)
        )

ax.text(
    0.8, 6.0,
    "Supply-chain\nrisks",
    ha="center", fontsize=10,
    fontweight="bold", color=ORANGE
)

ax.text(
    11.1, 2.8,
    "Runtime\nrisks",
    ha="center", fontsize=10,
    fontweight="bold", color=RED
)

plt.tight_layout()

show_collapsible_figure(
    "5.2 — Container Attack Surface and Image Dependencies",
    fig,
    """
    The attack surface of a containerized
    AI application extends beyond the
    model and inference code.

    <br><br>

    A container image may inherit operating
    system packages from a base image and
    include additional Python libraries,
    machine learning frameworks, and
    application dependencies.

    <br><br>

    Each dependency may contain known
    vulnerabilities or become outdated
    after the image is built.

    <br><br>

    The container registry is another
    important security component.
    An attacker may attempt to distribute
    a malicious image or substitute
    an expected artifact.

    <br><br>

    Once the image is deployed,
    runtime configuration introduces
    additional risks.

    <br><br>

    Examples include privileged execution,
    unrestricted host mounts, excessive
    Linux capabilities, and unnecessary
    network access.

    <br><br>

    The inference application may also
    contain ordinary software vulnerabilities
    or AI-specific weaknesses.

    <br><br>

    Defenders should therefore consider
    the origin of an image, the software
    it contains, its runtime privileges,
    and how it interacts with the host.

    <br><br>

    <b>Key takeaway:</b> Container security
    requires protecting the entire
    deployment chain, not just checking
    the application's source code.
    """,
    PURPLE
)


# ============================================================
# VISUALIZATION 3
# MINIMAL IMAGES AND VULNERABILITY SCANNING
# ============================================================

# Synthetic example package counts and scanner findings.
# These values are NOT measured vulnerabilities.

image_types = [
    "General-Purpose\nDevelopment Image",
    "Slim Production\nImage",
    "Minimal Production\nImage"
]

package_counts = np.array([320, 110, 35])
scanner_findings = np.array([42, 13, 4])

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

fig.suptitle(
    "Illustrative Container Minimization and Scanning",
    fontsize=16, fontweight="bold"
)

ax = axes[0]
bars = ax.bar(
    image_types,
    package_counts,
    color=[RED, ORANGE, GREEN]
)

ax.set_title("Installed Software Packages", fontweight="bold")
ax.set_ylabel("Package Count")
ax.grid(axis="y", alpha=0.2)

for bar, value in zip(bars, package_counts):
    ax.text(
        bar.get_x()+bar.get_width()/2,
        value+5, str(value),
        ha="center", fontweight="bold"
    )

ax = axes[1]
bars = ax.bar(
    image_types,
    scanner_findings,
    color=[RED, ORANGE, GREEN]
)

ax.set_title("Known Vulnerability Findings", fontweight="bold")
ax.set_ylabel("Illustrative Scanner Findings")
ax.grid(axis="y", alpha=0.2)

for bar, value in zip(bars, scanner_findings):
    ax.text(
        bar.get_x()+bar.get_width()/2,
        value+0.7, str(value),
        ha="center", fontweight="bold"
    )

plt.tight_layout()

show_collapsible_figure(
    "5.3 — Minimal Container Images and Vulnerability Scanning",
    fig,
    """
    Production AI containers should contain
    the software required to run their
    intended workload without unnecessarily
    including development tools.

    <br><br>

    A general-purpose development image may
    include compilers, editors, debugging
    utilities, package managers, and many
    unrelated libraries.

    <br><br>

    These tools may be useful during
    development but unnecessary when
    serving predictions in production.

    <br><br>

    Container minimization removes
    unnecessary components and reduces
    the amount of software that must
    be maintained and secured.

    <br><br>

    The first chart compares hypothetical
    package counts for three container images.

    <br><br>

    The second chart shows illustrative
    vulnerability scanner findings.
    The numbers are synthetic and do
    not represent actual image scans.

    <br><br>

    Fewer packages can reduce exposure,
    but image size alone does not prove
    that an image is secure.

    <br><br>

    Vulnerability scanners help identify
    packages associated with known security
    vulnerabilities. Their results should
    be reviewed according to severity,
    exploitability, and application context.

    <br><br>

    Scanners may miss unknown vulnerabilities
    or produce findings that are not exploitable
    in a particular deployment.

    <br><br>

    <b>Key takeaway:</b> Minimal images and
    vulnerability scanning are complementary
    defenses that help reduce and identify
    software supply-chain risks.
    """,
    GREEN
)


# ============================================================
# VISUALIZATION 4
# ROOT VS NON-ROOT AND LEAST PRIVILEGE
# ============================================================

permissions = [
    "Read Model",
    "Serve Predictions",
    "Write Approved Logs",
    "Modify Host Files",
    "Control Other Containers",
    "Access Unrelated Secrets"
]

# Conceptual permissions in two hypothetical deployments.
# Actual effective access depends on runtime configuration.

overprivileged = np.array([1, 1, 1, 1, 1, 1])
restricted = np.array([1, 1, 1, 0, 0, 0])

matrix = np.column_stack([overprivileged, restricted])

fig, ax = plt.subplots(figsize=(12, 8))

ax.imshow(
    matrix,
    cmap=ListedColormap(["#DCFCE7", "#FEE2E2"]),
    vmin=0, vmax=1, aspect="auto"
)

ax.set_xticks([0, 1])
ax.set_xticklabels([
    "Overprivileged Runtime",
    "Restricted Non-Root Runtime"
])

ax.set_yticks(range(len(permissions)))
ax.set_yticklabels(permissions)

ax.set_title(
    "Example Runtime Permissions for an AI Inference Container",
    fontsize=15, fontweight="bold", pad=18
)

for i in range(len(permissions)):
    for j in range(2):
        granted = matrix[i, j] == 1
        ax.text(
            j, i,
            "GRANTED" if granted else "DENIED",
            ha="center", va="center",
            fontsize=10, fontweight="bold",
            color=RED if granted and i >= 3 else (
                GREEN if granted else GRAY
            )
        )

ax.set_xticks(np.arange(-0.5, 2, 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(permissions), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", bottom=False, left=False)

plt.tight_layout()

show_collapsible_figure(
    "5.4 — Root vs. Non-Root Containers and Least Privilege",
    fig,
    """
    The principle of least privilege applies
    to containerized AI workloads just as
    it applies to users and service identities.

    <br><br>

    An inference container normally needs
    to read its approved model artifact,
    receive requests, generate predictions,
    and write permitted logs.

    <br><br>

    It generally does not need permission
    to modify host operating system files,
    control other containers, or retrieve
    unrelated secrets.

    <br><br>

    The matrix compares two hypothetical
    runtime permission policies.

    <br><br>

    The overprivileged example grants
    unnecessary capabilities that could
    increase the impact of an application
    compromise.

    <br><br>

    The restricted example allows the
    intended inference functions while
    denying unrelated administrative actions.

    <br><br>

    Running as a non-root user can reduce
    privileges inside the container.
    However, non-root execution alone
    does not enforce every restriction
    shown in the matrix.

    <br><br>

    Additional protections may include
    dropping Linux capabilities, avoiding
    privileged mode, restricting host
    mounts, using read-only filesystems,
    and enforcing appropriate security profiles.

    <br><br>

    <b>Key takeaway:</b> A compromised
    inference application should inherit
    as few permissions as possible.
    Container security depends on the
    complete runtime configuration,
    not simply whether the process
    runs as root or non-root.
    """,
    ORANGE
)


# ============================================================
# VISUALIZATION 5
# RUNTIME ISOLATION AND RESOURCE LIMITS
# ============================================================

# Synthetic CPU demand for two AI workloads.
time = np.arange(0, 30)

workload_a = np.array([
    20, 22, 25, 28, 32, 38, 45, 60, 78, 92,
    110, 125, 140, 130, 120, 100, 85, 70,
    60, 50, 45, 40, 35, 30, 28, 26, 25, 24, 22, 20
])

workload_b = np.array([
    18, 20, 19, 21, 22, 24, 23, 25, 24, 26,
    27, 26, 25, 24, 25, 23, 24, 22, 21, 22,
    23, 22, 21, 20, 21, 20, 19, 20, 18, 19
])

cpu_limit = 80
limited_a = np.minimum(workload_a, cpu_limit)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle(
    "Runtime Resource Isolation — Illustrative CPU Demand",
    fontsize=16, fontweight="bold"
)

ax = axes[0]
ax.plot(time, workload_a, color=RED, linewidth=2.3,
        label="Workload A Demand")
ax.plot(time, workload_b, color=BLUE, linewidth=2.3,
        label="Workload B Demand")
ax.set_title("Without a Workload-Specific Limit", fontweight="bold")
ax.set_xlabel("Time Step")
ax.set_ylabel("Illustrative CPU Units")
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

ax = axes[1]
ax.plot(time, limited_a, color=GREEN, linewidth=2.3,
        label="Workload A Allocated")
ax.plot(time, workload_b, color=BLUE, linewidth=2.3,
        label="Workload B Demand")
ax.plot(time, workload_a, color=RED, linestyle=":",
        alpha=0.45, label="Workload A Requested")
ax.axhline(cpu_limit, color=ORANGE, linestyle="--",
           label="Workload A CPU Limit")
ax.set_title("With a Workload-Specific CPU Limit", fontweight="bold")
ax.set_xlabel("Time Step")
ax.set_ylabel("Illustrative CPU Units")
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "5.5 — Runtime Isolation and Resource Limits",
    fig,
    """
    Container runtime security includes
    both isolation and resource management.

    <br><br>

    Isolation mechanisms help restrict
    how a container interacts with the
    host and other containers.

    <br><br>

    On Linux, these mechanisms commonly
    include namespaces, control groups,
    capability restrictions, and
    additional security policies.

    <br><br>

    Resource limits help prevent one
    workload from consuming an excessive
    share of available CPU or memory.

    <br><br>

    This is particularly important for
    AI applications because inference
    requests may require substantial
    computing resources.

    <br><br>

    The left chart shows synthetic
    resource demand from two workloads.
    Workload A experiences a large
    increase in CPU demand.

    <br><br>

    The right chart demonstrates a
    simplified resource cap that limits
    the CPU allocated to Workload A.

    <br><br>

    This is a conceptual allocation
    example rather than an actual
    container scheduling simulation.

    <br><br>

    Real CPU limits may throttle
    workloads rather than simply
    discarding excess demand.

    <br><br>

    Memory limits, process limits,
    timeouts, and workload quotas
    provide additional protections.

    <br><br>

    Resource restrictions do not
    prevent all denial-of-service
    attacks, and poor limit selection
    can affect legitimate performance.

    <br><br>

    <b>Key takeaway:</b> Runtime isolation
    limits access between workloads,
    while resource restrictions help
    contain the operational impact
    of excessive resource consumption.
    """,
    TEAL
)


# ============================================================
# VISUALIZATION 6
# TRUSTED REGISTRIES, PATCHING, AND SECURE DEPLOYMENT
# ============================================================

fig, ax = plt.subplots(figsize=(12, 11))

setup_diagram(
    ax,
    "Secure AI Container Deployment Lifecycle",
    xlim=(0, 12), ylim=(0, 13)
)

stages = [
    ("Trusted Image Source",
     "Verify publisher and artifact provenance", BLUE),
    ("Build Minimal Image",
     "Include only required dependencies", PURPLE),
    ("Scan and Verify",
     "Check vulnerabilities, secrets, and integrity", ORANGE),
    ("Security Gate",
     "Review findings and enforce policy", RED),
    ("Deploy with Restrictions",
     "Least privilege and runtime isolation", GREEN),
    ("Monitor and Patch",
     "Detect new issues and rebuild images", TEAL)
]

for i, (title, subtitle, color) in enumerate(stages):
    y = 11.3 - i*1.75

    add_card(
        ax, 2.0, y, 8.0, 1.2,
        title, subtitle, color
    )

    if i < len(stages)-1:
        add_arrow(
            ax,
            (6.0, y-0.05),
            (6.0, y-0.55)
        )

ax.text(
    10.7, 5.9,
    "Failed checks:\nBlock or review",
    ha="center", va="center",
    fontsize=9.5, fontweight="bold", color=RED
)

ax.text(
    6.0, 0.65,
    "Security is maintained through repeated verification and updates",
    ha="center", fontsize=10, fontweight="bold", color=DARK
)

plt.tight_layout()

show_collapsible_figure(
    "5.6 — Trusted Registries, Patching, and Secure Deployment",
    fig,
    """
    Container security should be maintained
    throughout the deployment lifecycle.

    <br><br>

    The process begins by selecting
    container images from trusted sources.

    <br><br>

    Developers should verify the
    expected publisher and artifact
    integrity rather than assuming
    an image is trustworthy because
    it is publicly available.

    <br><br>

    The image should contain only
    the software necessary for
    the intended AI workload.

    <br><br>

    Before deployment, automated
    security checks can identify
    known vulnerabilities, exposed
    secrets, and configuration problems.

    <br><br>

    A security gate evaluates whether
    the artifact satisfies the
    organization's deployment policy.

    <br><br>

    An image that fails required
    checks should be blocked
    or reviewed before deployment.

    <br><br>

    Once deployed, the container
    should run with restricted
    privileges and appropriate
    runtime isolation.

    <br><br>

    Security maintenance continues
    after deployment.

    <br><br>

    New vulnerabilities may be
    discovered in previously
    approved dependencies.

    <br><br>

    Organizations should monitor
    vulnerability information,
    update affected dependencies,
    rebuild images, and redeploy
    corrected artifacts.

    <br><br>

    Updating an image in a registry
    does not automatically update
    every running container.
    Workloads generally need an
    appropriate replacement or
    redeployment process.

    <br><br>

    <b>Key takeaway:</b> Secure
    container deployment is a
    continuous lifecycle involving
    image trust, vulnerability
    management, runtime hardening,
    monitoring, and patching.
    """,
    GREEN
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("Section 5 — Container and Runtime Security Visualizations")
print("---------------------------------------------------------")
print("Six collapsible visualizations generated.")

print()
print("Concepts Covered")
print("----------------")
print("1. Containers vs. Virtual Machines")
print("2. Container Attack Surface and Image Dependencies")
print("3. Minimal Images and Vulnerability Scanning")
print("4. Root vs. Non-Root Containers and Least Privilege")
print("5. Runtime Isolation and Resource Limits")
print("6. Trusted Registries, Patching, and Secure Deployment")

print()
print("All numerical examples are synthetic.")
print("No container runtime, external registry, or cloud service was used.")


<details>

<summary><b>Section 5 Lab — Click to Expand</b></summary>

<br>

A development team has containerized an AI inference service.

The container packages:

    PYTHON

    ML FRAMEWORK

    INFERENCE APPLICATION

    PRODUCTION MODEL

    SUPPORTING LIBRARIES

The application works correctly, but the container was originally created for development rather than production.

The development configuration contains unnecessary packages, runs as root, has broad runtime privileges, contains an embedded credential, has no resource limits, and uses an unverified image source.

Your task is to review and harden this container before production deployment.

You will:

    Inspect the container attack surface

    Identify unnecessary software

    Simulate vulnerability scanning

    Remove unnecessary packages

    Compare root and non-root execution

    Inspect dangerous runtime privileges

    Apply resource limits

    Verify image trust

    Compare insecure and hardened containers

    Simulate compromise and measure the blast radius

The goal is to demonstrate an important principle:

> **A secure model running inside an insecure container is still part of an insecure AI system.**

This lab simulates container security concepts using standard Python so that no Docker installation or large container downloads are required.

All parts should run almost instantly.

</details>

In [ ]:
# @title ▶ Part 1: Inspect the AI Container { display-mode: "form" }

container = {
    "base_image": "python:3.12",

    "packages": [
        "python",
        "flask",
        "scikit-learn",
        "gcc",
        "vim",
        "curl",
        "debugger",
        "model-server"
    ],

    "user": "root",

    "privileged": True,

    "host_mount": True,

    "runtime_socket": True,

    "embedded_secret": True,

    "cpu_limit": None,

    "memory_limit": None,

    "trusted_registry": False
}


print("=" * 70)
print("AI CONTAINER CONFIGURATION")
print("=" * 70)

for setting, value in container.items():

    print(
        f"\n{setting}: {value}"
    )

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

The container contains everything required to run the AI inference service.

However, it also contains several settings that should receive security attention.

Notice that the container:

    RUNS AS ROOT

    USES PRIVILEGED MODE

    MOUNTS HOST RESOURCES

    ACCESSES THE RUNTIME SOCKET

    CONTAINS AN EMBEDDED SECRET

    HAS NO CPU LIMIT

    HAS NO MEMORY LIMIT

    USES AN UNVERIFIED REGISTRY

The package list also contains development tools that may not be required during production inference.

The application may function correctly with this configuration, but functionality alone does not mean the container is securely configured.

</details>

In [ ]:
# @title ▶ Part 2: Map the Attack Surface { display-mode: "form" }

attack_surface = {
    "Base Image": [
        "Inherited vulnerabilities",
        "Outdated packages"
    ],

    "Installed Packages": [
        "Vulnerable dependencies",
        "Unnecessary software"
    ],

    "Runtime": [
        "Excessive privileges",
        "Host access"
    ],

    "Secrets": [
        "Credential exposure"
    ],

    "Registry": [
        "Untrusted or modified image"
    ],

    "Resources": [
        "CPU exhaustion",
        "Memory exhaustion"
    ]
}


print("=" * 70)
print("CONTAINER ATTACK SURFACE")
print("=" * 70)

for component, risks in attack_surface.items():

    print(f"\n{component}")

    for risk in risks:
        print(f"  • {risk}")

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

The container attack surface extends beyond the AI application itself.

A simplified path is:

    CONTAINER REGISTRY
          ↓
       BASE IMAGE
          ↓
       PACKAGES
          ↓
    AI APPLICATION
          ↓
       RUNTIME
          ↓
         HOST

Weaknesses can appear at each stage.

For example, the application developer may write secure code while still inheriting a vulnerable library from the base image.

Likewise, a secure image can still become dangerous if it is launched with excessive runtime privileges.

This is why Section 5 distinguishes between:

    SECURE IMAGE

and:

    SECURE RUNTIME CONFIGURATION

Both are necessary.

</details>

In [ ]:
# @title ▶ Part 3: Scan the Container { display-mode: "form" }

vulnerability_database = {
    "python": 0,
    "flask": 0,
    "scikit-learn": 0,
    "gcc": 2,
    "vim": 1,
    "curl": 1,
    "debugger": 2,
    "model-server": 0
}


total_vulnerabilities = 0

print("=" * 70)
print("SIMULATED CONTAINER SCAN")
print("=" * 70)

for package in container["packages"]:

    findings = vulnerability_database.get(
        package,
        0
    )

    total_vulnerabilities += findings

    status = (
        "✓ No findings"
        if findings == 0
        else f"⚠ {findings} finding(s)"
    )

    print(
        f"{package:<20} {status}"
    )


print(
    f"\nTotal simulated findings: "
    f"{total_vulnerabilities}"
)

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

This is a simplified simulation of container vulnerability scanning.

The scanner compares installed packages against a small simulated vulnerability database.

Several packages produced findings.

Notice that some of them are development utilities rather than software required for production inference.

Real container scanners examine operating-system packages, language dependencies, base-image components, and other software for known vulnerabilities.

However:

> **A scanner finding does not automatically mean a vulnerability is exploitable.**

Likewise:

> **No scanner findings does not prove that the container is completely secure.**

Scanning is one security control within a larger defense-in-depth strategy.

</details>

In [ ]:
# @title ▶ Part 4: Minimize the Container { display-mode: "form" }

required_packages = [
    "python",
    "flask",
    "scikit-learn",
    "model-server"
]

original_count = len(
    container["packages"]
)

minimal_count = len(
    required_packages
)

removed_packages = [
    package
    for package in container["packages"]
    if package not in required_packages
]


print("=" * 70)
print("CONTAINER MINIMIZATION")
print("=" * 70)

print("\nRemoved packages:")

for package in removed_packages:
    print(f"  • {package}")

print(
    f"\nOriginal packages: "
    f"{original_count}"
)

print(
    f"Production packages: "
    f"{minimal_count}"
)

reduction = (
    (original_count - minimal_count)
    / original_count
) * 100

print(
    f"Package reduction: "
    f"{reduction:.1f}%"
)

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

The development image contained tools such as:

    GCC

    VIM

    CURL

    DEBUGGER

These may be useful while developing or troubleshooting an application.

The production inference service does not require them for normal operation.

Removing unnecessary packages creates:

    FEWER SOFTWARE COMPONENTS
              ↓
    FEWER COMPONENTS TO PATCH
              ↓
    SMALLER ATTACK SURFACE

This illustrates the distinction from your reading:

    DEVELOPMENT CONTAINER
    → Tools needed to build and debug

    PRODUCTION CONTAINER
    → Only what is needed to run

Container minimization does not guarantee security.

It reduces unnecessary exposure.

</details>

In [ ]:
# @title ▶ Part 5: Rescan the Minimal Image { display-mode: "form" }

before_findings = sum(
    vulnerability_database.get(package, 0)
    for package in container["packages"]
)

after_findings = sum(
    vulnerability_database.get(package, 0)
    for package in required_packages
)


print("=" * 70)
print("SCAN COMPARISON")
print("=" * 70)

print(
    f"\nOriginal image findings: "
    f"{before_findings}"
)

print(
    f"Minimal image findings: "
    f"{after_findings}"
)


if after_findings < before_findings:

    reduction = (
        (before_findings - after_findings)
        / before_findings
    ) * 100

    print(
        f"\n✓ Findings reduced by "
        f"{reduction:.1f}%"
    )

else:

    print(
        "\nNo reduction detected."
    )

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

The minimal image produces fewer simulated vulnerability findings.

This happened because unnecessary vulnerable packages were removed.

The result demonstrates an important security relationship:

    MORE SOFTWARE
         ↓
    MORE COMPONENTS
         ↓
    MORE POTENTIAL VULNERABILITIES

compared with:

    REQUIRED SOFTWARE ONLY
             ↓
    SMALLER ATTACK SURFACE

In a real environment, required packages could still contain vulnerabilities.

Minimization therefore works alongside vulnerability scanning, patching, and other security controls.

</details>

In [ ]:
# @title ▶ Part 6: Compare Runtime Users { display-mode: "form" }

runtime_users = {
    "Root Container": {
        "user": "root",
        "admin_privileges": True
    },

    "Hardened Container": {
        "user": "ai-service",
        "admin_privileges": False
    }
}


print("=" * 70)
print("CONTAINER USER COMPARISON")
print("=" * 70)

for name, configuration in runtime_users.items():

    print(f"\n{name}")
    print(
        f"User: {configuration['user']}"
    )

    print(
        f"Administrative privileges: "
        f"{configuration['admin_privileges']}"
    )

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

The original AI application runs as:

    ROOT

The hardened version runs as:

    ai-service

The application only needs enough permission to read its model, receive requests, perform inference, and write approved logs.

It does not need unrestricted administrative privileges.

If the application becomes compromised:

    APPLICATION COMPROMISE
            ↓
    PROCESS PRIVILEGES

The attacker may inherit the permissions of the compromised process.

Running as a dedicated non-root user therefore reduces the authority available after compromise.

This is another application of the least-privilege principle introduced earlier in the module.

</details>

In [ ]:
# @title ▶ Part 7: Audit Runtime Privileges { display-mode: "form" }

runtime_settings = {
    "Privileged Mode":
        container["privileged"],

    "Host Directory Mount":
        container["host_mount"],

    "Runtime Socket Access":
        container["runtime_socket"],

    "Embedded Secret":
        container["embedded_secret"]
}


print("=" * 70)
print("RUNTIME SECURITY AUDIT")
print("=" * 70)

risk_count = 0

for setting, enabled in runtime_settings.items():

    if enabled:
        status = "⚠ RISK ENABLED"
        risk_count += 1
    else:
        status = "✓ RESTRICTED"

    print(
        f"{setting:<25} {status}"
    )


print(
    f"\nRuntime risks detected: "
    f"{risk_count}"
)

<details>

<summary><b>Part 7 Explanation — Click to Expand</b></summary>

<br>

The original container receives several powerful capabilities.

It runs in privileged mode, accesses host resources, and can interact with the runtime socket.

These settings can weaken container isolation.

The intended security boundary is:

    COMPROMISED APPLICATION
            ↓
        CONTAINER
            ↓
    X ISOLATION BOUNDARY X
            ↓
          HOST

The objective is to keep compromise contained inside the container.

Powerful runtime permissions may increase the attacker's opportunities to interact with the host or other workloads.

This is why containers should not automatically be treated as perfect security boundaries.

Their security depends heavily on runtime configuration.

</details>

In [ ]:
# @title ▶ Part 8: Harden Runtime Configuration { display-mode: "form" }

hardened_container = {
    "packages":
        required_packages,

    "user":
        "ai-service",

    "privileged":
        False,

    "host_mount":
        False,

    "runtime_socket":
        False,

    "embedded_secret":
        False,

    "cpu_limit":
        2,

    "memory_limit":
        "2GB",

    "trusted_registry":
        True
}


print("=" * 70)
print("HARDENED CONTAINER")
print("=" * 70)

for setting, value in hardened_container.items():

    print(
        f"\n{setting}: {value}"
    )

<details>

<summary><b>Part 8 Explanation — Click to Expand</b></summary>

<br>

The hardened configuration changes several important properties.

The application now:

    RUNS AS NON-ROOT

    DOES NOT USE PRIVILEGED MODE

    DOES NOT MOUNT UNNECESSARY HOST DIRECTORIES

    CANNOT ACCESS THE RUNTIME SOCKET

    DOES NOT CONTAIN AN EMBEDDED SECRET

    HAS CPU LIMITS

    HAS MEMORY LIMITS

    REQUIRES A TRUSTED REGISTRY

These controls do not make compromise impossible.

Instead, they reduce the attack surface and restrict what a compromised workload can do.

This is the same security strategy used throughout this module:

> **Assume a component may eventually be compromised and limit what happens next.**

</details>

In [ ]:
# @title ▶ Part 9: Apply Resource Limits { display-mode: "form" }

host_memory_gb = 16

unrestricted_request = 14

container_memory_limit = 2


print("=" * 70)
print("RESOURCE EXHAUSTION SIMULATION")
print("=" * 70)

print(
    f"\nHost memory: "
    f"{host_memory_gb} GB"
)

print(
    f"Compromised workload attempts: "
    f"{unrestricted_request} GB"
)


without_limit = min(
    unrestricted_request,
    host_memory_gb
)

with_limit = min(
    unrestricted_request,
    container_memory_limit
)


print("\nWithout Container Limit:")
print(
    f"Workload can consume up to "
    f"{without_limit} GB"
)

print("\nWith Container Limit:")
print(
    f"Workload restricted to "
    f"{with_limit} GB"
)

protected_memory = (
    without_limit - with_limit
)

print(
    f"\nHost memory protected by limit: "
    f"{protected_memory} GB"
)

<details>

<summary><b>Part 9 Explanation — Click to Expand</b></summary>

<br>

The compromised workload attempts to consume 14 GB of memory.

Without a resource limit, the workload may consume most of the host's available memory.

This could affect other workloads running on the same system.

With a 2 GB container limit:

    WORKLOAD REQUESTS 14 GB
             ↓
        RESOURCE LIMIT
             ↓
        MAXIMUM 2 GB

The container cannot consume the full amount it requested.

This complements the API rate limiting you saw earlier.

    API RATE LIMIT
    → Controls how many requests reach the application

    CONTAINER RESOURCE LIMIT
    → Controls how many infrastructure resources the workload can consume

Using both provides stronger defense against resource exhaustion.

</details>

In [ ]:
# @title ▶ Part 10: Check Image Trust { display-mode: "form" }

images = [
    {
        "name": "ai-inference:v4",
        "registry": "company-registry",
        "trusted": True,
        "scanned": True
    },

    {
        "name": "ai-inference:latest",
        "registry": "unknown-public-user",
        "trusted": False,
        "scanned": False
    }
]


print("=" * 70)
print("CONTAINER IMAGE TRUST CHECK")
print("=" * 70)

for image in images:

    approved = (
        image["trusted"]
        and image["scanned"]
    )

    print(
        f"\nImage:    {image['name']}"
    )

    print(
        f"Registry: {image['registry']}"
    )

    print(
        f"Trusted:  {image['trusted']}"
    )

    print(
        f"Scanned:  {image['scanned']}"
    )

    print(
        "Decision: "
        + (
            "✓ APPROVED"
            if approved
            else "✗ BLOCK DEPLOYMENT"
        )
    )

<details>

<summary><b>Part 10 Explanation — Click to Expand</b></summary>

<br>

The deployment process evaluates more than the image name.

An image called:

    ai-inference:latest

does not prove that the image is trustworthy.

The lab requires the image to come from an approved source and pass the simulated security scan.

This reflects the principle:

> **Available for download does not mean trusted.**

Container registries are part of the software supply chain.

If an attacker replaces an image or convinces a deployment system to retrieve an untrusted image, malicious code could enter production before the AI application even starts.

Production deployment should therefore verify what artifact is actually being deployed rather than relying only on familiar names or tags.

</details>

In [ ]:
# @title ▶ Part 11: Compare Compromise Impact { display-mode: "form" }

insecure_capabilities = [
    "Run as root",
    "Access host directories",
    "Access runtime socket",
    "Use embedded credential",
    "Consume unrestricted memory",
    "Execute unnecessary tools",
    "Use privileged runtime"
]

hardened_capabilities = [
    "Read production model",
    "Execute inference",
    "Write approved logs"
]


print("=" * 70)
print("CONTAINER COMPROMISE COMPARISON")
print("=" * 70)

print("\nINSECURE CONTAINER")

for capability in insecure_capabilities:
    print(f"  ⚠ {capability}")

print(
    f"\nAttacker opportunities: "
    f"{len(insecure_capabilities)}"
)


print("\n" + "=" * 70)

print("\nHARDENED CONTAINER")

for capability in hardened_capabilities:
    print(f"  • {capability}")

print(
    f"\nAvailable workload capabilities: "
    f"{len(hardened_capabilities)}"
)


reduction = (
    (
        len(insecure_capabilities)
        - len(hardened_capabilities)
    )
    / len(insecure_capabilities)
) * 100

print(
    f"\nSimulated capability reduction: "
    f"{reduction:.1f}%"
)

<details>

<summary><b>Part 11 Explanation — Click to Expand</b></summary>

<br>

Assume that the AI inference application is successfully compromised in both configurations.

The initial attack is the same.

What changes is what the attacker can do afterward.

### Insecure Container

The attacker inherits access to powerful runtime features, unnecessary tools, host resources, and credentials.

This creates opportunities for the compromise to expand beyond the inference application.

### Hardened Container

The attacker is much more restricted.

The workload retains only the capabilities needed for normal inference.

This produces:

    APPLICATION COMPROMISE
            ↓
    LIMITED CONTAINER ACCESS
            ↓
    SMALLER BLAST RADIUS

The hardened configuration does not make the application impossible to compromise.

It attempts to **contain the impact of compromise**.

</details>

In [ ]:
# @title ▶ Part 12: Container Security Lifecycle { display-mode: "form" }

lifecycle = [
    "Build",
    "Scan",
    "Test",
    "Deploy",
    "Monitor",
    "Update",
    "Rebuild",
    "Redeploy"
]


print("=" * 70)
print("CONTAINER SECURITY LIFECYCLE")
print("=" * 70)

for number, stage in enumerate(
    lifecycle,
    start=1
):

    print(
        f"{number}. {stage}"
    )


print(
    "\n✓ Container security continues "
    "after the initial deployment."
)

<details>

<summary><b>Part 12 Explanation — Click to Expand</b></summary>

<br>

A container that passes security checks today may contain a vulnerable dependency tomorrow.

New vulnerabilities continue to be discovered in:

    OPERATING-SYSTEM PACKAGES

    PYTHON LIBRARIES

    ML FRAMEWORKS

    API FRAMEWORKS

    CONTAINER RUNTIMES

    SUPPORTING DEPENDENCIES

This means container security is a lifecycle:

    BUILD
      ↓
    SCAN
      ↓
    TEST
      ↓
    DEPLOY
      ↓
    MONITOR
      ↓
    UPDATE
      ↓
    REBUILD
      ↓
    REDEPLOY

Updating the upstream base image does not automatically repair an already deployed container image.

Organizations may need to rebuild, test, and redeploy the application using updated components.

</details>

In [ ]:
# @title ▶ Part 13: Knowledge Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. Why did we remove GCC, Vim, Curl, and the debugger?",
        "options": [
            "A. They were unnecessary for production inference and increased attack surface",
            "B. Containers cannot contain software tools",
            "C. They prevented the model from loading",
            "D. They were required for production"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. What happened after container minimization?",
        "options": [
            "A. Simulated vulnerability findings decreased",
            "B. Administrator privileges increased",
            "C. The container gained more packages",
            "D. Resource limits disappeared"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. Why did we change the application from root to ai-service?",
        "options": [
            "A. To reduce privileges available after compromise",
            "B. To give the application host administrator access",
            "C. To disable isolation",
            "D. To increase the attack surface"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Which setting could weaken container isolation?",
        "options": [
            "A. Privileged mode",
            "B. Running as non-root",
            "C. Memory limits",
            "D. Minimal packages"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. What did the memory limit accomplish?",
        "options": [
            "A. Restricted how much host memory the workload could consume",
            "B. Increased the container to unlimited memory",
            "C. Disabled the AI model",
            "D. Removed authentication"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. Why was the unknown public image blocked?",
        "options": [
            "A. Its source was untrusted and it had not been scanned",
            "B. Public images are always malicious",
            "C. Containers cannot use registries",
            "D. The image had a readable name"
        ],
        "answer": "A"
    },
    {
        "question":
        "7. What was the main effect of hardening the container?",
        "options": [
            "A. It reduced the attacker's opportunities after compromise",
            "B. It guaranteed that compromise was impossible",
            "C. It gave the container more privileges",
            "D. It removed all runtime restrictions"
        ],
        "answer": "A"
    },
    {
        "question":
        "8. Why must containers be rebuilt and redeployed over time?",
        "options": [
            "A. New vulnerabilities may be discovered in their dependencies",
            "B. Containers automatically delete themselves",
            "C. Models cannot run twice",
            "D. Security scanning only works once"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("LAB 5 — KNOWLEDGE CHECK")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1

    else:
        print(
            f"✗ Incorrect. "
            f"Correct answer: {q['answer']}"
        )

percentage = (
    score / len(questions)
) * 100

print("\n" + "=" * 70)
print(f"Lab Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Lab complete!")

else:
    print(
        "△ Review the lab explanations "
        "and try again."
    )

<details>

<summary><b>Lab 5 — Practical Takeaways — Click to Expand</b></summary>

<br>

## Lab 5 — Practical Takeaways

This lab demonstrated that container security involves both:

    WHAT IS INSIDE THE CONTAINER

and:

    HOW THE CONTAINER IS ALLOWED TO RUN

You began with a development-oriented AI container containing unnecessary packages, excessive privileges, an embedded credential, unrestricted resources, and an untrusted image source.

You then applied several layers of protection.

### Image Security

You reduced unnecessary packages and simulated vulnerability scanning.

This changed:

    LARGE DEVELOPMENT IMAGE
              ↓
    MANY SOFTWARE COMPONENTS
              ↓
    LARGER ATTACK SURFACE

into:

    MINIMAL PRODUCTION IMAGE
              ↓
    REQUIRED DEPENDENCIES
              ↓
    SMALLER ATTACK SURFACE

### Runtime Security

You changed the workload from root to a dedicated non-root identity and removed unnecessary host access and privileged runtime capabilities.

The objective was to keep a compromise contained:

    COMPROMISED AI APPLICATION
              ↓
           CONTAINER
              ↓
       ISOLATION BOUNDARY
              X
             HOST

### Resource Security

You also placed limits on resource consumption.

Even if the application attempts to consume excessive memory, the container is restricted by the runtime.

### Supply-Chain Security

Finally, you examined image trust.

A familiar image name does not prove that the underlying artifact is trustworthy.

Images should come from approved sources and be verified before deployment.

The central lesson is:

> **Containerization provides useful isolation, but a container is only as secure as its image, dependencies, privileges, runtime configuration, and maintenance process.**

A compromised container can expose AI-specific assets such as production models, credentials, inputs, predictions, and other infrastructure.

Protecting the AI model therefore requires protecting the environment that executes it.

</details>

In [ ]:
# @title ▶ Section 5 Checkpoint — Container & Runtime Security { display-mode: "form" }

questions = [
    {
        "question":
        "1. Why are containers commonly used for AI applications?",
        "options": [
            "A. They package applications and dependencies into portable environments",
            "B. They eliminate all vulnerabilities",
            "C. They replace machine learning models",
            "D. They automatically secure applications"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. What is an important difference between containers and virtual machines?",
        "options": [
            "A. Containers commonly share the host operating-system kernel",
            "B. Containers always contain a complete guest operating system",
            "C. Virtual machines cannot run applications",
            "D. Containers provide perfect isolation"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. Why should base images be reviewed carefully?",
        "options": [
            "A. Vulnerabilities in the base image can be inherited by the final image",
            "B. Base images cannot contain software",
            "C. They only affect model accuracy",
            "D. They automatically disappear after deployment"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. What is the goal of container minimization?",
        "options": [
            "A. Include only software required for the workload",
            "B. Install every available development tool",
            "C. Give the container administrator privileges",
            "D. Disable vulnerability scanning"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. Why should an AI application run as a non-root user when possible?",
        "options": [
            "A. To reduce privileges available if the application is compromised",
            "B. To increase administrator access",
            "C. To disable isolation",
            "D. To allow access to every host resource"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. What is a container escape?",
        "options": [
            "A. Activity crosses the intended container isolation boundary",
            "B. A container shuts down normally",
            "C. A model produces an incorrect prediction",
            "D. A user logs out"
        ],
        "answer": "A"
    },
    {
        "question":
        "7. What do container resource limits help control?",
        "options": [
            "A. CPU, memory, GPU, and other resource consumption",
            "B. Model labels",
            "C. User passwords only",
            "D. Training accuracy"
        ],
        "answer": "A"
    },
    {
        "question":
        "8. Why must deployed containers continue to be maintained?",
        "options": [
            "A. New vulnerabilities may be discovered after deployment",
            "B. Containers automatically remain secure forever",
            "C. Dependencies cannot contain vulnerabilities",
            "D. Updating containers always reduces model accuracy"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("SECTION 5 CHECKPOINT")
print("CONTAINER AND RUNTIME SECURITY")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 70)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Checkpoint complete!")
else:
    print("△ Review Section 5 and try again.")

---

<details>
<summary><b>Section 6 — Network and Cloud Infrastructure Security — Click to Expand</b></summary>
<br>

## Section 6 — Network and Cloud Infrastructure Security

AI workloads are increasingly deployed across cloud platforms, virtual networks, containers, databases, storage systems, and distributed services. These environments provide scalability and flexibility, but they also create many communication paths that attackers may attempt to exploit.

A compromised inference service should not automatically provide an attacker with access to training data, model registries, secrets, databases, administrative interfaces, or other production workloads.

Network and cloud security controls help determine **which components can communicate, which resources are exposed, and how far an attacker can move after a compromise**.

This section examines public and private endpoints, network segmentation, firewalls, ingress and egress controls, encrypted service communication, zero-trust architecture, cloud permissions, storage security, workload isolation, and network monitoring.

The central security question is:

**If one AI component becomes compromised, what prevents the attacker from reaching everything else?**

<br>

<details>
<summary><b>6.1 Why AI Workloads Move to the Cloud</b></summary>
<br>

### Why AI Workloads Move to the Cloud

Artificial intelligence systems can require significant computational and storage resources.

Organizations may need:

- CPUs
- GPUs
- Large amounts of memory
- Model storage
- Dataset storage
- Databases
- Scalable inference infrastructure
- Monitoring platforms
- Distributed processing
- High network availability

Cloud platforms allow organizations to obtain these resources without building every component themselves.

A cloud-hosted AI environment might contain:

**Internet Users**

↓

**Cloud Load Balancer / API Gateway**

↓

**Inference Services**

↓

**Production Models**

along with:

**Databases | Storage | Model Registry | Logging | Secret Management**

Cloud platforms can also dynamically scale workloads when inference demand changes.

However, moving AI workloads into cloud infrastructure changes the security environment.

Instead of protecting a single server, defenders may need to secure:

- Cloud identities
- Virtual networks
- Storage permissions
- Security groups
- Service accounts
- Containers
- APIs
- Databases
- Managed AI services
- External integrations

Qayyum et al. examined security and privacy challenges associated with machine learning in cloud environments and highlighted the need to protect the broader infrastructure supporting cloud-based machine learning services [11].

Cloud computing can make AI easier to deploy and scale, but it also makes **cloud security part of AI security**.

</details>

<br>

<details>
<summary><b>6.2 Understanding the Cloud AI Attack Surface</b></summary>
<br>

### Understanding the Cloud AI Attack Surface

A cloud-hosted AI system may contain many services distributed across different infrastructure components.

Consider:

**Internet**

↓

**API Gateway**

↓

**Inference Service**

↓

**Model**

The inference service may also communicate with:

- Model storage
- Application databases
- Logging services
- Secret-management systems
- Model registries
- External APIs

Training infrastructure may separately communicate with:

- Training datasets
- Model registries
- Experiment tracking systems
- Cloud storage

Each service and connection contributes to the cloud attack surface.

Potential weaknesses include:

- Publicly exposed services
- Misconfigured storage
- Excessive permissions
- Weak network restrictions
- Unencrypted communication
- Exposed administrative interfaces
- Unnecessary service connections
- Compromised cloud credentials

An attacker does not necessarily need to compromise the model directly.

For example:

**Stolen Cloud Credential**

↓

**Cloud Storage Access**

↓

**Production Model Downloaded**

Or:

**Compromised Inference Service**

↓

**Unrestricted Internal Network**

↓

**Training Database Reached**

The security of a cloud AI system therefore depends on both the individual services and the relationships between those services.

</details>

<br>

<details>
<summary><b>6.3 Public vs. Private AI Endpoints</b></summary>
<br>

### Public vs. Private AI Endpoints

One of the first network-security decisions is determining which services actually need to be reachable from the Internet.

A **public endpoint** can be reached through a public network.

A **private endpoint** is restricted to approved internal networks or communication paths.

Some AI services legitimately require public access.

For example:

**Customer → Public AI API**

However, many supporting services may not.

A secure architecture might use:

**Internet**

↓

**Public API Gateway**

↓

**Private Inference Service**

↓

**Private Model Storage**

The user can reach the API gateway but cannot directly communicate with the model storage or internal inference infrastructure.

A poorly designed architecture might expose:

**Public API**

**Public Inference Server**

**Public Database**

**Public Model Storage**

**Public Administrative Interface**

Every unnecessary public service increases the externally reachable attack surface.

A useful design principle is:

**Expose only what must be exposed.**

Internal services should remain private unless a legitimate requirement exists for public access.

</details>

<br>

<details>
<summary><b>6.4 Network Segmentation</b></summary>
<br>

### Network Segmentation

**Network segmentation** divides infrastructure into separate network zones and restricts communication between them.

Consider an AI environment containing:

- Public API
- Inference service
- Training environment
- Training database
- Model registry
- Administrative systems

In a flat network, every component may be able to communicate freely:

**API ↔ Inference ↔ Training ↔ Database ↔ Registry ↔ Administration**

This creates significant risk.

If an attacker compromises the inference service, the attacker may be able to directly reach the training environment or administrative systems.

A segmented architecture could instead use:

**Public Zone**

API Gateway

↓

**Inference Zone**

Inference Service

↓

**Model Access**

Approved Production Model

while:

**Training Zone**

and:

**Administrative Zone**

remain separately restricted.

Communication between zones should only be permitted when required.

NIST's guidance for securing microservices emphasizes controlling communication between distributed services rather than treating an application environment as a single trusted network [6].

Segmentation does not necessarily prevent the first system from being compromised.

Its purpose is to make:

**One Compromised Component**

less likely to become:

**Entire Environment Compromised**

</details>

<br>

<details>
<summary><b>6.5 Security Groups, Firewalls, and Network Policies</b></summary>
<br>

### Security Groups, Firewalls, and Network Policies

Network segmentation must be enforced by security controls.

Depending on the environment, these controls may include:

- Firewalls
- Cloud security groups
- Network access-control lists
- Kubernetes network policies
- Service-mesh policies
- Host firewalls

These controls determine which communication is allowed.

For example:

**Internet → API Gateway: ALLOW HTTPS**

**Internet → Database: DENY**

**Internet → Model Registry: DENY**

**Inference Service → Production Model Storage: ALLOW**

**Inference Service → Training Database: DENY**

This follows a principle similar to least privilege:

**Only allow the network communication required for legitimate operation.**

A common insecure approach is to allow broad ranges of systems or ports simply because doing so is easier during deployment.

For example:

**Source: Anywhere**

**Destination: Database**

**Action: Allow**

may make troubleshooting easy but also creates unnecessary exposure.

Network policies should be specific enough to restrict communication without preventing legitimate application functionality.

</details>

<br>

<details>
<summary><b>6.6 Ingress and Egress Controls</b></summary>
<br>

### Ingress and Egress Controls

Network security should control both traffic entering a workload and traffic leaving it.

**Ingress** refers to incoming network traffic.

**Egress** refers to outgoing network traffic.

For an inference service:

### Ingress

The service may need to receive:

**API Gateway → Inference Request**

It may not need to accept:

**Internet → Direct Administrative Connection**

### Egress

The service may need to communicate with:

**Inference Service → Model Storage**

**Inference Service → Logging Service**

It may not need unrestricted access to:

**Inference Service → Anywhere on the Internet**

Egress restrictions are particularly important after a compromise.

Suppose an attacker gains code execution inside an inference container.

If unrestricted outbound communication is allowed:

**Compromised Container → Attacker-Controlled Server**

The attacker may attempt to exfiltrate information or download additional malicious tools.

With egress controls:

**Compromised Container → Unauthorized Destination → DENIED**

Ingress controls help reduce what attackers can reach.

Egress controls help reduce what compromised systems can reach.

Both are important for limiting attack paths.

</details>

<br>

<details>
<summary><b>6.7 Secure Service-to-Service Communication</b></summary>
<br>

### Secure Service-to-Service Communication

AI systems frequently contain services that communicate automatically with one another.

Examples include:

**API Gateway → Inference Service**

**Inference Service → Model Storage**

**Training Service → Dataset Storage**

**Deployment Pipeline → Model Registry**

**Monitoring Agent → Logging Platform**

These connections should not automatically be trusted simply because they occur inside the same cloud environment.

Secure service-to-service communication should consider:

- Service identity
- Authentication
- Authorization
- Encryption
- Network policy
- Logging

A request may follow:

**Service A**

↓

**Authenticate Service Identity**

↓

**Check Authorization**

↓

**Encrypted Connection**

↓

**Service B**

This combines the IAM principles from Section 3 with the network controls in this section.

NIST guidance for microservice and zero-trust architectures supports protecting interactions between individual services rather than relying entirely on network location [3], [5], [6].

A service should therefore be able to answer:

**Which service is communicating with me?**

and:

**Is that service permitted to perform this action?**

</details>

<br>

<details>
<summary><b>6.8 TLS and Encryption in Transit</b></summary>
<br>

### TLS and Encryption in Transit

Network segmentation determines which systems can communicate.

Encryption helps protect the information being transmitted during those communications.

AI systems may transmit sensitive information such as:

- User inputs
- Model predictions
- Authentication tokens
- API keys
- Embeddings
- Training data
- Model artifacts
- Application metadata

TLS can help protect network communication against unauthorized observation and manipulation.

For example:

**Inference Service**

↓

**TLS-Encrypted Connection**

↓

**Internal API**

Encryption should not be limited only to Internet-facing communication.

Sensitive internal communication may also require protection.

This becomes especially important in distributed cloud environments where services may communicate across multiple networks, hosts, or infrastructure layers.

Encryption provides one layer of defense.

However:

**Encrypted Connection ≠ Authorized Connection**

An attacker with valid credentials could still establish an encrypted connection.

This is why secure communication combines:

**Encryption + Authentication + Authorization + Network Policy**

rather than relying on TLS alone.

</details>

<br>

<details>
<summary><b>6.9 Zero Trust for AI Infrastructure</b></summary>
<br>

### Zero Trust for AI Infrastructure

Zero-trust architecture challenges the assumption that systems should automatically be trusted because they exist inside an organization's network.

NIST SP 800-207 describes zero trust as an approach focused on protecting resources rather than granting broad trust based on network location [3].

For AI infrastructure:

**Inside Cloud Network ≠ Automatically Trusted**

Consider an inference service attempting to retrieve a production model.

A traditional approach might allow the request because both systems exist inside the same private network.

A zero-trust-oriented approach asks:

- What service is requesting access?
- Has the service authenticated?
- Is the service authorized?
- Which model can it access?
- Is the connection allowed?
- Is the request being logged?

The resulting path becomes:

**Inference Service**

↓

**Verify Identity**

↓

**Evaluate Policy**

↓

**Authorize Specific Resource**

↓

**Allow Required Connection**

This approach can also be applied between:

- Training systems and datasets
- Deployment pipelines and registries
- Monitoring systems and logs
- Applications and databases

Zero trust does not mean that no communication is allowed.

It means that **trust should be explicitly established rather than implicitly assumed**.

</details>

<br>

<details>
<summary><b>6.10 Cloud IAM and Infrastructure Permissions</b></summary>
<br>

### Cloud IAM and Infrastructure Permissions

Network controls determine which systems can communicate, but cloud IAM determines what authenticated identities can do with cloud resources.

For example, an inference workload may have network access to cloud storage.

However, IAM can further determine whether it can:

**Read Model: ALLOW**

**Upload Model: DENY**

**Delete Model: DENY**

This provides another layer of defense.

Consider a compromised inference service.

If network rules allow communication with model storage but IAM permits only read access:

**Compromised Inference Service**

↓

**Model Storage Reached**

↓

**Read Production Model: ALLOW**

**Replace Production Model: DENY**

The attacker still has access to a sensitive resource, but the available actions are restricted.

This demonstrates how controls from different sections work together:

**Network Segmentation**

+

**Service Identity**

+

**Least Privilege**

+

**Monitoring**

=

**Reduced Blast Radius**

Cloud permissions should therefore follow the same least-privilege principles introduced in Section 3.

</details>

<br>

<details>
<summary><b>6.11 Protecting AI Data and Storage</b></summary>
<br>

### Protecting AI Data and Storage

AI systems may depend on several types of storage.

Examples include:

- Training datasets
- Production models
- Model checkpoints
- User inputs
- Inference results
- Embeddings
- Application databases
- Logs
- Backups

Different storage resources may require different protections.

For example:

**Training Dataset**

may require strong confidentiality and write restrictions.

**Production Model**

may require confidentiality and integrity protection.

**Security Logs**

may require strong integrity controls to prevent attackers from altering evidence.

Storage security may include:

- IAM restrictions
- Encryption at rest
- Private access
- Network restrictions
- Backup protection
- Versioning
- Integrity controls
- Audit logging

Public storage should be used only when there is a legitimate requirement for the stored information to be publicly accessible.

A misconfigured storage resource can bypass other security controls entirely.

For example:

**Secure API + Strong Authentication + Hardened Container**

does little to protect a proprietary model if:

**Production Model Storage → Publicly Accessible**

Storage should therefore be treated as part of the AI system's security boundary.

</details>

<br>

<details>
<summary><b>6.12 Multi-Tenancy and Workload Isolation</b></summary>
<br>

### Multi-Tenancy and Workload Isolation

Cloud infrastructure often allows multiple users, applications, containers, or organizations to share underlying resources.

This is known as **multi-tenancy**.

A platform might host:

**Customer A Workload**

**Customer B Workload**

**Customer C Workload**

on shared infrastructure.

Multi-tenancy improves resource utilization but creates an important security requirement:

**One tenant should not be able to access another tenant's resources.**

For AI systems, isolation may need to protect:

- Models
- Training data
- Inference inputs
- Predictions
- Credentials
- Memory
- Storage
- Logs

Organizations should consider how workloads are isolated at multiple levels, including:

- Identity
- Network
- Container
- Storage
- Compute
- Application

Research into secure machine-learning-as-a-service environments has examined the privacy and security challenges created when ML computation is performed on shared or externally managed infrastructure [11]–[14].

The objective is to prevent:

**Compromise of Workload A**

from becoming:

**Access to Workload B**

</details>

<br>

<details>
<summary><b>6.13 Limiting the Blast Radius of a Compromise</b></summary>
<br>

### Limiting the Blast Radius of a Compromise

The concept of **blast radius** has appeared repeatedly throughout this module because it is fundamental to secure infrastructure design.

Consider an insecure AI deployment:

**Internet**

↓

**Inference Service**

↓

**Flat Internal Network**

↓

**Training Database**

↓

**Model Registry**

↓

**Secret Store**

↓

**Administrative Services**

If the inference service becomes compromised, the attacker may have direct network paths to many sensitive resources.

Now consider a segmented environment:

**Internet**

↓

**API Gateway**

↓

**Inference Zone**

↓

**Approved Production Model**

while:

**Training Zone — Restricted**

**Management Zone — Restricted**

**Secret Store — Identity Restricted**

**Registry — Deployment Identity Only**

The same inference service may still become compromised.

However, the attacker encounters multiple barriers.

**Training Database → DENIED**

**Administrative Services → DENIED**

**Model Upload → DENIED**

**Unrelated Secrets → DENIED**

The security goal is not:

**"Assume nothing will ever be compromised."**

A more resilient goal is:

**"Assume individual components can be compromised and design the environment so that compromise remains contained."**

This is the combined purpose of segmentation, least privilege, zero trust, workload isolation, and defense in depth.

</details>

<br>

<details>
<summary><b>6.14 Network Monitoring and Threat Detection</b></summary>
<br>

### Network Monitoring and Threat Detection

Preventive controls cannot guarantee that every attack will be stopped.

Network activity should therefore also be monitored for suspicious behavior.

Potential indicators include:

- Unexpected outbound connections
- Connections to unusual destinations
- Repeated denied connections
- Unexpected communication between network zones
- Large data transfers
- New public services
- Unusual administrative connections
- Changes to firewall rules
- Changes to security groups
- Unexpected ports or protocols

Consider an inference service that normally communicates only with:

**API Gateway**

**Model Storage**

**Logging Service**

If monitoring suddenly observes:

**Inference Service → Training Database**

or:

**Inference Service → Unknown Internet Host**

the behavior may deserve investigation.

Monitoring therefore depends partly on understanding what **normal communication** looks like.

A simple baseline might be:

**Expected**

Inference → Model Storage

Inference → Logging

Gateway → Inference

and:

**Unexpected**

Inference → Administrative Server

Inference → Training Database

Inference → Unknown External Server

Network monitoring adds another layer to defense in depth by helping defenders identify suspicious behavior when preventive controls fail or credentials are abused.

The broader monitoring and incident-response process will be explored in Section 8.

</details>

<br>

<details>
<summary><b>6.15 Connecting Network and Cloud Security to AI Attacks</b></summary>
<br>

### Connecting Network and Cloud Security to AI Attacks

Network and cloud vulnerabilities can provide attackers with paths to AI assets even when the model itself contains no exploitable vulnerability.

Consider several examples.

#### Public Model Storage

**Cloud Misconfiguration**

↓

**Public Storage**

↓

**Production Model Download**

↓

**Model Theft**

#### Flat Internal Network

**Inference Service Compromised**

↓

**Training Environment Reached**

↓

**Potential Dataset Manipulation**

#### Excessive Cloud Permissions

**Service Credential Stolen**

↓

**Broad Cloud Access**

↓

**Models, Data, or Infrastructure Modified**

#### Unrestricted Egress

**Inference Container Compromised**

↓

**External Attacker Server Reached**

↓

**Potential Data Exfiltration**

#### Unprotected Internal Communication

**Internal Traffic Intercepted**

↓

**Sensitive AI Data or Credentials Exposed**

These attack paths demonstrate why traditional infrastructure controls remain essential to AI security.

The attacker may begin with:

**Container Vulnerability**

**Credential Theft**

**API Weakness**

or:

**Cloud Misconfiguration**

but eventually reach:

**Model**

**Dataset**

**Inference Data**

or:

**AI Deployment Pipeline**

The infrastructure surrounding the model determines which paths are available.

</details>

<br>

<details>
<summary><b>Section 6 Summary</b></summary>
<br>

### Section 6 Summary

Cloud and network infrastructure connects the many services that make a production AI system function. These connections provide necessary capabilities, but they can also create paths for attackers to move between systems.

Throughout this section, you learned that:

- Cloud platforms provide scalable compute, storage, networking, and managed services for AI workloads.
- Cloud AI deployments create attack surfaces involving identities, networks, storage, services, and configuration.
- Only services that legitimately require public access should be exposed to the Internet.
- Network segmentation separates AI components and restricts unnecessary communication.
- Firewalls, security groups, and network policies can enforce approved communication paths.
- Ingress controls restrict incoming traffic while egress controls restrict outbound communication.
- Internal service-to-service communication should be authenticated, authorized, encrypted, and monitored.
- TLS protects sensitive AI information while it moves across networks.
- Zero trust avoids granting automatic trust based solely on network location.
- Cloud IAM can restrict what workloads are permitted to do even when network communication is allowed.
- Models, datasets, logs, embeddings, and other AI storage resources require appropriate access and encryption controls.
- Multi-tenant environments require strong workload and data isolation.
- Segmentation and least privilege can significantly reduce the blast radius of a compromised AI component.
- Network monitoring can identify unusual communication and possible attacker movement.
- Infrastructure weaknesses can provide paths to model theft, data manipulation, credential exposure, and other AI-specific security consequences.

The next section examines **Model Artifact and Deployment Pipeline Security**, where you will explore how organizations can verify that the model entering production is authentic, approved, unmodified, and traceable to a trusted development and deployment process.

</details>

</details>

In [ ]:

# @title Section 6 — Network and Cloud Infrastructure Security Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.colors import ListedColormap
from IPython.display import display, HTML
from io import BytesIO
import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"
TEAL = "#0891B2"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):
    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")
    buffer.close()

    display(HTML(f"""
    <details style="margin:12px 0;border:1px solid #cbd5e1;
                    border-radius:10px;overflow:hidden;">
      <summary style="cursor:pointer;padding:16px;font-size:16px;
                      font-weight:bold;background:#eff6ff;
                      color:#1e3a8a;border-left:5px solid {color};">
        {html.escape(title)} — Click to Expand
      </summary>
      <div style="padding:15px;background:white;text-align:center;">
        <img src="data:image/png;base64,{encoded}"
             style="width:100%;max-width:950px;height:auto;">
        <div style="max-width:850px;margin:15px auto;padding:16px;
                    background:#f8fafc;border-radius:8px;color:#334155;
                    font-size:14px;line-height:1.7;text-align:left;">
          {explanation}
        </div>
      </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def setup_diagram(ax, title, xlim=(0, 12), ylim=(0, 10)):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.axis("off")
    ax.set_title(
        title,
        fontsize=15,
        fontweight="bold",
        pad=18
    )


def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC",
             title_size=10):

    card = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.8
    )
    ax.add_patch(card)

    ax.text(
        x+w/2,
        y+h*(0.66 if subtitle else 0.5),
        title,
        ha="center",
        va="center",
        fontsize=title_size,
        fontweight="bold",
        color=DARK
    )

    if subtitle:
        ax.text(
            x+w/2, y+h*0.28,
            subtitle,
            ha="center",
            va="center",
            fontsize=8.5,
            color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):
    ax.add_patch(FancyArrowPatch(
        start, end,
        arrowstyle="-|>",
        mutation_scale=16,
        linewidth=1.8,
        color=color
    ))


# ============================================================
# VISUALIZATION 1
# CLOUD AI DEPLOYMENT ARCHITECTURE
# ============================================================

fig, ax = plt.subplots(figsize=(12, 10))

setup_diagram(
    ax,
    "Cloud-Based AI Deployment Architecture",
    xlim=(0, 12),
    ylim=(0, 12)
)

# Cloud boundary
cloud = FancyBboxPatch(
    (3.0, 1.1), 8.5, 9.8,
    boxstyle="round,pad=0.08,rounding_size=0.25",
    facecolor="#F8FAFC",
    edgecolor=BLUE,
    linewidth=2,
    linestyle="--"
)
ax.add_patch(cloud)

ax.text(
    7.2, 10.45,
    "Cloud Infrastructure Boundary",
    ha="center",
    fontsize=12,
    fontweight="bold",
    color=BLUE
)

add_card(
    ax, 0.2, 7.6, 2.3, 1.3,
    "External Client",
    "User or application",
    RED
)

add_card(
    ax, 3.5, 7.6, 3.1, 1.3,
    "Public API Gateway",
    "Controlled entry point",
    BLUE
)

add_card(
    ax, 7.7, 7.6, 3.1, 1.3,
    "Private Inference",
    "Model-serving workload",
    GREEN
)

add_arrow(ax, (2.5, 8.25), (3.5, 8.25), BLUE)
add_arrow(ax, (6.6, 8.25), (7.7, 8.25), GREEN)

supporting = [
    (3.5, 4.7, "Cloud IAM", "Identity and permissions", PURPLE),
    (7.7, 4.7, "Model Storage", "Protected artifacts", ORANGE),
    (3.5, 2.1, "Network Controls", "Firewall and segmentation", TEAL),
    (7.7, 2.1, "Monitoring", "Logs and alerts", GREEN)
]

for x, y, title, subtitle, color in supporting:
    add_card(
        ax, x, y, 3.1, 1.3,
        title, subtitle, color
    )

add_arrow(ax, (9.25, 7.6), (9.25, 6.0), ORANGE)
add_arrow(ax, (7.7, 8.0), (6.6, 6.0), PURPLE)
add_arrow(ax, (5.05, 4.7), (5.05, 3.4), TEAL)
add_arrow(ax, (9.25, 4.7), (9.25, 3.4), GREEN)

plt.tight_layout()

show_collapsible_figure(
    "6.1 — Cloud AI Deployment Architecture",
    fig,
    """
    Cloud platforms provide infrastructure
    for deploying AI applications without
    requiring an organization to operate
    every physical server itself.

    <br><br>

    A cloud-based AI deployment may contain
    a public API gateway, private inference
    services, protected model storage,
    identity management, and monitoring.

    <br><br>

    The public gateway receives requests
    from external users and applications.

    <br><br>

    The inference service processes
    authorized requests and accesses
    the approved model artifact.

    <br><br>

    Cloud IAM controls which identities
    can access infrastructure resources.

    <br><br>

    Network controls restrict communication
    between services, while monitoring
    systems collect security and
    operational information.

    <br><br>

    Cloud deployments introduce a
    <b>shared responsibility model</b>.
    The cloud provider secures certain
    underlying infrastructure, while
    the customer remains responsible
    for applicable application,
    identity, data, and configuration
    security.

    <br><br>

    The exact responsibilities depend
    on the cloud service being used.

    <br><br>

    <b>Key takeaway:</b> Cloud AI security
    requires protecting the connections
    between the inference application,
    cloud identities, storage,
    and network infrastructure.
    """,
    BLUE
)


# ============================================================
# VISUALIZATION 2
# PUBLIC VS PRIVATE AI ENDPOINTS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

fig.suptitle(
    "Public vs. Private AI Inference Endpoints",
    fontsize=16,
    fontweight="bold"
)

# Public endpoint
ax = axes[0]
setup_diagram(ax, "Public Endpoint", (0, 6), (0, 10))

public_components = [
    ("Internet Clients", "Potentially reachable externally", RED),
    ("Public API Endpoint", "Internet-facing interface", ORANGE),
    ("Authentication / Gateway", "Required access controls", BLUE),
    ("Inference Service", "Protected model execution", GREEN)
]

for i, (title, subtitle, color) in enumerate(public_components):
    y = 8.0 - i*2.0

    add_card(
        ax, 0.6, y, 4.8, 1.2,
        title, subtitle, color
    )

    if i < len(public_components)-1:
        add_arrow(
            ax, (3.0, y-0.05),
            (3.0, y-0.75)
        )

# Private endpoint
ax = axes[1]
setup_diagram(ax, "Private Endpoint", (0, 6), (0, 10))

private_components = [
    ("Authorized Workload", "Approved network origin", BLUE),
    ("Private Network Path", "Private routing or connectivity", TEAL),
    ("Identity / Network Policy", "Verify and restrict access", PURPLE),
    ("Private Inference Service", "No direct public endpoint", GREEN)
]

for i, (title, subtitle, color) in enumerate(private_components):
    y = 8.0 - i*2.0

    add_card(
        ax, 0.6, y, 4.8, 1.2,
        title, subtitle, color
    )

    if i < len(private_components)-1:
        add_arrow(
            ax, (3.0, y-0.05),
            (3.0, y-0.75)
        )

plt.tight_layout()

show_collapsible_figure(
    "6.2 — Public vs. Private AI Endpoints",
    fig,
    """
    A public AI endpoint is reachable
    through an internet-facing
    network interface.

    <br><br>

    Public endpoints are useful
    when external applications
    or users must access an
    AI service.

    <br><br>

    However, public exposure
    also creates opportunities
    for unauthorized requests,
    automated probing,
    and denial-of-service attempts.

    <br><br>

    A private endpoint restricts
    network reachability to
    approved private networks
    or connectivity mechanisms.

    <br><br>

    Private endpoints are
    commonly useful for
    internal AI applications,
    model-serving backends,
    and sensitive workloads.

    <br><br>

    Private network placement
    does not automatically
    guarantee security.

    <br><br>

    A compromised internal
    service may still attempt
    unauthorized access.

    <br><br>

    Both public and private
    endpoints should enforce
    appropriate authentication,
    authorization, encryption,
    and monitoring.

    <br><br>

    <b>Key takeaway:</b> Network
    exposure should be intentional.
    Services that do not require
    direct public access should
    generally remain inaccessible
    from the public internet.
    """,
    GREEN
)


# ============================================================
# VISUALIZATION 3
# NETWORK SEGMENTATION AND SECURITY POLICIES
# ============================================================

services = [
    "API Gateway",
    "Inference Service",
    "Model Registry",
    "Data Store",
    "Monitoring",
    "Admin Interface"
]

# Hypothetical network policy:
# Rows = source services, columns = destinations.
# 1 = permitted communication; 0 = blocked.
# Diagonal values represent local/same-service access
# and are excluded from the inter-service comparison.

policy = np.array([
    [1, 1, 0, 0, 1, 0],
    [0, 1, 1, 1, 1, 0],
    [0, 0, 1, 0, 1, 0],
    [0, 0, 0, 1, 1, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 0, 1, 1, 1, 1]
])

fig, ax = plt.subplots(figsize=(12, 8))

ax.imshow(
    policy,
    cmap=ListedColormap(["#FEE2E2", "#DCFCE7"]),
    vmin=0,
    vmax=1,
    aspect="auto"
)

ax.set_xticks(range(len(services)))
ax.set_xticklabels(services, rotation=30, ha="right")

ax.set_yticks(range(len(services)))
ax.set_yticklabels(services)

ax.set_xlabel("Destination Service")
ax.set_ylabel("Source Service")

ax.set_title(
    "Illustrative Inter-Service Network Policy",
    fontsize=16,
    fontweight="bold",
    pad=18
)

for i in range(len(services)):
    for j in range(len(services)):
        label = "SELF" if i == j else (
            "ALLOW" if policy[i, j] else "BLOCK"
        )
        ax.text(
            j, i, label,
            ha="center",
            va="center",
            fontsize=8.5,
            fontweight="bold",
            color=DARK
        )

ax.set_xticks(
    np.arange(-0.5, len(services), 1), minor=True
)
ax.set_yticks(
    np.arange(-0.5, len(services), 1), minor=True
)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", bottom=False, left=False)

plt.tight_layout()

show_collapsible_figure(
    "6.3 — Network Segmentation and Security Policies",
    fig,
    """
    Network segmentation divides
    infrastructure into smaller
    communication zones.

    <br><br>

    Rather than allowing every
    service to communicate
    with every other service,
    network policies define
    permitted connections.

    <br><br>

    The matrix shows a
    hypothetical network policy
    for a deployed AI application.

    <br><br>

    Each row represents the
    service initiating a connection,
    while each column represents
    the destination service.

    <br><br>

    For example, the API gateway
    may be allowed to communicate
    with the inference service
    but not directly access
    the model registry.

    <br><br>

    The inference service may
    require access to approved
    model artifacts and application
    data, but not administrative
    interfaces.

    <br><br>

    Security groups, firewalls,
    and network policies can
    enforce these restrictions.

    <br><br>

    Real deployments must also
    account for connection state,
    ports, protocols, and
    application requirements.

    <br><br>

    Network segmentation helps
    reduce lateral movement
    after a service is compromised.

    <br><br>

    <b>Key takeaway:</b> Communication
    between AI infrastructure
    components should be explicitly
    permitted according to
    operational requirements.
    """,
    PURPLE
)


# ============================================================
# VISUALIZATION 4
# INGRESS, EGRESS, AND SECURE COMMUNICATION
# ============================================================

fig, ax = plt.subplots(figsize=(12, 9))

setup_diagram(
    ax,
    "Ingress and Egress Controls for AI Infrastructure",
    xlim=(0, 12),
    ylim=(0, 11)
)

add_card(
    ax, 0.3, 7.8, 2.7, 1.3,
    "External Clients",
    "Incoming requests",
    RED
)

add_card(
    ax, 3.4, 7.8, 2.7, 1.3,
    "Ingress Controls",
    "Gateway, firewall, TLS",
    BLUE
)

add_card(
    ax, 6.5, 7.8, 2.7, 1.3,
    "Inference Service",
    "Protected workload",
    GREEN
)

add_arrow(ax, (3.0, 8.45), (3.4, 8.45), BLUE)
add_arrow(ax, (6.1, 8.45), (6.5, 8.45), GREEN)

add_card(
    ax, 6.5, 4.8, 2.7, 1.3,
    "Egress Controls",
    "Approved outbound access",
    ORANGE
)

add_arrow(ax, (7.85, 7.8), (7.85, 6.1), ORANGE)

add_card(
    ax, 9.5, 4.8, 2.2, 1.3,
    "External API",
    "Approved destination",
    PURPLE
)

add_arrow(ax, (9.2, 5.45), (9.5, 5.45), GREEN)

add_card(
    ax, 0.3, 4.8, 2.7, 1.3,
    "Unapproved Host",
    "Potential exfiltration target",
    RED
)

ax.text(
    4.4, 5.4,
    "Outbound traffic\nshould be restricted",
    ha="center",
    fontsize=10,
    fontweight="bold",
    color=ORANGE
)

add_card(
    ax, 3.4, 1.7, 5.8, 1.2,
    "Service-to-Service Protection",
    "TLS/mTLS, workload identity, and authorization",
    TEAL
)

ax.text(
    6.0, 0.8,
    "Ingress controls what enters; egress controls what leaves",
    ha="center",
    fontsize=10,
    fontweight="bold",
    color=DARK
)

plt.tight_layout()

show_collapsible_figure(
    "6.4 — Ingress, Egress, and Secure Service Communication",
    fig,
    """
    Network traffic can be
    classified according to
    its direction.

    <br><br>

    <b>Ingress traffic</b>
    enters an application
    or network environment.

    <br><br>

    For example, a user
    submitting a prediction
    request generates ingress
    traffic to an AI API.

    <br><br>

    <b>Egress traffic</b>
    leaves the application
    or network environment.

    <br><br>

    An inference service
    contacting an external
    AI provider or downloading
    an approved artifact
    generates egress traffic.

    <br><br>

    Unrestricted egress
    may create opportunities
    for data exfiltration
    or unauthorized communication.

    <br><br>

    Organizations can restrict
    outbound connections to
    approved destinations
    and required protocols.

    <br><br>

    TLS protects information
    in transit. Mutual TLS
    can additionally authenticate
    both communicating endpoints
    when correctly configured.

    <br><br>

    However, encrypted
    communication does not
    automatically mean
    the requested operation
    is authorized.

    <br><br>

    Identity verification
    and access control
    remain necessary.

    <br><br>

    <b>Key takeaway:</b> AI
    infrastructure should
    control both inbound
    and outbound communication
    while protecting authorized
    service-to-service traffic.
    """,
    ORANGE
)


# ============================================================
# VISUALIZATION 5
# ZERO TRUST, CLOUD IAM, AND WORKLOAD ISOLATION
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

fig.suptitle(
    "Cloud Workload Compromise — Limiting the Blast Radius",
    fontsize=16,
    fontweight="bold"
)

resources = [
    ("Model Registry", 7.3),
    ("Training Data", 5.3),
    ("Secret Manager", 3.3),
    ("Cloud Administration", 1.3)
]

# Overly permissive
ax = axes[0]
setup_diagram(
    ax,
    "Broad Cloud Permissions",
    xlim=(0, 6),
    ylim=(0, 10)
)

add_card(
    ax, 0.8, 8.5, 4.4, 1.0,
    "Compromised Inference Workload",
    accent=RED,
    title_size=9
)

for title, y in resources:
    add_card(
        ax, 1.1, y, 3.8, 1.0,
        title, "Potentially accessible",
        RED
    )
    add_arrow(
        ax, (3.0, 8.5),
        (3.0, y+1.0),
        RED
    )

# Restricted
ax = axes[1]
setup_diagram(
    ax,
    "Zero Trust / Least Privilege",
    xlim=(0, 6),
    ylim=(0, 10)
)

add_card(
    ax, 0.8, 8.5, 4.4, 1.0,
    "Compromised Inference Workload",
    accent=ORANGE,
    title_size=9
)

for title, y in resources:
    subtitle = (
        "Read-only approved artifacts"
        if title == "Model Registry"
        else "Access denied by policy"
    )

    color = ORANGE if title == "Model Registry" else GREEN

    add_card(
        ax, 1.1, y, 3.8, 1.0,
        title, subtitle,
        color
    )

add_arrow(
    ax, (3.0, 8.5),
    (3.0, 8.3),
    ORANGE
)

plt.tight_layout()

show_collapsible_figure(
    "6.5 — Zero Trust, Cloud IAM, and Workload Isolation",
    fig,
    """
    Zero Trust applies to
    cloud AI infrastructure
    by requiring access
    decisions based on
    verified identity
    and authorization.

    <br><br>

    A service should not
    receive unrestricted
    access merely because
    it operates within
    the same cloud network.

    <br><br>

    Consider an attacker
    who compromises an
    inference workload.

    <br><br>

    If that workload
    possesses broad cloud
    permissions, the attacker
    may gain access to
    sensitive model storage,
    training data, secrets,
    or administrative resources.

    <br><br>

    In a least-privilege
    architecture, the
    inference workload
    receives only the
    permissions required
    for its intended operation.

    <br><br>

    Cloud IAM policies,
    network segmentation,
    workload identities,
    and storage permissions
    help enforce restrictions.

    <br><br>

    Multi-tenant environments
    require additional attention
    to isolation between
    customers or workloads.

    <br><br>

    The diagram is conceptual.
    Real compromise impact
    depends on effective
    permissions, vulnerabilities,
    and other security controls.

    <br><br>

    <b>Key takeaway:</b> Cloud
    infrastructure should
    assume that individual
    services may become
    compromised and limit
    what those services
    can access afterward.
    """,
    GREEN
)


# ============================================================
# VISUALIZATION 6
# NETWORK MONITORING AND THREAT DETECTION
# ============================================================

rng = np.random.default_rng(42)

minutes = np.arange(1, 61)

# Synthetic baseline traffic.
baseline = rng.poisson(35, size=60)

# Synthetic observed traffic with an unusual burst.
observed = baseline.copy()
observed[35:46] += np.array([
    20, 35, 50, 65, 85, 95,
    80, 70, 55, 40, 25
])

# Illustrative fixed alert threshold.
threshold = 65

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(
    minutes,
    baseline,
    color=BLUE,
    linewidth=2,
    label="Expected Baseline"
)

ax.plot(
    minutes,
    observed,
    color=ORANGE,
    linewidth=2.3,
    label="Observed Requests"
)

ax.axhline(
    threshold,
    color=RED,
    linestyle="--",
    linewidth=2,
    label="Illustrative Alert Threshold"
)

ax.fill_between(
    minutes,
    observed,
    threshold,
    where=observed > threshold,
    color=RED,
    alpha=0.15
)

ax.annotate(
    "Unusual traffic burst",
    xy=(41, observed[40]),
    xytext=(23, 145),
    fontsize=10,
    fontweight="bold",
    color=RED,
    arrowprops=dict(
        arrowstyle="->",
        color=RED,
        linewidth=1.5
    )
)

ax.set_xlabel("Minute")
ax.set_ylabel("Requests per Minute")

ax.set_title(
    "Illustrative AI Endpoint Traffic Monitoring",
    fontsize=16,
    fontweight="bold"
)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "6.6 — Network Monitoring and Threat Detection",
    fig,
    """
    Network monitoring helps
    defenders understand how
    AI infrastructure is
    being accessed and used.

    <br><br>

    A monitoring system
    may collect request
    counts, source addresses,
    connection failures,
    authentication events,
    and service communication
    patterns.

    <br><br>

    Establishing a baseline
    helps identify activity
    that differs from
    expected behavior.

    <br><br>

    This graph shows
    synthetic requests
    to an AI inference
    endpoint over one hour.

    <br><br>

    The blue line represents
    expected traffic,
    while the orange line
    represents observed
    requests.

    <br><br>

    A sudden traffic increase
    exceeds an illustrative
    alert threshold.

    <br><br>

    Such behavior may
    warrant investigation
    for automated probing,
    excessive API usage,
    or denial-of-service
    activity.

    <br><br>

    However, a traffic
    spike does not
    automatically prove
    that an attack
    is occurring.

    <br><br>

    Legitimate events,
    application updates,
    or increased user
    demand may produce
    similar patterns.

    <br><br>

    Defenders should
    combine traffic
    metrics with identity
    information, logs,
    request characteristics,
    and application context.

    <br><br>

    <b>Key takeaway:</b> Network
    monitoring provides
    visibility into suspicious
    activity, but alerts
    require investigation
    before concluding that
    a security incident
    has occurred.
    """,
    RED
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("Section 6 — Network and Cloud Infrastructure Security Visualizations")
print("--------------------------------------------------------------------")
print("Six collapsible visualizations generated.")

print()
print("Concepts Covered")
print("----------------")
print("1. Cloud AI Deployment Architecture")
print("2. Public vs. Private AI Endpoints")
print("3. Network Segmentation and Security Policies")
print("4. Ingress, Egress, and Secure Communication")
print("5. Zero Trust, Cloud IAM, and Workload Isolation")
print("6. Network Monitoring and Threat Detection")

print()
print("All examples and traffic metrics are illustrative.")
print("No external APIs, cloud accounts, or network services were used.")


<details>

<summary><b>Section 6 Lab — Click to Expand</b></summary>

<br>

A company operates an AI application in a cloud environment.

The environment contains:

    INTERNET

    API GATEWAY

    INFERENCE SERVICE

    MODEL REGISTRY

    PRODUCTION DATABASE

    TRAINING STORAGE

    SECRET MANAGER

    LOGGING SERVICE

Only the API gateway needs to accept requests directly from Internet users.

The other services support the AI application internally.

During a security review, the team discovers that internal network rules are extremely permissive.

If the inference service becomes compromised, it may be able to communicate with resources that it does not actually need.

Your task is to examine this environment and reduce the attacker's ability to move through it.

You will:

    Identify public and private services

    Examine a flat cloud network

    Simulate compromise of the inference service

    Measure lateral-movement opportunities

    Segment the environment

    Apply ingress and egress restrictions

    Examine encrypted service communication

    Restrict access to cloud storage

    Apply Zero-Trust decisions

    Compare the final blast radius

The goal is not to configure a real cloud provider.

Instead, you will practice the security decisions that determine which AI components should be allowed to communicate.

All code uses standard Python and should run almost instantly.

</details>

In [ ]:
# @title ▶ Part 1: Examine the Cloud Environment { display-mode: "form" }

cloud_services = {
    "API Gateway": {
        "public": True,
        "purpose": "Receive user requests"
    },

    "Inference Service": {
        "public": False,
        "purpose": "Run model inference"
    },

    "Model Registry": {
        "public": False,
        "purpose": "Store approved models"
    },

    "Production Database": {
        "public": False,
        "purpose": "Store application data"
    },

    "Training Storage": {
        "public": False,
        "purpose": "Store training datasets"
    },

    "Secret Manager": {
        "public": False,
        "purpose": "Store credentials"
    },

    "Logging Service": {
        "public": False,
        "purpose": "Collect security logs"
    }
}


print("=" * 70)
print("CLOUD AI ENVIRONMENT")
print("=" * 70)

for service, info in cloud_services.items():

    exposure = (
        "PUBLIC"
        if info["public"]
        else "PRIVATE"
    )

    print(f"\n{service}")
    print(f"Exposure: {exposure}")
    print(f"Purpose:  {info['purpose']}")

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

The environment contains several services, but only one needs direct Internet exposure:

    API GATEWAY

The remaining services are internal components.

A simplified architecture is:

    INTERNET
       ↓
    PUBLIC API GATEWAY
       ↓
    PRIVATE INFERENCE SERVICE
       ↓
    INTERNAL RESOURCES

This follows an important network-security principle:

> **A service should not be publicly reachable simply because it exists in the cloud.**

Model registries, databases, secret-management systems, and training storage normally have no reason to accept arbitrary Internet traffic.

Reducing public exposure removes unnecessary attack paths.

</details>

In [ ]:
# @title ▶ Part 2: Create a Flat Network { display-mode: "form" }

services = list(
    cloud_services.keys()
)

flat_network = {}

for source in services:

    flat_network[source] = [
        destination
        for destination in services
        if destination != source
    ]


print("=" * 70)
print("FLAT CLOUD NETWORK")
print("=" * 70)

for service, reachable in flat_network.items():

    print(f"\n{service} can reach:")

    for destination in reachable:
        print(f"  • {destination}")

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

The simulated cloud network is completely flat.

Every service can communicate with every other service.

This is convenient because developers do not need to think carefully about network rules.

However, it creates a serious security problem.

If:

    INFERENCE SERVICE
          ↓
      COMPROMISED

the attacker may inherit network paths to:

    MODEL REGISTRY

    DATABASE

    TRAINING STORAGE

    SECRET MANAGER

    LOGGING SERVICE

A workload compromise can therefore become an infrastructure compromise.

The next part measures this blast radius.

</details>

In [ ]:
# @title ▶ Part 3: Simulate Compromise { display-mode: "form" }

compromised_service = "Inference Service"

reachable = flat_network[
    compromised_service
]


print("=" * 70)
print("COMPROMISED INFERENCE SERVICE")
print("=" * 70)

print(
    f"\nInitial compromise: "
    f"{compromised_service}"
)

print(
    "\nAttacker can directly reach:"
)

for service in reachable:
    print(f"  ⚠ {service}")

print(
    f"\nTotal reachable services: "
    f"{len(reachable)}"
)

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

The attacker initially compromises only one component:

    INFERENCE SERVICE

However, the flat network allows that component to communicate with nearly everything else.

This creates opportunities for **lateral movement**.

For example:

    COMPROMISED INFERENCE SERVICE
                ↓
          SECRET MANAGER
                ↓
         STOLEN CREDENTIAL

or:

    COMPROMISED INFERENCE SERVICE
                ↓
         TRAINING STORAGE
                ↓
         DATA MANIPULATION

The original vulnerability may exist in the inference service, but unrestricted network connectivity increases the potential impact.

</details>

In [ ]:
# @title ▶ Part 4: Apply Network Segmentation { display-mode: "form" }

segmented_network = {

    "API Gateway": [
        "Inference Service"
    ],

    "Inference Service": [
        "Model Registry",
        "Production Database",
        "Logging Service"
    ],

    "Model Registry": [],

    "Production Database": [],

    "Training Storage": [
        "Model Registry"
    ],

    "Secret Manager": [],

    "Logging Service": []
}


print("=" * 70)
print("SEGMENTED AI NETWORK")
print("=" * 70)

for source, destinations in segmented_network.items():

    print(f"\n{source}")

    if destinations:

        for destination in destinations:
            print(
                f"  → {destination}"
            )

    else:
        print(
            "  No unnecessary outgoing paths"
        )

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

The network now permits communication according to workload requirements.

The API gateway communicates with the inference service.

The inference service can reach resources needed for production inference.

It cannot directly communicate with:

    TRAINING STORAGE

    SECRET MANAGER

This changes the architecture from:

    EVERYTHING
       ↕
    EVERYTHING

to:

    REQUIRED COMMUNICATION
            ONLY

Network segmentation does not prevent the initial inference-service compromise.

It limits what the compromised service can reach afterward.

</details>

In [ ]:
# @title ▶ Part 5: Compare Lateral Movement { display-mode: "form" }

flat_reach = len(
    flat_network["Inference Service"]
)

segmented_reach = len(
    segmented_network["Inference Service"]
)

reduction = (
    (flat_reach - segmented_reach)
    / flat_reach
) * 100


print("=" * 70)
print("LATERAL MOVEMENT COMPARISON")
print("=" * 70)

print(
    f"\nFlat network reachable services: "
    f"{flat_reach}"
)

print(
    f"Segmented network reachable services: "
    f"{segmented_reach}"
)

print(
    f"Reachability reduction: "
    f"{reduction:.1f}%"
)


if segmented_reach < flat_reach:

    print(
        "\n✓ Segmentation reduced "
        "lateral-movement opportunities."
    )

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

The compromise is identical in both scenarios.

What changes is the surrounding architecture.

With the flat network:

    ONE COMPROMISE
          ↓
    MANY NETWORK PATHS

With segmentation:

    ONE COMPROMISE
          ↓
    LIMITED NETWORK PATHS

This is another example of reducing blast radius.

Network security should assume that individual workloads may eventually become compromised.

The objective is to prevent that compromise from automatically providing access to unrelated systems.

</details>

In [ ]:
# @title ▶ Part 6: Control Ingress Traffic { display-mode: "form" }

ingress_rules = {
    "API Gateway": [
        "Internet"
    ],

    "Inference Service": [
        "API Gateway"
    ],

    "Model Registry": [
        "Inference Service",
        "Training Service"
    ],

    "Production Database": [
        "Inference Service"
    ],

    "Logging Service": [
        "Inference Service"
    ]
}


def ingress_allowed(
    source,
    destination
):

    allowed_sources = ingress_rules.get(
        destination,
        []
    )

    return source in allowed_sources


tests = [
    ("Internet", "API Gateway"),
    ("Internet", "Inference Service"),
    ("API Gateway", "Inference Service"),
    ("Internet", "Production Database"),
    ("Inference Service", "Production Database")
]


print("=" * 70)
print("INGRESS SECURITY TEST")
print("=" * 70)

for source, destination in tests:

    allowed = ingress_allowed(
        source,
        destination
    )

    result = (
        "✓ ALLOWED"
        if allowed
        else "✗ BLOCKED"
    )

    print(
        f"\n{source} → {destination}"
        f"\n{result}"
    )

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

Ingress rules control traffic attempting to **enter** a service.

The Internet can reach:

    API GATEWAY

but cannot directly reach:

    INFERENCE SERVICE

    PRODUCTION DATABASE

The inference service accepts traffic from the API gateway because that communication is required.

This creates:

    INTERNET
       ↓
    API GATEWAY
       ↓
    INFERENCE SERVICE

rather than:

    INTERNET
      ↙ ↓ ↘
    API MODEL DATABASE

Public exposure is concentrated at the component specifically designed to receive external requests.

</details>

In [ ]:
# @title ▶ Part 7: Control Egress Traffic { display-mode: "form" }

egress_rules = {
    "Inference Service": [
        "Model Registry",
        "Production Database",
        "Logging Service"
    ]
}


def egress_allowed(
    source,
    destination
):

    allowed_destinations = (
        egress_rules.get(
            source,
            []
        )
    )

    return (
        destination
        in allowed_destinations
    )


tests = [
    (
        "Inference Service",
        "Production Database"
    ),

    (
        "Inference Service",
        "Logging Service"
    ),

    (
        "Inference Service",
        "Training Storage"
    ),

    (
        "Inference Service",
        "Secret Manager"
    ),

    (
        "Inference Service",
        "Unknown Internet Server"
    )
]


print("=" * 70)
print("EGRESS SECURITY TEST")
print("=" * 70)

for source, destination in tests:

    allowed = egress_allowed(
        source,
        destination
    )

    print(
        f"\n{source} → {destination}"
    )

    print(
        "✓ ALLOWED"
        if allowed
        else "✗ BLOCKED"
    )

<details>

<summary><b>Part 7 Explanation — Click to Expand</b></summary>

<br>

Ingress asks:

> **Who can connect to this service?**

Egress asks:

> **Where can this service connect?**

Both matter.

Imagine that malicious code successfully executes inside the inference service.

Without egress restrictions, it may attempt to:

    CONTACT ATTACKER SERVER

    ACCESS SECRET MANAGER

    REACH TRAINING STORAGE

    EXFILTRATE DATA

The egress policy restricts the service to destinations required for legitimate inference.

This can make it more difficult for a compromised workload to communicate with unauthorized resources.

</details>

In [ ]:
# @title ▶ Part 8: Check Service Encryption { display-mode: "form" }

connections = [
    {
        "source": "API Gateway",
        "destination": "Inference Service",
        "encrypted": True
    },

    {
        "source": "Inference Service",
        "destination": "Production Database",
        "encrypted": True
    },

    {
        "source": "Inference Service",
        "destination": "Logging Service",
        "encrypted": False
    }
]


print("=" * 70)
print("SERVICE ENCRYPTION AUDIT")
print("=" * 70)

for connection in connections:

    print(
        f"\n{connection['source']} "
        f"→ {connection['destination']}"
    )

    if connection["encrypted"]:

        print(
            "✓ Encrypted communication"
        )

    else:

        print(
            "⚠ Unencrypted communication"
        )

<details>

<summary><b>Part 8 Explanation — Click to Expand</b></summary>

<br>

Network restrictions determine which systems can communicate.

Encryption protects the information exchanged during that communication.

For example:

    INFERENCE SERVICE
           ↓
      ENCRYPTED CHANNEL
           ↓
        DATABASE

Sensitive information may travel between cloud services even when both systems are considered internal.

Internal network location should not automatically mean that communication can be trusted or transmitted without protection.

This supports the Zero-Trust principle:

> **Internal does not automatically mean trusted.**

</details>

In [ ]:
# @title ▶ Part 9: Audit Cloud Storage { display-mode: "form" }

storage_resources = {
    "Training Dataset": {
        "public": False,
        "authorized_services": [
            "Training Service"
        ]
    },

    "Production Models": {
        "public": False,
        "authorized_services": [
            "Inference Service",
            "Deployment Service"
        ]
    },

    "Public Demo Data": {
        "public": True,
        "authorized_services": []
    }
}


print("=" * 70)
print("CLOUD STORAGE SECURITY")
print("=" * 70)

for resource, config in storage_resources.items():

    print(f"\n{resource}")

    print(
        f"Public: "
        f"{config['public']}"
    )

    if config[
        "authorized_services"
    ]:

        print(
            "Authorized services:"
        )

        for service in config[
            "authorized_services"
        ]:
            print(f"  • {service}")

    else:

        print(
            "No restricted service list"
        )

<details>

<summary><b>Part 9 Explanation — Click to Expand</b></summary>

<br>

Cloud storage may contain some of the most valuable assets in an AI environment.

Examples include:

    TRAINING DATA

    PRODUCTION MODELS

    MODEL ARTIFACTS

    APPLICATION DATA

Sensitive storage should normally be private and accessible only to workloads that require it.

For example:

    INFERENCE SERVICE
           ↓
    PRODUCTION MODEL
           ✓

but:

    INFERENCE SERVICE
           ↓
    TRAINING DATASET
           ✗

Restricting storage access combines network security with identity and authorization controls from earlier sections.

</details>

In [ ]:
# @title ▶ Part 10: Apply Zero Trust { display-mode: "form" }

def zero_trust_request(
    authenticated,
    authorized,
    network_allowed,
    encrypted
):

    checks = {
        "Authenticated":
            authenticated,

        "Authorized":
            authorized,

        "Network Allowed":
            network_allowed,

        "Encrypted":
            encrypted
    }

    print("\nSecurity Checks:")

    for check, passed in checks.items():

        print(
            f"{check:<20} "
            f"{'✓ PASS' if passed else '✗ FAIL'}"
        )

    return all(
        checks.values()
    )


print("=" * 70)
print("ZERO-TRUST SERVICE REQUEST")
print("=" * 70)

allowed = zero_trust_request(
    authenticated=True,
    authorized=True,
    network_allowed=True,
    encrypted=True
)

print(
    "\nFinal Decision:",
    "✓ ALLOW" if allowed
    else "✗ DENY"
)

<details>

<summary><b>Part 10 Explanation — Click to Expand</b></summary>

<br>

The request is not trusted simply because it originates inside the cloud environment.

The simulated Zero-Trust decision checks several conditions:

    IDENTITY AUTHENTICATED?

    ACTION AUTHORIZED?

    NETWORK PATH ALLOWED?

    COMMUNICATION ENCRYPTED?

Only after the required controls succeed is the request permitted.

This combines concepts from several sections of the module.

    IAM
      +

    NETWORK POLICY
      +

    ENCRYPTION
      +

    LEAST PRIVILEGE

Security controls become stronger when they reinforce one another rather than depending on a single boundary.

</details>

In [ ]:
# @title ▶ Part 11: Compare Cloud Blast Radius { display-mode: "form" }

flat_targets = set(
    flat_network[
        "Inference Service"
    ]
)

hardened_targets = set(
    segmented_network[
        "Inference Service"
    ]
)


print("=" * 70)
print("CLOUD COMPROMISE COMPARISON")
print("=" * 70)

print("\nFLAT NETWORK")

for target in sorted(
    flat_targets
):
    print(f"  ⚠ {target}")

print(
    f"\nReachable resources: "
    f"{len(flat_targets)}"
)


print("\n" + "=" * 70)
print("\nHARDENED NETWORK")

for target in sorted(
    hardened_targets
):
    print(f"  • {target}")

print(
    f"\nReachable resources: "
    f"{len(hardened_targets)}"
)


blocked = (
    flat_targets
    - hardened_targets
)

print("\nResources no longer reachable:")

for target in sorted(blocked):
    print(f"  ✓ {target}")

<details>

<summary><b>Part 11 Explanation — Click to Expand</b></summary>

<br>

Assume again that the inference service has been successfully compromised.

The vulnerable architecture gives the attacker many network paths.

The hardened architecture still allows the inference service to perform its legitimate responsibilities, but unrelated resources are no longer directly reachable.

The goal is:

    REQUIRED ACCESS
          ✓

while removing:

    UNNECESSARY ACCESS
          ✗

This is especially important in cloud environments because a single workload may exist alongside databases, storage systems, model registries, administrative services, and other sensitive resources.

Network architecture determines whether compromise remains isolated or becomes an opportunity for lateral movement.

</details>

In [ ]:
# @title ▶ Part 12: Knowledge Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. Which service needed direct Internet exposure in the lab?",
        "options": [
            "A. API Gateway",
            "B. Training Storage",
            "C. Secret Manager",
            "D. Production Database"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. What was the main weakness of the flat network?",
        "options": [
            "A. Services could communicate with many unnecessary resources",
            "B. It used too much encryption",
            "C. It had too much segmentation",
            "D. The API gateway was private"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. What did segmentation reduce?",
        "options": [
            "A. Lateral-movement opportunities",
            "B. Authentication",
            "C. Model accuracy",
            "D. Encryption"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Why was the Internet blocked from directly reaching the inference service?",
        "options": [
            "A. External traffic should enter through the intended public gateway",
            "B. Inference services cannot use networks",
            "C. Models cannot process Internet requests",
            "D. It improves training accuracy"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. What did the egress policy prevent?",
        "options": [
            "A. The inference service from contacting unnecessary destinations",
            "B. Users from authenticating",
            "C. Models from loading",
            "D. Logs from being created"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. Why was encryption checked between internal services?",
        "options": [
            "A. Internal traffic may still contain sensitive information",
            "B. Internal systems never require security",
            "C. Encryption replaces authorization",
            "D. Encryption makes storage public"
        ],
        "answer": "A"
    },
    {
        "question":
        "7. Why was the inference service denied access to training data?",
        "options": [
            "A. It did not require training data for its normal function",
            "B. Training data cannot be stored in the cloud",
            "C. AI services cannot access storage",
            "D. The dataset was too small"
        ],
        "answer": "A"
    },
    {
        "question":
        "8. What was the main lesson from the Zero-Trust test?",
        "options": [
            "A. Internal requests should still satisfy explicit security checks",
            "B. Internal workloads should automatically trust each other",
            "C. Authentication is unnecessary inside cloud networks",
            "D. Every service should be publicly accessible"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("LAB 6 — KNOWLEDGE CHECK")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1

    else:
        print(
            f"✗ Incorrect. "
            f"Correct answer: {q['answer']}"
        )

percentage = (
    score / len(questions)
) * 100

print("\n" + "=" * 70)
print(f"Lab Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Lab complete!")

else:
    print(
        "△ Review the lab explanations "
        "and try again."
    )

<details>

<summary><b>Lab 6 — Practical Takeaways — Click to Expand</b></summary>

<br>

## Lab 6 — Practical Takeaways

This lab demonstrated how cloud and network architecture can determine the impact of an AI workload compromise.

You began with a flat network where the inference service could communicate with nearly every other cloud resource.

This created:

    COMPROMISED INFERENCE SERVICE
                ↓
       MANY NETWORK PATHS
                ↓
       LATERAL MOVEMENT
                ↓
       LARGER BLAST RADIUS

You then applied network segmentation.

The inference service retained access to resources required for normal operation while losing unnecessary paths to unrelated systems.

You also examined the difference between:

    INGRESS
    → What can connect to the service

and:

    EGRESS
    → Where the service can connect

Both directions matter.

A compromised workload may attempt to reach sensitive internal systems or communicate externally with an attacker.

You also required encrypted service communication and restricted access to cloud storage.

Finally, the Zero-Trust example combined multiple controls:

    AUTHENTICATION

    AUTHORIZATION

    NETWORK POLICY

    ENCRYPTION

The central lesson is:

> **A compromised AI component should not automatically become a path to the rest of the cloud environment.**

Strong network and cloud security limits public exposure, restricts communication to required paths, protects data in transit, isolates workloads, and reduces opportunities for lateral movement.

The important question remains:

> **If this component is compromised, what can the attacker reach next?**

A secure architecture should make the answer:

> **Only the minimum resources the workload actually needs.**

</details>

In [ ]:
# @title ▶ Section 6 Checkpoint — Network & Cloud Infrastructure Security { display-mode: "form" }

questions = [
    {
        "question":
        "1. Why does cloud deployment expand the AI attack surface?",
        "options": [
            "A. AI workloads interact with networks, storage, identities, APIs, and other cloud services",
            "B. Cloud systems do not use networks",
            "C. Models automatically become public",
            "D. Cloud deployment removes infrastructure"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. Which component most likely needs a public endpoint?",
        "options": [
            "A. Customer-facing API gateway",
            "B. Internal model registry",
            "C. Training database",
            "D. Secret-management service"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. What is the purpose of network segmentation?",
        "options": [
            "A. Separate workloads and restrict unnecessary communication",
            "B. Allow every service to communicate freely",
            "C. Remove authentication",
            "D. Make all storage public"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. What does an ingress control manage?",
        "options": [
            "A. Traffic entering a service or network",
            "B. Model accuracy",
            "C. Training labels",
            "D. Password length"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. What does an egress control manage?",
        "options": [
            "A. Traffic leaving a service or network",
            "B. Only incoming Internet traffic",
            "C. Model training",
            "D. User authentication only"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. Why should communication between AI services be encrypted?",
        "options": [
            "A. To protect data while it travels between services",
            "B. To eliminate authorization",
            "C. To make every endpoint public",
            "D. To prevent all software vulnerabilities"
        ],
        "answer": "A"
    },
    {
        "question":
        "7. What is lateral movement?",
        "options": [
            "A. An attacker moving from one compromised resource to other systems",
            "B. A model changing predictions",
            "C. Moving a container between screens",
            "D. Uploading a training dataset"
        ],
        "answer": "A"
    },
    {
        "question":
        "8. What is the goal of workload isolation?",
        "options": [
            "A. Limit how one compromised workload can affect others",
            "B. Give every workload administrator access",
            "C. Allow unrestricted internal traffic",
            "D. Disable monitoring"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("SECTION 6 CHECKPOINT")
print("NETWORK AND CLOUD INFRASTRUCTURE SECURITY")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(
            f"✗ Incorrect. Correct answer: "
            f"{q['answer']}"
        )

percentage = score / len(questions) * 100

print("\n" + "=" * 70)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Checkpoint complete!")
else:
    print("△ Review Section 6 and try again.")

---

<details>
<summary><b>Section 7 — Model Artifact and Deployment Pipeline Security — Click to Expand</b></summary>
<br>

## Section 7 — Model Artifact and Deployment Pipeline Security

A machine learning model is not secure simply because it performed well during training and evaluation. Before a model reaches production, it may pass through experiment tracking systems, model registries, automated pipelines, storage locations, approval processes, and deployment infrastructure.

Each stage creates an opportunity for the model artifact to be replaced, modified, misidentified, or deployed without appropriate verification.

This makes the **model artifact and the deployment pipeline part of the AI supply chain**.

The central security question for this section is:

**How can an organization prove that the model running in production is the same trusted, approved, and tested model that was intended for deployment?**

This section examines model integrity, provenance, lineage, registries, third-party models, cryptographic hashes, digital signatures, CI/CD security, deployment gates, versioning, separation of duties, and secure rollback.

<br>

<details>
<summary><b>7.1 From Trained Model to Production</b></summary>
<br>

### From Trained Model to Production

Training a machine learning model is only one step in the model lifecycle.

After training, a model may move through several stages before it is used in production.

A simplified lifecycle may look like:

**Training**

↓

**Evaluation**

↓

**Model Artifact Created**

↓

**Registry or Storage**

↓

**Security / Quality Review**

↓

**Approval**

↓

**Deployment**

↓

**Production**

At each stage, the model may be copied, stored, renamed, scanned, tested, approved, or transferred between systems.

This creates several important questions:

- Is this the correct model?
- Has the model changed since evaluation?
- Who created it?
- Which dataset was used?
- Which code and dependencies were used?
- Who approved it?
- Where did it come from?
- Was the deployment authorized?
- Can the organization prove its history?

Atlas, a framework for ML lifecycle provenance and transparency, emphasizes the value of tracking model artifacts and lifecycle events so that organizations can better understand how a deployed model was created and handled [26].

A secure deployment process should therefore preserve trust from:

**Model Creation**

to:

**Production Execution**

</details>

<br>

<details>
<summary><b>7.2 Understanding the ML Deployment Pipeline</b></summary>
<br>

### Understanding the ML Deployment Pipeline

An ML deployment pipeline automates or coordinates the steps required to move a model from development into production.

A pipeline may include:

**Source Code**

↓

**Training**

↓

**Model Evaluation**

↓

**Security Testing**

↓

**Artifact Storage**

↓

**Approval**

↓

**Deployment**

MLOps practices use automation to make these processes more repeatable and consistent [23].

Automation provides important benefits, but it also means that compromise of the pipeline itself can have serious consequences.

If an attacker controls the deployment pipeline, the attacker may be able to:

- Deploy an unauthorized model
- Replace an approved artifact
- Modify configuration
- Steal deployment credentials
- Disable security tests
- Change production infrastructure

The pipeline therefore becomes a high-value security target.

A useful principle is:

**The deployment system should not be trusted simply because it is automated.**

Automation must still enforce identity, authorization, integrity, testing, and approval.

</details>

<br>

<details>
<summary><b>7.3 The Model Artifact as a Security Asset</b></summary>
<br>

### The Model Artifact as a Security Asset

A trained model is usually stored as some form of artifact.

Examples may include:

- Serialized model files
- Checkpoints
- Weight files
- Saved neural networks
- ONNX models
- Pickled models
- Framework-specific model formats

The artifact may contain valuable intellectual property and the learned behavior of the model.

It should therefore be treated as a sensitive security asset.

Potential risks include:

- Model theft
- Unauthorized modification
- Model replacement
- Accidental corruption
- Deployment of an outdated model
- Distribution of a malicious or backdoored model

For example:

**Approved Model**

↓

**Artifact Replaced**

↓

**Modified Model**

↓

**Production Deployment**

The application and deployment infrastructure may function normally, yet the actual model behavior may no longer be trustworthy.

This is why model security includes both:

**Confidentiality**

and:

**Integrity**

Confidentiality helps prevent unauthorized access to the artifact.

Integrity helps ensure the artifact has not been changed without authorization.

</details>

<br>

<details>
<summary><b>7.4 Model Integrity and Authenticity</b></summary>
<br>

### Model Integrity and Authenticity

Two closely related concepts are **integrity** and **authenticity**.

Integrity asks:

**Has the model changed?**

Authenticity asks:

**Is this model really from the expected source?**

Consider a production system that expects:

**approved_model_v4**

An attacker replaces the file with another model but keeps the same filename.

The deployment system sees:

`approved_model_v4`

and assumes it is legitimate.

This demonstrates an important lesson:

**Filename ≠ Proof of Identity**

Similarly:

**Storage Location ≠ Proof of Integrity**

A model stored in the expected folder may still have been modified.

Secure deployment therefore needs stronger evidence than names and locations.

Cryptographic mechanisms such as hashing and digital signatures can help verify model artifacts before deployment.

This allows the deployment system to ask:

**Does this artifact match the trusted artifact we approved?**

rather than simply:

**Does a file with the expected name exist?**

</details>

<br>

<details>
<summary><b>7.5 Model Provenance and Lineage</b></summary>
<br>

### Model Provenance and Lineage

Model security is not only about the final file.

Defenders may also need to understand how the model was created.

**Provenance** refers to information about the origin and history of an artifact.

**Lineage** tracks the chain of data, code, processes, and decisions that contributed to the final model.

Useful lineage information may include:

- Training dataset version
- Training code version
- Hyperparameters
- ML framework version
- Dependency versions
- Model evaluation results
- Security test results
- Model creator
- Approval records
- Deployment history

A simplified lineage record may look like:

**Dataset v3**

+

**Training Code v8**

+

**PyTorch Version X**

+

**Configuration Y**

↓

**Model v5**

↓

**Evaluation Passed**

↓

**Security Review Passed**

↓

**Approved for Production**

Atlas focuses on provenance and transparency across the machine learning lifecycle so that model artifacts can be connected to trusted processes and lifecycle records [26].

Without provenance, defenders may know:

**"We have a model."**

With provenance, they can begin answering:

**"Where did it come from, how was it created, and why should we trust it?"**

</details>

<br>

<details>
<summary><b>7.6 Model Registries and Model Hubs</b></summary>
<br>

### Model Registries and Model Hubs

A **model registry** provides centralized storage and management for model artifacts and associated metadata.

A registry may track:

- Model versions
- Approval status
- Evaluation results
- Deployment stage
- Ownership
- Metadata
- Model lineage

A simplified workflow could be:

**Training**

↓

**Candidate Model**

↓

**Model Registry**

↓

**Approved Production Version**

↓

**Deployment**

This provides more control than storing production models in arbitrary folders or manually copying them between systems.

However, the registry itself becomes a sensitive resource.

Attackers may attempt to:

- Upload unauthorized models
- Replace approved versions
- Delete models
- Modify metadata
- Steal proprietary models

Access to the registry should therefore follow least privilege.

For example:

**Data Scientist → Upload Candidate Model**

**Approver → Change Approval Status**

**Deployment Service → Read Approved Model**

**Inference Service → Read Production Artifact**

These responsibilities should not automatically be combined into one unrestricted account.

</details>

<br>

<details>
<summary><b>7.7 Risks of Third-Party and Pretrained Models</b></summary>
<br>

### Risks of Third-Party and Pretrained Models

Modern AI development frequently begins with models created by someone else.

Organizations may download:

- Pretrained neural networks
- Foundation models
- Embedding models
- Computer-vision models
- NLP models
- Community checkpoints

These models can significantly reduce development time.

However:

**Downloaded Model ≠ Trusted Model**

Meiklejohn et al. argue that machine learning models face a supply-chain problem involving artifacts, frameworks, datasets, and distribution mechanisms [25].

Third-party models may contain:

- Backdoors
- Malicious behavior
- Unknown training data
- Vulnerable serialization formats
- Licensing restrictions
- Hidden dependencies
- Poorly documented provenance

Research into model supply-chain poisoning has also demonstrated that malicious behavior can potentially be introduced into pretrained models before downstream organizations use them [24].

Third-party models should therefore be treated like other external software dependencies.

Before production use, organizations should consider:

- Source reputation
- Model provenance
- Integrity
- Documentation
- Security testing
- Licensing
- Evaluation
- Known vulnerabilities
- Serialization risks

Convenience should not eliminate verification.

</details>

<br>

<details>
<summary><b>7.8 Hashing and Integrity Verification</b></summary>
<br>

### Hashing and Integrity Verification

A cryptographic hash produces a fixed-length value derived from the contents of a file.

For example:

**Model File**

↓

**SHA-256**

↓

`ExampleHashValue`

If even a small part of the model file changes, the resulting hash should also change.

This allows defenders to compare:

**Approved Model Hash**

with:

**Deployment Model Hash**

A deployment decision might look like:

**Calculate Hash**

↓

**Matches Approved Hash?**

**Yes → Continue**

**No → Block Deployment**

For example:

**Approved Artifact**

`SHA256 = ABC123`

**Production Candidate**

`SHA256 = ABC123`

→ Integrity Match

If instead:

`SHA256 = XYZ999`

→ Artifact Changed

Hashing can provide evidence that two files are identical.

However, hashing alone does not prove who created the trusted value.

If an attacker can replace both:

**Model Artifact**

and:

**Expected Hash**

then the comparison may still succeed.

This is why hashing is often combined with controlled registries, access controls, digital signatures, and provenance records.

</details>

<br>

<details>
<summary><b>7.9 Digital Signatures and Model Signing</b></summary>
<br>

### Digital Signatures and Model Signing

Digital signatures can provide stronger evidence about both artifact integrity and origin.

A simplified signing process looks like:

**Approved Model**

↓

**Cryptographic Hash**

↓

**Private Signing Key**

↓

**Digital Signature**

The deployment system can then verify the signature using the corresponding trusted public key.

The process becomes:

**Model + Signature**

↓

**Verify**

↓

**Valid?**

**Yes → Trusted Artifact**

**No → Reject**

If the model changes after signing, verification should fail.

This provides protection against unauthorized artifact modification.

Meiklejohn et al. discuss the relevance of software supply-chain technologies such as Sigstore to machine learning artifacts and argue for stronger provenance and signing mechanisms for ML models [25].

Model signing therefore supports an important deployment principle:

**Production should accept verified artifacts rather than arbitrary model files.**

Signing is not a substitute for model testing.

A signed model may still contain insecure behavior if an authorized party signs a bad model.

Signatures answer:

**"Is this the artifact that an authorized source signed?"**

They do not automatically answer:

**"Is this model safe?"**

Both security testing and artifact verification are necessary.

</details>

<br>

<details>
<summary><b>7.10 Secure CI/CD for AI/ML</b></summary>
<br>

### Secure CI/CD for AI/ML

Continuous Integration and Continuous Delivery or Deployment, commonly called **CI/CD**, automates software build, test, and deployment processes.

MLOps applies similar principles to machine learning systems [23].

A secure ML pipeline might perform:

**Code Change**

↓

**Train / Build**

↓

**Model Evaluation**

↓

**Security Testing**

↓

**Artifact Verification**

↓

**Approval**

↓

**Deploy**

This automation can improve consistency because security checks can occur every time a new model version is produced.

However, CI/CD pipelines often require powerful credentials.

A deployment pipeline may have permission to:

- Access registries
- Upload artifacts
- Modify cloud resources
- Deploy production models
- Restart services

If attackers compromise the pipeline:

**Pipeline Compromise**

↓

**Deployment Credentials**

↓

**Production Access**

For this reason, pipeline security should include:

- Strong authentication
- Least-privilege service identities
- Protected secrets
- Audit logging
- Restricted pipeline modification
- Security gates
- Artifact verification
- Controlled production permissions

The deployment pipeline should be treated as production infrastructure rather than merely a convenience tool.

</details>

<br>

<details>
<summary><b>7.11 Security Testing and Deployment Gates</b></summary>
<br>

### Security Testing and Deployment Gates

A **deployment gate** is a condition that must be satisfied before an artifact can proceed to the next stage.

Instead of:

**Model Created → Production**

a secure process can require:

**Model Created**

↓

**Performance Check**

↓

**Security Check**

↓

**Integrity Verification**

↓

**Approval**

↓

**Production**

Possible gates include:

- Minimum performance requirements
- Robustness testing
- Malware scanning
- Dependency scanning
- Container scanning
- Model hash verification
- Signature verification
- Bias or policy checks
- Approval requirements

If a gate fails:

**Deployment Stops**

This prevents security checks from becoming optional steps that can easily be skipped.

A pipeline may implement:

**Critical Vulnerability Found?**

**Yes → BLOCK**

**No → Continue**

or:

**Artifact Signature Valid?**

**No → BLOCK**

The important security principle is:

**Security requirements should be enforced by the deployment process whenever practical, rather than depending entirely on someone remembering to perform them manually.**

</details>

<br>

<details>
<summary><b>7.12 Model Versioning</b></summary>
<br>

### Model Versioning

Production AI systems may go through many model versions.

For example:

**Model v1**

↓

**Model v2**

↓

**Model v3**

↓

**Model v4**

Each version may use different:

- Training data
- Code
- Hyperparameters
- Dependencies
- Security controls
- Evaluation results

Without versioning, organizations may struggle to determine which model is actually running.

A weak deployment might simply use:

`model.pkl`

A stronger system may track:

**Model ID:** fraud_model

**Version:** 4.2

**Hash:** ABC123

**Status:** Production Approved

**Deployment Date:** Recorded

Versioning allows defenders to answer:

- Which model is currently deployed?
- Which model was deployed during an incident?
- When did the version change?
- Which version produced a prediction?
- Which version should be restored during rollback?

This becomes essential for troubleshooting, auditability, incident response, and secure recovery.

</details>

<br>

<details>
<summary><b>7.13 Production Approval and Separation of Duties</b></summary>
<br>

### Production Approval and Separation of Duties

Section 3 introduced the principle of separation of duties.

This principle is especially important for model deployment.

A developer who creates a model should not necessarily have unrestricted authority to place that model directly into production.

Consider:

**Developer**

↓

**Creates Model**

↓

**Self-Approves**

↓

**Deploys to Production**

This creates a single point of control.

A stronger workflow is:

**Developer → Creates Model**

↓

**Automated Tests → Validate**

↓

**Reviewer / Approver → Authorizes**

↓

**Deployment Service → Deploys**

This makes unauthorized production deployment more difficult.

Permissions can also be separated:

**Developer**

- Create candidate
- Upload candidate
- Cannot deploy production

**Approver**

- Review candidate
- Approve or reject
- Cannot modify model

**Deployment Service**

- Deploy approved artifact
- Cannot approve artifact

This structure reduces both accidental and malicious misuse.

It also provides clearer audit records showing who performed each stage.

</details>

<br>

<details>
<summary><b>7.14 Secure Rollback and Recovery</b></summary>
<br>

### Secure Rollback and Recovery

Even carefully reviewed models can cause problems after deployment.

A new model may produce:

- Unexpected predictions
- Performance degradation
- Security weaknesses
- Operational failures
- Compatibility problems

Organizations therefore need the ability to **rollback** to a trusted previous version.

A secure deployment history might contain:

**Model v3 — Trusted Previous**

**Model v4 — Current**

If v4 creates a serious problem:

**Detect Problem**

↓

**Stop / Contain v4**

↓

**Redeploy Verified v3**

↓

**Validate Recovery**

Rollback should not mean selecting any older file that happens to exist.

The previous model should still be:

- Approved
- Versioned
- Integrity verified
- Available from trusted storage
- Compatible with the current environment

Attackers could potentially exploit rollback mechanisms if older vulnerable artifacts can be deployed without control.

Therefore:

**Rollback must be controlled just like forward deployment.**

A rollback event should also be logged so investigators can determine:

- Who initiated it
- Why it occurred
- Which artifact was restored
- When the change occurred

</details>

<br>

<details>
<summary><b>7.15 Connecting Pipeline Security to AI Attacks</b></summary>
<br>

### Connecting Pipeline Security to AI Attacks

Deployment-pipeline weaknesses can provide attackers with a direct path to production AI behavior.

Consider several scenarios.

#### Artifact Replacement

**Approved Model**

↓

**Attacker Replaces File**

↓

**Deployment Does Not Verify Hash**

↓

**Malicious Model Deployed**

#### Compromised Model Registry

**Registry Credential Stolen**

↓

**Production Artifact Modified**

↓

**Unauthorized Model Distributed**

#### Compromised CI/CD Pipeline

**Deployment Service Compromised**

↓

**Production Access Obtained**

↓

**Unauthorized Model Deployed**

#### Untrusted Third-Party Model

**Model Downloaded**

↓

**No Provenance or Security Review**

↓

**Backdoored Model Introduced**

#### Missing Version Control

**Production Model Changes**

↓

**No Reliable Version Record**

↓

**Incident Investigation Becomes Difficult**

These scenarios demonstrate that attackers may not need to manipulate the model after it reaches production.

Instead, they may attack:

**The process used to place the model into production.**

This makes model provenance, artifact integrity, deployment authorization, and pipeline security essential components of AI defense.

</details>

<br>

<details>
<summary><b>Section 7 Summary</b></summary>
<br>

### Section 7 Summary

A model deployment pipeline determines how trained models move from development into production. If that pipeline cannot verify the identity, integrity, origin, and approval status of an artifact, attackers may be able to introduce unauthorized models into production.

Throughout this section, you learned that:

- Models pass through multiple systems and processes before reaching production.
- The model artifact itself should be treated as a sensitive security asset.
- Integrity determines whether an artifact has changed.
- Authenticity helps determine whether an artifact came from the expected source.
- Provenance and lineage record how a model was created and handled throughout its lifecycle.
- Model registries provide centralized management but must be secured against unauthorized modification.
- Third-party and pretrained models introduce software supply-chain risks.
- Cryptographic hashes can detect unexpected artifact changes.
- Digital signatures can help verify artifact integrity and origin.
- CI/CD and MLOps pipelines require strong identities, protected credentials, logging, and least privilege.
- Deployment gates can automatically enforce security requirements before production release.
- Model versioning improves traceability, auditability, incident investigation, and rollback.
- Separation of duties can prevent one identity from controlling model creation, approval, and deployment.
- Rollback procedures should restore known, verified, and approved artifacts.
- Attackers may target the deployment process itself rather than attacking the model after production deployment.

The next and final section examines **Logging, Monitoring, Incident Response, and Secure Operations**, where you will learn how organizations detect suspicious behavior, identify AI security incidents, contain compromised components, restore trusted models, and improve defenses after an incident.

</details>

</details>

In [ ]:

# @title Section 7 — Model Artifact and Deployment Pipeline Security Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import ListedColormap
from IPython.display import display, HTML
from io import BytesIO
import base64
import hashlib
import html

# ============================================================
# CONFIGURATION
# ============================================================

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"
TEAL = "#0891B2"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):
    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")
    buffer.close()

    display(HTML(f"""
    <details style="margin:12px 0;border:1px solid #cbd5e1;
                    border-radius:10px;overflow:hidden;">
      <summary style="cursor:pointer;padding:16px;font-size:16px;
                      font-weight:bold;background:#eff6ff;
                      color:#1e3a8a;border-left:5px solid {color};">
        {html.escape(title)} — Click to Expand
      </summary>
      <div style="padding:15px;background:white;text-align:center;">
        <img src="data:image/png;base64,{encoded}"
             style="width:100%;max-width:950px;height:auto;">
        <div style="max-width:850px;margin:15px auto;padding:16px;
                    background:#f8fafc;border-radius:8px;color:#334155;
                    font-size:14px;line-height:1.7;text-align:left;">
          {explanation}
        </div>
      </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def setup_diagram(ax, title, xlim=(0, 12), ylim=(0, 10)):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.axis("off")
    ax.set_title(
        title,
        fontsize=15,
        fontweight="bold",
        pad=18
    )


def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC",
             title_size=10):

    card = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.8
    )
    ax.add_patch(card)

    ax.text(
        x+w/2,
        y+h*(0.66 if subtitle else 0.5),
        title,
        ha="center",
        va="center",
        fontsize=title_size,
        fontweight="bold",
        color=DARK
    )

    if subtitle:
        ax.text(
            x+w/2,
            y+h*0.28,
            subtitle,
            ha="center",
            va="center",
            fontsize=8.5,
            color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):
    ax.add_patch(FancyArrowPatch(
        start, end,
        arrowstyle="-|>",
        mutation_scale=16,
        linewidth=1.8,
        color=color
    ))


# ============================================================
# VISUALIZATION 1
# SECURE ML DEPLOYMENT PIPELINE
# ============================================================

fig, ax = plt.subplots(figsize=(12, 11))

setup_diagram(
    ax,
    "Secure Machine Learning Deployment Pipeline",
    xlim=(0, 12),
    ylim=(0, 13)
)

stages = [
    ("Training Data",
     "Approved datasets and preprocessing", BLUE),
    ("Model Training",
     "Reproducible training process", PURPLE),
    ("Model Evaluation",
     "Performance and security testing", ORANGE),
    ("Artifact Packaging",
     "Save model and required metadata", TEAL),
    ("Model Registry",
     "Store versioned, verified artifacts", BLUE),
    ("Deployment Approval",
     "Security gates and authorized review", RED),
    ("Production Deployment",
     "Release approved model version", GREEN)
]

for i, (title, subtitle, color) in enumerate(stages):
    y = 11.3 - i*1.55

    add_card(
        ax, 2.0, y, 8.0, 1.1,
        title, subtitle, color
    )

    if i < len(stages)-1:
        add_arrow(
            ax,
            (6.0, y-0.05),
            (6.0, y-0.45)
        )

ax.text(
    6.0, 0.65,
    "Security verification should occur throughout the pipeline",
    ha="center",
    fontsize=10,
    fontweight="bold",
    color=DARK
)

plt.tight_layout()

show_collapsible_figure(
    "7.1 — Secure ML Deployment Pipeline",
    fig,
    """
    A machine learning deployment pipeline
    describes how a trained model moves
    from development into production.

    <br><br>

    The process begins with approved
    training data and a controlled
    model training environment.

    <br><br>

    After training, the model
    undergoes evaluation to determine
    whether it satisfies performance
    and security requirements.

    <br><br>

    The model is then packaged
    into a deployable artifact.

    <br><br>

    This artifact may contain
    model parameters, architecture
    information, preprocessing
    components, and metadata.

    <br><br>

    A model registry stores
    approved versions and
    provides a controlled
    location for retrieving them.

    <br><br>

    Before production deployment,
    the organization should verify
    that the artifact is authorized
    and has passed the required
    security checks.

    <br><br>

    The pipeline should also
    support logging, traceability,
    and recovery if a deployed
    model is later found to
    be compromised.

    <br><br>

    <b>Key takeaway:</b> Securing
    model deployment requires
    protecting every transition
    between training, evaluation,
    artifact storage, approval,
    and production.
    """,
    BLUE
)


# ============================================================
# VISUALIZATION 2
# MODEL ARTIFACT INTEGRITY AND AUTHENTICITY
# ============================================================

# Harmless demonstration using synthetic model-file bytes.
# These are NOT actual serialized ML models.

approved_artifact = b"illustrative-model-v1-weights"
modified_artifact = b"illustrative-model-v1-weights-modified"

approved_hash = hashlib.sha256(
    approved_artifact
).hexdigest()

modified_hash = hashlib.sha256(
    modified_artifact
).hexdigest()

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

fig.suptitle(
    "Model Artifact Integrity Verification with SHA-256",
    fontsize=16,
    fontweight="bold"
)

for ax, title, observed_hash, color in [
    (axes[0], "Unmodified Artifact", approved_hash, GREEN),
    (axes[1], "Modified Artifact", modified_hash, RED)
]:

    setup_diagram(ax, title, (0, 6), (0, 10))

    add_card(
        ax, 0.6, 8.0, 4.8, 1.1,
        "Stored Model Artifact",
        "Illustrative file contents",
        BLUE
    )

    add_arrow(ax, (3.0, 8.0), (3.0, 7.0))

    add_card(
        ax, 0.6, 5.8, 4.8, 1.1,
        "SHA-256 Calculation",
        "Compute artifact digest",
        PURPLE
    )

    add_arrow(ax, (3.0, 5.8), (3.0, 4.8))

    matches = observed_hash == approved_hash

    add_card(
        ax, 0.6, 3.6, 4.8, 1.1,
        "MATCH — Integrity Verified"
        if matches else "MISMATCH — Reject Artifact",
        "Compared with trusted expected digest",
        color,
        title_size=9.5
    )

    ax.text(
        3.0, 2.3,
        "Observed SHA-256 prefix:",
        ha="center", fontsize=10, color=GRAY
    )

    ax.text(
        3.0, 1.7,
        observed_hash[:24] + "...",
        ha="center", fontsize=10,
        fontweight="bold", color=color,
        family="monospace"
    )

plt.tight_layout()

show_collapsible_figure(
    "7.2 — Model Artifact Integrity and Authenticity",
    fig,
    """
    A model artifact is a valuable
    security asset because it contains
    information needed to execute
    a trained model.

    <br><br>

    If an attacker modifies an artifact,
    the deployed model may behave
    differently from the approved version.

    <br><br>

    A cryptographic hash provides
    a way to detect changes
    to the artifact's contents.

    <br><br>

    In this example, SHA-256
    produces a digest from
    illustrative model-file bytes.

    <br><br>

    The unmodified artifact matches
    the expected digest.

    <br><br>

    The modified artifact produces
    a different digest and fails
    the integrity comparison.

    <br><br>

    However, <b>hashing alone does
    not establish authenticity</b>.

    <br><br>

    An attacker who can replace
    both the artifact and its
    expected hash may defeat
    an unprotected comparison.

    <br><br>

    Digital signatures provide
    stronger authenticity assurances
    when the signing key is trusted
    and properly protected.

    <br><br>

    A digital signature can help
    verify that an artifact was
    approved by the expected
    signing identity and has
    not been altered afterward.

    <br><br>

    <b>Key takeaway:</b> Hashes help
    verify integrity, while trusted
    signatures and provenance
    information help establish
    artifact authenticity.
    """,
    GREEN
)


# ============================================================
# VISUALIZATION 3
# MODEL PROVENANCE AND LINEAGE
# ============================================================

fig, ax = plt.subplots(figsize=(12, 9))

setup_diagram(
    ax,
    "Model Provenance and Lineage Tracking",
    xlim=(0, 12),
    ylim=(0, 11)
)

sources = [
    (0.3, 8.3, "Dataset Version", "training-data-v3", BLUE),
    (4.4, 8.3, "Training Code", "commit: abc123", PURPLE),
    (8.5, 8.3, "Dependencies", "environment-v2", ORANGE)
]

for x, y, title, subtitle, color in sources:
    add_card(
        ax, x, y, 3.2, 1.2,
        title, subtitle, color
    )

add_card(
    ax, 4.4, 5.6, 3.2, 1.2,
    "Training Run",
    "run-1042",
    TEAL
)

for x in [1.9, 6.0, 10.1]:
    add_arrow(
        ax, (x, 8.3), (6.0, 6.8)
    )

add_card(
    ax, 4.4, 3.1, 3.2, 1.2,
    "Model Artifact",
    "classifier-v1.4",
    GREEN
)

add_arrow(
    ax, (6.0, 5.6), (6.0, 4.3)
)

add_card(
    ax, 4.4, 0.7, 3.2, 1.2,
    "Production Release",
    "deployment-2026-07",
    BLUE
)

add_arrow(
    ax, (6.0, 3.1), (6.0, 1.9)
)

plt.tight_layout()

show_collapsible_figure(
    "7.3 — Model Provenance and Lineage",
    fig,
    """
    Model provenance describes
    where a model came from
    and how it was produced.

    <br><br>

    Model lineage records the
    relationships between datasets,
    training code, dependencies,
    training runs, model artifacts,
    and deployed versions.

    <br><br>

    For example, an organization
    may record which dataset version
    was used to train a classifier,
    which source code commit
    executed the training process,
    and which software dependencies
    were installed.

    <br><br>

    This information helps
    establish traceability.

    <br><br>

    If a training dataset
    is later found to contain
    poisoned records, defenders
    can identify which model
    versions may be affected.

    <br><br>

    Similarly, if a dependency
    is found to be compromised,
    lineage records can help
    locate artifacts built
    using that dependency.

    <br><br>

    Provenance information also
    supports reproducibility,
    auditing, and incident
    response investigations.

    <br><br>

    The identifiers in this
    diagram are fictional
    teaching examples.

    <br><br>

    <b>Key takeaway:</b> A deployed
    model should be traceable
    to the approved data,
    code, environment, and
    training process that
    produced it.
    """,
    PURPLE
)


# ============================================================
# VISUALIZATION 4
# MODEL REGISTRY AND THIRD-PARTY TRUST
# ============================================================

fig, ax = plt.subplots(figsize=(12, 10))

setup_diagram(
    ax,
    "Trusted Model Registry and External Artifact Verification",
    xlim=(0, 12),
    ylim=(0, 12)
)

add_card(
    ax, 0.3, 9.0, 3.3, 1.2,
    "Internal Training",
    "Organization-controlled models",
    BLUE
)

add_card(
    ax, 8.4, 9.0, 3.3, 1.2,
    "External Model Hub",
    "Third-party pretrained models",
    ORANGE
)

add_card(
    ax, 4.4, 6.8, 3.2, 1.3,
    "Security Verification",
    "Source, signature, hash, tests",
    PURPLE
)

add_arrow(
    ax, (3.6, 9.6), (4.8, 8.1)
)

add_arrow(
    ax, (8.4, 9.6), (7.2, 8.1)
)

add_card(
    ax, 4.4, 4.4, 3.2, 1.2,
    "Approved Model Registry",
    "Versioned and access-controlled",
    GREEN
)

add_arrow(
    ax, (6.0, 6.8), (6.0, 5.6),
    GREEN
)

add_card(
    ax, 4.4, 1.8, 3.2, 1.2,
    "Production Deployment",
    "Approved model version",
    TEAL
)

add_arrow(
    ax, (6.0, 4.4), (6.0, 3.0),
    GREEN
)

ax.text(
    9.6, 5.5,
    "Failed verification:\nDo not promote",
    ha="center",
    fontsize=10,
    fontweight="bold",
    color=RED
)

ax.text(
    6.0, 0.7,
    "External availability does not establish model trustworthiness",
    ha="center",
    fontsize=10,
    fontweight="bold",
    color=DARK
)

plt.tight_layout()

show_collapsible_figure(
    "7.4 — Model Registries and Third-Party Model Trust",
    fig,
    """
    Model registries provide
    centralized storage for
    versioned machine learning
    artifacts.

    <br><br>

    A secure registry should
    restrict who can upload,
    modify, approve, and
    retrieve production models.

    <br><br>

    Organizations may develop
    models internally or obtain
    pretrained models from
    external model hubs.

    <br><br>

    External models can
    accelerate development,
    but they introduce
    supply-chain security risks.

    <br><br>

    A model may contain
    malicious functionality,
    hidden backdoor behavior,
    vulnerable dependencies,
    or unsafe serialization code.

    <br><br>

    Before approving an
    external artifact,
    defenders should verify
    its source and integrity
    and evaluate it in an
    appropriately isolated
    environment.

    <br><br>

    Some model serialization
    formats can execute code
    when loaded through
    unsafe deserialization.

    <br><br>

    Artifact verification
    should therefore occur
    before loading untrusted
    model files into
    sensitive environments.

    <br><br>

    Security testing should
    also evaluate the
    model's behavior,
    because a valid signature
    does not prove that
    a model is harmless.

    <br><br>

    <b>Key takeaway:</b> Model
    registries should promote
    only authorized, verified,
    and appropriately tested
    artifacts into production.
    """,
    ORANGE
)


# ============================================================
# VISUALIZATION 5
# CI/CD SECURITY GATES AND SEPARATION OF DUTIES
# ============================================================

checks = [
    "Artifact Integrity",
    "Dependency Scan",
    "Secret Scan",
    "Security Evaluation",
    "Authorized Approval",
    "Deployment Policy"
]

# Two fictional release candidates.
# 1 = passed, 0 = failed.

candidate_a = np.array([1, 1, 1, 1, 1, 1])
candidate_b = np.array([1, 0, 1, 1, 0, 1])

matrix = np.column_stack([candidate_a, candidate_b])

fig, ax = plt.subplots(figsize=(12, 8))

ax.imshow(
    matrix,
    cmap=ListedColormap(["#FEE2E2", "#DCFCE7"]),
    vmin=0,
    vmax=1,
    aspect="auto"
)

ax.set_xticks([0, 1])
ax.set_xticklabels([
    "Candidate A — Approved",
    "Candidate B — Blocked"
])

ax.set_yticks(range(len(checks)))
ax.set_yticklabels(checks)

ax.set_title(
    "Illustrative CI/CD Deployment Security Gates",
    fontsize=16,
    fontweight="bold",
    pad=18
)

for i in range(len(checks)):
    for j in range(2):
        passed = matrix[i, j] == 1

        ax.text(
            j, i,
            "PASS" if passed else "FAIL",
            ha="center",
            va="center",
            fontsize=11,
            fontweight="bold",
            color=GREEN if passed else RED
        )

ax.set_xticks(np.arange(-0.5, 2, 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(checks), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", bottom=False, left=False)

plt.tight_layout()

show_collapsible_figure(
    "7.5 — CI/CD Security Gates and Separation of Duties",
    fig,
    """
    Continuous Integration and
    Continuous Deployment,
    commonly called CI/CD,
    automate parts of the
    software delivery process.

    <br><br>

    AI deployment pipelines
    may use CI/CD to test,
    package, verify, and
    release model artifacts.

    <br><br>

    Security gates determine
    whether a release candidate
    satisfies the requirements
    for production deployment.

    <br><br>

    These checks may include
    artifact integrity verification,
    dependency vulnerability
    scanning, secret scanning,
    and model security evaluation.

    <br><br>

    Organizations may also
    require approval from
    an authorized reviewer.

    <br><br>

    This supports separation
    of duties by preventing
    a single developer
    from independently
    training, approving,
    and deploying every model.

    <br><br>

    The matrix compares
    two fictional candidates.

    <br><br>

    Candidate A passes
    all required checks.

    <br><br>

    Candidate B fails
    the dependency scan
    and approval requirements.

    <br><br>

    Under the example policy,
    Candidate B should
    not be deployed.

    <br><br>

    A failed check may
    require remediation,
    further review, or
    an explicitly authorized
    exception process.

    <br><br>

    <b>Key takeaway:</b> Automated
    security gates help
    prevent unverified or
    unauthorized artifacts
    from reaching production.
    """,
    GREEN
)


# ============================================================
# VISUALIZATION 6
# MODEL VERSIONING, ROLLBACK, AND RECOVERY
# ============================================================

# Synthetic service quality over a deployment event.
# These values are illustrative, not experimental results.

time = np.arange(1, 31)

quality = np.array([
    95, 96, 95, 96, 95,
    96, 95, 95, 96, 95,
    94, 93, 90, 85, 78,
    70, 64, 60, 58, 62,
    90, 94, 95, 96, 95,
    96, 95, 96, 95, 96
])

deployment_time = 12
rollback_time = 21

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(
    time,
    quality,
    color=BLUE,
    linewidth=2.5,
    marker="o",
    markersize=4,
    label="Illustrative Service Quality"
)

ax.axvline(
    deployment_time,
    color=ORANGE,
    linestyle="--",
    linewidth=2,
    label="New Version Deployed"
)

ax.axvline(
    rollback_time,
    color=GREEN,
    linestyle="--",
    linewidth=2,
    label="Rollback Executed"
)

ax.axhline(
    90,
    color=GRAY,
    linestyle=":",
    linewidth=1.5,
    label="Example Quality Threshold"
)

ax.fill_between(
    time,
    quality,
    90,
    where=quality < 90,
    color=RED,
    alpha=0.15
)

ax.annotate(
    "Performance degradation",
    xy=(18, 60),
    xytext=(12, 53),
    fontsize=10,
    fontweight="bold",
    color=RED,
    arrowprops=dict(
        arrowstyle="->",
        color=RED
    )
)

ax.annotate(
    "Recovery after rollback",
    xy=(23, 95),
    xytext=(20, 78),
    fontsize=10,
    fontweight="bold",
    color=GREEN,
    arrowprops=dict(
        arrowstyle="->",
        color=GREEN
    )
)

ax.set_xlim(1, 30)
ax.set_ylim(45, 105)

ax.set_xlabel("Illustrative Time Step")
ax.set_ylabel("Service Quality Score (%)")

ax.set_title(
    "Model Deployment Failure and Secure Rollback",
    fontsize=16,
    fontweight="bold"
)

ax.legend(fontsize=8, loc="lower left")
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "7.6 — Model Versioning, Rollback, and Recovery",
    fig,
    """
    Model versioning allows
    organizations to maintain
    identifiable records of
    different model releases.

    <br><br>

    Each version should be
    associated with the
    relevant artifact,
    configuration, approval
    information, and
    deployment history.

    <br><br>

    If a newly deployed
    model performs incorrectly
    or exhibits suspicious
    behavior, defenders
    may need to restore
    an earlier version.

    <br><br>

    This process is called
    <b>rollback</b>.

    <br><br>

    The graph shows a
    synthetic deployment
    scenario.

    <br><br>

    After a new model
    version is deployed,
    the service quality
    score begins to decline.

    <br><br>

    Once the problem
    is identified, the
    organization rolls
    back to a previously
    approved version.

    <br><br>

    The illustrative
    quality score then
    returns toward its
    earlier level.

    <br><br>

    In real environments,
    rollback requires more
    than selecting an
    older model file.

    <br><br>

    Defenders must verify
    that the earlier artifact
    is trustworthy and
    compatible with the
    current application,
    preprocessing pipeline,
    and infrastructure.

    <br><br>

    A compromised version
    should be quarantined
    for investigation
    rather than silently
    reused.

    <br><br>

    <b>Key takeaway:</b> Secure
    versioning and rollback
    support incident response
    by allowing organizations
    to restore known-good
    model deployments.
    """,
    TEAL
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("Section 7 — Model Artifact and Deployment Pipeline Security Visualizations")
print("---------------------------------------------------------------------------")
print("Six collapsible visualizations generated.")

print()
print("Concepts Covered")
print("----------------")
print("1. Secure ML Deployment Pipeline")
print("2. Model Artifact Integrity and Authenticity")
print("3. Model Provenance and Lineage")
print("4. Model Registries and Third-Party Model Trust")
print("5. CI/CD Security Gates and Separation of Duties")
print("6. Model Versioning, Rollback, and Recovery")

print()
print("All deployment scenarios and numerical values are illustrative.")
print("No external model files, registries, or cloud services were used.")


<details>

<summary><b>Section 7 Lab — Click to Expand</b></summary>

<br>

A machine learning team has completed training a new fraud-detection model.

The model performed well during evaluation and is now ready to move toward production.

However, several systems and people will interact with the model before deployment.

The path looks like:

    TRAINING
       ↓
    MODEL ARTIFACT
       ↓
    MODEL REGISTRY
       ↓
    SECURITY TESTING
       ↓
    APPROVAL
       ↓
    DEPLOYMENT
       ↓
    PRODUCTION

A successful training run does not prove that the correct model will eventually reach production.

An attacker or deployment mistake could:

    MODIFY THE ARTIFACT

    REPLACE THE MODEL

    INTRODUCE AN UNTRUSTED MODEL

    BYPASS SECURITY TESTING

    DEPLOY THE WRONG VERSION

    SKIP REQUIRED APPROVAL

Your job is to secure this pipeline.

You will create a model artifact, calculate its cryptographic hash, simulate tampering, verify provenance, inspect registry information, evaluate a third-party model, simulate model signing, enforce deployment gates, require approval, deploy a specific version, and perform a secure rollback.

The goal is to preserve trust from:

    MODEL CREATION

to:

    PRODUCTION EXECUTION

All exercises use small simulated artifacts and standard Python so they should run almost instantly.

</details>

In [ ]:
# @title ▶ Part 1: Create the Model Artifact { display-mode: "form" }

model_artifact = {
    "model_name": "fraud-detector",
    "version": "2.1",
    "framework": "scikit-learn",
    "accuracy": 0.92,
    "training_dataset": "fraud-training-v4",
    "training_code": "commit-a82f91",
    "created_by": "training-pipeline"
}

print("=" * 70)
print("MODEL ARTIFACT")
print("=" * 70)

for key, value in model_artifact.items():
    print(f"{key:<20}: {value}")

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

Training is only the beginning of the deployment lifecycle.

The simulated model now has identifying information including its:

    NAME

    VERSION

    FRAMEWORK

    TRAINING DATASET

    TRAINING CODE

    CREATOR

This information helps distinguish one artifact from another.

A model file named:

    model.pkl

provides very little information by itself.

A controlled deployment process should be able to answer questions such as:

**Which model is this?**

**Which dataset created it?**

**Which code version created it?**

**Who or what created it?**

This information becomes important when establishing model provenance and lineage later in the lab.

</details>

In [ ]:
# @title ▶ Part 2: Hash the Model Artifact { display-mode: "form" }

import hashlib
import json

def calculate_hash(artifact):
    artifact_bytes = json.dumps(
        artifact,
        sort_keys=True
    ).encode()

    return hashlib.sha256(
        artifact_bytes
    ).hexdigest()


approved_hash = calculate_hash(
    model_artifact
)

print("=" * 70)
print("MODEL INTEGRITY RECORD")
print("=" * 70)

print("\nModel:")
print(model_artifact["model_name"])

print("\nVersion:")
print(model_artifact["version"])

print("\nSHA-256:")
print(approved_hash)

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

A cryptographic hash creates a fingerprint of the model artifact.

The lab uses:

    SHA-256

The resulting value represents the exact artifact being approved.

Conceptually:

    MODEL ARTIFACT
          ↓
       SHA-256
          ↓
    UNIQUE HASH VALUE

If the artifact changes, its calculated hash should also change.

The organization can therefore record the hash of the approved model and compare it against the artifact presented for deployment.

Hashing does not determine whether a model is safe, accurate, or trustworthy.

It answers a narrower but extremely useful question:

> **Is this artifact still the same artifact that was previously verified?**

</details>

In [ ]:
# @title ▶ Part 3: Detect Artifact Tampering { display-mode: "form" }

tampered_model = model_artifact.copy()

# Simulate unauthorized modification
tampered_model["training_code"] = "unknown-modified-code"

tampered_hash = calculate_hash(
    tampered_model
)

print("=" * 70)
print("MODEL INTEGRITY CHECK")
print("=" * 70)

print("\nApproved Hash:")
print(approved_hash)

print("\nCurrent Hash:")
print(tampered_hash)

print("\nVerification Result:")

if approved_hash == tampered_hash:
    print("✓ INTEGRITY VERIFIED")
else:
    print("✗ INTEGRITY FAILURE")
    print("Artifact has changed!")

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

The simulated artifact was modified after its original hash was recorded.

The change may appear small, but the resulting hash is different.

The pipeline therefore detects:

    APPROVED ARTIFACT
          ↓
       HASH A

compared with:

    MODIFIED ARTIFACT
          ↓
       HASH B

Because:

    HASH A ≠ HASH B

the pipeline should not assume that the artifact is still the approved model.

This is useful against accidental modification as well as intentional model replacement or tampering.

The important behavior is:

    INTEGRITY FAILURE
          ↓
    STOP DEPLOYMENT
          ↓
      INVESTIGATE

rather than silently deploying the changed artifact.

</details>

In [ ]:
# @title ▶ Part 4: Verify Provenance { display-mode: "form" }

required_provenance = [
    "model_name",
    "version",
    "framework",
    "training_dataset",
    "training_code",
    "created_by"
]

print("=" * 70)
print("MODEL PROVENANCE CHECK")
print("=" * 70)

missing = []

for field in required_provenance:

    if field in model_artifact:
        print(f"✓ {field}: {model_artifact[field]}")
    else:
        print(f"✗ {field}: MISSING")
        missing.append(field)

print("\nResult:")

if not missing:
    print("✓ Required provenance information available.")
else:
    print("✗ Provenance incomplete.")

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

Integrity tells us whether an artifact changed.

Provenance helps answer:

> **Where did this artifact come from?**

The model contains information connecting it to its development history.

For example:

    MODEL VERSION 2.1
          ↓
    TRAINING CODE commit-a82f91
          ↓
    DATASET fraud-training-v4
          ↓
    TRAINING PIPELINE

This creates a simplified model lineage.

If an organization discovers a problem with a particular dataset or code version, lineage information can help determine which models may have been affected.

Without provenance, defenders may possess a model artifact without being able to confidently explain how it was produced.

</details>

In [ ]:
# @title ▶ Part 5: Store the Model in a Registry { display-mode: "form" }

model_registry = {
    "fraud-detector:2.0": {
        "status": "production",
        "approved": True,
        "hash": "previous-trusted-hash"
    },

    "fraud-detector:2.1": {
        "status": "candidate",
        "approved": False,
        "hash": approved_hash
    }
}

print("=" * 70)
print("MODEL REGISTRY")
print("=" * 70)

for model, info in model_registry.items():

    print(f"\n{model}")

    for key, value in info.items():
        print(f"  {key}: {value}")

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

A model registry helps organize and control model artifacts.

The registry currently contains:

    VERSION 2.0
    → Current production model

and:

    VERSION 2.1
    → Candidate model

Notice that version 2.1 exists in the registry but is not yet approved.

This distinction is important.

    STORED

does not mean:

    APPROVED FOR PRODUCTION

A secure registry can help organizations track model versions, lifecycle states, integrity information, and approval status.

The deployment pipeline should select a specific approved artifact rather than simply deploying whichever model file happens to be newest.

</details>

In [ ]:
# @title ▶ Part 6: Review a Third-Party Model { display-mode: "form" }

third_party_model = {
    "name": "community-fraud-model",
    "known_source": True,
    "hash_verified": False,
    "security_scan_passed": False,
    "provenance_available": False
}

requirements = [
    "known_source",
    "hash_verified",
    "security_scan_passed",
    "provenance_available"
]

print("=" * 70)
print("THIRD-PARTY MODEL REVIEW")
print("=" * 70)

approved = True

for requirement in requirements:

    passed = third_party_model[
        requirement
    ]

    print(
        f"{requirement:<25} "
        f"{'✓ PASS' if passed else '✗ FAIL'}"
    )

    if not passed:
        approved = False

print("\nDeployment Decision:")

if approved:
    print("✓ APPROVED")
else:
    print("✗ BLOCKED")

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

Pretrained models and model hubs can significantly accelerate AI development.

However, a downloadable model should not automatically be considered trusted.

The simulated model has a known source, but several important security checks are missing.

The organization cannot yet verify:

    ARTIFACT INTEGRITY

    SECURITY TESTING

    PROVENANCE

Therefore:

    AVAILABLE MODEL
          ≠
    TRUSTED MODEL

Third-party models become part of the AI supply chain.

They should be reviewed before being introduced into trusted development or production environments.

</details>

In [ ]:
# @title ▶ Part 7: Sign the Approved Artifact { display-mode: "form" }

import hmac

# Demonstration key only.
# A production system would protect signing keys securely.
signing_key = b"demo-secure-signing-key"

def sign_artifact(artifact_hash):

    return hmac.new(
        signing_key,
        artifact_hash.encode(),
        hashlib.sha256
    ).hexdigest()


model_signature = sign_artifact(
    approved_hash
)

print("=" * 70)
print("MODEL SIGNING")
print("=" * 70)

print("\nArtifact Hash:")
print(approved_hash)

print("\nSignature:")
print(model_signature)

<details>

<summary><b>Part 7 Explanation — Click to Expand</b></summary>

<br>

Hashing helps detect whether an artifact changed.

Signing adds another trust question:

> **Was this artifact approved by the expected trusted source?**

For this lightweight lab, we simulate signing with an HMAC so students can practice the verification workflow without installing cryptographic packages or managing real certificates.

Conceptually:

    APPROVED MODEL
          ↓
        HASH
          ↓
    TRUSTED SIGNING PROCESS
          ↓
      SIGNATURE

A real production environment may use digital signatures and protected private signing keys.

The important distinction is:

    HASH
    → Helps verify integrity

    SIGNATURE
    → Helps establish integrity and authenticity

The signing mechanism is only trustworthy if its signing credentials are properly protected.

</details>

In [ ]:
# @title ▶ Part 8: Verify Model Signature { display-mode: "form" }

def verify_signature(
    artifact,
    expected_signature
):

    current_hash = calculate_hash(
        artifact
    )

    current_signature = sign_artifact(
        current_hash
    )

    return hmac.compare_digest(
        current_signature,
        expected_signature
    )


print("=" * 70)
print("PRE-DEPLOYMENT SIGNATURE CHECK")
print("=" * 70)

trusted_result = verify_signature(
    model_artifact,
    model_signature
)

tampered_result = verify_signature(
    tampered_model,
    model_signature
)

print(
    "\nOriginal Artifact:",
    "✓ VERIFIED"
    if trusted_result
    else "✗ REJECTED"
)

print(
    "Tampered Artifact:",
    "✓ VERIFIED"
    if tampered_result
    else "✗ REJECTED"
)

<details>

<summary><b>Part 8 Explanation — Click to Expand</b></summary>

<br>

The pipeline now verifies the artifact before deployment.

The original artifact passes.

The modified artifact fails.

This creates an important control:

    MODEL PRESENTED FOR DEPLOYMENT
               ↓
       VERIFY SIGNATURE
           ↙       ↘
        VALID     INVALID
          ↓          ↓
       CONTINUE     STOP

The pipeline does not simply trust the filename:

    fraud-detector-2.1

It verifies the artifact itself.

This helps reduce the risk that a replaced or modified model reaches production while pretending to be an approved version.

</details>

In [ ]:
# @title ▶ Part 9: Apply Deployment Gates { display-mode: "form" }

deployment_checks = {
    "Evaluation Passed": True,
    "Security Tests Passed": True,
    "Integrity Verified": True,
    "Signature Verified": True,
    "Provenance Complete": True,
    "Approved Registry Source": True
}

print("=" * 70)
print("DEPLOYMENT SECURITY GATES")
print("=" * 70)

pipeline_passed = True

for check, passed in deployment_checks.items():

    print(
        f"{check:<30} "
        f"{'✓ PASS' if passed else '✗ FAIL'}"
    )

    if not passed:
        pipeline_passed = False

print("\nPipeline Decision:")

if pipeline_passed:
    print("✓ SECURITY GATES PASSED")
else:
    print("✗ DEPLOYMENT BLOCKED")

<details>

<summary><b>Part 9 Explanation — Click to Expand</b></summary>

<br>

A secure CI/CD pipeline should not automatically deploy every model produced by development.

Instead, the candidate must pass required checks.

In this simulation, deployment requires:

    EVALUATION

    SECURITY TESTING

    INTEGRITY VERIFICATION

    SIGNATURE VERIFICATION

    PROVENANCE

    APPROVED REGISTRY SOURCE

These checks form **deployment gates**.

The important principle is:

    BUILD SUCCESS
          ≠
    DEPLOYMENT APPROVAL

Security requirements become part of the automated deployment process rather than optional manual suggestions.

</details>

In [ ]:
# @title ▶ Part 10: Block an Unsafe Deployment { display-mode: "form" }

failed_checks = deployment_checks.copy()

# Simulate failed security testing
failed_checks[
    "Security Tests Passed"
] = False

print("=" * 70)
print("FAILED DEPLOYMENT TEST")
print("=" * 70)

deployment_allowed = True

for check, passed in failed_checks.items():

    print(
        f"{check:<30} "
        f"{'✓ PASS' if passed else '✗ FAIL'}"
    )

    if not passed:
        deployment_allowed = False

print("\nFinal Decision:")

if deployment_allowed:
    print("✓ DEPLOY")
else:
    print("✗ DEPLOYMENT BLOCKED")

<details>

<summary><b>Part 10 Explanation — Click to Expand</b></summary>

<br>

Only one security check failed.

The model may still:

    LOAD CORRECTLY

    PRODUCE PREDICTIONS

    HAVE GOOD ACCURACY

However, the deployment is blocked.

This demonstrates why security gates matter.

Without enforcement, a team may be tempted to deploy a model because it appears operationally ready.

A secure pipeline follows:

    REQUIRED CHECK FAILS
            ↓
      PIPELINE STOPS
            ↓
      ISSUE CORRECTED
            ↓
       TEST AGAIN

The pipeline should fail safely rather than silently bypassing security requirements.

</details>

In [ ]:
# @title ▶ Part 11: Enforce Separation of Duties { display-mode: "form" }

deployment_request = {
    "model": "fraud-detector:2.1",
    "developer": "developer-alex",
    "approver": "security-jordan",
    "developer_approved": True,
    "independent_approval": True
}

print("=" * 70)
print("PRODUCTION APPROVAL")
print("=" * 70)

print(
    f"\nDeveloper: "
    f"{deployment_request['developer']}"
)

print(
    f"Approver:  "
    f"{deployment_request['approver']}"
)

separate_people = (
    deployment_request["developer"]
    != deployment_request["approver"]
)

approved = (
    deployment_request[
        "developer_approved"
    ]
    and deployment_request[
        "independent_approval"
    ]
    and separate_people
)

print(
    "\nSeparation of Duties:",
    "✓ PASS"
    if separate_people
    else "✗ FAIL"
)

print(
    "Production Approval:",
    "✓ APPROVED"
    if approved
    else "✗ DENIED"
)

<details>

<summary><b>Part 11 Explanation — Click to Expand</b></summary>

<br>

The developer who prepares the deployment is not the only person involved in approving production release.

This provides separation of duties.

Instead of:

    ONE PERSON
       ↓
    CREATE MODEL
       ↓
    APPROVE MODEL
       ↓
    DEPLOY MODEL

the process becomes:

    DEVELOPMENT
         ↓
    SECURITY / REVIEW
         ↓
    PRODUCTION APPROVAL
         ↓
      DEPLOYMENT

Separation of duties reduces the amount of control held by a single identity.

It can also help catch mistakes before they reach production.

</details>

In [ ]:
# @title ▶ Part 12: Deploy the Approved Version { display-mode: "form" }

if pipeline_passed and approved:

    model_registry[
        "fraud-detector:2.1"
    ]["approved"] = True

    model_registry[
        "fraud-detector:2.1"
    ]["status"] = "production"

    model_registry[
        "fraud-detector:2.0"
    ]["status"] = "previous-production"

    production_model = (
        "fraud-detector:2.1"
    )

    print("=" * 70)
    print("PRODUCTION DEPLOYMENT")
    print("=" * 70)

    print(
        f"\n✓ Deployed: "
        f"{production_model}"
    )

else:

    print(
        "✗ Deployment requirements "
        "not satisfied."
    )

<details>

<summary><b>Part 12 Explanation — Click to Expand</b></summary>

<br>

The pipeline deploys a specific model version:

    fraud-detector:2.1

rather than an ambiguous label such as:

    latest

Specific versioning makes it easier to determine exactly which artifact is running in production.

The registry now preserves both:

    CURRENT PRODUCTION
    → Version 2.1

and:

    PREVIOUS PRODUCTION
    → Version 2.0

This becomes important when responding to a failed or unsafe deployment.

</details>

In [ ]:
# @title ▶ Part 13: Roll Back the Deployment { display-mode: "form" }

print("=" * 70)
print("PRODUCTION INCIDENT")
print("=" * 70)

print(
    "\n⚠ Unexpected behavior detected "
    "in fraud-detector:2.1"
)

previous_version = (
    "fraud-detector:2.0"
)

if model_registry[
    previous_version
]["approved"]:

    production_model = previous_version

    model_registry[
        previous_version
    ]["status"] = "production"

    model_registry[
        "fraud-detector:2.1"
    ]["status"] = "quarantined"

    print(
        f"\n✓ Rolled back to: "
        f"{production_model}"
    )

    print(
        "✓ Version 2.1 quarantined "
        "for investigation."
    )

else:

    print(
        "\n✗ Previous version "
        "is not trusted."
    )

<details>

<summary><b>Part 13 Explanation — Click to Expand</b></summary>

<br>

The new deployment produces unexpected behavior.

Instead of selecting an arbitrary older model, the system rolls back to a previously trusted and approved version.

The response is:

    PROBLEM DETECTED
          ↓
    STOP USING VERSION 2.1
          ↓
    RESTORE TRUSTED VERSION 2.0
          ↓
    QUARANTINE VERSION 2.1
          ↓
       INVESTIGATE

Rollback is therefore part of deployment security.

Version history allows the organization to recover to a known state while investigating the problematic artifact.

</details>

In [ ]:
# @title ▶ Part 14: Review the Secure Pipeline { display-mode: "form" }

secure_pipeline = [
    "Train Model",
    "Create Artifact",
    "Record Provenance",
    "Calculate Hash",
    "Store in Registry",
    "Run Security Tests",
    "Verify Integrity",
    "Verify Signature",
    "Pass Deployment Gates",
    "Receive Independent Approval",
    "Deploy Specific Version",
    "Monitor Production",
    "Rollback if Necessary"
]

print("=" * 70)
print("SECURE MODEL DEPLOYMENT PIPELINE")
print("=" * 70)

for number, stage in enumerate(
    secure_pipeline,
    start=1
):
    print(
        f"{number:>2}. {stage}"
    )

<details>

<summary><b>Part 14 Explanation — Click to Expand</b></summary>

<br>

The individual controls from the lab now form one complete security process.

The model moves through:

    TRAIN
      ↓
    CREATE ARTIFACT
      ↓
    RECORD PROVENANCE
      ↓
    HASH
      ↓
    REGISTER
      ↓
    SECURITY TEST
      ↓
    VERIFY
      ↓
    APPROVE
      ↓
    DEPLOY
      ↓
    MONITOR
      ↓
    ROLLBACK IF REQUIRED

No individual control provides complete security.

Instead, each control protects a different part of the model lifecycle.

Together they help answer the central question from this section:

> **Can we prove that the model running in production is the trusted, tested, and approved model we intended to deploy?**

</details>

In [ ]:
# @title ▶ Part 15: Knowledge Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. What did the SHA-256 hash represent?",
        "options": [
            "A. A fingerprint of the model artifact",
            "B. Model accuracy",
            "C. GPU utilization",
            "D. User permissions"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. What happened after the artifact was modified?",
        "options": [
            "A. Its hash changed and integrity verification failed",
            "B. Its hash remained identical",
            "C. It automatically became approved",
            "D. Its accuracy became 100%"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. What information did provenance provide?",
        "options": [
            "A. Information about the model's origin and development history",
            "B. Only the model's prediction",
            "C. The user's password",
            "D. Network firewall rules"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Why was the third-party model blocked?",
        "options": [
            "A. Important trust and security checks were missing",
            "B. Third-party models are always prohibited",
            "C. It was too accurate",
            "D. It had a model name"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. What did signature verification help establish?",
        "options": [
            "A. Artifact integrity and authenticity",
            "B. Model accuracy only",
            "C. GPU performance",
            "D. Dataset size"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. What happened when one required deployment gate failed?",
        "options": [
            "A. Deployment was blocked",
            "B. The failure was ignored",
            "C. The model automatically entered production",
            "D. All testing was disabled"
        ],
        "answer": "A"
    },
    {
        "question":
        "7. Why did the lab use an independent approver?",
        "options": [
            "A. To provide separation of duties",
            "B. To remove security testing",
            "C. To eliminate version control",
            "D. To make the registry public"
        ],
        "answer": "A"
    },
    {
        "question":
        "8. Why was version 2.0 retained?",
        "options": [
            "A. It provided a previously trusted rollback target",
            "B. Every model must run simultaneously",
            "C. Versions prevent all vulnerabilities",
            "D. Version 2.0 did not need approval"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("LAB 7 — KNOWLEDGE CHECK")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1

    else:
        print(
            f"✗ Incorrect. "
            f"Correct answer: {q['answer']}"
        )

percentage = (
    score / len(questions)
) * 100

print("\n" + "=" * 70)
print(f"Lab Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Lab complete!")
else:
    print(
        "△ Review the lab explanations "
        "and try again."
    )

<details>

<summary><b>Lab 7 — Practical Takeaways — Click to Expand</b></summary>

<br>

## Lab 7 — Practical Takeaways

This lab followed a machine learning model from creation to production.

The most important lesson is that training a secure model is not enough.

After training, the model becomes an artifact that moves through multiple systems:

    TRAINING
       ↓
    ARTIFACT
       ↓
    REGISTRY
       ↓
    TESTING
       ↓
    APPROVAL
       ↓
    DEPLOYMENT
       ↓
    PRODUCTION

Each transition creates an opportunity for mistakes or malicious modification.

### Integrity

You used a SHA-256 hash to create a fingerprint of the approved artifact.

When the artifact changed:

    ORIGINAL HASH
          ≠
    MODIFIED HASH

the deployment process detected the modification.

### Provenance and Lineage

You recorded information connecting the model to its:

    TRAINING DATA

    TRAINING CODE

    VERSION

    CREATOR

This helps establish where the model came from and how it was produced.

### Artifact Trust

You examined a third-party model and learned that:

    DOWNLOADABLE
         ≠
    TRUSTED

External models should be evaluated before entering a trusted AI pipeline.

### Signing and Verification

The lab then simulated artifact signing.

Before deployment, the pipeline verified that the artifact still matched the trusted signed version.

### CI/CD Security

Security checks became required deployment gates.

If an important check failed:

    SECURITY FAILURE
          ↓
    DEPLOYMENT BLOCKED

The model was not allowed into production simply because it functioned correctly.

### Approval and Separation of Duties

Independent approval reduced the amount of control held by a single person or process.

### Versioning and Recovery

Finally, you retained a previously trusted production model.

When the new deployment experienced a problem:

    DETECT PROBLEM
          ↓
    QUARANTINE NEW MODEL
          ↓
    RESTORE TRUSTED VERSION

This demonstrates why rollback is part of secure deployment rather than merely an operational convenience.

The central lesson is:

> **Organizations should be able to prove what model is running, where it came from, whether it changed, who approved it, and how to safely replace it if something goes wrong.**

Model security therefore extends beyond the model itself.

The entire deployment pipeline is part of the AI supply chain and must preserve trust from model creation through production execution.

</details>

In [ ]:
# @title ▶ Section 7 Checkpoint — Model Artifact & Pipeline Security { display-mode: "form" }

questions = [
    {
        "question": "1. Why should a model artifact be treated as a security asset?",
        "options": [
            "A. It may contain valuable functionality and is ultimately executed in production",
            "B. Models cannot be modified",
            "C. Model files are always public",
            "D. Only training data requires protection"
        ],
        "answer": "A"
    },
    {
        "question": "2. What does model provenance help establish?",
        "options": [
            "A. Where a model came from and how it was created",
            "B. Only its file size",
            "C. Only its prediction speed",
            "D. The user's password"
        ],
        "answer": "A"
    },
    {
        "question": "3. What can a cryptographic hash help detect?",
        "options": [
            "A. Whether a model artifact has changed",
            "B. Whether a model is accurate",
            "C. Whether users like the model",
            "D. Whether the GPU is fast enough"
        ],
        "answer": "A"
    },
    {
        "question": "4. What additional property can a digital signature provide?",
        "options": [
            "A. Evidence of artifact integrity and authenticity",
            "B. Unlimited model permissions",
            "C. Guaranteed model accuracy",
            "D. Automatic training"
        ],
        "answer": "A"
    },
    {
        "question": "5. Why are third-party pretrained models a supply-chain concern?",
        "options": [
            "A. Their origin, contents, or integrity may not be fully trusted",
            "B. Third-party models cannot run",
            "C. They never contain vulnerabilities",
            "D. They eliminate deployment pipelines"
        ],
        "answer": "A"
    },
    {
        "question": "6. What is a deployment gate?",
        "options": [
            "A. A required check that must pass before deployment continues",
            "B. A model-training algorithm",
            "C. A public API endpoint",
            "D. A type of dataset"
        ],
        "answer": "A"
    },
    {
        "question": "7. Why is separation of duties useful?",
        "options": [
            "A. It prevents one person or process from controlling every critical deployment step",
            "B. It removes approval requirements",
            "C. It gives developers unlimited production access",
            "D. It eliminates logging"
        ],
        "answer": "A"
    },
    {
        "question": "8. Why maintain previous trusted model versions?",
        "options": [
            "A. They can support rollback if a deployment causes problems",
            "B. Old models should always remain in production",
            "C. Versioning prevents all attacks",
            "D. Versions replace security testing"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("SECTION 7 CHECKPOINT")
print("MODEL ARTIFACT AND DEPLOYMENT PIPELINE SECURITY")
print("=" * 70)

for q in questions:
    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input("\nYour answer: ").strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(f"✗ Incorrect. Correct answer: {q['answer']}")

percentage = score / len(questions) * 100

print("\n" + "=" * 70)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Checkpoint complete!")
else:
    print("△ Review Section 7 and try again.")

---

<details>
<summary><b>Section 8 — Logging, Monitoring, Incident Response, and Secure Operations — Click to Expand</b></summary>
<br>

## Section 8 — Logging, Monitoring, Incident Response, and Secure Operations

Security does not end when an AI system is successfully deployed.

Production AI systems continuously receive requests, access data, communicate with other services, consume infrastructure resources, load model artifacts, and generate outputs. Their behavior can change over time because of attacks, software failures, configuration changes, model drift, infrastructure problems, or unexpected user behavior.

For this reason, organizations need visibility into what their AI systems are doing.

Logging and monitoring help defenders identify abnormal activity, determine whether a security incident is occurring, understand what happened, and respond appropriately.

This section examines infrastructure monitoring, AI-specific telemetry, behavioral baselines, drift detection, suspicious query activity, authentication events, resource monitoring, alerting, incident response, containment, rollback, recovery, and post-incident improvement.

The central question is:

**How do we know when a deployed AI system is behaving abnormally, and what should we do when something goes wrong?**

<br>

<details>
<summary><b>8.1 Security Does Not End at Deployment</b></summary>
<br>

### Security Does Not End at Deployment

A system can pass security testing before deployment and still experience security problems later.

New vulnerabilities may be discovered.

Credentials may be stolen.

Attackers may begin probing an inference endpoint.

Configuration settings may change.

Model behavior may drift.

A new deployment may introduce unexpected problems.

For these reasons, security must continue throughout production operation.

A simplified lifecycle is:

**Build**

↓

**Test**

↓

**Deploy**

↓

**Monitor**

↓

**Detect**

↓

**Respond**

↓

**Recover**

↓

**Improve**

Monitoring Machine Learning Systems research has shown that production ML monitoring includes concerns such as model performance, operational health, data quality, drift, and security-related behavior [27].

This means that a production AI system should not be treated as a static artifact.

It should be treated as a continuously operating system whose behavior must be observed over time.

A useful principle is:

**If an organization cannot observe what its AI system is doing, it will have difficulty determining when something is wrong.**

</details>

<br>

<details>
<summary><b>8.2 Observability in AI Systems</b></summary>
<br>

### Observability in AI Systems

**Observability** refers to the ability to understand the internal state and behavior of a system based on the information it produces.

This information may come from:

- Logs
- Metrics
- Traces
- Alerts
- Model telemetry
- Security events

Traditional applications may monitor:

- CPU usage
- Memory usage
- Network traffic
- Error rates
- Authentication failures

AI systems need these same signals, but they may also require model-specific information.

Examples include:

- Prediction distributions
- Confidence scores
- Input distributions
- Model version
- Drift indicators
- Inference volume
- Abnormal query patterns

A useful way to think about AI observability is:

**Infrastructure Observability**

+

**Application Observability**

+

**Model Observability**

+

**Security Observability**

Together, these signals help defenders understand whether a system is operating normally.

Research on ML monitoring emphasizes that effective monitoring must account for both traditional software behavior and characteristics unique to machine learning systems [27], [28].

</details>

<br>

<details>
<summary><b>8.3 What Should an AI System Log?</b></summary>
<br>

### What Should an AI System Log?

Logs provide a historical record of activity.

Useful AI security logs may include:

- Timestamp
- Request identifier
- User or service identity
- Authentication result
- Authorization result
- Endpoint accessed
- Model version
- Request size
- Response status
- Inference latency
- Rate-limit events
- Errors
- Administrative actions
- Model deployment events
- Configuration changes
- Artifact verification results

For example:

**2026-09-04 09:15:22**

**User:** client_42

**Endpoint:** /predict

**Authentication:** SUCCESS

**Model:** fraud_model_v4

**Status:** 200

Logs should support investigation without unnecessarily collecting sensitive information.

For example, storing every raw user input may introduce privacy or confidentiality risks.

Organizations should therefore balance:

**Security Visibility**

with:

**Data Minimization and Privacy**

Logs should also be protected against unauthorized modification.

If an attacker compromises an inference service and can delete or alter the security logs, incident investigation becomes much more difficult.

</details>

<br>

<details>
<summary><b>8.4 Infrastructure and Application Monitoring</b></summary>
<br>

### Infrastructure and Application Monitoring

AI systems still depend on traditional computing infrastructure.

Defenders should monitor resources such as:

- CPU usage
- Memory usage
- GPU usage
- Disk usage
- Network activity
- API latency
- HTTP error rates
- Container restarts
- Service availability

These signals can reveal security or reliability problems.

For example:

**Normal GPU Usage: 35–50%**

Suddenly:

**GPU Usage: 100% for Extended Period**

This could indicate:

- Legitimate traffic increase
- Resource exhaustion
- Automated abuse
- Misconfiguration
- Software failure

Similarly:

**Normal Authentication Failures: 2 per hour**

Suddenly:

**Authentication Failures: 2,000 per hour**

may indicate automated credential attacks.

Monitoring should therefore establish expected patterns and identify significant deviations.

</details>

<br>

<details>
<summary><b>8.5 AI and Model-Specific Monitoring</b></summary>
<br>

### AI and Model-Specific Monitoring

Traditional infrastructure monitoring may show that a service is online, but it may not reveal that the model itself is behaving incorrectly.

AI systems may therefore require additional telemetry.

Useful model-specific signals may include:

- Input distribution
- Prediction distribution
- Confidence distribution
- Accuracy
- Precision
- Recall
- Error patterns
- Drift indicators
- Model version
- Inference volume

Suppose a classifier normally predicts:

**Class A: 45%**

**Class B: 35%**

**Class C: 20%**

Then suddenly produces:

**Class A: 2%**

**Class B: 3%**

**Class C: 95%**

This may indicate:

- Data drift
- Pipeline failure
- Model corruption
- Attack activity
- Application error

Monitoring model behavior therefore gives defenders visibility that infrastructure metrics alone cannot provide.

Research on ML monitoring highlights the importance of combining model-performance and system-level indicators when operating machine learning systems in production [27], [28].

</details>

<br>

<details>
<summary><b>8.6 Establishing Normal Behavior and Baselines</b></summary>
<br>

### Establishing Normal Behavior and Baselines

Monitoring is more useful when defenders know what **normal behavior** looks like.

A baseline records expected ranges or patterns for system activity.

For example:

**Normal Requests per Minute:** 200–400

**Normal Authentication Failures per Hour:** 0–10

**Normal GPU Usage:** 30–60%

**Normal Prediction Distribution:** Stable within historical range

Without a baseline, defenders may observe a value but have no context for whether it is unusual.

For example:

**5,000 requests per hour**

may be normal for one system and extremely suspicious for another.

Baselines can be created from:

- Historical activity
- Testing
- Known workloads
- Business expectations
- Model performance records

Monitoring then compares current behavior against expected behavior.

The basic process is:

**Observe**

↓

**Compare to Baseline**

↓

**Significant Deviation?**

**Yes → Investigate**

Baselines should also be updated carefully because legitimate system behavior may change over time.

</details>

<br>

<details>
<summary><b>8.7 Data Drift and Distribution Shift</b></summary>
<br>

### Data Drift and Distribution Shift

Machine learning systems depend on patterns in data.

Production data may change over time.

This is commonly referred to as **data drift** or **distribution shift**.

Suppose a model was trained on inputs with a particular distribution.

Months later, user behavior changes.

The production data no longer resembles the original training data.

This may cause:

- Reduced accuracy
- Unexpected predictions
- Increased uncertainty
- Changed error patterns

Drift does not automatically mean that an attack has occurred.

It may result from legitimate changes in the environment.

However, unexpected distribution changes can also be relevant to security.

An attacker may intentionally submit unusual inputs or manipulate data flows.

Cobb and Van Looveren discuss approaches for detecting drift while accounting for the context in which data changes occur [22].

Monitoring therefore needs to distinguish between:

**Normal Change**

and:

**Potentially Suspicious Change**

Drift detection should be treated as one signal among many rather than definitive proof of an attack.

</details>

<br>

<details>
<summary><b>8.8 Detecting Suspicious Query Patterns</b></summary>
<br>

### Detecting Suspicious Query Patterns

Attackers may interact with an AI model through many repeated queries.

This behavior may support:

- Model extraction
- Membership inference
- Adversarial probing
- Resource exhaustion
- Reconnaissance

A normal user might submit:

**20 requests per hour**

An attacker may submit:

**10,000 carefully varied requests per hour**

Monitoring can help identify these differences.

Useful indicators include:

- Extremely high request volume
- Repetitive requests
- Systematic input variation
- Requests near decision boundaries
- Unusual query timing
- Distributed requests across many accounts
- Repeated rate-limit violations

For example:

**User A**

**Requests:** 50/minute

**Normal Limit:** 5/minute

↓

**Alert**

Monitoring can complement the rate limiting introduced in Section 2.

Rate limiting attempts to prevent excessive activity.

Monitoring helps determine:

**Who is repeatedly attempting that activity?**

and:

**Is the behavior part of a broader attack?**

</details>

<br>

<details>
<summary><b>8.9 Monitoring Authentication and Access</b></summary>
<br>

### Monitoring Authentication and Access

Authentication and authorization systems generate valuable security signals.

Important events may include:

- Failed logins
- Successful logins
- Token failures
- Permission denials
- Role changes
- Service-account activity
- Administrative access
- Privilege changes
- Secret-access events

Consider:

**Normal User**

**Failed Login Attempts: 1**

versus:

**Unknown Source**

**Failed Login Attempts: 5,000**

The second pattern may indicate automated credential attacks.

Another example:

**Inference Service**

normally accesses:

**Production Model Storage**

but suddenly attempts to access:

**Administrative Secrets**

The request may be denied by IAM, but the attempt itself is still important.

A denied action can provide valuable security information.

Monitoring should therefore record both:

**Successful Access**

and:

**Suspicious Failed Access**

</details>

<br>

<details>
<summary><b>8.10 Resource and Availability Monitoring</b></summary>
<br>

### Resource and Availability Monitoring

Availability is an important part of AI security.

Attackers may attempt to make AI services unavailable by consuming excessive resources.

Monitoring should therefore track:

- Request volume
- CPU utilization
- Memory usage
- GPU utilization
- Queue length
- Response latency
- Timeouts
- Error rates
- Container restarts
- Service failures

For example:

**Normal Latency:** 120 ms

Suddenly:

**Latency:** 4,000 ms

combined with:

**GPU Utilization:** 100%

and:

**Requests:** 10× normal volume

may indicate a resource-exhaustion event.

This connects several controls introduced earlier:

**Rate Limiting**

+

**Container Resource Limits**

+

**Cloud Scaling Controls**

+

**Monitoring**

Together, these controls can provide stronger protection against availability attacks.

</details>

<br>

<details>
<summary><b>8.11 Alerting and Security Thresholds</b></summary>
<br>

### Alerting and Security Thresholds

Monitoring systems may collect enormous amounts of data.

Defenders cannot manually inspect every event.

**Alerts** identify conditions that may require human attention.

An alert may trigger when:

- Failed authentication exceeds a threshold
- Request volume becomes unusually high
- Model hash verification fails
- A new administrative role is assigned
- Resource utilization remains abnormally high
- Model output distribution changes significantly
- An inference service accesses an unexpected resource

For example:

**Failed Authentication Threshold: 20 per 5 Minutes**

If:

**Failures > 20**

↓

**Generate Alert**

Thresholds should be chosen carefully.

If alerts are too sensitive:

**Normal Activity → Alert**

**Normal Activity → Alert**

**Normal Activity → Alert**

This creates **alert fatigue**.

Research on practitioners' experiences with ML monitoring has identified challenges involving thresholds, monitoring complexity, and excessive alerts [28].

Security monitoring should therefore attempt to prioritize meaningful events rather than generating unnecessary noise.

</details>

<br>

<details>
<summary><b>8.12 Recognizing an AI Security Incident</b></summary>
<br>

### Recognizing an AI Security Incident

Not every error is a security incident.

A model may produce a poor prediction without an attacker being involved.

A container may restart because of a software failure.

A user may enter the wrong password.

A security incident occurs when evidence suggests that confidentiality, integrity, availability, or another important security property may have been compromised.

Potential AI security indicators include:

- Large numbers of unauthorized queries
- Repeated authentication failures
- Unexpected model-file changes
- Failed model signature verification
- Suspicious secret access
- Unexpected cloud privilege changes
- Unusual outbound traffic
- Large-scale model probing
- Unexplained model behavior changes
- Unauthorized deployment events

A useful investigation process is:

**Signal**

↓

**Context**

↓

**Correlation**

↓

**Determine Whether Incident Exists**

For example:

**High GPU Usage Alone**

may not indicate an attack.

But:

**High GPU Usage**

+

**Extreme Request Volume**

+

**Rate-Limit Violations**

+

**Unknown Source**

may provide stronger evidence of malicious activity.

</details>

<br>

<details>
<summary><b>8.13 AI Incident Response</b></summary>
<br>

### AI Incident Response

Incident response provides a structured process for handling security events.

A simplified AI incident-response lifecycle can include:

**Detect**

↓

**Analyze**

↓

**Contain**

↓

**Eradicate**

↓

**Recover**

↓

**Learn**

Tuscano and Disso propose an incident-response framework specifically for generative AI systems and connect AI-specific security problems with established incident-response practices [29].

AI incident response may need to consider assets not always present in traditional systems, such as:

- Models
- Training data
- Model registries
- Inference endpoints
- Prompt or query logs
- Model artifacts
- AI provider credentials
- Model versions

For example, a compromised web server might require traditional host containment.

A compromised AI deployment may additionally require:

- Blocking inference access
- Revoking model-service credentials
- Verifying the model artifact
- Rolling back to a trusted version
- Reviewing suspicious query behavior

AI incident response therefore extends established cybersecurity practices to AI-specific resources.

</details>

<br>

<details>
<summary><b>8.14 Containment of Compromised AI Components</b></summary>
<br>

### Containment of Compromised AI Components

The goal of containment is to prevent an incident from causing additional damage.

Suppose monitoring identifies that an inference service has been compromised.

Possible containment actions may include:

- Disable the affected endpoint
- Isolate the container
- Block network communication
- Revoke service credentials
- Remove compromised instances
- Restrict suspicious accounts
- Preserve relevant logs

A simplified response may look like:

**Compromised Inference Service**

↓

**Network Isolation**

↓

**Credential Revocation**

↓

**Traffic Redirected to Trusted Service**

Containment decisions should consider business impact.

Immediately shutting down a critical AI service may prevent an attack but also disrupt legitimate operations.

Organizations may therefore choose different strategies depending on severity.

The network segmentation and least-privilege controls introduced earlier help make containment easier.

A well-segmented workload can often be isolated without disconnecting the entire environment.

</details>

<br>

<details>
<summary><b>8.15 Model Rollback and Recovery</b></summary>
<br>

### Model Rollback and Recovery

If a production model has been modified, corrupted, or deployed incorrectly, recovery may require restoring a trusted previous artifact.

Section 7 introduced secure rollback.

During an incident, rollback may become part of recovery.

For example:

**Model v5 → Suspected Compromise**

↓

**Verify Model v4**

↓

**Deploy Trusted v4**

↓

**Monitor Behavior**

Recovery should not rely on arbitrary backup files.

The restored model should be:

- Known
- Approved
- Versioned
- Integrity verified
- Stored in trusted infrastructure

Credentials associated with the incident may also need to be rotated before the service returns to normal operation.

Recovery may therefore include:

**Replace Model**

+

**Rotate Credentials**

+

**Rebuild Container**

+

**Verify Configuration**

+

**Restore Service**

The objective is not simply to make the system operational again.

The objective is to restore it to a **trusted state**.

</details>

<br>

<details>
<summary><b>8.16 Post-Incident Analysis and Improvement</b></summary>
<br>

### Post-Incident Analysis and Improvement

Incident response should not end when the affected service returns online.

Organizations should determine:

- What happened?
- How did the attacker gain access?
- Which systems were affected?
- Which controls failed?
- Which controls worked?
- What evidence was available?
- What evidence was missing?
- How can recurrence be prevented?

Consider an incident involving a stolen service credential.

The investigation may discover:

**Credential Hardcoded in Application**

↓

**Credential Extracted**

↓

**Excessive Permissions**

↓

**Model Registry Modified**

Several improvements may follow:

- Remove hardcoded secrets
- Use managed credentials
- Reduce service permissions
- Add model signature verification
- Improve registry logging
- Add alerts for unauthorized model changes

This creates a cycle:

**Incident**

↓

**Learn**

↓

**Improve Controls**

↓

**Improve Monitoring**

↓

**Reduce Future Risk**

Incident response should therefore contribute to continuous security improvement rather than only restoring operations.

</details>

<br>

<details>
<summary><b>8.17 Connecting Monitoring and Response to Earlier Module Controls</b></summary>
<br>

### Connecting Monitoring and Response to Earlier Module Controls

The controls introduced throughout Module 6 are strongest when they work together.

Consider an attacker attempting to abuse an AI inference service.

### Section 2 — API Security

**Rate Limit Exceeded**

↓

Request restricted.

### Section 3 — IAM

**Unauthorized Resource Access**

↓

Permission denied.

### Section 4 — Secrets

**Credential Access Logged**

↓

Suspicious activity detected.

### Section 5 — Container Security

**Resource Limit Reached**

↓

Container cannot consume unlimited resources.

### Section 6 — Network Security

**Unexpected Outbound Connection**

↓

Network policy blocks communication.

### Section 7 — Pipeline Security

**Model Hash Mismatch**

↓

Deployment blocked.

### Section 8 — Monitoring

All of these events can generate evidence.

That evidence allows defenders to determine:

**What happened?**

**Who was involved?**

**Which control responded?**

**Was the attack successful?**

This is the final layer of defense in depth.

Security controls do not only need to:

**Prevent**

They also need to:

**Detect**

**Record**

**Support Response**

**Support Recovery**

</details>

<br>

<details>
<summary><b>Section 8 Summary</b></summary>
<br>

### Section 8 Summary

Secure deployment requires continuous visibility after an AI system reaches production. Logging, monitoring, alerting, and incident response allow defenders to detect suspicious behavior, understand security events, contain compromised systems, and restore trusted operation.

Throughout this section, you learned that:

- Security continues throughout the production lifecycle.
- Observability combines logs, metrics, traces, model telemetry, and security signals.
- AI systems should log important authentication, authorization, inference, deployment, and configuration events.
- Infrastructure monitoring tracks CPU, memory, GPU, network activity, latency, availability, and errors.
- AI-specific monitoring tracks model outputs, input distributions, confidence, performance, drift, and model version.
- Baselines help defenders distinguish expected activity from significant deviations.
- Data drift may indicate legitimate change, model degradation, or potentially suspicious behavior.
- Suspicious query patterns may indicate extraction, inference, probing, or resource-exhaustion activity.
- Authentication and authorization events provide valuable indicators of attempted misuse.
- Availability monitoring can help identify resource-exhaustion attacks.
- Alert thresholds should identify meaningful events without creating excessive alert fatigue.
- Multiple signals should be correlated before concluding that a security incident has occurred.
- AI incident response extends traditional incident-response practices to models, datasets, inference services, and deployment infrastructure.
- Containment may include isolating workloads, revoking credentials, blocking traffic, and disabling affected endpoints.
- Recovery should restore verified models, trusted configurations, and protected credentials.
- Post-incident analysis should identify failed controls and improve the security architecture.
- Prevention, detection, response, and recovery should work together as part of defense in depth.

This section completes Module 6 by bringing together the full secure-deployment lifecycle:

**Architecture**

↓

**API Security**

↓

**Identity and Access Control**

↓

**Secrets and Configuration**

↓

**Container Security**

↓

**Network and Cloud Security**

↓

**Model and Pipeline Integrity**

↓

**Logging, Monitoring, and Incident Response**

A secure AI deployment is not defined by a single control. It is created by **multiple layers of protection that reduce attack surface, limit attacker movement, preserve model integrity, detect abnormal behavior, and support recovery when incidents occur.**

</details>

</details>

In [ ]:

# @title Section 8 — Logging, Monitoring, Incident Response, and Secure Operations Visualizations { display-mode: "form" }

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display, HTML
from io import BytesIO
import base64
import html

# ============================================================
# CONFIGURATION
# ============================================================

BLUE = "#2563EB"
GREEN = "#16A34A"
RED = "#DC2626"
ORANGE = "#EA580C"
PURPLE = "#7C3AED"
GRAY = "#64748B"
DARK = "#0F172A"
TEAL = "#0891B2"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})


# ============================================================
# COLLAPSIBLE VISUALIZATION HELPER
# ============================================================

def show_collapsible_figure(title, fig, explanation, color=BLUE):
    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=135,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.close(fig)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")
    buffer.close()

    display(HTML(f"""
    <details style="margin:12px 0;border:1px solid #cbd5e1;
                    border-radius:10px;overflow:hidden;">
      <summary style="cursor:pointer;padding:16px;font-size:16px;
                      font-weight:bold;background:#eff6ff;
                      color:#1e3a8a;border-left:5px solid {color};">
        {html.escape(title)} — Click to Expand
      </summary>
      <div style="padding:15px;background:white;text-align:center;">
        <img src="data:image/png;base64,{encoded}"
             style="width:100%;max-width:950px;height:auto;">
        <div style="max-width:850px;margin:15px auto;padding:16px;
                    background:#f8fafc;border-radius:8px;color:#334155;
                    font-size:14px;line-height:1.7;text-align:left;">
          {explanation}
        </div>
      </div>
    </details>
    """))


# ============================================================
# DIAGRAM HELPERS
# ============================================================

def setup_diagram(ax, title, xlim=(0, 12), ylim=(0, 10)):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.axis("off")
    ax.set_title(
        title,
        fontsize=15,
        fontweight="bold",
        pad=18
    )


def add_card(ax, x, y, w, h, title, subtitle="",
             accent=BLUE, background="#F8FAFC",
             title_size=10):

    card = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.12",
        facecolor=background,
        edgecolor=accent,
        linewidth=1.8
    )
    ax.add_patch(card)

    ax.text(
        x+w/2,
        y+h*(0.66 if subtitle else 0.5),
        title,
        ha="center",
        va="center",
        fontsize=title_size,
        fontweight="bold",
        color=DARK
    )

    if subtitle:
        ax.text(
            x+w/2,
            y+h*0.28,
            subtitle,
            ha="center",
            va="center",
            fontsize=8.5,
            color=GRAY
        )


def add_arrow(ax, start, end, color=GRAY):
    ax.add_patch(FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=16,
        linewidth=1.8,
        color=color
    ))


# ============================================================
# VISUALIZATION 1
# AI OBSERVABILITY AND SECURITY LOGGING
# ============================================================

fig, ax = plt.subplots(figsize=(12, 10))

setup_diagram(
    ax,
    "AI System Observability and Security Monitoring",
    xlim=(0, 12),
    ylim=(0, 12)
)

sources = [
    (0.3, 9.0, "Infrastructure",
     "CPU, GPU, memory, network", BLUE),
    (4.4, 9.0, "Application",
     "Requests, errors, latency", PURPLE),
    (8.5, 9.0, "Identity and Access",
     "Logins, permissions, failures", ORANGE),
    (2.3, 6.5, "Model Behavior",
     "Predictions, confidence, drift", GREEN),
    (6.5, 6.5, "Security Events",
     "Policy violations, suspicious queries", RED)
]

for x, y, title, subtitle, color in sources:
    add_card(
        ax, x, y, 3.2, 1.2,
        title, subtitle, color
    )

add_card(
    ax, 4.2, 3.8, 3.6, 1.3,
    "Monitoring Platform",
    "Collect, correlate, and analyze",
    TEAL
)

for x in [1.9, 6.0, 10.1]:
    add_arrow(
        ax, (x, 9.0), (6.0, 5.1),
        GRAY
    )

for x in [3.9, 8.1]:
    add_arrow(
        ax, (x, 6.5), (6.0, 5.1),
        GRAY
    )

add_card(
    ax, 4.2, 1.1, 3.6, 1.3,
    "Alerts and Investigation",
    "Review abnormal activity",
    RED
)

add_arrow(
    ax, (6.0, 3.8), (6.0, 2.4),
    RED
)

plt.tight_layout()

show_collapsible_figure(
    "8.1 — AI Observability and Security Logging",
    fig,
    """
    Observability provides visibility into
    the behavior and condition of a
    deployed AI system.

    <br><br>

    Traditional applications are often
    monitored using infrastructure
    metrics, application logs,
    and network activity.

    <br><br>

    AI systems require these same
    capabilities, but also benefit
    from monitoring model-specific
    behavior.

    <br><br>

    <b>Infrastructure monitoring</b>
    tracks CPU usage, GPU utilization,
    memory consumption, and
    network activity.

    <br><br>

    <b>Application monitoring</b>
    records request volume,
    response times, errors,
    and service availability.

    <br><br>

    <b>Identity monitoring</b>
    helps identify failed
    authentication attempts,
    unusual access patterns,
    and unauthorized operations.

    <br><br>

    <b>Model monitoring</b>
    evaluates changes in
    predictions, input
    distributions, and
    model performance.

    <br><br>

    A monitoring platform
    can correlate information
    from these sources to
    support investigation.

    <br><br>

    Logs should include
    useful context such as
    timestamps, request IDs,
    service identities,
    and event outcomes.

    <br><br>

    However, defenders should
    avoid unnecessarily logging
    credentials, sensitive
    prompts, personal data,
    or confidential outputs.

    <br><br>

    <b>Key takeaway:</b> AI
    observability combines
    traditional infrastructure
    monitoring with visibility
    into model behavior
    and security events.
    """,
    BLUE
)


# ============================================================
# VISUALIZATION 2
# ESTABLISHING MONITORING BASELINES
# ============================================================

rng = np.random.default_rng(42)

minutes = np.arange(1, 61)

# Synthetic baseline and observed latency values.
baseline = 110 + rng.normal(0, 7, 60)
observed = baseline.copy()

# Illustrative latency anomaly.
observed[35:47] += np.array([
    20, 30, 45, 65, 90, 115,
    125, 110, 90, 70, 45, 25
])

threshold = 155

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(
    minutes,
    baseline,
    color=BLUE,
    linewidth=2,
    label="Expected Latency Baseline"
)

ax.plot(
    minutes,
    observed,
    color=ORANGE,
    linewidth=2.4,
    label="Observed Latency"
)

ax.axhline(
    threshold,
    color=RED,
    linestyle="--",
    linewidth=2,
    label="Illustrative Alert Threshold"
)

ax.fill_between(
    minutes,
    observed,
    threshold,
    where=observed > threshold,
    color=RED,
    alpha=0.15
)

ax.annotate(
    "Unusual latency increase",
    xy=(41, observed[40]),
    xytext=(19, 245),
    fontsize=10,
    fontweight="bold",
    color=RED,
    arrowprops=dict(
        arrowstyle="->",
        color=RED,
        linewidth=1.5
    )
)

ax.set_xlabel("Minute")
ax.set_ylabel("Inference Latency (ms)")

ax.set_title(
    "Illustrative AI Inference Latency Monitoring",
    fontsize=16,
    fontweight="bold"
)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "8.2 — Establishing Normal Behavior and Monitoring Baselines",
    fig,
    """
    A monitoring baseline
    describes the expected
    behavior of a system
    under normal conditions.

    <br><br>

    Without a baseline,
    defenders may struggle
    to distinguish ordinary
    workload changes from
    potentially suspicious
    activity.

    <br><br>

    For an AI inference
    service, useful baseline
    measurements may include
    response latency,
    request volume,
    error rates, and
    resource utilization.

    <br><br>

    This graph shows
    synthetic inference
    latency over a
    60-minute period.

    <br><br>

    The blue line represents
    expected latency,
    while the orange line
    represents observed
    latency.

    <br><br>

    During the middle
    of the observation
    period, response
    times increase
    significantly.

    <br><br>

    The increase exceeds
    an illustrative
    alert threshold.

    <br><br>

    This may indicate
    excessive requests,
    resource contention,
    application problems,
    or malicious activity.

    <br><br>

    However, an anomaly
    does not automatically
    establish that a
    security incident
    has occurred.

    <br><br>

    Baselines should account
    for legitimate workload
    variations, such as
    peak usage periods
    and scheduled processing.

    <br><br>

    <b>Key takeaway:</b>
    Monitoring baselines
    help identify unusual
    behavior, but anomalies
    require investigation
    and contextual analysis.
    """,
    ORANGE
)


# ============================================================
# VISUALIZATION 3
# DATA DRIFT AND DISTRIBUTION SHIFT
# ============================================================

rng = np.random.default_rng(21)

# Synthetic feature values.
training_data = rng.normal(
    loc=50,
    scale=10,
    size=1200
)

production_data = rng.normal(
    loc=66,
    scale=13,
    size=1200
)

fig, ax = plt.subplots(figsize=(12, 7))

bins = np.linspace(10, 110, 35)

ax.hist(
    training_data,
    bins=bins,
    density=True,
    alpha=0.65,
    color=BLUE,
    label="Training Data Distribution"
)

ax.hist(
    production_data,
    bins=bins,
    density=True,
    alpha=0.55,
    color=ORANGE,
    label="Production Data Distribution"
)

ax.axvline(
    training_data.mean(),
    color=BLUE,
    linestyle="--",
    linewidth=2,
    label="Training Mean"
)

ax.axvline(
    production_data.mean(),
    color=RED,
    linestyle="--",
    linewidth=2,
    label="Production Mean"
)

ax.set_xlabel("Illustrative Input Feature Value")
ax.set_ylabel("Probability Density")

ax.set_title(
    "Synthetic Example of Input Data Distribution Shift",
    fontsize=16,
    fontweight="bold"
)

ax.legend()
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "8.3 — Data Drift and Distribution Shift",
    fig,
    """
    Machine learning models
    are trained using data
    that represents patterns
    present during development.

    <br><br>

    After deployment,
    incoming data may
    gradually change.

    <br><br>

    This can occur because
    of changes in user
    behavior, application
    environments, data
    collection processes,
    or attacker activity.

    <br><br>

    <b>Data drift</b> commonly
    refers to changes in
    the statistical properties
    of incoming input data.

    <br><br>

    This graph compares
    two synthetic distributions.

    <br><br>

    The blue distribution
    represents a feature
    observed during training.

    <br><br>

    The orange distribution
    represents the same
    feature after deployment.

    <br><br>

    The production data
    has a higher average
    value and greater
    variability.

    <br><br>

    Such changes may
    affect model performance,
    particularly when the
    model encounters inputs
    that differ substantially
    from its training data.

    <br><br>

    However, data drift
    does not automatically
    mean that a model
    has been attacked
    or that its accuracy
    has decreased.

    <br><br>

    Defenders should
    investigate distribution
    changes and evaluate
    model performance
    using appropriate
    labeled data when
    available.

    <br><br>

    Monitoring can include
    statistical tests,
    feature summaries,
    and comparisons
    against historical
    reference distributions.

    <br><br>

    <b>Key takeaway:</b>
    Data drift monitoring
    helps identify changes
    in the environment
    that may affect the
    reliability and
    security of a
    deployed AI model.
    """,
    PURPLE
)


# ============================================================
# VISUALIZATION 4
# DETECTING SUSPICIOUS QUERY PATTERNS
# ============================================================

# Synthetic requests per minute from three client identities.

minutes = np.arange(1, 31)

rng = np.random.default_rng(7)

normal_client = rng.poisson(12, size=30)

automated_client = np.array([
    10, 11, 13, 12, 15,
    14, 17, 25, 38, 55,
    70, 85, 95, 105, 110,
    108, 112, 107, 110, 106,
    95, 80, 65, 48, 32,
    22, 18, 15, 13, 12
])

batch_client = np.array([
    5, 5, 6, 7, 6,
    5, 6, 7, 6, 5,
    45, 50, 48, 6, 5,
    7, 6, 5, 6, 5,
    45, 48, 46, 7, 6,
    5, 6, 7, 5, 6
])

threshold = 60

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(
    minutes,
    normal_client,
    marker="o",
    markersize=3,
    linewidth=2,
    color=GREEN,
    label="Client A — Routine Activity"
)

ax.plot(
    minutes,
    automated_client,
    linewidth=2.4,
    color=RED,
    label="Client B — Sustained High-Volume Activity"
)

ax.plot(
    minutes,
    batch_client,
    linewidth=2,
    color=PURPLE,
    label="Client C — Periodic Batches"
)

ax.axhline(
    threshold,
    color=ORANGE,
    linestyle="--",
    linewidth=2,
    label="Illustrative Review Threshold"
)

ax.set_xlabel("Minute")
ax.set_ylabel("Requests per Minute")

ax.set_title(
    "Synthetic AI API Query Patterns",
    fontsize=16,
    fontweight="bold"
)

ax.legend(fontsize=9)
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "8.4 — Detecting Suspicious Query Patterns",
    fig,
    """
    AI inference endpoints
    may receive requests
    from legitimate users,
    automated applications,
    and potentially
    malicious clients.

    <br><br>

    Monitoring query patterns
    can help defenders
    identify activity that
    deserves further
    investigation.

    <br><br>

    The graph illustrates
    three synthetic clients.

    <br><br>

    <b>Client A</b> submits
    a relatively small
    number of requests
    at a consistent rate.

    <br><br>

    <b>Client B</b> generates
    sustained high-volume
    traffic that exceeds
    the illustrative
    review threshold.

    <br><br>

    <b>Client C</b> submits
    periodic batches
    of requests.

    <br><br>

    High-volume querying
    may be associated
    with automated model
    probing, model extraction,
    or attempts to exhaust
    inference resources.

    <br><br>

    However, legitimate
    applications may also
    generate large or
    periodic request
    volumes.

    <br><br>

    Defenders should
    examine additional
    indicators, including
    authentication identity,
    request diversity,
    timing patterns,
    resource consumption,
    and policy violations.

    <br><br>

    Rate limiting,
    request quotas,
    authentication,
    and monitoring can
    work together to
    reduce abusive usage.

    <br><br>

    <b>Key takeaway:</b>
    Suspicious query
    patterns are useful
    investigation signals,
    but request volume
    alone does not prove
    that model extraction
    or another attack
    is occurring.
    """,
    RED
)


# ============================================================
# VISUALIZATION 5
# AI SECURITY INCIDENT RESPONSE LIFECYCLE
# ============================================================

fig, ax = plt.subplots(figsize=(12, 11))

setup_diagram(
    ax,
    "AI Security Incident Response Lifecycle",
    xlim=(0, 12),
    ylim=(0, 13)
)

stages = [
    ("Preparation",
     "Plans, responsibilities, logging, and backups", BLUE),
    ("Detection and Analysis",
     "Identify and investigate suspicious activity", ORANGE),
    ("Containment",
     "Restrict compromised identities or services", RED),
    ("Eradication",
     "Remove the cause and address weaknesses", PURPLE),
    ("Recovery",
     "Restore trusted services and validate operation", GREEN),
    ("Post-Incident Improvement",
     "Document lessons and improve defenses", TEAL)
]

for i, (title, subtitle, color) in enumerate(stages):
    y = 11.1 - i*1.8

    add_card(
        ax, 2.0, y, 8.0, 1.25,
        title, subtitle, color
    )

    if i < len(stages)-1:
        add_arrow(
            ax,
            (6.0, y-0.05),
            (6.0, y-0.55),
            GRAY
        )

ax.text(
    6.0, 0.7,
    "Incident response is iterative: teams may revisit earlier activities",
    ha="center",
    fontsize=10,
    fontweight="bold",
    color=DARK
)

plt.tight_layout()

show_collapsible_figure(
    "8.5 — AI Security Incident Response Lifecycle",
    fig,
    """
    Incident response is
    the structured process
    organizations use
    to investigate and
    recover from security
    incidents.

    <br><br>

    AI systems require
    many of the same
    incident response
    capabilities as
    traditional applications.

    <br><br>

    However, incidents
    may also involve
    model artifacts,
    inference behavior,
    training data,
    or compromised
    deployment pipelines.

    <br><br>

    <b>Preparation</b> involves
    establishing procedures,
    assigning responsibilities,
    and ensuring that
    monitoring and recovery
    capabilities are available.

    <br><br>

    <b>Detection and analysis</b>
    involves investigating
    unusual behavior and
    determining whether
    a security incident
    has occurred.

    <br><br>

    <b>Containment</b> limits
    ongoing damage.
    Examples include
    disabling compromised
    credentials, restricting
    network access, or
    isolating affected
    inference services.

    <br><br>

    <b>Eradication</b> addresses
    the underlying cause,
    such as removing
    malicious artifacts
    or correcting insecure
    configurations.

    <br><br>

    <b>Recovery</b> restores
    trusted operation
    and verifies that
    the system is
    functioning correctly.

    <br><br>

    <b>Post-incident improvement</b>
    examines what happened
    and identifies ways
    to strengthen future
    prevention and response.

    <br><br>

    These activities may
    overlap or repeat
    during an investigation.

    <br><br>

    <b>Key takeaway:</b>
    Effective AI incident
    response combines
    traditional cybersecurity
    procedures with
    model-specific
    investigation and
    recovery activities.
    """,
    ORANGE
)


# ============================================================
# VISUALIZATION 6
# MODEL ROLLBACK, RECOVERY, AND CONTINUOUS IMPROVEMENT
# ============================================================

# Synthetic recovery timeline.
# Values represent a fictional service quality score.

time = np.arange(1, 31)

quality = np.array([
    96, 95, 96, 97, 96,
    95, 96, 96, 95, 96,
    94, 90, 83, 75, 66,
    60, 58, 61, 65, 73,
    87, 93, 95, 96, 96,
    97, 96, 96, 97, 96
])

incident_detected = 14
rollback_started = 20
recovery_verified = 24

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(
    time,
    quality,
    color=BLUE,
    linewidth=2.5,
    marker="o",
    markersize=4,
    label="Illustrative Service Quality"
)

ax.axvline(
    incident_detected,
    color=RED,
    linestyle="--",
    linewidth=2,
    label="Incident Detected"
)

ax.axvline(
    rollback_started,
    color=ORANGE,
    linestyle="--",
    linewidth=2,
    label="Rollback Started"
)

ax.axvline(
    recovery_verified,
    color=GREEN,
    linestyle="--",
    linewidth=2,
    label="Recovery Verified"
)

ax.axhline(
    90,
    color=GRAY,
    linestyle=":",
    linewidth=1.5,
    label="Illustrative Quality Threshold"
)

ax.fill_between(
    time,
    quality,
    90,
    where=quality < 90,
    color=RED,
    alpha=0.15
)

ax.annotate(
    "Degraded operation",
    xy=(17, 58),
    xytext=(8, 53),
    fontsize=10,
    fontweight="bold",
    color=RED,
    arrowprops=dict(
        arrowstyle="->",
        color=RED
    )
)

ax.annotate(
    "Trusted service restored",
    xy=(25, 96),
    xytext=(21, 77),
    fontsize=10,
    fontweight="bold",
    color=GREEN,
    arrowprops=dict(
        arrowstyle="->",
        color=GREEN
    )
)

ax.set_xlim(1, 30)
ax.set_ylim(45, 105)

ax.set_xlabel("Illustrative Time Step")
ax.set_ylabel("Service Quality Score (%)")

ax.set_title(
    "AI Incident Recovery and Model Rollback",
    fontsize=16,
    fontweight="bold"
)

ax.legend(fontsize=8, loc="lower left")
ax.grid(alpha=0.2)

plt.tight_layout()

show_collapsible_figure(
    "8.6 — Model Rollback, Recovery, and Continuous Improvement",
    fig,
    """
    Recovery is an important
    part of securing deployed
    AI systems.

    <br><br>

    If an AI model becomes
    compromised, behaves
    unexpectedly, or produces
    unreliable results,
    defenders may need
    to restore an earlier
    trusted deployment.

    <br><br>

    This graph illustrates
    a fictional incident
    affecting service quality.

    <br><br>

    Initially, the service
    operates near its
    expected performance
    level.

    <br><br>

    After a problem develops,
    the quality score
    begins to decline.

    <br><br>

    The incident is detected,
    investigated, and
    contained.

    <br><br>

    The organization then
    initiates rollback
    to a previously
    approved model version.

    <br><br>

    Before restoring
    production traffic,
    defenders should verify
    the integrity and
    authenticity of the
    selected artifact.

    <br><br>

    They should also
    confirm that the
    supporting application,
    infrastructure, and
    credentials are trustworthy.

    <br><br>

    Recovery testing may
    include checking model
    behavior, application
    availability, security
    controls, and monitoring.

    <br><br>

    The graph shows
    a synthetic improvement
    after rollback.
    Real recovery may
    take longer or require
    additional remediation.

    <br><br>

    Once service is restored,
    the incident should
    be reviewed to identify
    weaknesses in deployment,
    monitoring, or response
    procedures.

    <br><br>

    <b>Key takeaway:</b>
    Secure AI recovery
    requires restoring
    verified artifacts,
    validating system
    behavior, and applying
    lessons learned to
    reduce future risk.
    """,
    GREEN
)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("Section 8 — Logging, Monitoring, Incident Response, and Secure Operations")
print("-------------------------------------------------------------------------")
print("Six collapsible visualizations generated.")

print()
print("Concepts Covered")
print("----------------")
print("1. AI Observability and Security Logging")
print("2. Establishing Monitoring Baselines")
print("3. Data Drift and Distribution Shift")
print("4. Detecting Suspicious Query Patterns")
print("5. AI Security Incident Response Lifecycle")
print("6. Model Rollback, Recovery, and Continuous Improvement")

print()
print("All numerical values and incident scenarios are illustrative.")
print("No external datasets, cloud services, or AI models were used.")


<details>

<summary><b>Section 8 Lab — Click to Expand</b></summary>

<br>

Your organization operates a production AI inference service.

The service has already passed deployment security testing and is actively serving users.

However, successful deployment does not mean that security work is finished.

During normal operation, the system generates information about:

    USER REQUESTS

    AUTHENTICATION

    MODEL PREDICTIONS

    CPU AND GPU USAGE

    NETWORK ACTIVITY

    MODEL VERSION

    ERRORS

    SECURITY EVENTS

During this lab, unusual activity will begin appearing in the environment.

Your job is to determine whether the behavior represents normal variation or a possible security incident.

You will:

    Examine AI security logs

    Establish normal behavioral baselines

    Monitor infrastructure metrics

    Monitor model behavior

    Detect simulated distribution drift

    Identify suspicious query activity

    Detect authentication anomalies

    Monitor resource exhaustion

    Generate alerts using thresholds

    Correlate multiple security signals

    Determine whether an incident exists

    Contain the affected service

    Roll back to a trusted model

    Recover the system

    Perform post-incident analysis

The central question is:

> **How do we know when a deployed AI system is behaving abnormally, and what should we do when something goes wrong?**

All activity is simulated using standard Python so each part should execute almost instantly.

</details>

In [ ]:
# @title ▶ Part 1: Examine Security Logs { display-mode: "form" }

security_logs = [
    {
        "time": "09:00",
        "user": "client_12",
        "endpoint": "/predict",
        "auth": "SUCCESS",
        "model": "fraud-v5",
        "status": 200
    },
    {
        "time": "09:01",
        "user": "client_27",
        "endpoint": "/predict",
        "auth": "SUCCESS",
        "model": "fraud-v5",
        "status": 200
    },
    {
        "time": "09:02",
        "user": "unknown",
        "endpoint": "/predict",
        "auth": "FAILED",
        "model": "fraud-v5",
        "status": 401
    },
    {
        "time": "09:03",
        "user": "client_41",
        "endpoint": "/predict",
        "auth": "SUCCESS",
        "model": "fraud-v5",
        "status": 200
    }
]

print("=" * 70)
print("AI SECURITY LOG")
print("=" * 70)

for event in security_logs:

    print(
        f"{event['time']} | "
        f"{event['user']:<10} | "
        f"{event['endpoint']:<10} | "
        f"{event['auth']:<7} | "
        f"{event['model']} | "
        f"{event['status']}"
    )

<details>

<summary><b>Part 1 Explanation — Click to Expand</b></summary>

<br>

Logs provide a historical record of activity within the AI system.

The entries contain enough information to answer several useful questions:

    WHEN DID THE EVENT OCCUR?

    WHO MADE THE REQUEST?

    WHICH ENDPOINT WAS USED?

    DID AUTHENTICATION SUCCEED?

    WHICH MODEL VERSION RESPONDED?

    WHAT WAS THE RESPONSE STATUS?

Notice that the log does not contain the complete raw contents of every prediction request.

This demonstrates the balance discussed in Section 8 between:

    SECURITY VISIBILITY

and:

    DATA MINIMIZATION

Logging enough information to investigate activity is important, but collecting unnecessary sensitive information can create additional privacy and security risks.

</details>

In [ ]:
# @title ▶ Part 2: Establish Normal Behavior { display-mode: "form" }

baseline = {
    "requests_per_minute": (200, 400),
    "auth_failures_per_hour": (0, 10),
    "gpu_usage": (30, 60),
    "error_rate": (0, 5),
    "class_c_prediction_pct": (15, 25)
}

print("=" * 70)
print("NORMAL AI SYSTEM BASELINE")
print("=" * 70)

for metric, normal_range in baseline.items():

    print(
        f"{metric:<28} "
        f"{normal_range[0]} - {normal_range[1]}"
    )

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

A metric has limited meaning without context.

For example:

    350 REQUESTS PER MINUTE

could represent extremely high traffic for one application but normal traffic for another.

The baseline establishes expected ranges for this particular AI service.

Monitoring can now ask:

    CURRENT VALUE
          ↓
    COMPARE TO BASELINE
          ↓
    NORMAL OR ABNORMAL?

The baseline does not prove that behavior inside the range is safe or that behavior outside the range is malicious.

It provides context that helps defenders decide when additional investigation may be necessary.

</details>

In [ ]:
# @title ▶ Part 3: Monitor Infrastructure { display-mode: "form" }

current_metrics = {
    "CPU Usage": 48,
    "Memory Usage": 55,
    "GPU Usage": 46,
    "Disk Usage": 41,
    "API Latency": 82,
    "HTTP Error Rate": 2
}

print("=" * 70)
print("INFRASTRUCTURE MONITORING")
print("=" * 70)

for metric, value in current_metrics.items():

    print(
        f"{metric:<20}: {value}"
    )

print("\n✓ Infrastructure appears operational.")

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

AI applications still depend on traditional infrastructure.

Even if the model itself is functioning correctly, defenders should observe:

    CPU

    MEMORY

    GPU

    STORAGE

    NETWORK ACTIVITY

    LATENCY

    ERRORS

    SERVICE AVAILABILITY

These metrics help identify reliability problems and may also reveal security-related behavior.

For example, sustained GPU or CPU exhaustion could result from legitimate demand, software failure, misconfiguration, or malicious resource abuse.

Infrastructure metrics therefore provide useful signals, but they must be interpreted with context.

</details>

In [ ]:
# @title ▶ Part 4: Monitor Model Predictions { display-mode: "form" }

normal_predictions = {
    "Class A": 45,
    "Class B": 35,
    "Class C": 20
}

current_predictions = {
    "Class A": 43,
    "Class B": 36,
    "Class C": 21
}

print("=" * 70)
print("MODEL PREDICTION MONITORING")
print("=" * 70)

print("\nBaseline → Current\n")

for label in normal_predictions:

    print(
        f"{label}: "
        f"{normal_predictions[label]}% "
        f"→ {current_predictions[label]}%"
    )

print("\n✓ Prediction behavior remains close to baseline.")

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

Infrastructure monitoring tells us whether the service is operating.

Model monitoring tells us whether the model itself appears to be behaving normally.

The current prediction distribution remains similar to historical behavior.

This distinction is important.

A server could report:

    CPU = NORMAL

    MEMORY = NORMAL

    SERVICE = ONLINE

while the model produces dramatically abnormal predictions.

AI observability therefore combines traditional infrastructure monitoring with model-specific telemetry.

</details>

In [ ]:
# @title ▶ Part 5: Detect Distribution Drift { display-mode: "form" }

drifted_predictions = {
    "Class A": 3,
    "Class B": 5,
    "Class C": 92
}

drift_threshold = 25

print("=" * 70)
print("MODEL DRIFT CHECK")
print("=" * 70)

drift_detected = False

for label in normal_predictions:

    change = abs(
        drifted_predictions[label]
        - normal_predictions[label]
    )

    print(
        f"{label}: "
        f"{normal_predictions[label]}% "
        f"→ {drifted_predictions[label]}% "
        f"(change = {change}%)"
    )

    if change >= drift_threshold:
        drift_detected = True

print("\nResult:")

if drift_detected:
    print("⚠ SIGNIFICANT DISTRIBUTION CHANGE DETECTED")
else:
    print("✓ Distribution within expected range")

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

The prediction distribution has changed dramatically.

Previously:

    CLASS C ≈ 20%

Now:

    CLASS C = 92%

The monitoring system correctly identifies a significant deviation.

However, the correct conclusion is **not**:

    DRIFT = ATTACK

Drift may result from:

    LEGITIMATE DATA CHANGES

    PIPELINE PROBLEMS

    MODEL FAILURE

    MODEL CORRUPTION

    ATTACK ACTIVITY

The drift alert tells defenders that something unusual is happening and additional evidence should be examined.

Drift is a security-relevant signal, not definitive proof of malicious activity.

</details>

In [ ]:
# @title ▶ Part 6: Detect Suspicious Queries { display-mode: "form" }

query_activity = {
    "client_12": 22,
    "client_27": 31,
    "client_41": 18,
    "unknown_203": 8750
}

query_threshold = 500

print("=" * 70)
print("QUERY ACTIVITY MONITOR")
print("=" * 70)

suspicious_clients = []

for client, requests in query_activity.items():

    suspicious = requests > query_threshold

    print(
        f"{client:<15} "
        f"{requests:>5} requests | "
        f"{'⚠ SUSPICIOUS' if suspicious else '✓ NORMAL'}"
    )

    if suspicious:
        suspicious_clients.append(client)

print(
    f"\nSuspicious clients detected: "
    f"{len(suspicious_clients)}"
)

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

Most clients generate a small number of requests.

One identity generates:

    8,750 REQUESTS

This differs substantially from normal behavior.

Extremely high or systematically varied request volumes can be associated with activities such as:

    MODEL EXTRACTION

    MEMBERSHIP INFERENCE

    ADVERSARIAL PROBING

    RECONNAISSANCE

    RESOURCE EXHAUSTION

High request volume alone does not prove which attack is occurring.

It provides another security signal that can be correlated with other evidence.

</details>

In [ ]:
# @title ▶ Part 7: Detect Authentication Anomalies { display-mode: "form" }

authentication_activity = {
    "normal_hour_1": 3,
    "normal_hour_2": 5,
    "normal_hour_3": 2,
    "current_hour": 1840
}

normal_auth_failure_max = 10

print("=" * 70)
print("AUTHENTICATION MONITOR")
print("=" * 70)

auth_alert = False

for period, failures in authentication_activity.items():

    status = (
        "⚠ ALERT"
        if failures > normal_auth_failure_max
        else "✓ NORMAL"
    )

    print(
        f"{period:<16}: "
        f"{failures:>4} failures | {status}"
    )

    if period == "current_hour" and failures > normal_auth_failure_max:
        auth_alert = True

<details>

<summary><b>Part 7 Explanation — Click to Expand</b></summary>

<br>

The normal environment experiences only a few failed authentication attempts each hour.

The current hour contains:

    1,840 FAILURES

This represents a major deviation from the established baseline.

Authentication and authorization events can provide evidence of:

    CREDENTIAL ATTACKS

    UNAUTHORIZED ACCESS ATTEMPTS

    COMPROMISED ACCOUNTS

    PRIVILEGE MISUSE

The monitoring system should preserve these events so defenders can correlate them with activity elsewhere in the environment.

</details>

In [ ]:
# @title ▶ Part 8: Monitor Resource Usage { display-mode: "form" }

resource_history = [
    42, 45, 47, 51, 48,
    96, 99, 100, 100, 99
]

gpu_alert_threshold = 90

print("=" * 70)
print("GPU RESOURCE MONITOR")
print("=" * 70)

high_usage_events = 0

for minute, usage in enumerate(
    resource_history,
    start=1
):

    status = (
        "⚠ HIGH"
        if usage >= gpu_alert_threshold
        else "✓ NORMAL"
    )

    print(
        f"Minute {minute:>2}: "
        f"{usage:>3}% | {status}"
    )

    if usage >= gpu_alert_threshold:
        high_usage_events += 1

resource_alert = high_usage_events >= 3

print("\nResult:")

if resource_alert:
    print("⚠ SUSTAINED RESOURCE EXHAUSTION DETECTED")
else:
    print("✓ No sustained exhaustion detected")

<details>

<summary><b>Part 8 Explanation — Click to Expand</b></summary>

<br>

A single spike in GPU usage may be normal.

The more concerning behavior is sustained usage near maximum capacity.

The system records several consecutive periods above the threshold.

This could indicate:

    LEGITIMATE TRAFFIC INCREASE

    RESOURCE-INTENSIVE WORKLOAD

    SOFTWARE FAILURE

    AUTOMATED ABUSE

    RESOURCE-EXHAUSTION ATTACK

Again, the monitoring system detects abnormal behavior without immediately assuming the cause.

Other telemetry is needed to understand what is happening.

</details>

In [ ]:
# @title ▶ Part 9: Generate Alerts { display-mode: "form" }

alerts = []

if drift_detected:
    alerts.append(
        "MODEL_DISTRIBUTION_ANOMALY"
    )

if suspicious_clients:
    alerts.append(
        "EXCESSIVE_QUERY_ACTIVITY"
    )

if auth_alert:
    alerts.append(
        "AUTHENTICATION_FAILURE_SPIKE"
    )

if resource_alert:
    alerts.append(
        "RESOURCE_EXHAUSTION"
    )

print("=" * 70)
print("SECURITY ALERTS")
print("=" * 70)

for alert in alerts:
    print(f"⚠ {alert}")

print(
    f"\nTotal active alerts: "
    f"{len(alerts)}"
)

<details>

<summary><b>Part 9 Explanation — Click to Expand</b></summary>

<br>

Raw telemetry becomes more useful when monitoring systems convert meaningful deviations into alerts.

The environment now contains alerts for:

    MODEL BEHAVIOR

    QUERY ACTIVITY

    AUTHENTICATION

    RESOURCE USAGE

Alert thresholds should be chosen carefully.

If thresholds are too sensitive:

    TOO MANY ALERTS
          ↓
    ALERT FATIGUE

If thresholds are too permissive:

    IMPORTANT EVENTS
          ↓
       MISSED

The purpose of alerting is to identify activity that deserves attention without overwhelming defenders with unnecessary noise.

</details>

In [ ]:
# @title ▶ Part 10: Correlate Security Signals { display-mode: "form" }

evidence = {
    "Model Distribution Changed":
        drift_detected,

    "Extreme Query Volume":
        len(suspicious_clients) > 0,

    "Authentication Failure Spike":
        auth_alert,

    "Sustained GPU Exhaustion":
        resource_alert
}

print("=" * 70)
print("SECURITY EVENT CORRELATION")
print("=" * 70)

signal_count = 0

for signal, present in evidence.items():

    print(
        f"{signal:<32} "
        f"{'⚠ DETECTED' if present else '✓ NOT DETECTED'}"
    )

    if present:
        signal_count += 1

print(
    f"\nCorrelated signals: "
    f"{signal_count}"
)

<details>

<summary><b>Part 10 Explanation — Click to Expand</b></summary>

<br>

Any one of these events might have an innocent explanation.

For example:

    HIGH GPU USAGE

could result from legitimate demand.

However, the environment simultaneously shows:

    EXTREME QUERY VOLUME

        +

    AUTHENTICATION FAILURE SPIKE

        +

    SUSTAINED GPU EXHAUSTION

        +

    ABNORMAL MODEL BEHAVIOR

The combination provides stronger evidence that defenders should investigate.

This demonstrates the process described in Section 8:

    SIGNAL
       ↓
    CONTEXT
       ↓
    CORRELATION
       ↓
    INCIDENT DETERMINATION

Monitoring should help defenders reason about related events rather than treating every alert as independent proof of an attack.

</details>

In [ ]:
# @title ▶ Part 11: Classify the Security Event { display-mode: "form" }

if signal_count >= 3:
    incident_status = "HIGH"
elif signal_count == 2:
    incident_status = "MEDIUM"
elif signal_count == 1:
    incident_status = "INVESTIGATE"
else:
    incident_status = "NORMAL"

print("=" * 70)
print("INCIDENT ASSESSMENT")
print("=" * 70)

print(
    f"\nCorrelated security signals: "
    f"{signal_count}"
)

print(
    f"Assessment: "
    f"{incident_status}"
)

if incident_status == "HIGH":
    print(
        "\n⚠ Multiple related anomalies "
        "justify incident response."
    )

<details>

<summary><b>Part 11 Explanation — Click to Expand</b></summary>

<br>

Not every application error or unusual metric should automatically be declared a security incident.

In this simulated environment, however, multiple security-relevant indicators occur together.

The lab therefore escalates the event for incident response.

The simplified process is:

    DETECT
       ↓
    ANALYZE
       ↓
    CORRELATE
       ↓
    DETERMINE INCIDENT
       ↓
    RESPOND

Real organizations may use more detailed severity classifications and investigation procedures.

The important lesson is that incident decisions should be based on evidence and context rather than a single isolated metric.

</details>

In [ ]:
# @title ▶ Part 12: Contain the Incident { display-mode: "form" }

containment_actions = {
    "Inference Endpoint": "DISABLED",
    "Compromised Container": "ISOLATED",
    "Suspicious Client": "BLOCKED",
    "Service Credential": "REVOKED",
    "Outbound Network Access": "BLOCKED",
    "Security Logs": "PRESERVED"
}

print("=" * 70)
print("INCIDENT CONTAINMENT")
print("=" * 70)

for control, action in containment_actions.items():

    print(
        f"{control:<28} → {action}"
    )

print(
    "\n✓ Affected service contained."
)

<details>

<summary><b>Part 12 Explanation — Click to Expand</b></summary>

<br>

The immediate objective is no longer simply detecting the attack.

The priority becomes preventing additional damage.

The response isolates the affected service by:

    DISABLING THE ENDPOINT

    ISOLATING THE CONTAINER

    BLOCKING THE SUSPICIOUS CLIENT

    REVOKING CREDENTIALS

    RESTRICTING NETWORK COMMUNICATION

    PRESERVING LOGS

Preserving evidence is important.

Deleting the compromised workload without retaining useful logs could make later investigation more difficult.

Containment should reduce attacker access while preserving enough evidence to understand what happened.

</details>

In [ ]:
# @title ▶ Part 13: Perform Secure Rollback { display-mode: "form" }

models = {
    "fraud-v5": {
        "status": "suspected",
        "integrity_verified": False
    },

    "fraud-v4": {
        "status": "previous-production",
        "integrity_verified": True
    }
}

current_model = "fraud-v5"
rollback_model = "fraud-v4"

print("=" * 70)
print("MODEL RECOVERY")
print("=" * 70)

print(
    f"\nCurrent model: "
    f"{current_model}"
)

if models[rollback_model]["integrity_verified"]:

    current_model = rollback_model

    models["fraud-v5"]["status"] = "quarantined"
    models["fraud-v4"]["status"] = "production"

    print(
        f"✓ Trusted rollback model verified."
    )

    print(
        f"✓ Production restored to "
        f"{current_model}."
    )

    print(
        "✓ fraud-v5 quarantined."
    )

else:

    print(
        "✗ Rollback model could not "
        "be verified."
    )

<details>

<summary><b>Part 13 Explanation — Click to Expand</b></summary>

<br>

Recovery does not simply choose any older model.

The previous model must first be known and integrity verified.

The process follows:

    SUSPECT MODEL v5
          ↓
       QUARANTINE
          ↓
    VERIFY MODEL v4
          ↓
    RESTORE TRUSTED v4
          ↓
    CONTINUE MONITORING

This connects directly to the secure model versioning and rollback controls from Section 7.

The goal is not simply:

    GET SERVICE RUNNING AGAIN

The goal is:

    RESTORE A TRUSTED STATE

</details>

In [ ]:
# @title ▶ Part 14: Restore Trusted Operation { display-mode: "form" }

recovery_steps = {
    "Trusted Model Restored": True,
    "Credential Rotated": True,
    "Container Rebuilt": True,
    "Configuration Verified": True,
    "Network Rules Verified": True,
    "Monitoring Enabled": True
}

print("=" * 70)
print("RECOVERY VERIFICATION")
print("=" * 70)

recovery_complete = True

for step, completed in recovery_steps.items():

    print(
        f"{step:<28} "
        f"{'✓ COMPLETE' if completed else '✗ INCOMPLETE'}"
    )

    if not completed:
        recovery_complete = False

print("\nRecovery Status:")

if recovery_complete:
    print("✓ TRUSTED OPERATION RESTORED")
else:
    print("✗ RECOVERY INCOMPLETE")

<details>

<summary><b>Part 14 Explanation — Click to Expand</b></summary>

<br>

Replacing the model alone may not remove every part of the compromise.

The recovery process also:

    ROTATES CREDENTIALS

    REBUILDS THE CONTAINER

    VERIFIES CONFIGURATION

    CHECKS NETWORK CONTROLS

    RESTORES MONITORING

This prevents the organization from returning a partially compromised environment to production.

Recovery therefore combines controls introduced throughout the module.

The system should return to service only after defenders have reasonable confidence that the production environment is once again trusted.

</details>

In [ ]:
# @title ▶ Part 15: Post-Incident Review { display-mode: "form" }

incident_review = {
    "Initial Indicator":
        "Extreme query volume",

    "Additional Evidence":
        "Authentication failures and GPU exhaustion",

    "Model Indicator":
        "Abnormal prediction distribution",

    "Containment":
        "Endpoint isolated and credential revoked",

    "Recovery":
        "Verified model v4 restored",

    "Improvement":
        "Lower query thresholds and strengthen access monitoring"
}

print("=" * 70)
print("POST-INCIDENT REVIEW")
print("=" * 70)

for category, finding in incident_review.items():

    print(
        f"\n{category}:"
    )

    print(
        f"  {finding}"
    )

<details>

<summary><b>Part 15 Explanation — Click to Expand</b></summary>

<br>

Incident response should not end when production returns to normal.

The organization should determine:

    WHAT HAPPENED?

    HOW WAS IT DETECTED?

    WHICH CONTROLS WORKED?

    WHICH CONTROLS FAILED?

    WHAT EVIDENCE WAS AVAILABLE?

    WHAT SHOULD CHANGE?

The result should be continuous improvement:

    INCIDENT
       ↓
    INVESTIGATE
       ↓
    LEARN
       ↓
    IMPROVE CONTROLS
       ↓
    IMPROVE MONITORING
       ↓
    REDUCE FUTURE RISK

This turns incident response into part of the organization's long-term security process.

</details>

In [ ]:
# @title ▶ Part 16: Defense-in-Depth Review { display-mode: "form" }

module_controls = {
    "API Security":
        "Rate limits suspicious traffic",

    "IAM":
        "Blocks unauthorized access",

    "Secrets":
        "Protects and rotates credentials",

    "Containers":
        "Restricts compromised workloads",

    "Network Security":
        "Limits lateral movement",

    "Pipeline Security":
        "Protects model integrity",

    "Monitoring":
        "Detects abnormal behavior",

    "Incident Response":
        "Contains and recovers"
}

print("=" * 70)
print("AI DEPLOYMENT DEFENSE IN DEPTH")
print("=" * 70)

for control, purpose in module_controls.items():

    print(
        f"\n{control}"
    )

    print(
        f"  → {purpose}"
    )

<details>

<summary><b>Part 16 Explanation — Click to Expand</b></summary>

<br>

Section 8 brings the entire module together.

Earlier controls attempt to prevent or limit attacks.

For example:

    API RATE LIMIT
          ↓
    RESTRICT ABUSE

    IAM
          ↓
    DENY UNAUTHORIZED ACCESS

    CONTAINER LIMIT
          ↓
    RESTRICT RESOURCE USAGE

    NETWORK POLICY
          ↓
    BLOCK LATERAL MOVEMENT

    MODEL HASH
          ↓
    DETECT ARTIFACT CHANGE

Section 8 adds another requirement.

These controls should also produce evidence.

That evidence allows defenders to:

    DETECT

    INVESTIGATE

    RESPOND

    RECOVER

Security controls are therefore strongest when prevention and detection work together.

</details>

In [ ]:
# @title ▶ Part 17: Knowledge Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. Why did the lab establish a baseline before detecting anomalies?",
        "options": [
            "A. To provide context for what normal behavior looks like",
            "B. To guarantee that no attack occurs",
            "C. To disable alerts",
            "D. To increase GPU usage"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. Did the prediction distribution change alone prove an attack?",
        "options": [
            "A. No, drift is a signal requiring additional context",
            "B. Yes, all drift proves malicious activity",
            "C. Yes, because Class C increased",
            "D. No, because drift should never be monitored"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. Why was unknown_203 considered suspicious?",
        "options": [
            "A. It generated extremely high query volume",
            "B. It made one normal request",
            "C. It used the approved model",
            "D. It had low GPU usage"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Why was sustained GPU usage more meaningful than one brief spike?",
        "options": [
            "A. Sustained abnormal activity provides stronger evidence of a continuing problem",
            "B. One spike always proves an attack",
            "C. GPU usage has no security relevance",
            "D. High GPU usage always means model drift"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. Why were multiple alerts correlated?",
        "options": [
            "A. Related signals provide more context for determining whether an incident exists",
            "B. Every alert independently proves compromise",
            "C. Correlation removes the need for logs",
            "D. Correlation prevents attacks automatically"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. What was the main purpose of containment?",
        "options": [
            "A. Prevent additional damage while preserving evidence",
            "B. Immediately delete every log",
            "C. Increase attacker access",
            "D. Train a new model"
        ],
        "answer": "A"
    },
    {
        "question":
        "7. Why was fraud-v4 selected for rollback?",
        "options": [
            "A. It was a known previous version whose integrity was verified",
            "B. It was randomly selected",
            "C. Older models are always safer",
            "D. It did not need security verification"
        ],
        "answer": "A"
    },
    {
        "question":
        "8. Why did recovery include credential rotation and container rebuilding?",
        "options": [
            "A. Recovery should restore the entire environment to a trusted state",
            "B. Model rollback automatically repairs every compromised component",
            "C. Credentials never need replacement",
            "D. Containers cannot be compromised"
        ],
        "answer": "A"
    },
    {
        "question":
        "9. What is the purpose of post-incident analysis?",
        "options": [
            "A. Learn from the incident and improve future controls",
            "B. Hide evidence of the incident",
            "C. Disable monitoring",
            "D. Restore the compromised configuration"
        ],
        "answer": "A"
    },
    {
        "question":
        "10. What does Section 8 add to defense in depth?",
        "options": [
            "A. Detection, investigation, response, recovery, and improvement",
            "B. Only model training",
            "C. Only container creation",
            "D. Removal of preventive controls"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("LAB 8 — KNOWLEDGE CHECK")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1

    else:
        print(
            f"✗ Incorrect. "
            f"Correct answer: {q['answer']}"
        )

percentage = (
    score / len(questions)
) * 100

print("\n" + "=" * 70)
print(f"Lab Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Lab complete!")
else:
    print(
        "△ Review the lab explanations "
        "and try again."
    )

<details>

<summary><b>Lab 8 — Practical Takeaways — Click to Expand</b></summary>

<br>

## Lab 8 — Practical Takeaways

This lab demonstrated what happens **after an AI system reaches production**.

Deployment was not the end of the security process.

You began by collecting logs and establishing normal behavior for the production AI service.

This created:

    OBSERVABILITY
          ↓
    BASELINE
          ↓
    CURRENT BEHAVIOR
          ↓
    COMPARISON

You then monitored both traditional infrastructure and AI-specific behavior.

Infrastructure monitoring examined signals such as:

    CPU

    MEMORY

    GPU

    ERRORS

    LATENCY

Model monitoring examined signals such as:

    PREDICTION DISTRIBUTION

    MODEL VERSION

    DRIFT

    INFERENCE ACTIVITY

This distinction is important because a service can appear operational while the model itself behaves abnormally.

The lab then introduced several suspicious signals:

    ABNORMAL MODEL DISTRIBUTION

            +

    EXTREME QUERY VOLUME

            +

    AUTHENTICATION FAILURE SPIKE

            +

    SUSTAINED GPU EXHAUSTION

No individual signal automatically proved that an attack occurred.

Instead, the monitoring process followed:

    SIGNAL
       ↓
    CONTEXT
       ↓
    CORRELATION
       ↓
    INCIDENT DETERMINATION

Once enough evidence existed, the lab moved from detection into incident response.

The affected AI service was contained by isolating the workload, blocking suspicious activity, revoking credentials, restricting communication, and preserving logs.

Recovery then restored a verified previous model and rebuilt other affected components.

The objective was not simply:

    MAKE THE SERVICE WORK AGAIN

It was:

    RESTORE A TRUSTED STATE

Finally, the post-incident review identified lessons that could improve future monitoring and security controls.

This creates the continuing security lifecycle:

    DEPLOY
       ↓
    MONITOR
       ↓
    DETECT
       ↓
    RESPOND
       ↓
    CONTAIN
       ↓
    RECOVER
       ↓
    LEARN
       ↓
    IMPROVE
       ↓
    MONITOR AGAIN

The central lesson from Section 8 is:

> **A secure AI system must not only resist attacks. Defenders must also be able to observe abnormal behavior, understand what happened, contain the damage, restore trusted operation, and improve the system afterward.**

This also completes the module's defense-in-depth architecture.

    API SECURITY
          ↓
    IDENTITY AND ACCESS
          ↓
    SECRETS MANAGEMENT
          ↓
    CONTAINER SECURITY
          ↓
    NETWORK SECURITY
          ↓
    MODEL AND PIPELINE INTEGRITY
          ↓
    MONITORING AND RESPONSE

No single layer is expected to stop every attack.

Together, the layers reduce attack opportunities, limit attacker movement, preserve model integrity, generate evidence, detect abnormal behavior, and support recovery when prevention fails.

</details>

In [ ]:
# @title ▶ Section 8 Checkpoint — Monitoring & Incident Response { display-mode: "form" }

questions = [
    {
        "question": "1. Why must security monitoring continue after deployment?",
        "options": [
            "A. Production systems can experience attacks, drift, failures, and configuration changes",
            "B. Deployment guarantees permanent security",
            "C. Models cannot change after deployment",
            "D. Monitoring is only useful during training"
        ],
        "answer": "A"
    },
    {
        "question": "2. What does observability help defenders understand?",
        "options": [
            "A. The internal state and behavior of a system using its telemetry",
            "B. Only model accuracy",
            "C. Only user passwords",
            "D. Only training speed"
        ],
        "answer": "A"
    },
    {
        "question": "3. Why establish a behavioral baseline?",
        "options": [
            "A. To compare current behavior with expected normal behavior",
            "B. To make every alert critical",
            "C. To disable logging",
            "D. To eliminate authentication"
        ],
        "answer": "A"
    },
    {
        "question": "4. Does data drift automatically prove that an attack occurred?",
        "options": [
            "A. No, drift is a signal that requires context and investigation",
            "B. Yes, all drift is malicious",
            "C. Yes, if accuracy changes at all",
            "D. No, because drift has no security relevance"
        ],
        "answer": "A"
    },
    {
        "question": "5. Which behavior may indicate automated model probing?",
        "options": [
            "A. Extremely high volumes of systematically varied queries",
            "B. One normal prediction request",
            "C. A scheduled backup",
            "D. A successful normal login"
        ],
        "answer": "A"
    },
    {
        "question": "6. Why should defenders correlate multiple security signals?",
        "options": [
            "A. Multiple related signals can provide stronger evidence than one event alone",
            "B. Every individual alert proves an attack",
            "C. Correlation eliminates logging",
            "D. Correlation prevents every attack"
        ],
        "answer": "A"
    },
    {
        "question": "7. What is the primary goal of containment?",
        "options": [
            "A. Prevent an incident from causing additional damage",
            "B. Delete every security log",
            "C. Increase attacker access",
            "D. Ignore the affected service"
        ],
        "answer": "A"
    },
    {
        "question": "8. What should recovery restore?",
        "options": [
            "A. A verified and trusted operational state",
            "B. Any available model file",
            "C. The compromised configuration",
            "D. Unverified credentials"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("SECTION 8 CHECKPOINT")
print("LOGGING, MONITORING, INCIDENT RESPONSE, AND SECURE OPERATIONS")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input("\nYour answer: ").strip().upper()

    if response == q["answer"]:
        print("✓ Correct!")
        score += 1
    else:
        print(f"✗ Incorrect. Correct answer: {q['answer']}")

percentage = score / len(questions) * 100

print("\n" + "=" * 70)
print(f"Score: {score}/{len(questions)}")
print(f"Percentage: {percentage:.1f}%")

if percentage >= 80:
    print("✓ Checkpoint complete!")
else:
    print("△ Review Section 8 and try again.")

---

<details>

<summary><b>Final Lab Overview — Click to Expand</b></summary>

<br>

Throughout this module, you examined individual controls used to securely deploy and operate AI systems.

In this final lab, those controls will no longer be examined individually.

You are now responsible for the security of a fictional production AI service called:

    SentinelAI

SentinelAI provides fraud predictions through a cloud-hosted API.

The production architecture contains:

    INTERNET
       ↓
    API GATEWAY
       ↓
    INFERENCE SERVICE
       ↓
    MODEL REGISTRY
       ↓
    PRODUCTION DATABASE

Supporting infrastructure also includes:

    SECRET MANAGER

    LOGGING SERVICE

    TRAINING STORAGE

    CI/CD PIPELINE

The development team believes the system is ready for production.

Your security team must determine whether that is actually true.

During the lab, you will:

    Audit the deployment configuration

    Identify unnecessary exposure

    Harden the API

    Apply least-privilege access

    Protect secrets

    Restrict the inference workload

    Segment the network

    Verify the production model

    Enforce deployment gates

    Establish monitoring baselines

    Detect suspicious activity

    Correlate security signals

    Respond to a simulated incident

    Contain the affected workload

    Recover using a trusted model

    Evaluate the final security posture

The purpose of the lab is not simply to make every test display:

    PASS

Instead, you should understand why each security control exists and what could happen if that control were missing.

The central question for the final lab is:

> **Can you securely deploy an AI system, detect when something goes wrong, limit the damage, and restore the system to a trusted state?**

All exercises are simulated with standard Python and should execute almost instantly.

</details>

In [ ]:
# @title ▶ Part 1: Inspect the Production Environment { display-mode: "form" }

environment = {
    "API Gateway": {
        "public": True,
        "purpose": "Receive external prediction requests"
    },

    "Inference Service": {
        "public": True,
        "purpose": "Run SentinelAI model"
    },

    "Model Registry": {
        "public": False,
        "purpose": "Store approved models"
    },

    "Production Database": {
        "public": False,
        "purpose": "Store application data"
    },

    "Training Storage": {
        "public": True,
        "purpose": "Store training datasets"
    },

    "Secret Manager": {
        "public": False,
        "purpose": "Store production credentials"
    },

    "Logging Service": {
        "public": False,
        "purpose": "Collect operational and security logs"
    }
}

print("=" * 70)
print("SENTINELAI PRODUCTION ENVIRONMENT")
print("=" * 70)

for service, config in environment.items():

    exposure = (
        "PUBLIC"
        if config["public"]
        else "PRIVATE"
    )

    print(f"\n{service}")
    print(f"  Exposure: {exposure}")
    print(f"  Purpose:  {config['purpose']}")

<details>

<summary><b>Part 1 — Student Task — Click to Expand</b></summary>

<br>

Review the production environment.

Ask yourself:

**Which service actually needs direct Internet exposure?**

The API gateway exists specifically to receive external requests.

Compare that purpose with the inference service and training storage.

Before continuing, identify which resources appear unnecessarily exposed.

You will verify your answer in the next part.

</details>

In [ ]:
# @title ▶ Part 2: Audit Public Exposure { display-mode: "form" }

expected_public_services = {
    "API Gateway"
}

exposure_failures = []

print("=" * 70)
print("PUBLIC EXPOSURE AUDIT")
print("=" * 70)

for service, config in environment.items():

    should_be_public = (
        service in expected_public_services
    )

    actually_public = config["public"]

    secure = (
        should_be_public == actually_public
    )

    print(
        f"{service:<25} "
        f"{'✓ PASS' if secure else '✗ FAIL'}"
    )

    if not secure:
        exposure_failures.append(service)

print("\nMisconfigured Resources:")

for service in exposure_failures:
    print(f"  ⚠ {service}")

<details>

<summary><b>Part 2 Explanation — Click to Expand</b></summary>

<br>

The audit discovered two unnecessary public resources:

    INFERENCE SERVICE

    TRAINING STORAGE

External users should normally interact with:

    INTERNET
       ↓
    API GATEWAY
       ↓
    PRIVATE INFERENCE SERVICE

Training storage also has no reason to accept arbitrary Internet traffic.

This demonstrates an important deployment principle:

> **A resource should not be publicly accessible simply because it exists in a cloud environment.**

Reducing exposure removes unnecessary attack paths before attackers can use them.

</details>

In [ ]:
# @title ▶ Part 3: Correct the Exposure { display-mode: "form" }

environment[
    "Inference Service"
]["public"] = False

environment[
    "Training Storage"
]["public"] = False

print("=" * 70)
print("DEPLOYMENT HARDENING")
print("=" * 70)

for service, config in environment.items():

    print(
        f"{service:<25} "
        f"{'PUBLIC' if config['public'] else 'PRIVATE'}"
    )

print(
    "\n✓ Unnecessary public exposure removed."
)

<details>

<summary><b>Part 3 Explanation — Click to Expand</b></summary>

<br>

The production architecture now follows a safer design:

    INTERNET
       ↓
    API GATEWAY
       ↓
    INFERENCE SERVICE

Only the component intended to receive external requests remains publicly reachable.

This does not make the environment completely secure.

It simply removes two unnecessary attack paths.

Secure deployment is built by combining many controls rather than depending on a single configuration change.

</details>

In [ ]:
# @title ▶ Part 4: Apply API Security { display-mode: "form" }

api_requests = [
    {
        "client": "customer_12",
        "authenticated": True,
        "requests": 25
    },

    {
        "client": "customer_33",
        "authenticated": True,
        "requests": 41
    },

    {
        "client": "unknown_91",
        "authenticated": False,
        "requests": 18
    },

    {
        "client": "client_887",
        "authenticated": True,
        "requests": 9400
    }
]

RATE_LIMIT = 500

print("=" * 70)
print("API SECURITY GATEWAY")
print("=" * 70)

for request in api_requests:

    if not request["authenticated"]:
        decision = "✗ BLOCK — AUTHENTICATION"

    elif request["requests"] > RATE_LIMIT:
        decision = "✗ BLOCK — RATE LIMIT"

    else:
        decision = "✓ ALLOW"

    print(
        f"{request['client']:<15} "
        f"{request['requests']:>5} requests | "
        f"{decision}"
    )

<details>

<summary><b>Part 4 Explanation — Click to Expand</b></summary>

<br>

The API gateway applies more than one control.

Requests must first come from an authenticated client.

Authenticated clients are still subject to rate limits.

This matters because:

    AUTHENTICATED
          ≠
    UNLIMITED TRUST

A legitimate credential could be compromised or intentionally abused.

Rate limiting helps reduce activities such as automated probing, excessive querying, resource exhaustion, and some model-extraction attempts.

</details>

In [ ]:
# @title ▶ Part 5: Apply Least Privilege { display-mode: "form" }

permissions = {
    "Inference Service": {
        "read_model": True,
        "read_database": True,
        "write_logs": True,
        "read_training_data": False,
        "modify_model_registry": False,
        "manage_users": False
    }
}

print("=" * 70)
print("INFERENCE SERVICE PERMISSIONS")
print("=" * 70)

for permission, allowed in permissions[
    "Inference Service"
].items():

    print(
        f"{permission:<25} "
        f"{'✓ ALLOWED' if allowed else '✗ DENIED'}"
    )

<details>

<summary><b>Part 5 Explanation — Click to Expand</b></summary>

<br>

The inference service needs enough permission to perform predictions.

It needs to:

    READ THE PRODUCTION MODEL

    ACCESS REQUIRED APPLICATION DATA

    WRITE LOGS

It does not need to:

    MODIFY THE MODEL REGISTRY

    ACCESS TRAINING DATA

    MANAGE USERS

Least privilege asks:

> **What is the minimum access this workload needs to perform its legitimate function?**

If the inference service is later compromised, unnecessary permissions are not automatically inherited by the attacker.

</details>

In [ ]:
# @title ▶ Part 6: Protect Production Secrets { display-mode: "form" }

secret_configuration = {
    "Database Password": "SECRET_MANAGER",
    "API Signing Key": "SECRET_MANAGER",
    "Registry Token": "SECRET_MANAGER",
    "Developer Test Key": "SOURCE_CODE"
}

print("=" * 70)
print("SECRET SECURITY AUDIT")
print("=" * 70)

secret_failures = []

for secret, location in secret_configuration.items():

    secure = (
        location == "SECRET_MANAGER"
    )

    print(
        f"{secret:<25} "
        f"{location:<15} "
        f"{'✓ PASS' if secure else '✗ FAIL'}"
    )

    if not secure:
        secret_failures.append(secret)

print(
    f"\nSecrets requiring remediation: "
    f"{len(secret_failures)}"
)

<details>

<summary><b>Part 6 Explanation — Click to Expand</b></summary>

<br>

One credential is stored directly in source code.

This creates unnecessary exposure.

Secrets embedded in:

    SOURCE CODE

    NOTEBOOKS

    CONTAINER IMAGES

    CONFIGURATION FILES

may accidentally enter repositories, logs, backups, or other systems.

The production application should retrieve sensitive credentials from an approved secrets-management system rather than permanently embedding them into application code.

</details>

In [ ]:
# @title ▶ Part 7: Audit the Inference Workload { display-mode: "form" }

container_config = {
    "Runs as Root": False,
    "Privileged Mode": False,
    "Read-Only Filesystem": True,
    "CPU Limit": True,
    "Memory Limit": True,
    "GPU Limit": True,
    "Unnecessary Capabilities": False
}

print("=" * 70)
print("INFERENCE WORKLOAD SECURITY")
print("=" * 70)

secure_container = True

for control, value in container_config.items():

    if control in [
        "Runs as Root",
        "Privileged Mode",
        "Unnecessary Capabilities"
    ]:
        passed = not value

    else:
        passed = value

    print(
        f"{control:<28} "
        f"{'✓ PASS' if passed else '✗ FAIL'}"
    )

    if not passed:
        secure_container = False

<details>

<summary><b>Part 7 Explanation — Click to Expand</b></summary>

<br>

The workload is configured to reduce the privileges available inside the runtime environment.

It does not run as root or use privileged mode.

Resource limits are also enforced.

These controls become especially important if an attacker successfully exploits the application.

The objective is:

    APPLICATION COMPROMISE
             ↓
    RESTRICTED WORKLOAD
             ↓
    LIMITED DAMAGE

rather than allowing compromise of one process to automatically provide unrestricted control over its environment.

</details>

In [ ]:
# @title ▶ Part 8: Apply Network Segmentation { display-mode: "form" }

network_policy = {
    "API Gateway": [
        "Inference Service"
    ],

    "Inference Service": [
        "Model Registry",
        "Production Database",
        "Logging Service"
    ],

    "Training Storage": [
        "Model Registry"
    ],

    "Model Registry": [],

    "Production Database": [],

    "Secret Manager": [],

    "Logging Service": []
}

print("=" * 70)
print("PRODUCTION NETWORK POLICY")
print("=" * 70)

for source, destinations in network_policy.items():

    print(f"\n{source}")

    if destinations:
        for destination in destinations:
            print(f"  → {destination}")
    else:
        print("  No outgoing application paths")

<details>

<summary><b>Part 8 Explanation — Click to Expand</b></summary>

<br>

The inference service is not permitted to communicate with every resource in the environment.

For example, it cannot directly access:

    TRAINING STORAGE

    SECRET MANAGER

This reduces lateral-movement opportunities.

If the inference service becomes compromised, the attacker encounters additional network boundaries rather than automatically receiving access to the entire cloud environment.

</details>

In [ ]:
# @title ▶ Part 9: Verify Model Integrity { display-mode: "form" }

import hashlib
import json

approved_model = {
    "name": "SentinelAI",
    "version": "5.2",
    "training_data": "fraud-data-v8",
    "training_code": "commit-9af42",
    "approved": True
}

def model_hash(model):

    model_bytes = json.dumps(
        model,
        sort_keys=True
    ).encode()

    return hashlib.sha256(
        model_bytes
    ).hexdigest()


approved_hash = model_hash(
    approved_model
)

deployment_candidate = approved_model.copy()

candidate_hash = model_hash(
    deployment_candidate
)

print("=" * 70)
print("MODEL ARTIFACT VERIFICATION")
print("=" * 70)

print(
    f"\nApproved Hash:\n"
    f"{approved_hash}"
)

print(
    f"\nCandidate Hash:\n"
    f"{candidate_hash}"
)

integrity_verified = (
    approved_hash == candidate_hash
)

print(
    "\nIntegrity:",
    "✓ VERIFIED"
    if integrity_verified
    else "✗ FAILED"
)

<details>

<summary><b>Part 9 Explanation — Click to Expand</b></summary>

<br>

The candidate model matches the artifact previously approved by the organization.

The matching hashes provide evidence that the artifact has not changed since approval.

This does not prove that the model is perfect or free from every possible security problem.

It verifies something more specific:

> **The artifact presented for deployment is the same artifact that was approved.**

</details>

In [ ]:
# @title ▶ Part 10: Evaluate Deployment Readiness { display-mode: "form" }

deployment_gate = {
    "Public Exposure Secure":
        len(exposure_failures) == 2,

    "API Protection Enabled":
        True,

    "Least Privilege Applied":
        True,

    "Secrets Remediated":
        False,

    "Workload Hardened":
        secure_container,

    "Network Segmented":
        True,

    "Model Integrity Verified":
        integrity_verified,

    "Security Tests Passed":
        True,

    "Independent Approval":
        True
}

print("=" * 70)
print("PRODUCTION DEPLOYMENT GATE")
print("=" * 70)

deployment_ready = True

for check, passed in deployment_gate.items():

    print(
        f"{check:<30} "
        f"{'✓ PASS' if passed else '✗ FAIL'}"
    )

    if not passed:
        deployment_ready = False

print("\nDecision:")

if deployment_ready:
    print("✓ APPROVED FOR PRODUCTION")
else:
    print("✗ DEPLOYMENT BLOCKED")

<details>

<summary><b>Part 10 — Student Task — Click to Expand</b></summary>

<br>

The deployment has been blocked.

Review the output and identify the failed security requirement.

Ask yourself:

**Should the team bypass the failed check because every other security control passed?**

The answer should be no.

A deployment gate exists specifically to prevent required security controls from becoming optional.

Correct the failed requirement in the next part.

</details>

In [ ]:
# @title ▶ Part 11: Remediate the Deployment { display-mode: "form" }

# Move the exposed test key into the secret manager
secret_configuration[
    "Developer Test Key"
] = "SECRET_MANAGER"

secrets_secure = all(
    location == "SECRET_MANAGER"
    for location in secret_configuration.values()
)

deployment_gate[
    "Secrets Remediated"
] = secrets_secure

deployment_ready = all(
    deployment_gate.values()
)

print("=" * 70)
print("REMEDIATED DEPLOYMENT GATE")
print("=" * 70)

for check, passed in deployment_gate.items():

    print(
        f"{check:<30} "
        f"{'✓ PASS' if passed else '✗ FAIL'}"
    )

print("\nDecision:")

print(
    "✓ APPROVED FOR PRODUCTION"
    if deployment_ready
    else "✗ DEPLOYMENT BLOCKED"
)

<details>

<summary><b>Part 11 Explanation — Click to Expand</b></summary>

<br>

The failed control was corrected rather than bypassed.

The deployment now satisfies all required security gates.

This demonstrates an important secure-development behavior:

    SECURITY FAILURE
          ↓
       REMEDIATE
          ↓
       VERIFY
          ↓
       CONTINUE

rather than:

    SECURITY FAILURE
          ↓
        IGNORE
          ↓
        DEPLOY

SentinelAI can now enter production.

</details>

In [ ]:
# @title ▶ Part 12: Establish Monitoring Baselines { display-mode: "form" }

baseline = {
    "requests_per_minute": 300,
    "auth_failures": 5,
    "gpu_usage": 48,
    "error_rate": 2,
    "class_c_predictions": 20
}

print("=" * 70)
print("SENTINELAI NORMAL BASELINE")
print("=" * 70)

for metric, value in baseline.items():

    print(
        f"{metric:<25}: {value}"
    )

<details>

<summary><b>Part 12 Explanation — Click to Expand</b></summary>

<br>

SentinelAI has successfully entered production.

Security now shifts from:

    SECURE DEPLOYMENT

to:

    SECURE OPERATION

The security team records expected behavior for the production system.

These values create context for future monitoring.

The team can now compare current behavior against what the service normally looks like.

</details>

In [ ]:
# @title ▶ Part 13: Observe Production Anomalies { display-mode: "form" }

current_activity = {
    "requests_per_minute": 8700,
    "auth_failures": 1380,
    "gpu_usage": 99,
    "error_rate": 28,
    "class_c_predictions": 91
}

thresholds = {
    "requests_per_minute": 1000,
    "auth_failures": 50,
    "gpu_usage": 90,
    "error_rate": 10,
    "class_c_predictions": 50
}

alerts = []

print("=" * 70)
print("PRODUCTION SECURITY MONITOR")
print("=" * 70)

for metric, value in current_activity.items():

    abnormal = (
        value > thresholds[metric]
    )

    print(
        f"{metric:<25} "
        f"{value:>5} | "
        f"{'⚠ ALERT' if abnormal else '✓ NORMAL'}"
    )

    if abnormal:
        alerts.append(metric)

print(
    f"\nActive Alerts: "
    f"{len(alerts)}"
)

<details>

<summary><b>Part 13 — Student Investigation — Click to Expand</b></summary>

<br>

Several conditions changed at approximately the same time.

The service now has:

    EXTREME QUERY VOLUME

    LARGE AUTHENTICATION FAILURE SPIKE

    NEAR-MAXIMUM GPU UTILIZATION

    HIGH ERROR RATE

    MAJOR PREDICTION DISTRIBUTION CHANGE

Do not assume that one alert proves an attack.

Instead, consider the combined evidence.

Ask:

**Could high GPU usage alone have an innocent explanation?**

Yes.

**Could prediction drift alone have an innocent explanation?**

Yes.

But multiple related anomalies occurring simultaneously provide stronger evidence that something unusual is happening.

The next step is correlation.

</details>

In [ ]:
# @title ▶ Part 14: Correlate Security Signals { display-mode: "form" }

evidence = {
    "Excessive Query Activity":
        "requests_per_minute" in alerts,

    "Authentication Attack":
        "auth_failures" in alerts,

    "Resource Exhaustion":
        "gpu_usage" in alerts,

    "Service Instability":
        "error_rate" in alerts,

    "Model Behavior Anomaly":
        "class_c_predictions" in alerts
}

signal_count = 0

print("=" * 70)
print("INCIDENT CORRELATION")
print("=" * 70)

for signal, detected in evidence.items():

    print(
        f"{signal:<28} "
        f"{'⚠ DETECTED' if detected else '✓ NOT DETECTED'}"
    )

    if detected:
        signal_count += 1

print(
    f"\nCorrelated Signals: "
    f"{signal_count}"
)

incident_declared = (
    signal_count >= 3
)

print(
    "\nIncident Status:",
    "⚠ SECURITY INCIDENT"
    if incident_declared
    else "CONTINUE INVESTIGATION"
)

<details>

<summary><b>Part 14 Explanation — Click to Expand</b></summary>

<br>

The monitoring system does not rely on a single event.

Instead, several related signals are correlated.

The environment shows evidence affecting:

    ACCESS

    API ACTIVITY

    COMPUTE RESOURCES

    APPLICATION STABILITY

    MODEL BEHAVIOR

The combined evidence is strong enough to begin incident response.

The security lifecycle now changes from:

    MONITOR

to:

    RESPOND

</details>

In [ ]:
# @title ▶ Part 15: Test Attacker Reachability { display-mode: "form" }

compromised_service = "Inference Service"

reachable = network_policy[
    compromised_service
]

protected_resources = [
    "Training Storage",
    "Secret Manager"
]

print("=" * 70)
print("COMPROMISE BLAST RADIUS")
print("=" * 70)

print(
    f"\nCompromised workload: "
    f"{compromised_service}"
)

print("\nDirectly reachable:")

for resource in reachable:
    print(f"  ⚠ {resource}")

print("\nProtected by segmentation:")

for resource in protected_resources:

    if resource not in reachable:
        print(f"  ✓ {resource}")

<details>

<summary><b>Part 15 Explanation — Click to Expand</b></summary>

<br>

Assume the inference service has been compromised.

The earlier network controls now become extremely important.

The attacker cannot directly move from the inference service into:

    TRAINING STORAGE

    SECRET MANAGER

This demonstrates defense in depth.

The network controls did not necessarily prevent the initial compromise.

Instead, they limited what happened afterward.

Security architecture should assume that some preventive controls may eventually fail.

The surrounding controls determine whether one compromise remains isolated or spreads throughout the environment.

</details>

In [ ]:
# @title ▶ Part 16: Contain SentinelAI { display-mode: "form" }

containment = {
    "Public Prediction Endpoint":
        "TEMPORARILY DISABLED",

    "Inference Container":
        "ISOLATED",

    "Suspicious Client":
        "BLOCKED",

    "Inference Credential":
        "REVOKED",

    "Outbound Network Traffic":
        "BLOCKED",

    "Model Artifact":
        "QUARANTINED",

    "Security Logs":
        "PRESERVED"
}

print("=" * 70)
print("INCIDENT CONTAINMENT")
print("=" * 70)

for target, action in containment.items():

    print(
        f"{target:<30} → {action}"
    )

<details>

<summary><b>Part 16 Explanation — Click to Expand</b></summary>

<br>

Containment attempts to prevent additional damage while preserving information needed for investigation.

The team does not simply delete the affected service.

Instead, it:

    ISOLATES THE WORKLOAD

    REVOKES ACCESS

    BLOCKS SUSPICIOUS TRAFFIC

    RESTRICTS NETWORK COMMUNICATION

    QUARANTINES THE MODEL

    PRESERVES LOGS

This limits attacker activity while maintaining useful evidence.

</details>

In [ ]:
# @title ▶ Part 17: Investigate the Production Model { display-mode: "form" }

suspected_model = approved_model.copy()

# Simulate unauthorized modification discovered during investigation
suspected_model[
    "training_code"
] = "unknown-modified-build"

suspected_hash = model_hash(
    suspected_model
)

print("=" * 70)
print("MODEL FORENSIC CHECK")
print("=" * 70)

print(
    "\nExpected Hash:"
)
print(approved_hash)

print(
    "\nProduction Artifact Hash:"
)
print(suspected_hash)

model_compromised = (
    suspected_hash != approved_hash
)

print("\nResult:")

if model_compromised:
    print(
        "⚠ MODEL INTEGRITY FAILURE"
    )
else:
    print(
        "✓ MODEL INTEGRITY VERIFIED"
    )

<details>

<summary><b>Part 17 Explanation — Click to Expand</b></summary>

<br>

The investigation discovers that the production artifact no longer matches the approved artifact.

This provides concrete evidence of an integrity problem.

The expected response is not to continue running the suspicious model.

The artifact should remain quarantined while the organization restores a previously verified version.

This is why artifact integrity, versioning, and rollback were established before the incident occurred.

</details>

In [ ]:
# @title ▶ Part 18: Recover SentinelAI { display-mode: "form" }

trusted_versions = {
    "SentinelAI-5.2": {
        "integrity_verified": False,
        "status": "quarantined"
    },

    "SentinelAI-5.1": {
        "integrity_verified": True,
        "status": "previous-production"
    }
}

rollback_target = "SentinelAI-5.1"

print("=" * 70)
print("SECURE RECOVERY")
print("=" * 70)

if trusted_versions[
    rollback_target
]["integrity_verified"]:

    trusted_versions[
        rollback_target
    ]["status"] = "production"

    print(
        f"\n✓ Verified rollback target: "
        f"{rollback_target}"
    )

    print(
        "✓ New inference container deployed."
    )

    print(
        "✓ Credentials rotated."
    )

    print(
        "✓ Network policy reapplied."
    )

    print(
        "✓ Monitoring restored."
    )

    print(
        "\n✓ SENTINELAI RETURNED "
        "TO TRUSTED OPERATION"
    )

else:

    print(
        "✗ Recovery blocked: "
        "rollback artifact unverified."
    )

<details>

<summary><b>Part 18 Explanation — Click to Expand</b></summary>

<br>

Recovery uses a previously trusted model rather than an arbitrary available artifact.

The organization also:

    DEPLOYS A CLEAN WORKLOAD

    ROTATES CREDENTIALS

    REAPPLIES NETWORK CONTROLS

    RESTORES MONITORING

The objective is not simply availability.

The objective is:

> **Restore the service to a known and trusted state.**

</details>

In [ ]:
# @title ▶ Part 19: Evaluate the Final Architecture { display-mode: "form" }

final_controls = {
    "Minimal Public Exposure": True,
    "API Authentication": True,
    "API Rate Limiting": True,
    "Least-Privilege IAM": True,
    "Managed Secrets": True,
    "Restricted Workload": True,
    "Resource Limits": True,
    "Network Segmentation": True,
    "Model Integrity Verification": True,
    "Deployment Gates": True,
    "Security Monitoring": True,
    "Incident Detection": True,
    "Containment Capability": True,
    "Trusted Rollback": True,
    "Post-Incident Monitoring": True
}

passed = sum(
    final_controls.values()
)

total = len(
    final_controls
)

print("=" * 70)
print("FINAL SECURITY ASSESSMENT")
print("=" * 70)

for control, enabled in final_controls.items():

    print(
        f"{control:<32} "
        f"{'✓ PASS' if enabled else '✗ FAIL'}"
    )

score = (
    passed / total
) * 100

print("\n" + "=" * 70)

print(
    f"Security Controls: "
    f"{passed}/{total}"
)

print(
    f"Final Score: "
    f"{score:.1f}%"
)

if score >= 90:
    print(
        "\n✓ FINAL LAB SECURITY "
        "REQUIREMENTS SATISFIED"
    )
else:
    print(
        "\n△ Additional remediation required."
    )

In [ ]:
# @title ▶ Part 20: Final Knowledge Check { display-mode: "form" }

questions = [
    {
        "question":
        "1. Why was direct Internet access removed from the inference service?",
        "options": [
            "A. External requests should enter through the controlled API gateway",
            "B. AI models cannot use networks",
            "C. It improves model accuracy",
            "D. Private services cannot perform inference"
        ],
        "answer": "A"
    },
    {
        "question":
        "2. Why was rate limiting applied even to authenticated clients?",
        "options": [
            "A. Authenticated identities can still be compromised or abuse the service",
            "B. Authentication makes requests slower",
            "C. Rate limits replace authentication",
            "D. Authenticated users should never access models"
        ],
        "answer": "A"
    },
    {
        "question":
        "3. Why was the inference service denied access to training storage?",
        "options": [
            "A. It did not require that access for normal inference",
            "B. Training storage cannot exist in the cloud",
            "C. Models cannot read files",
            "D. Training data must always be public"
        ],
        "answer": "A"
    },
    {
        "question":
        "4. Why should secrets be removed from source code?",
        "options": [
            "A. Embedded secrets may leak through repositories, images, logs, or backups",
            "B. Source code cannot contain strings",
            "C. Secret managers improve model accuracy",
            "D. Credentials are never used by applications"
        ],
        "answer": "A"
    },
    {
        "question":
        "5. Why were container privileges and resources restricted?",
        "options": [
            "A. To reduce the impact of workload compromise and abuse",
            "B. To make the model public",
            "C. To eliminate authentication",
            "D. To increase attacker permissions"
        ],
        "answer": "A"
    },
    {
        "question":
        "6. What did network segmentation accomplish during the simulated compromise?",
        "options": [
            "A. It limited which internal resources the attacker could directly reach",
            "B. It guaranteed the inference service could never be compromised",
            "C. It disabled monitoring",
            "D. It removed IAM"
        ],
        "answer": "A"
    },
    {
        "question":
        "7. What did the model hash help defenders determine?",
        "options": [
            "A. Whether the production artifact still matched the approved artifact",
            "B. Whether the model was 100% accurate",
            "C. Whether the GPU was overloaded",
            "D. Whether users liked the model"
        ],
        "answer": "A"
    },
    {
        "question":
        "8. Why was the initial production deployment blocked?",
        "options": [
            "A. A required security gate had failed",
            "B. Every deployment must fail once",
            "C. The model was too accurate",
            "D. The API gateway was secure"
        ],
        "answer": "A"
    },
    {
        "question":
        "9. Why did the lab correlate multiple production anomalies?",
        "options": [
            "A. Multiple related signals provide stronger context than one alert alone",
            "B. Every single alert proves compromise",
            "C. Correlation replaces logging",
            "D. Monitoring cannot examine individual events"
        ],
        "answer": "A"
    },
    {
        "question":
        "10. What was the main purpose of containment?",
        "options": [
            "A. Limit additional damage while preserving useful evidence",
            "B. Immediately destroy all evidence",
            "C. Increase attacker access",
            "D. Retrain the model"
        ],
        "answer": "A"
    },
    {
        "question":
        "11. Why was SentinelAI-5.1 used during recovery?",
        "options": [
            "A. It was a previously trusted version whose integrity was verified",
            "B. Older models are automatically secure",
            "C. It did not require verification",
            "D. It was selected randomly"
        ],
        "answer": "A"
    },
    {
        "question":
        "12. What is the main security principle demonstrated by the final lab?",
        "options": [
            "A. AI deployment security requires multiple preventive, detective, and recovery controls working together",
            "B. One firewall is enough to secure an AI system",
            "C. Model accuracy determines whether infrastructure is secure",
            "D. Security ends once a model reaches production"
        ],
        "answer": "A"
    }
]

score = 0

print("=" * 70)
print("FINAL LAB — KNOWLEDGE CHECK")
print("=" * 70)

for q in questions:

    print("\n" + q["question"])

    for option in q["options"]:
        print(option)

    response = input(
        "\nYour answer: "
    ).strip().upper()

    if response == q["answer"]:

        print("✓ Correct!")
        score += 1

    else:

        print(
            f"✗ Incorrect. "
            f"Correct answer: {q['answer']}"
        )

percentage = (
    score / len(questions)
) * 100

print("\n" + "=" * 70)

print(
    f"Final Lab Score: "
    f"{score}/{len(questions)}"
)

print(
    f"Percentage: "
    f"{percentage:.1f}%"
)

if percentage >= 80:

    print(
        "\n✓ FINAL LAB COMPLETE!"
    )

else:

    print(
        "\n△ Review the lab explanations "
        "and try again."
    )

<details>

<summary><b>Final Lab — Practical Takeaways — Click to Expand</b></summary>

<br>

## Final Lab — Practical Takeaways

This final lab demonstrated that securely deploying an AI system requires more than protecting the model itself.

SentinelAI depended on an entire production environment.

That environment included:

    API ENDPOINTS

    IDENTITIES

    CREDENTIALS

    WORKLOADS

    NETWORKS

    MODEL ARTIFACTS

    DEPLOYMENT PIPELINES

    MONITORING SYSTEMS

Each component introduced its own attack surface.

You first reduced unnecessary exposure and placed the API gateway between external users and the internal inference service.

You then protected the API using authentication and rate limiting.

The inference service received only the permissions required for its legitimate responsibilities.

Secrets were moved out of source code and into managed storage.

The inference workload was restricted using reduced privileges and resource limits.

Network segmentation prevented the workload from communicating freely with unrelated infrastructure.

Before deployment, the model artifact was verified against its approved cryptographic hash.

The deployment pipeline then required all security gates to pass before the model could enter production.

This created several preventive layers:

    API SECURITY

          +

    LEAST PRIVILEGE

          +

    SECRET MANAGEMENT

          +

    WORKLOAD HARDENING

          +

    NETWORK SEGMENTATION

          +

    MODEL INTEGRITY

          +

    DEPLOYMENT SECURITY

However, prevention was not assumed to be perfect.

Once SentinelAI entered production, monitoring established normal behavior and watched for changes.

The simulated incident produced:

    EXCESSIVE QUERY ACTIVITY

          +

    AUTHENTICATION FAILURES

          +

    RESOURCE EXHAUSTION

          +

    SERVICE ERRORS

          +

    ABNORMAL MODEL BEHAVIOR

Rather than treating one signal as proof of an attack, the security process correlated multiple sources of evidence.

Once an incident was identified, the affected workload was isolated.

Existing network controls reduced the attacker's ability to move into other sensitive resources.

The model artifact was then checked and found to have failed integrity verification.

Recovery used a previously trusted model rather than an arbitrary available artifact.

The complete security lifecycle therefore became:

    DESIGN
       ↓
    HARDEN
       ↓
    VERIFY
       ↓
    DEPLOY
       ↓
    MONITOR
       ↓
    DETECT
       ↓
    CONTAIN
       ↓
    RECOVER
       ↓
    IMPROVE

The most important lesson from the module is:

> **AI security is a system property, not a single model property.**

A secure model running inside an insecure API, overprivileged identity, exposed container, flat network, compromised deployment pipeline, or unmonitored production environment can still become part of an insecure AI system.

The objective of secure AI deployment is therefore defense in depth.

Prevent what you can.

Restrict what an attacker can reach.

Verify what enters production.

Monitor what happens afterward.

Detect abnormal behavior.

Preserve evidence.

Contain compromise.

Recover to a trusted state.

</details>

---

<details>
<summary><b>Module 6 References — Click to Expand</b></summary>
<br>

# References

The following references provide the research, standards, frameworks, and technical guidance used throughout Module 6.

[1] National Institute of Standards and Technology, *Artificial Intelligence Risk Management Framework (AI RMF 1.0)*, NIST AI 100-1, Gaithersburg, MD, USA, Jan. 2023.

[2] A. Vassilev, A. Oprea, A. Fordyce, H. Anderson, X. Davies, and M. Hamin, *Adversarial Machine Learning: A Taxonomy and Terminology of Attacks and Mitigations*, NIST AI 100-2e2025, National Institute of Standards and Technology, Gaithersburg, MD, USA, 2025.

[3] S. Rose, O. Borchert, S. Mitchell, and S. Connelly, *Zero Trust Architecture*, NIST Special Publication 800-207, National Institute of Standards and Technology, Gaithersburg, MD, USA, Aug. 2020.

[4] V. C. Hu et al., *Guide to Attribute Based Access Control (ABAC) Definition and Considerations*, NIST Special Publication 800-162, National Institute of Standards and Technology, Gaithersburg, MD, USA.

[5] R. Chandramouli and Z. Butcher, *Building Secure Microservices-Based Applications Using Service-Mesh Architecture*, NIST Special Publication 800-204A, National Institute of Standards and Technology, Gaithersburg, MD, USA, May 2020.

[6] R. Chandramouli, *Security Strategies for Microservices-Based Application Systems*, NIST Special Publication 800-204, National Institute of Standards and Technology, Gaithersburg, MD, USA, Aug. 2019.

[7] R. S. Sandhu, E. J. Coyne, H. L. Feinstein, and C. E. Youman, “Role-based access control models,” *Computer*, vol. 29, no. 2, pp. 38–47, Feb. 1996.

[8] D. F. Ferraiolo and D. R. Kuhn, “Role-based access controls,” in *Proc. 15th National Computer Security Conference*, 1992, pp. 554–563.

[9] D. E. Denning, “A lattice model of secure information flow,” *Communications of the ACM*, vol. 19, no. 5, pp. 236–243, May 1976.

[10] F. Tramèr, F. Zhang, A. Juels, M. K. Reiter, and T. Ristenpart, “Stealing machine learning models via prediction APIs,” in *Proc. 25th USENIX Security Symposium (USENIX Security 16)*, Austin, TX, USA, 2016, pp. 601–618.

[11] M. Al-Rubaie and J. M. Chang, “Securing machine learning in the cloud: A systematic review of cloud machine learning security,” *Frontiers in Big Data*, vol. 3, 2020.

[12] T. Hunt, C. Song, R. Shokri, V. Shmatikov, and E. Witchel, “Chiron: Privacy-preserving machine learning as a service,” 2018.

[13] B. P. K. Reddy and A. Chatterjee, “SMLaaS: Secure machine learning as a service ensuring data and model parameter privacy,” *Security and Privacy*, vol. 8, 2025.

[14] J. Wang, W. Su, Z. Huang, J. Chen, C. Luo, and J. Li, “Practical privacy-preserving MLaaS: When compressive sensing meets generative networks,” in *Proc. AAAI Conference on Artificial Intelligence*, vol. 38, no. 14, 2024, pp. 15502–15510.

[15] M. Meli, M. R. McNiece, and B. Reaves, “How bad can it Git? Characterizing secret leakage in public GitHub repositories,” in *Proc. Network and Distributed System Security Symposium (NDSS)*, 2019.

[16] V. Sinha, D. Saha, P. Dhoolia, R. Padhye, and S. Mani, “Detecting and mitigating secret-key leaks in source code repositories,” in *Proc. 12th Working Conference on Mining Software Repositories (MSR)*, 2015.

[17] T. Combe, A. Martin, and R. Di Pietro, “To Docker or not to Docker: A security perspective,” *IEEE Cloud Computing*, vol. 3, no. 5, pp. 54–62, 2016.

[18] P. Liu et al., “Understanding the security risks of Docker Hub,” in *Computer Security – ESORICS 2020*, 2020.

[19] K. Wist, M. Helsem, and D. Gligoroski, “Vulnerability analysis of 2500 Docker Hub images,” 2020.

[20] T. Glatard et al., “An analysis of security vulnerabilities in container images for scientific data analysis,” *GigaScience*, vol. 10, no. 6, 2021.

[21] M. U. Haque and M. A. Babar, “Well begun is half done: An empirical study of exploitability and impact of base-image vulnerabilities,” 2021.

[22] O. Cobb and A. Van Looveren, “Context-aware drift detection,” in *Proc. 39th International Conference on Machine Learning (ICML)*, 2022.

[23] S. Garg et al., “On continuous integration / continuous delivery for automated deployment of machine learning models using MLOps,” 2022.

[24] H. Wang, S. Guo, J. He, H. Liu, T. Zhang, and T. Xiang, “Model supply chain poisoning: Backdooring pre-trained models via embedding indistinguishability,” 2024.

[25] S. Meiklejohn, H. Blauzvern, M. Maruseac, S. Schrock, L. Simon, and I. Shumailov, “Machine learning models have a supply chain problem,” in *Proc. International Conference on Machine Learning (ICML)*, PMLR, 2025.

[26] M. Spoczynski, M. Melara, and S. Szyller, “Atlas: A framework for ML lifecycle provenance and transparency,” in *Proc. IEEE European Symposium on Security and Privacy Workshops*, 2025.

[27] H. Naveed, S. Barnett, C. Arora, J. Grundy, H. Khalajzadeh, and O. Haggag, “Monitoring machine learning systems: A multivocal literature review,” 2025.

[28] H. Naveed et al., “Understanding practitioners' perspectives on monitoring machine learning systems,” 2025.

[29] D. Tuscano and J. P. Disso, “A practical incident-response framework for generative AI systems,” *Journal of Cybersecurity and Privacy*, vol. 6, no. 1, 2026.

[30] National Security Agency, Cybersecurity and Infrastructure Security Agency, and international partners, *AI/ML Supply Chain Risks and Mitigations*, 2026.

[31] OWASP Foundation, *OWASP API Security Top 10 – 2023*, 2023.

[32] OWASP Foundation, *OWASP Machine Learning Security Top 10*.

[33] Google, *Secure AI Framework (SAIF)*.

[34] Microsoft, *Secure AI and Zero Trust Security Guidance*.

</details>